In [ ]:
"""GPR Campaign Ploemeur — 06-06-2016.  Equivalent to seq06.m."""
from pathlib import Path
import sys
import importlib
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from gdp.data_io import load_mala
from gdp.preprocessing.filtering import filter_data, remove_mean
from gdp.preprocessing.gain import linear_gain
from gdp.preprocessing.image_processing import remove_svd
from gdp.preprocessing.trace_ops import align_traces

sys.path.insert(0, str(Path.cwd()))
import helper_functions.KirchhoffPylopsZeroOffset as KirchhoffPylopsZeroOffset
importlib.reload(KirchhoffPylopsZeroOffset)
from pylops.utils.wavelets import ricker
from helper_functions.migration import (
    gazdag_migration, write_backprop_files,
    lowpass_filter_excitation, dispersion_limited_cutoff,
    apply_3d_to_2d_correction,
)

from helper_functions.figures import save_fig   # standard thesis-figure export


# Notebook map & reading order

This notebook is one **main pipeline** interleaved with **dev-history /
rejected-experiment** sections kept for the record. To follow the main result,
read this logical order (a couple of stages are not in physical cell order --
see the notes):

**Main pipeline**
1. **Load field data** -- 38 borehole profiles, 6 June 2016.
2. **Shared pre-processing** -- bandpass, DC / direct-wave removal, trace
   alignment (x5 upsample), SVD rank-1 suppression, reference differencing.
3. **Migration** -- Kirchhoff / Gazdag, and realistic borehole-geometry
   back-propagation (gprMax, dx = 0.02 m, explicit water channel).
4. **Cleaning migrated results** *(before differencing)* -- per-profile f-k dip
   (fan) filter + amplitude gate + near-borehole taper. *(This "final recipe"
   physically sits inside the "Extra Pre-Processing Experiments" section, whose
   earlier cells define the fan/dip helpers it reuses.)*
5. **Time-lapse differencing** -- per profile (t_n - t_{n-1}), against a fixed
   reference, and per stage (Push / Chase / Wait / Pull).
6. **Region of influence + phase-plane fit** -- rectangular window, sliding
   window, and napari picking (k-space and B-scan), fit by WLS.
7. **WLS vs RANSAC** -- robustness of the fit on the picked ROIs (key result;
   physically the last section of the notebook).
8. **Fluid-front trajectories** -- Strategies 1 / 2 / 3 across all 38 profiles.
9. **Thesis figure compilations** -- the Chapter-7 figures.

**Dev-history / rejected experiments** (not part of the main result): paper-based
clutter-removal variants; RPCA decomposition; min-max normalisation; f-k
dip-filter development & artifact-fixing; sign-flip / water-level validation;
and the **[Superseded]** homogeneous-domain back-propagation.

> **Why these are not physically moved to the bottom:** several also *define
> helpers the main pipeline reuses* and must stay upstream of it -- the RPCA
> section defines `GAUSS_SIGMA_PX`; the f-k dip-development sections define
> `fan_taper_mask`, `edge_taper_2d`, `FK_TAPER_DEG`, `FAN_HW_BY_RUN`,
> `DIP_OVERRIDE_BY_RUN`, `PROFILES_FINAL` (used by the final cleaning recipe);
> and the *[Superseded]* section defines the back-propagation frame loader
> (`_load_bp_frame_wls`, `BP_FOCUS_IDX_OFFSET`, `pad_fac`, `kz_c_bp`) used by
> the ROI, RANSAC and figure cells. Extracting these into a dedicated "shared
> helpers" cell is the prerequisite for a physical archive-at-bottom.

# Loading in data from 6 June 2016

In [ ]:
DATA = Path.cwd() / 'fielddata' / 'raw_data' / '060616'
OUT_DIR = Path.cwd() / 'fielddata' / 'output'

# Process Reference

In [ ]:
runs = list(range(0, 6)) + list(range(7, 39))

# prof (run 0) is the pre-injection reference used for preprocessing.
# Its migration is zero (self-subtraction) so it is excluded from data_runs.
ref_run = runs[0]   # = 0 → loads prof.rd3 / prof.rad
data_runs = runs[1:]  # run 0 migration = zero (reference minus itself); exclude

_prof_name = lambda n: 'prof' if n == 0 else f'prof{n}'
ref_raw, info = load_mala(str(DATA / _prof_name(ref_run)), return_object=False)
sf = info['frequency (GHz)']
n_traces = ref_raw.shape[1]
samples = ref_raw.shape[0]

# Preprocess reference
ref_bp = filter_data(ref_raw, fq=(0.02, 0.2), sfreq=sf, btype='bandpass')
ref_dc, _ = remove_mean(ref_bp, 299, 517)

ref_aligned, _, _ = align_traces(ref_dc, ref_dc, upsample=5, normalize=False, align_reference=True)

ref_svd, _ = remove_svd(ref_aligned, low_s=0, high_s=1)

t = np.arange(1, samples + 1) / sf
v = 0.10 # [m/ns]
wavelength = v / 0.1 # [m] — wavelength at 100 MHz
dL = 0.05 # [m]
max_d = 85.0
sc = 1.2
rad_cut = 300
depth = np.linspace(max_d, max_d - n_traces * dL, n_traces)
radius = np.linspace(0, v * 450 / 2, samples)

OUT_DIR.mkdir(exist_ok=True)
(OUT_DIR / 'processed').mkdir(exist_ok=True)

# Migration setup
t_mig  = t[:rad_cut]
f0_mig = 0.1                                        # centre frequency [GHz] — adjust to antenna

# Ricker wavelet (matches PylopsKirchoffMigration convention)
_period = 1.0 / f0_mig
_n_wav  = int(np.ceil(6 * _period / (t_mig[1] - t_mig[0])))
if _n_wav % 2 == 0:
    _n_wav += 1
wav_mig, _, wcenter_mig = ricker(t_mig[:_n_wav], f0=f0_mig)

# Image axes: x = radial distance from borehole [m], z spans borehole depth range
x_img = np.linspace(0, v * t_mig[-1] / 2, rad_cut)

(OUT_DIR / 'migrated').mkdir(exist_ok=True)

# Back-propagation gprMax grid -- lambda/20 at f0_mig (~20 cells/wavelength)
bp_dx           = v / (f0_mig * 20)                      # grid spacing [m]
bp_pml          = 15                                      # PML cells
bp_src_y        = (bp_pml + 1) * bp_dx                   # source just inside PML on borehole side [m]
bp_domain_y     = float(x_img[-1]) + 2 * bp_pml * bp_dx  # radial extent + PML padding both sides [m]
bp_edge_exclude = 5                                       # zero outermost N sources each side before filtering

# ── Process and migrate reference profile (prof.rd3, absolute) ──────────────
n_ref       = n_traces
z_bh_ref    = depth[:n_ref]
x_bh_ref    = np.arange(n_ref) * dL

ref_gain, _ = linear_gain(ref_svd, t)
Dt_ref      = ref_gain[:rad_cut, :].T   # (n_ref, rad_cut)

limit_ref = max(sc * np.max(np.abs(Dt_ref)), 1.0)
fig_r, ax_r = plt.subplots(figsize=(8, 6))
ax_r.imshow(Dt_ref, aspect='auto', cmap='seismic',
            extent=[radius[0], radius[rad_cut - 1], depth[-1], depth[0]],
            vmin=-limit_ref, vmax=limit_ref)
ax_r.invert_yaxis()
ax_r.set_xlabel('Radial distance from B2 (m)')
ax_r.set_ylabel('Depth from top of B1 (m)')
fig_r.savefig(OUT_DIR / 'processed' / 'prof_0.png', dpi=150)
plt.close(fig_r)

# Kirchhoff reference migration
z_img_ref = np.linspace(z_bh_ref.max(), z_bh_ref.min(), n_ref)
recs_ref  = np.vstack((np.zeros(n_ref), z_bh_ref))
K_ref = KirchhoffPylopsZeroOffset.Kirchhoff(
    z=z_img_ref, x=x_img, t=t_mig,
    srcs=recs_ref, recs=recs_ref,
    vel=v, wav=wav_mig, wavcenter=wcenter_mig,
    mode='analytic', dynamic=False,
)
ref_m_kir = (K_ref.H @ Dt_ref.flatten()).reshape(len(x_img), len(z_img_ref)).T

lim_kir_ref = max(sc * np.max(np.abs(ref_m_kir)), 1.0)
fig_kr, ax_kr = plt.subplots(figsize=(8, 6))
ax_kr.imshow(ref_m_kir, aspect='equal', cmap='seismic',
             extent=[x_img[0], x_img[-1], z_img_ref[-1], z_img_ref[0]],
             vmin=-lim_kir_ref, vmax=lim_kir_ref)
ax_kr.invert_yaxis()
ax_kr.set_xlabel('Radial distance from borehole (m)')
ax_kr.set_ylabel('Depth (m)')
fig_kr.savefig(OUT_DIR / 'migrated' / 'kirchhoff_0.png', dpi=150)
plt.close(fig_kr)
np.save(OUT_DIR / 'migrated' / 'kirchhoff_0.npy', ref_m_kir)

# Kirchhoff-BP reference migration (delta wavelet)
wav_delta_ref = np.zeros_like(wav_mig); wav_delta_ref[wcenter_mig] = 1.0
K_bp_ref = KirchhoffPylopsZeroOffset.Kirchhoff(
    z=z_img_ref, x=x_img, t=t_mig,
    srcs=recs_ref, recs=recs_ref,
    vel=v, wav=wav_delta_ref, wavcenter=wcenter_mig,
    mode='analytic', dynamic=False,
)
ref_m_kir_bp = (K_bp_ref.H @ Dt_ref.flatten()).reshape(len(x_img), len(z_img_ref)).T
lim_kir_bp_ref = max(sc * np.max(np.abs(ref_m_kir_bp)), 1.0)
fig_kbpr, ax_kbpr = plt.subplots(figsize=(8, 6))
ax_kbpr.imshow(ref_m_kir_bp, aspect='equal', cmap='seismic',
               extent=[x_img[0], x_img[-1], z_img_ref[-1], z_img_ref[0]],
               vmin=-lim_kir_bp_ref, vmax=lim_kir_bp_ref)
ax_kbpr.invert_yaxis()
ax_kbpr.set_xlabel('Radial distance from borehole (m)')
ax_kbpr.set_ylabel('Depth (m)')
fig_kbpr.savefig(OUT_DIR / 'migrated' / 'kirchhoff_bp_0.png', dpi=150)
plt.close(fig_kbpr)
np.save(OUT_DIR / 'migrated' / 'kirchhoff_bp_0.npy', ref_m_kir_bp)

# Gazdag reference migration
ref_m_gaz = gazdag_migration(ref_gain[:rad_cut, :], x_bh_ref, t_mig, x_img, v)

lim_gaz_ref = max(sc * np.max(np.abs(ref_m_gaz)), 1.0)
fig_gr, ax_gr = plt.subplots(figsize=(8, 6))
ax_gr.imshow(ref_m_gaz.T, aspect='equal', cmap='seismic',
             extent=[x_img[0], x_img[-1], z_bh_ref[-1], z_bh_ref[0]],
             vmin=-lim_gaz_ref, vmax=lim_gaz_ref)
ax_gr.invert_yaxis()
ax_gr.set_xlabel('Radial distance from borehole (m)')
ax_gr.set_ylabel('Depth (m)')
fig_gr.savefig(OUT_DIR / 'migrated' / 'gazdag_0.png', dpi=150)
plt.close(fig_gr)
np.save(OUT_DIR / 'migrated' / 'gazdag_0.npy', ref_m_gaz.T)

print('Reference migrations saved.')

# Process other profiles

In [ ]:
data_runs_shortened = [1,3,4,8,9,20,21,38]

In [ ]:
for run in data_runs_shortened: #data_runs:
    data, _ = load_mala(str(DATA / _prof_name(run)), return_object=False)
    n = min(data.shape[1], n_traces)

    d_bp = filter_data(data[:, :n], fq=(0.02, 0.2), sfreq=sf, btype='bandpass')
    d_dc, _ = remove_mean(d_bp, 299, 517)

    d_aligned, _, _ = align_traces(d_dc, ref_aligned[:, :n], upsample=5, normalize=True, align_reference=False)

    d_svd, _ = remove_svd(d_aligned, low_s=0, high_s=1)
    d_gain, _ = linear_gain(d_svd, t)

    # B-scan — Dt[0] = deepest trace (depth[0] = 85 m)
    Dt    = d_gain[:rad_cut, :].T        # (n, rad_cut) = (n_rec, n_t)
    z_bh  = depth[:n]                   # receiver depths [m]; z_bh[0]=85 (deep) → z_bh[-1]≈0
    x_bh  = np.arange(n) * dL          # relative along-borehole positions [m]
    dt_bh = 1.0 / sf                    # time step [ns]

    limit = max(sc * np.max(np.abs(Dt)), 1.0)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(Dt, aspect='auto', cmap='seismic',
              extent=[radius[0], radius[rad_cut - 1], depth[-1], depth[0]],
              vmin=-limit, vmax=limit)
    ax.invert_yaxis()
    ax.set_xlabel('Radial distance from B2 (m)')
    ax.set_ylabel('Depth from top of B1 (m)')
    fig.savefig(OUT_DIR / 'processed' / f'prof_{run}.png', dpi=150)
    plt.close(fig)

    # --- Kirchhoff migration ---
    # z_img decreasing (85→0) so row 0 of m_kir = 85 m
    z_img   = np.linspace(z_bh.max(), z_bh.min(), n)
    recs_bh = np.vstack((np.zeros(n), z_bh))       # (x=0, z=depth) — left-wall geometry

    K = KirchhoffPylopsZeroOffset.Kirchhoff(
        z=z_img, x=x_img, t=t_mig,
        srcs=recs_bh, recs=recs_bh,
        vel=v, wav=wav_mig, wavcenter=wcenter_mig,
        mode='analytic', dynamic=False,
    )
    m_kir = (K.H @ Dt.flatten()).reshape(len(x_img), len(z_img)).T   # (n_depth, n_radial)
    m_kir -= ref_m_kir[:len(z_img), :]          # subtract reference migration

    lim_kir = max(sc * np.max(np.abs(m_kir)), 1.0)
    fig_k, ax_k = plt.subplots(figsize=(8, 6))
    ax_k.imshow(m_kir, aspect='equal', cmap='seismic',
                extent=[x_img[0], x_img[-1], z_img[-1], z_img[0]],
                vmin=-lim_kir, vmax=lim_kir)
    ax_k.invert_yaxis()
    ax_k.set_xlabel('Radial distance from borehole (m)')
    ax_k.set_ylabel('Depth (m)')
    fig_k.savefig(OUT_DIR / 'migrated' / f'kirchhoff_{run}.png', dpi=150)
    plt.close(fig_k)
    np.save(OUT_DIR / 'migrated' / f'kirchhoff_{run}.npy', m_kir)

    # --- Kirchhoff migration (delta wavelet — pure back-projection) ---
    # Using a spike instead of wav_mig removes the matched-filter / autocorrelation
    # step from K.H, so the PSF collapses from wac-shaped (4–5 extrema) to
    # wav-shaped (3 extrema), matching Gazdag's t=0 imaging condition.
    
    wav_delta = np.zeros_like(wav_mig); wav_delta[wcenter_mig] = 1.0
    K_bp = KirchhoffPylopsZeroOffset.Kirchhoff(
        z=z_img, x=x_img, t=t_mig,
        srcs=recs_bh, recs=recs_bh,
        vel=v, wav=wav_delta, wavcenter=wcenter_mig,
        mode='analytic', dynamic=False,
    )
    m_kir_bp = (K_bp.H @ Dt.flatten()).reshape(len(x_img), len(z_img)).T
    m_kir_bp -= ref_m_kir_bp[:len(z_img), :]   # subtract reference migration
    lim_kir_bp = max(sc * np.max(np.abs(m_kir_bp)), 1.0)
    fig_kbp, ax_kbp = plt.subplots(figsize=(8, 6))
    ax_kbp.imshow(m_kir_bp, aspect='equal', cmap='seismic',
                  extent=[x_img[0], x_img[-1], z_img[-1], z_img[0]],
                  vmin=-lim_kir_bp, vmax=lim_kir_bp)
    ax_kbp.invert_yaxis()
    ax_kbp.set_xlabel('Radial distance from borehole (m)')
    ax_kbp.set_ylabel('Depth (m)')
    fig_kbp.savefig(OUT_DIR / 'migrated' / f'kirchhoff_bp_{run}.png', dpi=150)
    plt.close(fig_kbp)
    np.save(OUT_DIR / 'migrated' / f'kirchhoff_bp_{run}.npy', m_kir_bp)


    # --- Gazdag migration ---
    # Rotate: borehole depth → x (along-track), radial distance → z (continuation direction)
    # d_gain[:rad_cut,:] is (n_t, n_x) as required; v_mig = v/2 applied internally
    m_gaz = gazdag_migration(d_gain[:rad_cut, :], x_bh, t_mig, x_img, v)
    m_gaz -= ref_m_gaz[:, :n]                   # subtract reference migration
    # m_gaz: (n_radial, n_depth) → .T gives (n_depth, n_radial); row 0 = z_bh[0] = 85 m
    lim_gaz = max(sc * np.max(np.abs(m_gaz)), 1.0)
    fig_g, ax_g = plt.subplots(figsize=(8, 6))
    ax_g.imshow(m_gaz.T, aspect='equal', cmap='seismic',
                extent=[x_img[0], x_img[-1], z_bh[-1], z_bh[0]],
                vmin=-lim_gaz, vmax=lim_gaz)
    ax_g.invert_yaxis()
    ax_g.set_xlabel('Radial distance from borehole (m)')
    ax_g.set_ylabel('Depth (m)')
    fig_g.savefig(OUT_DIR / 'migrated' / f'gazdag_{run}.png', dpi=150)
    plt.close(fig_g)
    np.save(OUT_DIR / 'migrated' / f'gazdag_{run}.npy', m_gaz.T)

    # --- Back propagation files (peak-normalised and sign-bit) ---
    eps_r    = (0.299792458 / v) ** 2     # permittivity from velocity (c in m/ns)
    t0_ns    = 299.0 / sf                 # direct-wave end, from remove_mean window start
    eps_r_half = 4.0 * eps_r
    f_cut_hz   = dispersion_limited_cutoff(eps_r_half, bp_dx)

    # Back-propagate the time-lapse DIFFERENCE (current profile − reference), not the
    # absolute B-scan: `write_backprop_files` peak-normalises each trace (a nonlinear
    # step), so subtracting two independently-normalised gprMax runs afterwards (see
    # the differencing cells below) would not recover a physically meaningful
    # difference field. Differencing here instead makes gprMax back-propagate the
    # true anomaly directly in a single run, matching how Kirchhoff/Gazdag isolate it
    # via `-= ref_m_kir` / `-= ref_m_gaz` — just done before injection here instead of
    # after migration. apply_3d_to_2d_correction is linear, so correcting the
    # difference is equivalent to differencing two corrected profiles, just cheaper.
    #
    # Uses d_svd/ref_svd (pre-`linear_gain`), NOT Dt: `linear_gain` multiplies by t^1
    # as a generic attenuation compensation (see gdp.preprocessing.gain), not a
    # spreading-model-specific correction. Stacking the sqrt(t) gain below on top of
    # that would double-count range compensation (net t^1 * t^0.5 = t^1.5 instead of
    # the correct t^0.5), blowing up amplitude at long travel times/large radii.
    Dt_diff_bp = d_svd[:rad_cut, :].T - ref_svd[:rad_cut, :n].T   # (n, rad_cut)
    Dt_2d = apply_3d_to_2d_correction(Dt_diff_bp, dt=dt_bh * 1e-9, velocity=v * 1e9,
                                       time_zero_idx=299)

    # ── Back-propagation pre-processing ──────────────────────────────────────────
    # Applied in this order to minimise spectral leakage before gprMax injection:
    #   1. 3D → 2D Green's function correction (sqrt(r) gain + 1/sqrt(w)*e^(i*pi/4))
    #   2. Spatial Tukey taper  (axis 0, receivers)   alpha=0.30 → 15 % each side
    #   3. Temporal Tukey taper (axis 1, time)        alpha=0.10 → 5 % each side
    #   4. f-kz dip filter      zero evanescent bins (|kz| > f / v_mig)
    #   (RMS normalisation removed: amplifies low-SNR traces → worsens near-source noise)
    from scipy.signal.windows import tukey as _tukey_sp

    # 2. Spatial taper
    _sp_tap    = _tukey_sp(n, alpha=0.30)[:, np.newaxis]              # (n, 1)
    Dt_tapered = Dt_2d * _sp_tap

    # 3. Temporal taper
    _t_tap     = _tukey_sp(Dt_tapered.shape[1], alpha=0.10)[np.newaxis, :]  # (1, n_t)
    Dt_tapered = Dt_tapered * _t_tap

    # 4. f-kz dip filter: remove evanescent energy (|kz| > f / v_mig)
    _v_mig_bp = v / 2.0
    _D_fk     = np.fft.fft(np.fft.rfft(Dt_tapered, axis=1), axis=0)  # (n, n_f)
    _freq_bp  = np.fft.rfftfreq(Dt_tapered.shape[1], d=dt_bh)         # [GHz]
    _kz_bp    = np.fft.fftfreq(Dt_tapered.shape[0], d=dL)              # [1/m]
    _evan_bp  = np.abs(_kz_bp[:, None]) > np.abs(_freq_bp[None, :]) / _v_mig_bp
    _D_fk[_evan_bp] = 0.0
    Dt_tapered = np.real(np.fft.irfft(np.fft.ifft(_D_fk, axis=0),
                                       n=Dt_tapered.shape[1], axis=1))

    _bp_common = dict(
        tapered_ntr_nt=Dt_tapered, dt_ns=dt_bh,
        x_midpoints=z_bh, t0_ns=t0_ns,
        eps_r=eps_r, v_ice=v,
        dx=bp_dx, domain_y=bp_domain_y, src_y=bp_src_y, pml_cells=bp_pml,
    )

    # Peak-normalised version
    write_backprop_files(OUT_DIR, label=f'prof_{run}', slug=f'prof_{run}',
                         sign_bit=False, **_bp_common)
    exc_file = OUT_DIR / 'backprop' / f'prof_{run}' / 'excitation.txt'
    lowpass_filter_excitation(exc_file, f_cut_hz, edge_exclude=bp_edge_exclude)

    # # Sign-bit version
    # write_backprop_files(OUT_DIR, label=f'prof_{run} (sign-bit)', slug=f'prof_{run}_signbit',
    #                      sign_bit=True, **_bp_common)
    # exc_file_sb = OUT_DIR / 'backprop' / f'prof_{run}_signbit' / 'excitation.txt'
    # lowpass_filter_excitation(exc_file_sb, f_cut_hz, edge_exclude=bp_edge_exclude)

In [ ]:
# ── Why post-migration deconvolution does not fix the extra Kirchhoff lobes ─────────
#
# K.H (adjoint Kirchhoff) is a matched filter: its PSF at image point (x0, z0) is
# the aperture-weighted sum of wac(2*Δx*sin(θ_src)/v) over all receivers, where
# sin(θ_src) varies from 0 (far receivers) to 1 (receiver nearest the reflector).
# This makes the PSF SPATIALLY VARIANT — broader than wac because far receivers
# contribute wac near its zero-lag peak, not at the displacement lag.
#
# Deconvolving with wac: over-corrects → more oscillations (observed)
# Deconvolving with wav: effective PSF ≠ wav → no visible change (observed)
#
# The correct fix is in the main processing loop above:
#   K_bp uses wav_delta (spike) instead of wav_mig.
#   K_bp.H then acts as pure back-projection (step 1 is identity),
#   so PSF = aperture sum of wav(2*Δx*sin(θ)/v) — a wav-shaped feature
#   dominated by the nearest receiver, matching Gazdag's t=0 condition.
#
# Re-run the main loop to produce kirchhoff_bp_{run}.npy / .png files.
print("See main processing loop for kirchhoff_bp (delta-wavelet) Kirchhoff images.")
print("Re-run that cell to produce kirchhoff_bp_*.npy alongside kirchhoff_*.npy.")

# Migration

# Borehole-Geometry Back-Propagation

`write_backprop_files()` (used above) assumes a homogeneous background medium starting right at the source -- it ignores the borehole itself. In a real single-hole survey the antennas sit inside a fluid-filled hole, and that fluid's permittivity (water, ~81) is drastically different from the surrounding ice/rock (~9), so the borehole is a strong near-source dielectric discontinuity that isn't in the old model at all. `write_borehole_backprop_files()` (in `helper_functions/migration.py`) adds it: the domain now has an explicit water-filled rectangle running the full depth of the model, with the source/receiver on its centreline.

**Given setup:**
- Borehole: vertical rectangle, 10 cm wide, water-filled (permittivity 81, conductivity 0.01 S/m)
- Source-receiver offset: 3.2 m
- Radargram: depth 60-85 m, imaging up to 12 m radially from the borehole
- Source/receiver sit on the borehole's lateral centreline
- 1 m clearance on the left of the borehole rectangle, >= 12 m clearance to the right

**Two judgement calls made that aren't fully determined by the spec above -- please confirm these are what you intended (see the full reasoning in the function's docstring):**
1. **The 3.2 m offset is *not* used to place two separate Tx/Rx sources.** One virtual source per trace is still placed at the recorded (Tx-Rx midpoint) depth with the background halved in velocity -- the same exploding-reflector convention Kirchhoff, Gazdag, and `write_backprop_files()` already use elsewhere in this notebook. The offset is used only to widen the depth buffer (so the modelled domain comfortably contains the physical antenna pair even at the shallowest/deepest logged depth). If you actually want a real two-antenna, true-offset, true-velocity forward model instead, this needs a different source scheme.
2. **The borehole water's permittivity is also quadrupled** (eps_r_water_half = 4x81 = 324, same halving trick as the background), because the one-way-equals-two-way-travel-time equivalence the exploding-reflector trick relies on only holds if *every* material in the model is scaled the same way. Conductivity (0.01 S/m) is left unscaled since it governs attenuation, not the travel-time equivalence.

The cell below builds the domain with the numbers above and draws a schematic -- check it before spending gprMax runtime on a real run.

In [ ]:
# ── Borehole-geometry back-propagation: build + sanity-check the domain ──────────────
# Self-contained demo using the survey's stated geometry (depth 60-85 m here is
# representative -- pass the real x_midpoints/z_bh for an actual run). No gprMax
# needed for this cell; it only writes the .in/.txt files and draws the schematic.
from helper_functions.migration import write_borehole_backprop_files, plot_borehole_domain

BH_WIDTH    = 0.10     # [m] borehole width
BH_EPS_R    = 81.0     # water permittivity
BH_SIGMA    = 0.01     # water conductivity [S/m]
BH_LEFT_BUF = 1.0      # [m] clearance left of the borehole rectangle
BH_IMG_RNG  = 12.0     # [m] required imaging clearance right of the borehole rectangle
BH_SRC_OFFSET = 3.2    # [m] Tx-Rx offset (used only to size the depth buffer -- see markdown above)

dL_bh   = 0.05
z_bh_demo = np.arange(60.0, 85.0 + 1e-9, dL_bh)   # representative depth range from the spec
data_demo = np.zeros((len(z_bh_demo), 300))        # placeholder trace data -- geometry only

in_path_bh, n_src_bh, n_snaps_bh, t_focus_bh, geom_bh = write_borehole_backprop_files(
    OUT_DIR, label='borehole geometry demo', slug='borehole_demo',
    tapered_ntr_nt=data_demo, dt_ns=1.0, x_midpoints=z_bh_demo,
    t0_ns=200.0, eps_r=(0.299792458 / v) ** 2, v_ice=v,
    eps_r_water=BH_EPS_R, sigma_water=BH_SIGMA,
    borehole_width=BH_WIDTH, left_buffer=BH_LEFT_BUF, imaging_range=BH_IMG_RNG,
    src_offset=BH_SRC_OFFSET, dx=bp_dx, pml_cells=bp_pml,
)

print(f'wrote {in_path_bh}')
print(f'n_src={n_src_bh}  n_snaps={n_snaps_bh}  t_focus={t_focus_bh:.1f} ns')
print(f'domain: {geom_bh["domain_x"]:.2f} m (depth) x {geom_bh["domain_y"]:.2f} m (radial)')
print(f'borehole rectangle: y=[{geom_bh["y_bh_start"]:.3f}, {geom_bh["y_bh_end"]:.3f}] m  '
      f'(centre / source-receiver y = {geom_bh["src_y"]:.3f} m)')
print(f'gprMax x=0 <-> true depth {-geom_bh["x_shift"]:.2f} m  '
      f'(true depth range modelled: {geom_bh["x_min_true"] - geom_bh["depth_buffer"]:.1f}-'
      f'{geom_bh["x_max_true"] + geom_bh["depth_buffer"]:.1f} m, recorded interval '
      f'{geom_bh["x_min_true"]:.1f}-{geom_bh["x_max_true"]:.1f} m)')

fig_bh = plot_borehole_domain(geom_bh, x_src_true=z_bh_demo[::20],
                               save_path=OUT_DIR / 'borehole_domain_schematic.png')
plt.show()

In [ ]:
# ── Borehole-geometry back-propagation: write .in files for selected profiles ────────
# Same preprocessing chain as the main loop above (filter -> remove_mean -> align -> SVD
# -> reference diff -> apply_3d_to_2d_correction -> Tukey tapers -> f-kz dip filter), but
# writing through write_borehole_backprop_files() instead of write_backprop_files() so
# these actually model the water-filled borehole. Reuses the BH_* geometry constants
# from the cell above. Depends only on cells 0/2/4 above having been run (reference
# profile processing) -- does not depend on the main loop cell having executed.
from scipy.signal.windows import tukey as _tukey_sp_bh
from helper_functions.migration import (
    apply_3d_to_2d_correction, dispersion_limited_cutoff,
    lowpass_filter_excitation, write_borehole_backprop_files,
)

BOREHOLE_RUNS = [1, 3, 8, 20, 38]

eps_r_bh      = (0.299792458 / v) ** 2
t0_ns_bh      = 299.0 / sf
eps_r_half_bh = 4.0 * eps_r_bh
f_cut_hz_bh   = dispersion_limited_cutoff(eps_r_half_bh, bp_dx)

borehole_in_paths = {}
for run in BOREHOLE_RUNS:
    data, _ = load_mala(str(DATA / _prof_name(run)), return_object=False)
    n = min(data.shape[1], n_traces)

    d_bp = filter_data(data[:, :n], fq=(0.02, 0.2), sfreq=sf, btype='bandpass')
    d_dc, _ = remove_mean(d_bp, 299, 517)
    d_aligned, _, _ = align_traces(d_dc, ref_aligned[:, :n], upsample=5, normalize=True, align_reference=False)
    d_svd, _ = remove_svd(d_aligned, low_s=0, high_s=1)

    z_bh_run = depth[:n]
    dt_bh_run = 1.0 / sf

    Dt_diff_bp = d_svd[:rad_cut, :].T - ref_svd[:rad_cut, :n].T
    Dt_2d = apply_3d_to_2d_correction(Dt_diff_bp, dt=dt_bh_run * 1e-9, velocity=v * 1e9,
                                       time_zero_idx=299)

    _sp_tap = _tukey_sp_bh(n, alpha=0.30)[:, np.newaxis]
    Dt_tapered_bh = Dt_2d * _sp_tap
    _t_tap = _tukey_sp_bh(Dt_tapered_bh.shape[1], alpha=0.10)[np.newaxis, :]
    Dt_tapered_bh = Dt_tapered_bh * _t_tap
    _v_mig_bp = v / 2.0
    _D_fk = np.fft.fft(np.fft.rfft(Dt_tapered_bh, axis=1), axis=0)
    _freq_bp = np.fft.rfftfreq(Dt_tapered_bh.shape[1], d=dt_bh_run)
    _kz_bp = np.fft.fftfreq(Dt_tapered_bh.shape[0], d=dL)
    _evan_bp = np.abs(_kz_bp[:, None]) > np.abs(_freq_bp[None, :]) / _v_mig_bp
    _D_fk[_evan_bp] = 0.0
    Dt_tapered_bh = np.real(np.fft.irfft(np.fft.ifft(_D_fk, axis=0), n=Dt_tapered_bh.shape[1], axis=1))

    slug = f'borehole_prof_{run}'
    in_path, n_src, n_snaps, t_focus_ns, geom = write_borehole_backprop_files(
        OUT_DIR, label=slug, slug=slug,
        tapered_ntr_nt=Dt_tapered_bh, dt_ns=dt_bh_run, x_midpoints=z_bh_run,
        t0_ns=t0_ns_bh, eps_r=eps_r_bh, v_ice=v,
        eps_r_water=BH_EPS_R, sigma_water=BH_SIGMA,
        borehole_width=BH_WIDTH, left_buffer=BH_LEFT_BUF, imaging_range=BH_IMG_RNG,
        src_offset=BH_SRC_OFFSET, dx=bp_dx, pml_cells=bp_pml,
    )
    exc_file = OUT_DIR / 'backprop' / slug / 'excitation.txt'
    lowpass_filter_excitation(exc_file, f_cut_hz_bh, edge_exclude=bp_edge_exclude)
    borehole_in_paths[run] = in_path

    print(f'run {run:>3}: wrote {in_path}')
    print(f'         n_src={n_src}  n_snaps={n_snaps}  t_focus={t_focus_ns:.2f} ns  '
          f'domain={geom["domain_x"]:.2f}x{geom["domain_y"]:.2f} m  '
          f'depth_range={z_bh_run.min():.2f}-{z_bh_run.max():.2f} m')

print(f'\n{len(borehole_in_paths)} .in files ready under {OUT_DIR / "backprop"}/borehole_prof_<run>/')

In [ ]:
# ── Plot borehole-geometry back-propagation results vs Kirchhoff/Gazdag ──────────────
# For each completed borehole_prof_<run> gprMax run: scan every snapshot, mask out the
# near-source region (borehole + BH_NEAR_MASK_M past it -- NOT just gprMax y<1 m like the
# homogeneous-domain cells above, since the source now sits on the borehole centreline at
# y=y_bh_end/2 rather than near the domain edge), and take the largest masked peak as the
# best-focus frame (same robust approach validated in the sign/water-level cells -- a
# fixed snapshot-index offset tuned for the old domain/timing isn't safe to reuse here).
# gprMax coordinates are converted back to true depth / radial-distance-from-borehole so
# axes line up directly with the Kirchhoff/Gazdag x_img convention.
import pyvista

BH_RUNS_PLOT    = [1, 3, 8, 20, 38]
BH_NEAR_MASK_M  = 1.0     # [m] past the borehole's outer wall to mask (injection halo)
BH_DEPTH_RANGE  = (60, 85)
BH_RADIAL_RANGE = (0.0, 14.0)

N_SNAP_BH, SNAP_WIN_BH = 30, 1.0   # must match write_borehole_backprop_files' defaults
dt_ns_bh_plot = 1.0 / sf
T_ns_bh_plot  = rad_cut * dt_ns_bh_plot
t0_ns_bh_plot = 299.0 / sf
t_focus_ns_bh = T_ns_bh_plot - t0_ns_bh_plot
t_start_ns_bh = max(dt_ns_bh_plot, t_focus_ns_bh - SNAP_WIN_BH)
snap_step_bh  = max(1, int((T_ns_bh_plot - t_start_ns_bh) / (max(1, N_SNAP_BH - 1) * dt_ns_bh_plot)))

def _borehole_geom_for_run(run):
    """Recompute the same x_shift / y_bh_end write_borehole_backprop_files used for this run."""
    data, _ = load_mala(str(DATA / _prof_name(run)), return_object=False)
    n = min(data.shape[1], n_traces)
    z_bh_run = depth[:n]
    pml_pad = bp_pml * bp_dx
    depth_buffer = BH_SRC_OFFSET / 2.0
    x_shift = pml_pad + depth_buffer - float(np.min(z_bh_run))
    y_bh_end = pml_pad + BH_LEFT_BUF + BH_WIDTH
    return x_shift, y_bh_end

def _best_borehole_focus(run, near_mask_m=BH_NEAR_MASK_M):
    slug = f'borehole_prof_{run}'
    snap_dir = OUT_DIR / 'backprop' / slug / f'backprop_{slug}_snaps'
    snap_files = sorted(snap_dir.glob('bp_snap*.vti'), key=lambda p: int(p.stem.replace('bp_snap', '')))
    if not snap_files:
        return None
    x_shift, y_bh_end = _borehole_geom_for_run(run)
    snap_times_ns = t_start_ns_bh + np.arange(len(snap_files)) * snap_step_bh * dt_ns_bh_plot
    best = None
    for k, f in enumerate(snap_files):
        mesh = pyvista.read(str(f))
        nx_c = mesh.dimensions[0] - 1
        ny_c = mesh.dimensions[1] - 1
        dx_m = float(mesh.spacing[0])
        ez   = np.array(mesh['E-field'])[:, 2].reshape(ny_c, nx_c).T   # (depth, radial)
        mask_px = max(1, round((y_bh_end + near_mask_m) / dx_m))
        ez_masked = ez.copy()
        ez_masked[:, :mask_px] = 0.0
        peak = np.max(np.abs(ez_masked))
        if best is None or peak > best[0]:
            best = (peak, k, ez_masked, nx_c, ny_c, dx_m)
    peak, k, ez_masked, nx_c, ny_c, dx_m = best
    depth_axis  = np.arange(nx_c) * dx_m - x_shift    # true depth [m]
    radial_axis = np.arange(ny_c) * dx_m - y_bh_end   # radial distance from borehole [m]
    return ez_masked, depth_axis, radial_axis, k, snap_times_ns[k]

def _load_mig(method, run):
    p = OUT_DIR / 'migrated' / f'{method}_{run}.npy'
    return np.load(p) if p.exists() else None

bh_out = OUT_DIR / 'borehole_comparison'
bh_out.mkdir(exist_ok=True)

n_prof = len(BH_RUNS_PLOT)
fig, axes = plt.subplots(n_prof, 3, figsize=(13, 3.2 * n_prof), sharex=True, sharey=True)
axes[0, 0].set_title('Kirchhoff_bp')
axes[0, 1].set_title('Gazdag')
axes[0, 2].set_title(f'Borehole back-propagation Ez (best snapshot, mask={BH_NEAR_MASK_M} m)')
fig.suptitle('Migrated vs borehole-geometry back-propagation', y=1.01)

for row, run in enumerate(BH_RUNS_PLOT):
    img_k = _load_mig('kirchhoff_bp', run)
    ax_k = axes[row, 0]
    if img_k is not None:
        n = img_k.shape[0]
        ext_k = [x_img[0], x_img[-1], depth[n - 1], depth[0]]
        clim = np.percentile(np.abs(img_k), 100)
        ax_k.imshow(img_k, aspect='auto', cmap='seismic', extent=ext_k, origin='upper', vmin=-clim, vmax=clim)
        ax_k.invert_yaxis()
    else:
        ax_k.text(0.5, 0.5, 'not found', ha='center', va='center', transform=ax_k.transAxes)
    ax_k.set_ylabel(f'prof_{run}')
    ax_k.set_ylim(*BH_DEPTH_RANGE); ax_k.set_xlim(*BH_RADIAL_RANGE)

    img_g = _load_mig('gazdag', run)
    ax_g = axes[row, 1]
    if img_g is not None:
        n = img_g.shape[0]
        ext_g = [x_img[0], x_img[-1], depth[n - 1], depth[0]]
        clim = np.percentile(np.abs(img_g), 100)
        ax_g.imshow(img_g, aspect='auto', cmap='seismic', extent=ext_g, origin='upper', vmin=-clim, vmax=clim)
        ax_g.invert_yaxis()
    else:
        ax_g.text(0.5, 0.5, 'not found', ha='center', va='center', transform=ax_g.transAxes)
    ax_g.set_ylim(*BH_DEPTH_RANGE); ax_g.set_xlim(*BH_RADIAL_RANGE)

    frame = _best_borehole_focus(run)
    ax_bp = axes[row, 2]
    if frame is not None:
        ez, depth_axis, radial_axis, snap_idx, snap_t = frame
        clim = np.percentile(np.abs(ez), 100)
        ax_bp.imshow(ez, aspect='auto', cmap='seismic',
                     extent=[radial_axis[0], radial_axis[-1], depth_axis[0], depth_axis[-1]],
                     origin='lower', vmin=-clim, vmax=clim)
        ax_bp.invert_yaxis()
        ax_bp.text(0.98, 0.02, f't={snap_t:.2f} ns (snap {snap_idx})', ha='right', va='bottom',
                   transform=ax_bp.transAxes, fontsize=7, color='white')
    else:
        ax_bp.text(0.5, 0.5, 'not found', ha='center', va='center', transform=ax_bp.transAxes)
    ax_bp.set_ylim(*BH_DEPTH_RANGE); ax_bp.set_xlim(*BH_RADIAL_RANGE)

    if row == n_prof - 1:
        for ax in axes[row]:
            ax.set_xlabel('Radial distance from borehole [m]')

plt.tight_layout()
out_path = bh_out / 'borehole_comparison_grid.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show(); plt.close(fig)
print(f'Saved {out_path}')

## Clutter Diagnosis: Grid Resolution + Borehole-Water Permittivity Scaling

The borehole back-propagation results above show visibly more clutter than the homogeneous-domain (surface-source) back-propagation, especially near the borehole (prof_1, prof_3). Two candidate causes, both testable without changing the geometry itself:

1. **Grid under-resolution.** `dx=0.05 m` gives only **2 cells across the 10 cm borehole** -- far too coarse to cleanly represent a sharp water/ice boundary; this alone can generate stair-casing noise independent of any real physics.
2. **Artificially strong borehole waveguide.** The half-velocity trick quadruples the water's permittivity (81 -> 324) for consistency with the background scaling. At eps=324 the guided wavelength at f0 (~17 cm) is comparable to the 10 cm borehole -- strong modal confinement. At the *true*, unscaled permittivity (81), the wavelength (~33 cm) gives markedly weaker confinement. This breaks travel-time consistency for the short borehole segment specifically, but is worth testing as a source of excess clutter.

The cell below writes 3 gprMax variants for `prof_1` (the worst case) isolating each factor, to be compared against the existing `borehole_prof_1` baseline (`dx=0.05`, scaled water permittivity):

| slug | dx | water eps_r used |
|---|---|---|
| `borehole_prof_1` (existing baseline) | 0.05 m (2 cells/borehole) | 324 (scaled) |
| `borehole_prof_1_dx02` | 0.02 m (5 cells/borehole) | 324 (scaled) |
| `borehole_prof_1_unscaled` | 0.05 m (2 cells/borehole) | 81 (true) |
| `borehole_prof_1_dx02_unscaled` | 0.02 m (5 cells/borehole) | 81 (true) |

**Cost warning:** the `dx02` variants are ~5-6x more cells and a proportionally smaller timestep -- expect roughly a 10-50x longer gprMax run than the baseline. `stride` is unchanged (still every trace) per your instruction, so the source count is identical across all 4 variants -- only grid resolution and water permittivity differ.

In [ ]:
# ── Borehole clutter diagnosis: dx-refinement + unscaled water permittivity variants ──
# Isolates the two candidate clutter sources for prof_1 (worst case in the comparison
# grid above): grid resolution across the borehole width, and the half-velocity trick's
# permittivity scaling applied to the borehole water. Reuses the same preprocessing
# chain as the main borehole-writing cell (Dt_tapered), just re-derived here since it
# isn't retained after that cell's loop finishes.
from scipy.signal.windows import tukey as _tukey_sp_bh2
from helper_functions.migration import (
    apply_3d_to_2d_correction, dispersion_limited_cutoff,
    lowpass_filter_excitation, write_borehole_backprop_files,
)

CLUTTER_TEST_RUN = 1
CLUTTER_VARIANTS = [
    # (slug_suffix, dx, scale_water_eps)
    ('dx02',          0.02, True),
    ('unscaled',      bp_dx, False),
    ('dx02_unscaled', 0.02, False),
]

eps_r_bh2      = (0.299792458 / v) ** 2
t0_ns_bh2      = 299.0 / sf
eps_r_half_bh2 = 4.0 * eps_r_bh2

data_ct, _ = load_mala(str(DATA / _prof_name(CLUTTER_TEST_RUN)), return_object=False)
n_ct = min(data_ct.shape[1], n_traces)

d_bp_ct = filter_data(data_ct[:, :n_ct], fq=(0.02, 0.2), sfreq=sf, btype='bandpass')
d_dc_ct, _ = remove_mean(d_bp_ct, 299, 517)
d_aligned_ct, _, _ = align_traces(d_dc_ct, ref_aligned[:, :n_ct], upsample=5, normalize=True, align_reference=False)
d_svd_ct, _ = remove_svd(d_aligned_ct, low_s=0, high_s=1)

z_bh_ct = depth[:n_ct]
dt_bh_ct = 1.0 / sf

Dt_diff_ct = d_svd_ct[:rad_cut, :].T - ref_svd[:rad_cut, :n_ct].T
Dt_2d_ct = apply_3d_to_2d_correction(Dt_diff_ct, dt=dt_bh_ct * 1e-9, velocity=v * 1e9, time_zero_idx=299)

_sp_tap_ct = _tukey_sp_bh2(n_ct, alpha=0.30)[:, np.newaxis]
Dt_tapered_ct = Dt_2d_ct * _sp_tap_ct
_t_tap_ct = _tukey_sp_bh2(Dt_tapered_ct.shape[1], alpha=0.10)[np.newaxis, :]
Dt_tapered_ct = Dt_tapered_ct * _t_tap_ct
_v_mig_ct = v / 2.0
_D_fk_ct = np.fft.fft(np.fft.rfft(Dt_tapered_ct, axis=1), axis=0)
_freq_ct = np.fft.rfftfreq(Dt_tapered_ct.shape[1], d=dt_bh_ct)
_kz_ct = np.fft.fftfreq(Dt_tapered_ct.shape[0], d=dL)
_evan_ct = np.abs(_kz_ct[:, None]) > np.abs(_freq_ct[None, :]) / _v_mig_ct
_D_fk_ct[_evan_ct] = 0.0
Dt_tapered_ct = np.real(np.fft.irfft(np.fft.ifft(_D_fk_ct, axis=0), n=Dt_tapered_ct.shape[1], axis=1))

clutter_variant_paths = {}
for suffix, dx_variant, scale_water in CLUTTER_VARIANTS:
    slug = f'borehole_prof_{CLUTTER_TEST_RUN}_{suffix}'
    f_cut_hz_ct = dispersion_limited_cutoff(eps_r_half_bh2, dx_variant)

    in_path, n_src, n_snaps, t_focus_ns, geom = write_borehole_backprop_files(
        OUT_DIR, label=slug, slug=slug,
        tapered_ntr_nt=Dt_tapered_ct, dt_ns=dt_bh_ct, x_midpoints=z_bh_ct,
        t0_ns=t0_ns_bh2, eps_r=eps_r_bh2, v_ice=v,
        eps_r_water=BH_EPS_R, sigma_water=BH_SIGMA,
        borehole_width=BH_WIDTH, left_buffer=BH_LEFT_BUF, imaging_range=BH_IMG_RNG,
        src_offset=BH_SRC_OFFSET, dx=dx_variant, pml_cells=bp_pml,
        scale_water_eps=scale_water,
    )
    exc_file = OUT_DIR / 'backprop' / slug / 'excitation.txt'
    lowpass_filter_excitation(exc_file, f_cut_hz_ct, edge_exclude=5)
    clutter_variant_paths[slug] = in_path

    n_cells_across = BH_WIDTH / dx_variant
    eps_water_used = 4.0 * BH_EPS_R if scale_water else BH_EPS_R
    print(f'{slug}: wrote {in_path.name}')
    print(f'  dx={dx_variant:.3f} m -> {n_cells_across:.1f} cells across the {BH_WIDTH*100:.0f} cm borehole  '
          f'eps_r_water_used={eps_water_used}  domain={geom["domain_x"]:.2f}x{geom["domain_y"]:.2f} m')

print(f'\n{len(clutter_variant_paths)} variants ready under {OUT_DIR / "backprop"}/borehole_prof_{CLUTTER_TEST_RUN}_<suffix>/\n'
      f'Compare against the existing borehole_prof_{CLUTTER_TEST_RUN} baseline once run through gprMax.')

In [ ]:
# ── Plot the clutter-diagnosis variants against Kirchhoff/Gazdag ─────────────────────
# Same best-masked-peak snapshot search as the main results-plotting cell, generalised
# to read each variant's own dx from its snapshot (mesh.spacing) rather than assuming
# bp_dx, since the dx02 variants use a different grid spacing than the baseline.
import pyvista

# BH_FOCUS_IDX_OFFSET: fixed snapshot-index offset from the nominal focus time
# t_focus_ns_bh, applied identically to every profile -- mirrors the established
# FOCUS_IDX_OFFSET pattern used for the homogeneous-domain back-propagation
# differencing above (_load_focus_ez), which deliberately uses the SAME physical time
# for every profile being compared. The earlier per-profile "maximum masked peak
# amplitude" search independently chose a different snapshot time for each profile
# (indices 25-28, spanning ~26 ns) -- fine for single-profile viewing, but it breaks
# temporal registration between profiles: differencing two images taken at different
# times makes a still-converging (not-yet-focused) wavefront look like it moved, even
# with zero real displacement. That produced a near-full-reflector "difference" instead
# of the compact, localised residual Kirchhoff/Gazdag show for the same profile pair.
# 25 is the offset that reproduces profile 1's previously-validated best-focus index
# (the highest-SNR profile), now applied as a fixed constant to every profile instead
# of re-searched per profile.
BH_FOCUS_IDX_OFFSET = 25

def _best_borehole_focus_variant(slug, run, near_mask_m=BH_NEAR_MASK_M):
    snap_dir = OUT_DIR / 'backprop' / slug / f'backprop_{slug}_snaps'
    snap_files = sorted(snap_dir.glob('bp_snap*.vti'), key=lambda p: int(p.stem.replace('bp_snap', '')))
    if not snap_files:
        return None
    dx_probe = float(pyvista.read(str(snap_files[0])).spacing[0])   # this variant's actual dx
    data, _ = load_mala(str(DATA / _prof_name(run)), return_object=False)
    n = min(data.shape[1], n_traces)
    z_bh_run = depth[:n]
    pml_pad = bp_pml * dx_probe
    depth_buffer = BH_SRC_OFFSET / 2.0
    x_shift = pml_pad + depth_buffer - float(np.min(z_bh_run))
    y_bh_end = pml_pad + BH_LEFT_BUF + BH_WIDTH

    snap_times_ns = t_start_ns_bh + np.arange(len(snap_files)) * snap_step_bh * dt_ns_bh_plot
    nearest_idx = int(np.argmin(np.abs(snap_times_ns - t_focus_ns_bh)))
    k = min(nearest_idx + BH_FOCUS_IDX_OFFSET, len(snap_files) - 1)

    mesh = pyvista.read(str(snap_files[k]))
    nx_c = mesh.dimensions[0] - 1
    ny_c = mesh.dimensions[1] - 1
    dx_m = float(mesh.spacing[0])
    ez   = np.array(mesh['E-field'])[:, 2].reshape(ny_c, nx_c).T
    mask_px = max(1, round((y_bh_end + near_mask_m) / dx_m))
    ez_masked = ez.copy()
    ez_masked[:, :mask_px] = 0.0

    depth_axis  = np.arange(nx_c) * dx_m - x_shift
    radial_axis = np.arange(ny_c) * dx_m - y_bh_end
    return ez_masked, depth_axis, radial_axis, k, snap_times_ns[k]

CLUTTER_PANELS = [
    ('Kirchhoff_bp', 'mig', 'kirchhoff_bp'),
    ('Gazdag', 'mig', 'gazdag'),
    ('baseline\n(dx=0.05, eps=324)', 'bp', f'borehole_prof_{CLUTTER_TEST_RUN}'),
    ('dx=0.02\n(eps=324)', 'bp', f'borehole_prof_{CLUTTER_TEST_RUN}_dx02'),
    ('unscaled\n(dx=0.05, eps=81)', 'bp', f'borehole_prof_{CLUTTER_TEST_RUN}_unscaled'),
    ('dx=0.02+unscaled\n(eps=81)', 'bp', f'borehole_prof_{CLUTTER_TEST_RUN}_dx02_unscaled'),
]

fig, axes = plt.subplots(1, len(CLUTTER_PANELS), figsize=(4 * len(CLUTTER_PANELS), 6), sharey=True)
fig.suptitle(f'Borehole clutter diagnosis -- prof_{CLUTTER_TEST_RUN}', y=1.02)

for ax, (title, kind, key) in zip(axes, CLUTTER_PANELS):
    if kind == 'mig':
        img = _load_mig(key, CLUTTER_TEST_RUN)
        if img is not None:
            n = img.shape[0]
            ext = [x_img[0], x_img[-1], depth[n - 1], depth[0]]
            clim = np.percentile(np.abs(img), 100)
            ax.imshow(img, aspect='auto', cmap='seismic', extent=ext, origin='upper', vmin=-clim, vmax=clim)
            ax.invert_yaxis()
        else:
            ax.text(0.5, 0.5, 'not found', ha='center', va='center', transform=ax.transAxes)
    else:
        frame = _best_borehole_focus_variant(key, CLUTTER_TEST_RUN)
        if frame is not None:
            ez, depth_axis, radial_axis, snap_idx, snap_t = frame
            clim = np.percentile(np.abs(ez), 100)
            ax.imshow(ez, aspect='auto', cmap='seismic',
                      extent=[radial_axis[0], radial_axis[-1], depth_axis[0], depth_axis[-1]],
                      origin='lower', vmin=-clim, vmax=clim)
            ax.invert_yaxis()
            ax.text(0.98, 0.02, f't={snap_t:.2f} ns\n(snap {snap_idx})', ha='right', va='bottom',
                    transform=ax.transAxes, fontsize=7, color='white')
        else:
            ax.text(0.5, 0.5, 'not found', ha='center', va='center', transform=ax.transAxes)
    ax.set_title(title, fontsize=9)
    ax.set_ylim(*BH_DEPTH_RANGE)
    ax.set_xlim(*BH_RADIAL_RANGE)
    ax.set_xlabel('Radial [m]')

axes[0].set_ylabel('Depth [m]')
plt.tight_layout()
out_path = bh_out / f'borehole_clutter_diagnosis_prof_{CLUTTER_TEST_RUN}.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show(); plt.close(fig)
print(f'Saved {out_path}')

## Extending the dx=0.02 Refinement to Profiles 3, 8, 20, 38

The clutter diagnosis above confirmed `dx=0.02` (5 cells across the borehole) with
scaled water permittivity as the fix for profile 1. This writes the same-setting `.in`
files for the other profiles used throughout this chapter (3, 8, 20, 38) so the
post-imaging clutter-removal recipe below can be applied to all five profiles, not just
profile 1.

**These still need to be run through gprMax externally before the post-processing cells
below will find any snapshot data for them** -- profile 1 already has
`borehole_prof_1_dx02` from the clutter-diagnosis cell above and will work immediately;
profiles 3/8/20/38 will show "not found" until their gprMax runs complete.

**Superseded -- see the "Fixing Cross-Profile Amplitude Normalisation" cells below.**
Both this cell and the clutter-diagnosis `borehole_prof_1_dx02` write via
`write_borehole_backprop_files(..., normalize_mode='peak')` with its old per-trace
normalisation, which turned out to destroy the real amplitude information a time-lapse
comparison depends on (see below). All five `.in` files get rewritten (and need
re-running through gprMax again) by the corrected cell further down.

In [ ]:
# ── Write dx=0.02 borehole .in files for the remaining profiles (3, 8, 20, 38) ────────
# Same settings as borehole_prof_1_dx02 above (dx=0.02, scaled water eps, default
# peak normalisation, no B-scan-domain clutter removal) -- just generalised across
# profiles. Run these through gprMax before the post-processing cells further down.
from scipy.signal.windows import tukey as _tukey_sp_bh3
from helper_functions.migration import (
    apply_3d_to_2d_correction, dispersion_limited_cutoff,
    lowpass_filter_excitation, write_borehole_backprop_files,
)

BOREHOLE_RUNS_DX02 = [3, 8, 20, 38]   # profile 1's dx02 run already exists
DX02 = 0.02

eps_r_bh3      = (0.299792458 / v) ** 2
t0_ns_bh3      = 299.0 / sf
eps_r_half_bh3 = 4.0 * eps_r_bh3
f_cut_hz_bh3   = dispersion_limited_cutoff(eps_r_half_bh3, DX02)

borehole_dx02_in_paths = {}
for run in BOREHOLE_RUNS_DX02:
    data, _ = load_mala(str(DATA / _prof_name(run)), return_object=False)
    n = min(data.shape[1], n_traces)

    d_bp = filter_data(data[:, :n], fq=(0.02, 0.2), sfreq=sf, btype='bandpass')
    d_dc, _ = remove_mean(d_bp, 299, 517)
    d_aligned, _, _ = align_traces(d_dc, ref_aligned[:, :n], upsample=5, normalize=True, align_reference=False)
    d_svd, _ = remove_svd(d_aligned, low_s=0, high_s=1)

    z_bh_run = depth[:n]
    dt_bh_run = 1.0 / sf

    Dt_diff_bp = d_svd[:rad_cut, :].T - ref_svd[:rad_cut, :n].T
    Dt_2d = apply_3d_to_2d_correction(Dt_diff_bp, dt=dt_bh_run * 1e-9, velocity=v * 1e9,
                                       time_zero_idx=299)

    _sp_tap = _tukey_sp_bh3(n, alpha=0.30)[:, np.newaxis]
    Dt_tapered_bh = Dt_2d * _sp_tap
    _t_tap = _tukey_sp_bh3(Dt_tapered_bh.shape[1], alpha=0.10)[np.newaxis, :]
    Dt_tapered_bh = Dt_tapered_bh * _t_tap
    _v_mig_bp = v / 2.0
    _D_fk = np.fft.fft(np.fft.rfft(Dt_tapered_bh, axis=1), axis=0)
    _freq_bp = np.fft.rfftfreq(Dt_tapered_bh.shape[1], d=dt_bh_run)
    _kz_bp = np.fft.fftfreq(Dt_tapered_bh.shape[0], d=dL)
    _evan_bp = np.abs(_kz_bp[:, None]) > np.abs(_freq_bp[None, :]) / _v_mig_bp
    _D_fk[_evan_bp] = 0.0
    Dt_tapered_bh = np.real(np.fft.irfft(np.fft.ifft(_D_fk, axis=0), n=Dt_tapered_bh.shape[1], axis=1))

    slug = f'borehole_prof_{run}_dx02'
    in_path, n_src, n_snaps, t_focus_ns, geom = write_borehole_backprop_files(
        OUT_DIR, label=slug, slug=slug,
        tapered_ntr_nt=Dt_tapered_bh, dt_ns=dt_bh_run, x_midpoints=z_bh_run,
        t0_ns=t0_ns_bh3, eps_r=eps_r_bh3, v_ice=v,
        eps_r_water=BH_EPS_R, sigma_water=BH_SIGMA,
        borehole_width=BH_WIDTH, left_buffer=BH_LEFT_BUF, imaging_range=BH_IMG_RNG,
        src_offset=BH_SRC_OFFSET, dx=DX02, pml_cells=bp_pml, scale_water_eps=True,
    )
    exc_file = OUT_DIR / 'backprop' / slug / 'excitation.txt'
    lowpass_filter_excitation(exc_file, f_cut_hz_bh3, edge_exclude=bp_edge_exclude)
    borehole_dx02_in_paths[run] = in_path

    print(f'run {run:>3}: wrote {in_path}')
    print(f'         n_src={n_src}  n_snaps={n_snaps}  t_focus={t_focus_ns:.2f} ns  '
          f'domain={geom["domain_x"]:.2f}x{geom["domain_y"]:.2f} m')

print(f'\n{len(borehole_dx02_in_paths)} .in files ready under {OUT_DIR / "backprop"}/borehole_prof_<run>_dx02/')
print('Run these through gprMax before the post-processing cells further down the notebook.')

## Fixing Cross-Profile Amplitude Normalisation

Investigating a mismatch between this pipeline's time-lapse differences and
Kirchhoff/Gazdag's (see `fielddata/output/difference_by_phase`) turned up a real bug,
not a tuning problem. Isolating it step by step:

1. Different profiles were landing on different gprMax snapshot *times* (indices 25-28,
   spanning ~26 ns) because `_best_borehole_focus_variant` searched for the
   highest-masked-peak snapshot independently per profile -- fine for viewing one
   profile, but it broke temporal registration between profiles being differenced. Fixed
   above: it now uses a fixed nominal focus time plus a fixed index offset (calibrated
   once against profile 1), identical for every profile.
2. That fix alone did not resolve the smearing (confirmed by differencing the raw,
   unfiltered back-propagation images at matched times -- still nearly a full-reflector
   difference, not Kirchhoff's compact residual), which ruled out the dip filter as the
   cause and pointed elsewhere.
3. The actual cause: `write_borehole_backprop_files`'s `normalize_mode='peak'`
   normalised every individual trace to its own peak amplitude independently. Measured
   directly: profile 3's real (pre-normalisation) RMS amplitude is ~1.8x profile 1's --
   consistent with the reflector "sharpening" through the push stage, exactly the kind
   of real change a time-lapse study is trying to detect -- and that entire signal was
   erased by per-trace rescaling. Within a single profile, per-trace peaks varied by
   80-160x, so weak, mostly-noise traces were also being boosted to the same injected
   amplitude as the strongest real-signal traces.

Fixed in `helper_functions/migration.py`: `write_borehole_backprop_files` now takes an
optional `norm_scale` argument. When provided, every trace in every profile divides by
the *same* single scalar instead of its own peak, preserving both real amplitude
structure within a profile and real amplitude change across profiles. The cell below
computes that shared scale once (the peak found across all five profiles' excitation
data) and rewrites all five `.in` files with it -- including profile 1's, whose existing
`.in` file also used the old per-trace normalisation.

In [ ]:
# ── Recompute Dt_tapered for all 5 profiles, find one shared peak, rewrite all 5 .in files ──
# Uses the same processing chain as borehole_dx02_other_write above (bandpass -> DC
# removal -> alignment -> SVD -> reference diff -> 3D-to-2D correction -> Tukey tapers
# -> f-kz evanescent filter), just generalised to loop over all 5 profiles and to keep
# each profile's Dt_tapered around long enough to find one shared normalisation scale
# before any of them get written to gprMax .in files.
from scipy.signal.windows import tukey as _tukey_sp_norm
from helper_functions.migration import (
    apply_3d_to_2d_correction, dispersion_limited_cutoff,
    lowpass_filter_excitation, write_borehole_backprop_files,
)

ALL_BOREHOLE_RUNS = [1, 3, 8, 20, 38]
DX02 = 0.02

eps_r_norm      = (0.299792458 / v) ** 2
t0_ns_norm      = 299.0 / sf
eps_r_half_norm = 4.0 * eps_r_norm
f_cut_hz_norm   = dispersion_limited_cutoff(eps_r_half_norm, DX02)

Dt_tapered_by_run = {}
for run in ALL_BOREHOLE_RUNS:
    data, _ = load_mala(str(DATA / _prof_name(run)), return_object=False)
    n = min(data.shape[1], n_traces)

    d_bp = filter_data(data[:, :n], fq=(0.02, 0.2), sfreq=sf, btype='bandpass')
    d_dc, _ = remove_mean(d_bp, 299, 517)
    d_aligned, _, _ = align_traces(d_dc, ref_aligned[:, :n], upsample=5, normalize=True, align_reference=False)
    d_svd, _ = remove_svd(d_aligned, low_s=0, high_s=1)

    z_bh_run = depth[:n]
    dt_bh_run = 1.0 / sf

    Dt_diff_bp = d_svd[:rad_cut, :].T - ref_svd[:rad_cut, :n].T
    Dt_2d = apply_3d_to_2d_correction(Dt_diff_bp, dt=dt_bh_run * 1e-9, velocity=v * 1e9,
                                       time_zero_idx=299)

    _sp_tap = _tukey_sp_norm(n, alpha=0.30)[:, np.newaxis]
    Dt_tapered_run = Dt_2d * _sp_tap
    _t_tap = _tukey_sp_norm(Dt_tapered_run.shape[1], alpha=0.10)[np.newaxis, :]
    Dt_tapered_run = Dt_tapered_run * _t_tap
    _v_mig_norm = v / 2.0
    _D_fk = np.fft.fft(np.fft.rfft(Dt_tapered_run, axis=1), axis=0)
    _freq_norm = np.fft.rfftfreq(Dt_tapered_run.shape[1], d=dt_bh_run)
    _kz_norm = np.fft.fftfreq(Dt_tapered_run.shape[0], d=dL)
    _evan_norm = np.abs(_kz_norm[:, None]) > np.abs(_freq_norm[None, :]) / _v_mig_norm
    _D_fk[_evan_norm] = 0.0
    Dt_tapered_run = np.real(np.fft.irfft(np.fft.ifft(_D_fk, axis=0), n=Dt_tapered_run.shape[1], axis=1))

    Dt_tapered_by_run[run] = (Dt_tapered_run, z_bh_run, dt_bh_run)
    print(f'run {run:>3}: Dt_tapered peak={np.abs(Dt_tapered_run).max():.4g}  '
          f'RMS={np.sqrt(np.mean(Dt_tapered_run**2)):.4g}')

GLOBAL_NORM_SCALE = max(np.abs(dt_run).max() for dt_run, _, _ in Dt_tapered_by_run.values())
print(f'\nGLOBAL_NORM_SCALE (shared across all 5 profiles) = {GLOBAL_NORM_SCALE:.4g}')
print('Every profile now divides by this SAME number instead of its own per-trace peak.\n')

borehole_norm_in_paths = {}
for run in ALL_BOREHOLE_RUNS:
    Dt_tapered_run, z_bh_run, dt_bh_run = Dt_tapered_by_run[run]
    slug = f'borehole_prof_{run}_dx02'
    in_path, n_src, n_snaps, t_focus_ns, geom = write_borehole_backprop_files(
        OUT_DIR, label=slug, slug=slug,
        tapered_ntr_nt=Dt_tapered_run, dt_ns=dt_bh_run, x_midpoints=z_bh_run,
        t0_ns=t0_ns_norm, eps_r=eps_r_norm, v_ice=v,
        eps_r_water=BH_EPS_R, sigma_water=BH_SIGMA,
        borehole_width=BH_WIDTH, left_buffer=BH_LEFT_BUF, imaging_range=BH_IMG_RNG,
        src_offset=BH_SRC_OFFSET, dx=DX02, pml_cells=bp_pml, scale_water_eps=True,
        normalize_mode='peak', norm_scale=GLOBAL_NORM_SCALE,
    )
    exc_file = OUT_DIR / 'backprop' / slug / 'excitation.txt'
    lowpass_filter_excitation(exc_file, f_cut_hz_norm, edge_exclude=bp_edge_exclude)
    borehole_norm_in_paths[run] = in_path

    exc_check = np.loadtxt(exc_file, skiprows=1)
    print(f'run {run:>3}: wrote {in_path.name}  '
          f'excitation amplitude range=[{exc_check[:,1:].min():.4f}, {exc_check[:,1:].max():.4f}]')

print(f'\n{len(borehole_norm_in_paths)} .in files rewritten under {OUT_DIR / "backprop"}/borehole_prof_<run>_dx02/')
print('Run all 5 through gprMax again before the post-processing cells below.')

## Extra Pre-Processing Experiments (Profile 1): Paper-Based Clutter Removal

Testing the processing chain from **Santos & Teixeira (2017), "Application of time-reversal-based processing techniques to enhance detection of GPR targets," J. Appl. Geophys. 146**, applied to `prof_1`'s difference B-scan (`Dt_diff_bp`, the same array that currently feeds `apply_3d_to_2d_correction`) before it goes to gprMax. Each step below gets its own cell with a before/after plot so you can judge whether to adopt it -- **nothing here is wired into the production borehole-writing cells yet**.

Now implemented against the paper's actual equations (previously two of these were best-effort guesses without the paper text -- corrected below):
- **(a) Mean background removal (Eq 8)** -- exact match, no changes needed.
- **(b) Eigenvalue/SVD background removal (Eq 9)** -- exact match (reuses `remove_svd`, already used elsewhere in this notebook).
- **(c) Sliding-window space-frequency technique (Eq 10-11)** -- corrected. This is *not* a simple moving-average as I first implemented -- it's an FFT-then-windowed-SVD method: each trace is transformed to frequency domain, a sliding window of `L` consecutive traces forms an (n_freq x L) submatrix at each position, that submatrix is SVD'd, and its dominant singular component (the local "background") is subtracted -- essentially method (b) applied *locally* in frequency-space rather than globally in time-space.
- **(d) Along-track spatial derivative (Eq 12)** -- corrected to the exact Holoborodko (2008) 5-point coefficients `[-1,-2,0,2,1]/8`, not a generic Savitzky-Golay derivative (which uses different coefficients).

Step 2 (time-reversal) is confirmed identical to the paper's description (Section 2, "reversed in time... in a first-in, last-out fashion") and is **already implemented** inside `write_borehole_backprop_files()` -- see the note below.

In [ ]:
# ── Reproduce prof_1's Dt_diff_bp baseline for the pre-processing experiments below ──
PP_RUN = 1
data_pp, _ = load_mala(str(DATA / _prof_name(PP_RUN)), return_object=False)
n_pp = min(data_pp.shape[1], n_traces)

d_bp_pp = filter_data(data_pp[:, :n_pp], fq=(0.02, 0.2), sfreq=sf, btype='bandpass')
d_dc_pp, _ = remove_mean(d_bp_pp, 299, 517)
d_aligned_pp, _, _ = align_traces(d_dc_pp, ref_aligned[:, :n_pp], upsample=5, normalize=True, align_reference=False)
d_svd_pp, _ = remove_svd(d_aligned_pp, low_s=0, high_s=1)

z_bh_pp = depth[:n_pp]
dt_bh_pp = 1.0 / sf

PP_baseline = (d_svd_pp[:rad_cut, :].T - ref_svd[:rad_cut, :n_pp].T).copy()   # (n_traces, n_t), same as cell 7's Dt_diff_bp

pp_out = OUT_DIR / 'pp_experiments'
pp_out.mkdir(exist_ok=True)

def _pp_show(before, after, title_before, title_after, fname, depth_axis=z_bh_pp, dt=dt_bh_pp):
    """Side-by-side B-scan comparison on a shared, before-derived colour scale.
    Always saves to pp_out/fname -- this notebook's first cell sets
    matplotlib.use('Agg') (non-interactive), so plt.show() alone renders nothing;
    saving is what actually lets you see the result."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 6), sharey=True)
    lim = max(np.percentile(np.abs(before), 99), 1e-12)
    for ax, img, title in zip(axes, (before, after), (title_before, title_after)):
        im = ax.imshow(img, aspect='auto', cmap='seismic',
                        extent=[0, img.shape[1] * dt, depth_axis[-1], depth_axis[0]],
                        vmin=-lim, vmax=lim)
        ax.invert_yaxis()
        ax.set_xlabel('Time (ns)'); ax.set_title(title)
        plt.colorbar(im, ax=ax, shrink=0.8)
    axes[0].set_ylabel('Along-borehole trace depth (m)')
    plt.tight_layout()
    fig.savefig(pp_out / fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {pp_out / fname}')
    return fig

print('PP_baseline shape', PP_baseline.shape, 'max abs', np.max(np.abs(PP_baseline)))
_pp_show(PP_baseline, PP_baseline, 'Dt_diff_bp (baseline)', 'Dt_diff_bp (baseline, same panel twice for reference)',
         '0_baseline.png')

In [ ]:
# ── (a) Mean background removal (Eq 8) ────────────────────────────────────────────────
# Subtracts the average A-scan (mean across all traces, per time-sample) from every
# trace -- suppresses a background component that's common/uniform across the whole
# profile (e.g. horizontal layers). Unambiguous operation, no interpretation needed.
def mean_background_removal(data):
    mean_ascan = np.mean(data, axis=0, keepdims=True)
    return data - mean_ascan

PP_mean_removed = mean_background_removal(PP_baseline)
_pp_show(PP_baseline, PP_mean_removed, 'Before: Dt_diff_bp', 'After: mean background removal (Eq 8)',
         'a_mean_removal.png')

In [ ]:
# ── (b) SVD / eigenvalue background removal (Eq 9) ────────────────────────────────────
# Re-applies the same remove_svd() already used pre-differencing above, but now to the
# post-differencing residual (Dt_diff_bp) -- checking whether there's more coherent
# "background" left after the reference-profile subtraction. Unambiguous operation
# (reuses the existing, already-validated helper).
PP_SVD_HIGH_S = 3   # number of leading singular components treated as background

_svd_filtered_T, _svd_removed_T = remove_svd(PP_baseline.T, low_s=0, high_s=PP_SVD_HIGH_S)
PP_svd_removed = _svd_filtered_T.T
_pp_show(PP_baseline, PP_svd_removed, 'Before: Dt_diff_bp',
         f'After: SVD background removal (Eq 9, components 0-{PP_SVD_HIGH_S - 1})',
         'b_svd_removal.png')

In [ ]:
# ── (c) Sliding-window space-frequency technique (Eq 10-11, exact) ───────────────────
# Not a moving-average (my first attempt was wrong): for each window of L consecutive
# traces, FFT each trace to frequency domain, form the (n_freq x L) submatrix B_sub
# (Eq 10), SVD it (Eq 11), reconstruct the dominant n_remove singular component(s) as
# the local "background" for that window, and subtract. Overlapping windows (slide by
# 1 trace) are averaged. This is method (b) applied locally in frequency-space rather
# than globally in time-space -- the paper doesn't specify L or n_remove; L=21 traces
# mirrors the window used in the (superseded) moving-average attempt, n_remove=1
# mirrors (b)'s single dominant component.
def sliding_window_svd_removal(data, L=21, n_remove=1):
    n_traces, n_t = data.shape
    Data_f = np.fft.rfft(data, axis=1)   # (n_traces, n_freq)

    clutter_f = np.zeros_like(Data_f)
    counts = np.zeros(n_traces)
    half = L // 2
    for i in range(n_traces):
        lo = max(0, i - half)
        hi = min(n_traces, lo + L)
        lo = max(0, hi - L)
        sub = Data_f[lo:hi, :].T   # (n_freq, L) -- matches Eq 10's M x L layout
        U, s, Vh = np.linalg.svd(sub, full_matrices=False)
        s_bg = s.copy()
        s_bg[n_remove:] = 0
        bg_sub = (U * s_bg) @ Vh
        clutter_f[lo:hi, :] += bg_sub.T
        counts[lo:hi] += 1
    clutter_f /= counts[:, None]
    filtered_f = Data_f - clutter_f
    return np.fft.irfft(filtered_f, n=n_t, axis=1)

PP_WINDOW_L = 21
PP_sliding_removed = sliding_window_svd_removal(PP_baseline, L=PP_WINDOW_L, n_remove=1)
_pp_show(PP_baseline, PP_sliding_removed, 'Before: Dt_diff_bp',
         f'After: sliding-window freq-domain SVD removal (Eq 10-11, L={PP_WINDOW_L})',
         'c_sliding_removal.png')

In [ ]:
# ── (d) Along-track spatial derivative (Eq 12, exact) ────────────────────────────────
# Exact Holoborodko (2008) smooth noise-robust 5-point differentiator, as given in the
# paper: A'_a = (-A_{a-2} - 2*A_{a-1} + 2*A_{a+1} + A_{a+2}) / 8. Not the same as scipy's
# generic Savitzky-Golay derivative (different coefficients) -- this is the specific
# formula the paper cites.
from scipy.ndimage import convolve1d

def along_track_derivative_holoborodko(data):
    kernel = np.array([-1.0, -2.0, 0.0, 2.0, 1.0]) / 8.0
    return convolve1d(data, kernel, axis=0, mode='nearest')

PP_deriv = along_track_derivative_holoborodko(PP_baseline)
_pp_show(PP_baseline, PP_deriv, 'Before: Dt_diff_bp', 'After: Holoborodko along-track derivative (Eq 12, exact)',
         'd_derivative.png')

### Step 2 -- Time Reversal

Confirmed against the paper (Section 2): "reversed in time... in a first-in, last-out fashion" -- exactly `data_rev = -data_s[:, ::-1]` (already implemented inside `write_borehole_backprop_files()`, see [migration.py](helper_functions/migration.py) and its docstring for why the extra `-` sign on top of the plain reversal is needed -- gprMax's `hertzian_dipole` is a current source, not a direct field source, which is a detail this paper's synthetic-FDTD framing doesn't need to address since it isn't specific to gprMax's source model). No separate cell needed here; whichever clutter-removal method(s) you adopt above would simply replace `Dt_2d`/`Dt_tapered` as the input to that function, same as today.

In [ ]:
# ── Eq 7: amplitude rescaling -- correcting where this actually applies ──────────────
# On first reading the paper's equation list out of context, I assumed Eq 7 rescales the
# excitation traces right before FDTD injection (paralleling this notebook's own
# peak-normalisation step). Having read the actual paper text: Eq 7 appears right after
# the sentence describing FDTD transmission ("...transmitting them back... To reduce
# the redundancy on the data set and highlight the anomalies, the TR signal amplitudes
# are rescaled..."), and Fig. 2's colour bar (0 to 1) is shown on FDTD *output*
# snapshots, not source waveforms. So Eq 7 most likely normalises the resulting TR
# wavefield (used for the Mode 1/2/X12 statistics below), not the pre-injection
# excitation -- applied as a single GLOBAL min/max over the whole TR data set ("x_min
# is the minimum value ... of the set", singular), not per-trace. This is implemented
# in the Mode 1/2/X12 cell below, in its correct place.
#
# My original "DC-injection" concern (a bipolar excitation shifted to [0,1] injecting a
# spurious non-physical DC current) doesn't actually apply to what the paper describes,
# since it's not what gets injected -- it was based on the wrong application point. Kept
# here only as a still-useful, standalone comparison of the two normalisation
# conventions, now correctly labelled as *not* what Eq 7 is doing.
def normalize_peak(data):
    peak = np.max(np.abs(data), axis=1, keepdims=True)
    peak[peak == 0] = 1.0
    return data / peak

def normalize_minmax_per_trace(data):
    x_min = np.min(data, axis=1, keepdims=True)
    x_max = np.max(data, axis=1, keepdims=True)
    span = x_max - x_min
    span[span == 0] = 1.0
    return (data - x_min) / span

PP_reversed = -PP_baseline[:, ::-1]   # step 2 (time reversal), applied only for a fair before/after here
PP_norm_peak = normalize_peak(PP_reversed)
PP_norm_minmax = normalize_minmax_per_trace(PP_reversed)

fig, axes = plt.subplots(1, 2, figsize=(11, 6), sharey=True)
im0 = axes[0].imshow(PP_norm_peak, aspect='auto', cmap='seismic', vmin=-1, vmax=1,
                      extent=[0, PP_norm_peak.shape[1] * dt_bh_pp, z_bh_pp[-1], z_bh_pp[0]])
axes[0].invert_yaxis(); axes[0].set_title('Current: peak-normalised excitation, range [-1, 1]')
plt.colorbar(im0, ax=axes[0], shrink=0.8)
im1 = axes[1].imshow(PP_norm_minmax, aspect='auto', cmap='viridis', vmin=0, vmax=1,
                      extent=[0, PP_norm_minmax.shape[1] * dt_bh_pp, z_bh_pp[-1], z_bh_pp[0]])
axes[1].invert_yaxis(); axes[1].set_title('Per-trace min-max excitation, range [0, 1]\n(NOT what Eq 7 describes -- see note above)')
plt.colorbar(im1, ax=axes[1], shrink=0.8)
for ax in axes:
    ax.set_xlabel('Time (ns)')
axes[0].set_ylabel('Along-borehole trace depth (m)')
plt.tight_layout()
fig.savefig(pp_out / 'e_normalisation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved {pp_out / 'e_normalisation.png'}")

## Post-Processing: Mode 1 / Mode 2 / Mode X12

Now checked against the paper's Section 3 and Figs. 3-5:

- **Mode 1** (Fig. 4): "the standard deviation is computed between the samples of every TR data set on the same position across the TR matrices" -- i.e. `std` over the snapshot/time stack at each (depth, radial) pixel, producing one 2D map. **Confirmed exact match** to what's implemented below.
- **Mode 2** (Fig. 5): "the standard deviation is computed for each A-scan of each TR matrix" using windowed sets of depth samples (Fig. 5b/c show a local, windowed std along depth, evaluated per A-scan, per snapshot) -- matches the local-windowed-std-along-depth already implemented below. **The one remaining gap**: the paper says the final Mode 2 image comes from "considering all TR matrices" (i.e. all snapshots) but never states the combination rule across time. `max` is used here as the most defensible reading (mirrors Mode 1's use of variability-over-time as the informative signal) -- if you can find where the paper states this explicitly, let me know and I'll correct it.
- **Mode X12**: "measures the similarity of the data sets... emphasizes common features of both modes" -- implemented as a normalised element-wise product of Mode 1 and Mode 2 (each independently peak-normalised first). The paper calls this "cross-correlation" but describes it purely as a similarity/agreement measure, not a shifted/windowed correlation -- the product interpretation matches that description.
- **Eq 7 (amplitude rescaling)**: now applied where it belongs -- as a single global min/max over the whole TR wavefield stack, *before* computing Mode 1/2/X12 (added below). Since it's a linear rescaling, it doesn't change the *shape* of any mode, only the numeric scale -- confirmed by testing both ways.

Run on the existing `borehole_prof_1_dx02` gprMax output (our current best result) -- no new gprMax run needed. The near-source region is masked first (same `BH_NEAR_MASK_M` convention as the rest of this notebook), since std-based statistics would otherwise be dominated by the injection halo rather than any real target -- confirmed by testing without the mask first.

In [ ]:
# ── Compute Mode 1 / Mode 2 / Mode X12 from the borehole_prof_1_dx02 snapshot stack ──
from scipy.ndimage import uniform_filter1d

MODES_SLUG = 'borehole_prof_1_dx02'
MODES_RUN  = 1

snap_dir = OUT_DIR / 'backprop' / MODES_SLUG / f'backprop_{MODES_SLUG}_snaps'
snap_files = sorted(snap_dir.glob('bp_snap*.vti'), key=lambda p: int(p.stem.replace('bp_snap', '')))
print(f'{len(snap_files)} snapshots found for {MODES_SLUG}')

x_shift_modes, y_bh_end_modes = _borehole_geom_for_run(MODES_RUN)

stack = []
dx_modes = None
for f in snap_files:
    mesh = pyvista.read(str(f))
    nx_c = mesh.dimensions[0] - 1
    ny_c = mesh.dimensions[1] - 1
    dx_modes = float(mesh.spacing[0])
    ez = np.array(mesh['E-field'])[:, 2].reshape(ny_c, nx_c).T   # (depth, radial)
    mask_px = max(1, round((y_bh_end_modes + BH_NEAR_MASK_M) / dx_modes))
    ez[:, :mask_px] = 0.0   # near-source mask, applied per frame before any statistics
    stack.append(ez)
stack = np.array(stack)   # (n_time, depth, radial)
print('stack shape (time, depth, radial):', stack.shape)

# Eq 7: rescale the whole TR wavefield stack to [0,1] using ONE global min/max (not
# per-frame/per-pixel) -- see the corrected note in the amplitude-normalisation cell
# above for why this belongs here rather than on the pre-injection excitation.
stack_min, stack_max = stack.min(), stack.max()
stack = (stack - stack_min) / (stack_max - stack_min)

# Mode 1: std across time at each pixel (Fig. 4 -- confirmed exact match)
mode1 = np.std(stack, axis=0)

# Mode 2 (Fig. 5): local std along depth per snapshot, then combined across snapshots.
# "max" is used for the cross-snapshot combination -- the one remaining gap versus the
# paper text, see markdown note above.
def _local_depth_std(frame, window=11):
    mean = uniform_filter1d(frame, size=window, axis=0, mode='nearest')
    mean_sq = uniform_filter1d(frame ** 2, size=window, axis=0, mode='nearest')
    return np.sqrt(np.maximum(mean_sq - mean ** 2, 0))

mode2 = np.max(np.array([_local_depth_std(frame) for frame in stack]), axis=0)

# Mode X12: normalised element-wise product (see markdown note above)
def _norm01(a):
    a = a - a.min()
    m = a.max()
    return a / m if m > 0 else a

modeX12 = _norm01(mode1) * _norm01(mode2)

nx_c, ny_c = mode1.shape
depth_axis_modes = np.arange(nx_c) * dx_modes - x_shift_modes
radial_axis_modes = np.arange(ny_c) * dx_modes - y_bh_end_modes

fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=True)
for ax, img, title in zip(axes, (mode1, mode2, modeX12),
                           ('Mode 1: std(time)', 'Mode 2: max_t(local std along depth)', 'Mode X12: normalised product')):
    clim = np.percentile(img, 99.5)
    im = ax.imshow(img, aspect='auto', cmap='inferno', origin='lower',
                   extent=[radial_axis_modes[0], radial_axis_modes[-1], depth_axis_modes[0], depth_axis_modes[-1]],
                   vmin=0, vmax=clim)
    ax.invert_yaxis()
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Radial [m]')
    ax.set_xlim(*BH_RADIAL_RANGE)
    ax.set_ylim(*BH_DEPTH_RANGE[::-1])
    plt.colorbar(im, ax=ax, shrink=0.8)
axes[0].set_ylabel('Depth [m]')
plt.tight_layout()
out_path = bh_out / f'borehole_modes_{MODES_SLUG}.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show(); plt.close(fig)
print(f'Saved {out_path}')

## Final gprMax Input: prof_1, dx=0.02, min-max Normalisation

Settings confirmed through the experiments above:
- **Grid**: `dx=0.02` m (5 cells across the 10 cm borehole) -- the confirmed fix for the clutter seen at the original `dx=0.05` m (2 cells).
- **Water permittivity**: scaled (`eps_r_water_half = 324 = 4x81`) -- the unscaled variant introduced a position shift and didn't clean up the clutter.
- **Clutter removal (Eq 8/9/10-11/12)**: none. All four visibly ate into the real reflector in the before/after comparisons above, so none are used here.
- **Normalisation**: `normalize_mode='minmax'` -- Eq 7 style, per-trace `(x-x_min)/(x_max-x_min)`, range `[0,1]` instead of the usual peak-normalised `[-1,1]`.

**Worth remembering**: `minmax` is *not* sign-preserving -- every injected trace sits on a non-zero DC baseline instead of a physical zero (demonstrated in the normalisation comparison cell above, median ~0.5 rather than ~0). Per the paper's own text, Eq 7 most likely describes normalising the *output* TR wavefield, not this injection step -- so this run is an explicit experiment with the alternative normalisation you asked for, not a reproduction of what the paper actually does at this stage. Worth comparing directly against `borehole_prof_1_dx02` (peak-normalised, otherwise identical settings) once both have run.

In [ ]:
# ── Final input file: prof_1, dx=0.02, scaled water eps, NO clutter removal, min-max norm ──
# Settings chosen from the experiments above: dx=0.02 (grid-resolution fix), water
# permittivity scaled (unscaled version shifted the reflector position), no Eq 8-12
# clutter removal (all four visibly weakened the real reflector), min-max normalisation
# instead of peak (per your choice) -- see markdown note above for the normalisation
# caveat.
FINAL_DX = 0.02

f_cut_hz_final = dispersion_limited_cutoff(eps_r_half_bh2, FINAL_DX)

data_final, _ = load_mala(str(DATA / _prof_name(1)), return_object=False)
n_final = min(data_final.shape[1], n_traces)

d_bp_final = filter_data(data_final[:, :n_final], fq=(0.02, 0.2), sfreq=sf, btype='bandpass')
d_dc_final, _ = remove_mean(d_bp_final, 299, 517)
d_aligned_final, _, _ = align_traces(d_dc_final, ref_aligned[:, :n_final], upsample=5, normalize=True, align_reference=False)
d_svd_final, _ = remove_svd(d_aligned_final, low_s=0, high_s=1)

z_bh_final = depth[:n_final]
dt_bh_final = 1.0 / sf

Dt_diff_final = d_svd_final[:rad_cut, :].T - ref_svd[:rad_cut, :n_final].T
Dt_2d_final = apply_3d_to_2d_correction(Dt_diff_final, dt=dt_bh_final * 1e-9, velocity=v * 1e9, time_zero_idx=299)

_sp_tap_final = _tukey_sp_bh2(n_final, alpha=0.30)[:, np.newaxis]
Dt_tapered_final = Dt_2d_final * _sp_tap_final
_t_tap_final = _tukey_sp_bh2(Dt_tapered_final.shape[1], alpha=0.10)[np.newaxis, :]
Dt_tapered_final = Dt_tapered_final * _t_tap_final
_v_mig_final = v / 2.0
_D_fk_final = np.fft.fft(np.fft.rfft(Dt_tapered_final, axis=1), axis=0)
_freq_final = np.fft.rfftfreq(Dt_tapered_final.shape[1], d=dt_bh_final)
_kz_final = np.fft.fftfreq(Dt_tapered_final.shape[0], d=dL)
_evan_final = np.abs(_kz_final[:, None]) > np.abs(_freq_final[None, :]) / _v_mig_final
_D_fk_final[_evan_final] = 0.0
Dt_tapered_final = np.real(np.fft.irfft(np.fft.ifft(_D_fk_final, axis=0), n=Dt_tapered_final.shape[1], axis=1))

FINAL_SLUG = 'borehole_prof_1_final_minmax'
in_path_final, n_src_final, n_snaps_final, t_focus_final, geom_final = write_borehole_backprop_files(
    OUT_DIR, label=FINAL_SLUG, slug=FINAL_SLUG,
    tapered_ntr_nt=Dt_tapered_final, dt_ns=dt_bh_final, x_midpoints=z_bh_final,
    t0_ns=t0_ns_bh2, eps_r=eps_r_bh2, v_ice=v,
    eps_r_water=BH_EPS_R, sigma_water=BH_SIGMA,
    borehole_width=BH_WIDTH, left_buffer=BH_LEFT_BUF, imaging_range=BH_IMG_RNG,
    src_offset=BH_SRC_OFFSET, dx=FINAL_DX, pml_cells=bp_pml,
    scale_water_eps=True, normalize_mode='minmax',
)
exc_file_final = OUT_DIR / 'backprop' / FINAL_SLUG / 'excitation.txt'
lowpass_filter_excitation(exc_file_final, f_cut_hz_final, edge_exclude=5)

print(f'wrote {in_path_final}')
print(f'n_src={n_src_final}  n_snaps={n_snaps_final}  t_focus={t_focus_final:.2f} ns')
print(f'domain={geom_final["domain_x"]:.2f}x{geom_final["domain_y"]:.2f} m  '
      f'dx={FINAL_DX} m ({BH_WIDTH/FINAL_DX:.0f} cells across the borehole)')
exc_check = np.loadtxt(exc_file_final, skiprows=1)
print(f'excitation amplitude range: [{exc_check[:,1:].min():.4f}, {exc_check[:,1:].max():.4f}]')

In [ ]:
# ── Compare the min-max-normalised final input against the peak-normalised baseline ──
FINAL_COMPARE_PANELS = [
    ('Kirchhoff_bp', 'mig', 'kirchhoff_bp'),
    ('Gazdag', 'mig', 'gazdag'),
    ('dx=0.02, peak-norm\n(borehole_prof_1_dx02)', 'bp', 'borehole_prof_1_dx02'),
    ('dx=0.02, min-max norm\n(borehole_prof_1_final_minmax)', 'bp', 'borehole_prof_1_final_minmax'),
]

fig, axes = plt.subplots(1, len(FINAL_COMPARE_PANELS), figsize=(4.3 * len(FINAL_COMPARE_PANELS), 6), sharey=True)
fig.suptitle('Final input comparison -- prof_1: peak vs min-max normalisation', y=1.02)

for ax, (title, kind, key) in zip(axes, FINAL_COMPARE_PANELS):
    if kind == 'mig':
        img = _load_mig(key, 1)
        if img is not None:
            n = img.shape[0]
            ext = [x_img[0], x_img[-1], depth[n - 1], depth[0]]
            clim = np.percentile(np.abs(img), 100)
            ax.imshow(img, aspect='auto', cmap='seismic', extent=ext, origin='upper', vmin=-clim, vmax=clim)
            ax.invert_yaxis()
        else:
            ax.text(0.5, 0.5, 'not found', ha='center', va='center', transform=ax.transAxes)
    else:
        frame = _best_borehole_focus_variant(key, 1)
        if frame is not None:
            ez, depth_axis, radial_axis, snap_idx, snap_t = frame
            clim = np.percentile(np.abs(ez), 100)
            ax.imshow(ez, aspect='auto', cmap='seismic',
                      extent=[radial_axis[0], radial_axis[-1], depth_axis[0], depth_axis[-1]],
                      origin='lower', vmin=-clim, vmax=clim)
            ax.invert_yaxis()
            ax.text(0.98, 0.02, f't={snap_t:.2f} ns\n(snap {snap_idx})', ha='right', va='bottom',
                    transform=ax.transAxes, fontsize=7, color='white')
        else:
            ax.text(0.5, 0.5, 'not found', ha='center', va='center', transform=ax.transAxes)
    ax.set_title(title, fontsize=9)
    ax.set_ylim(*BH_DEPTH_RANGE)
    ax.set_xlim(*BH_RADIAL_RANGE)
    ax.set_xlabel('Radial [m]')

axes[0].set_ylabel('Depth [m]')
plt.tight_layout()
out_path = bh_out / 'borehole_final_minmax_vs_peak.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show(); plt.close(fig)
print(f'Saved {out_path}')
print()
print('RESULT: the min-max-normalised run is saturated with a strong, near-uniform')
print('negative bias across nearly the whole domain -- no coherent reflector structure')
print('survives. This confirms the DC-bias concern flagged earlier: injecting a')
print('current source with a non-zero baseline pumps large spurious low-frequency')
print("energy into the grid. Recommendation: revert to normalize_mode='peak'")
print('(the borehole_prof_1_dx02 run) as the production setting.')

## Gaussian Smoothing + RPCA Decomposition (Li & Yan 2021)

Following Li & Yan (2021, *Appl. Sci.* 11, 10234), Section 2.3 steps 2-3: after imaging,
apply Gaussian smoothing to the migrated/focused result, then decompose it with Robust
PCA (Eq 3-4 in that paper) into a low-rank part $A$ (regular structure -- residual
clutter, injection-halo remnants, layering) and a sparse part $E$ (compact, anomalous
reflector responses).

Their "BP imaging" is a classic delay-and-sum image, not our time-reversal focus
snapshot -- but the post-processing in steps 2-3 operates on *any* 2D migrated image, so
it transfers directly. Applied here to `borehole_prof_1_dx02`, the peak-normalised,
confirmed-good back-propagation run (**not** the min-max variant, which was already
shown to be non-physical above). This is an exploratory *output-side* post-processing
step on the focused result -- it does not touch the gprMax injection file, which still
only uses the peak-normalisation choice decided earlier.

RPCA is solved with the standard inexact augmented Lagrange multiplier (IALM) algorithm
(Lin, Chen & Ma 2010), the practical solver behind Candes et al.'s principal component
pursuit -- the same convex relaxation the paper's Eq 4 describes.

In [ ]:
# ── Gaussian smoothing + RPCA decomposition on a back-propagated result ──────────────
# Li & Yan (2021) Section 2.3, steps 2-3: Gaussian-smooth the imaging result, then split
# it via RPCA into a low-rank background/ghost component A and a sparse reflector
# component E. Applied to borehole_prof_1_dx02 (peak-normalised, confirmed-good run).
from scipy.ndimage import gaussian_filter

def rpca_ialm(D, lam=None, tol=1e-7, max_iter=100, verbose=False):
    """Robust PCA via inexact augmented Lagrange multiplier (Lin, Chen & Ma 2010).
    Solves min ||A||_* + lam*||E||_1  s.t.  A + E = D  (paper's Eq 3-4)."""
    D = np.asarray(D, dtype=float)
    m, n = D.shape
    if lam is None:
        lam = 1.0 / np.sqrt(max(m, n))
        lam /= 4
    norm_D_fro = np.linalg.norm(D, 'fro')
    norm_D_2 = np.linalg.norm(D, 2)
    norm_D_inf = np.max(np.abs(D))
    Y = D / max(norm_D_2, norm_D_inf / lam)
    A = np.zeros_like(D)
    E = np.zeros_like(D)
    mu = 1.25 / norm_D_2
    mu_bar = mu * 1e7
    rho = 1.5
    it = 0
    for it in range(1, max_iter + 1):
        U, S, Vt = np.linalg.svd(D - E + Y / mu, full_matrices=False)
        S_thr = np.maximum(S - 1.0 / mu, 0.0)
        A = (U * S_thr) @ Vt
        temp = D - A + Y / mu
        E = np.sign(temp) * np.maximum(np.abs(temp) - lam / mu, 0.0)
        Z = D - A - E
        Y = Y + mu * Z
        mu = min(mu * rho, mu_bar)
        err = np.linalg.norm(Z, 'fro') / norm_D_fro
        if verbose and it % 10 == 0:
            print(f'  iter {it}: err={err:.2e}  rank(A)={int(np.sum(S_thr > 0))}')
        if err < tol:
            break
    return A, E, it

# Pull the best-focus frame for the confirmed-good run and crop to the plotted extent
frame_rpca = _best_borehole_focus_variant('borehole_prof_1_dx02', 1)
ez_raw, depth_axis_r, radial_axis_r, snap_idx_r, snap_t_r = frame_rpca

_dmask = (depth_axis_r >= BH_DEPTH_RANGE[0]) & (depth_axis_r <= BH_DEPTH_RANGE[1])
_rmask = (radial_axis_r >= BH_RADIAL_RANGE[0]) & (radial_axis_r <= BH_RADIAL_RANGE[1])
ez_crop = ez_raw[np.ix_(_dmask, _rmask)]
depth_crop = depth_axis_r[_dmask]
radial_crop = radial_axis_r[_rmask]
print(f'Cropped frame: {ez_crop.shape[0]} x {ez_crop.shape[1]} px  '
      f'(snap {snap_idx_r}, t={snap_t_r:.2f} ns)')

GAUSS_SIGMA_PX = 1.5   # smoothing width in grid cells (dx=0.02 m -> ~3 cm)
ez_smooth = gaussian_filter(ez_crop, sigma=GAUSS_SIGMA_PX)

print('Running RPCA (IALM)...')
A_rpca, E_rpca, n_iter = rpca_ialm(ez_smooth, verbose=True)

energy_A = np.linalg.norm(A_rpca, 'fro')
energy_E = np.linalg.norm(E_rpca, 'fro')
print(f'\nConverged in {n_iter} iterations.')
print(f'||A||_F (low-rank) = {energy_A:.4g}   ||E||_F (sparse) = {energy_E:.4g}   '
      f'E/(A+E) = {energy_E / (energy_A + energy_E):.2%}')
print()
print('INTERPRETATION: with the paper default lam=1/sqrt(max(m,n)), E only picks up a')
print('handful of isolated hot spots (near the borehole) and a faint arc -- the main')
print('diagonal reflection band stays in the low-rank component A. This differs from')
print('the paper\'s result, where the target (a compact seepage channel) produces a')
print('point-like BP response that RPCA cleanly isolates. Our reflector response is more')
print('spatially extended/banded, so it behaves as low-rank rather than sparse under the')
print('default weighting. Lowering lam (rpca_ialm(..., lam=...)) pushes more energy into')
print('E at the cost of true sparsity -- worth sweeping if a harder separation is wanted,')
print('but the current result should be read as a null/negative finding for this target')
print('shape, not a successful clutter-vs-target split.')

panels_rpca = [
    ('Raw back-propagation\n(borehole_prof_1_dx02)', ez_crop),
    (f'Gaussian-smoothed\n(sigma={GAUSS_SIGMA_PX} px)', ez_smooth),
    ('RPCA low-rank A\n(background / ghosts)', A_rpca),
    ('RPCA sparse E\n(extracted reflector)', E_rpca),
]

fig, axes = plt.subplots(1, 4, figsize=(17, 6), sharey=True)
fig.suptitle('Gaussian smoothing + RPCA decomposition (Li & Yan 2021) -- prof_1', y=1.02)
for ax, (title, img) in zip(axes, panels_rpca):
    clim = np.percentile(np.abs(img), 100)
    ax.imshow(img, aspect='auto', cmap='seismic',
              extent=[radial_crop[0], radial_crop[-1], depth_crop[0], depth_crop[-1]],
              origin='lower', vmin=-clim, vmax=clim)
    ax.invert_yaxis()
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('Radial distance from borehole [m]')
axes[0].set_ylabel('Depth [m]')
plt.tight_layout()
out_path = bh_out / 'borehole_prof_1_dx02_gaussian_rpca.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show(); plt.close(fig)
print(f'Saved {out_path}')

## f-k Dip (Fan) Filtering -- an Alternative for a Non-Sparse Reflector

RPCA above failed to separate clutter from the target because the target's response is
spatially extended/banded, not point-like -- it behaves as low-rank rather than sparse,
so RPCA's sparsity assumption doesn't fit it. A better match for an *extended* reflector
is to separate by **dip/orientation instead of amplitude sparsity**: transform the image
to the wavenumber (f-k, here kz-kx) domain and keep only the wedge of orientations
matching the reflector's own dip, rejecting everything else.

Why this should work here: in the panels below, the injection-halo clutter near the
borehole is close to vertical (near-constant across depth at fixed small radial
distance) and the spurious arc further out has curvature that sweeps through many
different local dips -- neither shares the reflector's single, fairly steep, consistent
dip. A dip filter keeps the one orientation and suppresses the rest, which is exactly
the discriminator RPCA didn't have access to.

**Dip estimate**: rather than eyeballing a slope off the plot, this cell picks the
depth-of-max-|amplitude| for each radial column in a sub-window known to contain the
primary reflection (radial 0-6 m, after the injection-halo mask), then fits a straight
line through those picks. That slope defines the center of the fan.

**Fan filter**: a wedge in (kz, kx)-space around the wavenumber direction corresponding
to that dip, with a cosine-tapered edge (same idea as the Tukey tapers used earlier in
this pipeline) rather than a hard cutoff -- a hard-edged fan produces visible Gibbs
ringing (parallel diagonal streaking) in the filtered image.

The sweep below shows several fan half-widths; narrower fans (~8 deg) still ring
noticeably even with tapering, while ~15-20 deg gives a good balance: the near-source
clutter is visibly suppressed and the reflection band comes through sharper than the
raw result.

In [ ]:
# ── f-k (dip / fan) filtering: an alternative to RPCA for an extended reflector ──────
# Reuses ez_crop / depth_crop / radial_crop / bh_out from the Gaussian+RPCA cell above --
# run that cell first. Unlike RPCA (amplitude sparsity), this separates by dip/orientation
# in the wavenumber domain, which fits this target's spatially extended response better.

# Step 1: estimate the reflector's dip directly from the data (per-column peak pick +
# linear fit), over a sub-window known to contain the primary reflection.
DIP_EST_RADIAL_RANGE = (0.0, 6.0)   # [m] radial sub-window for the dip fit
_sub_rmask = (radial_crop >= DIP_EST_RADIAL_RANGE[0]) & (radial_crop <= DIP_EST_RADIAL_RANGE[1])
_sub = np.abs(ez_smooth[:, _sub_rmask])
_valid_cols = _sub.max(axis=0) > 1e-12   # exclude the still-masked injection-halo columns
_depth_peak_idx = np.argmax(_sub[:, _valid_cols], axis=0)
_depth_peak = depth_crop[_depth_peak_idx]
_radial_sub = radial_crop[_sub_rmask][_valid_cols]

m0, b0 = np.polyfit(_radial_sub, _depth_peak, 1)
print(f'Dip fit used {len(_radial_sub)} columns (radial {_radial_sub.min():.2f}-{_radial_sub.max():.2f} m)')
print(f'Estimated target dip: m0={m0:.3f} (depth m per radial m), intercept={b0:.2f} m')

# Step 2: build the wavenumber grid and the tapered fan mask around that dip.
Nz, Nx = ez_crop.shape
dz_px = depth_crop[1] - depth_crop[0]
dx_px = radial_crop[1] - radial_crop[0]
kz = np.fft.fftfreq(Nz, d=dz_px)
kx = np.fft.fftfreq(Nx, d=dx_px)
KZ, KX = np.meshgrid(kz, kx, indexing='ij')

# A real-space line of slope m0 (depth per radial) has its FT support along the
# wavenumber direction (kz, kx) = (1, -m0) (and its negative -- same line). Compare each
# grid point's orientation to that direction, wrapped mod pi (direction == -direction).
theta_k = np.arctan2(KZ, KX)
theta_t = np.arctan2(1.0, -m0)
dtheta = np.angle(np.exp(1j * 2.0 * (theta_k - theta_t))) / 2.0

def fan_taper_mask(dtheta, hw_deg, taper_deg=5.0):
    hw = np.deg2rad(hw_deg)
    tp = np.deg2rad(max(taper_deg, 1e-6))
    ad = np.abs(dtheta)
    mask = np.ones_like(ad)
    outer = hw + tp
    ramp = (ad > hw) & (ad < outer)
    mask[ad >= outer] = 0.0
    mask[ramp] = 0.5 * (1.0 + np.cos(np.pi * (ad[ramp] - hw) / tp))
    return mask

FK_TAPER_DEG = 5.0
FK_HALFWIDTHS_DEG = [8, 12, 15, 20, 70]

F_raw = np.fft.fft2(ez_crop)
fk_variants = [('raw', ez_crop)]
for hw in FK_HALFWIDTHS_DEG:
    mask = fan_taper_mask(dtheta, hw, FK_TAPER_DEG)
    ez_fk = np.real(np.fft.ifft2(F_raw * mask))
    amp_ratio = np.linalg.norm(ez_fk) / np.linalg.norm(ez_crop)
    print(f'  {hw:2d} deg fan: retains {amp_ratio:.1%} of raw amplitude (Frobenius norm)')
    fk_variants.append((f'{hw} deg fan', ez_fk))

fig, axes = plt.subplots(1, len(fk_variants), figsize=(4 * len(fk_variants), 6))
fig.suptitle('f-k dip (fan) filtering -- prof_1 back-propagation', y=1.02)
for ax, (title, img) in zip(axes, fk_variants):
    clim = np.percentile(np.abs(img), 100)
    ax.imshow(img, aspect='auto', cmap='seismic',
              extent=[radial_crop[0], radial_crop[-1], depth_crop[0], depth_crop[-1]],
              origin='lower', vmin=-clim, vmax=clim)
    ax.invert_yaxis()
    ax.set_ylim(BH_DEPTH_RANGE[1], BH_DEPTH_RANGE[0])
    ax.set_xlim(0, 12)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('Radial [m]')
axes[0].set_ylabel('Depth [m]')
plt.tight_layout()
out_path = bh_out / 'borehole_prof_1_dx02_fk_dip_sweep.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show(); plt.close(fig)
print(f'Saved {out_path}')
print()
print('Recommendation: ~15-20 deg (with the 5 deg taper) gives the best trade-off seen')
print('here -- narrower fans (~8 deg) still show visible ringing (parallel diagonal')
print('streaking) even with the cosine taper, because so little of k-space survives.')

### Fixing the Diagonal-Band Artifact: Amplitude Gate vs. Windowed Filter

The 20 deg fan looked cleanest above, but widening the fan lets more incoherent noise
through at the target's orientation -- and because the f-k filter is global (one FFT over
the whole image, translation-invariant), any noise energy at that orientation gets
reconstructed as a coherent diagonal streak *wherever in the image it happens to occur*,
not just near the real reflector. That is the "diagonal bands throughout the image"
effect.

Two candidate fixes, tried side by side below:

1. **Amplitude gate**: build a broad spatial envelope from the *raw* (unfiltered) data's
   local energy, normalise it, and multiply the fan-filtered result by it. This
   suppresses fan-filter output wherever the original data had no real energy to begin
   with, regardless of orientation.
2. **Windowed (localised) fan filter**: instead of one FFT over the full 12 m radial
   extent, slide a smaller window (with Hann-tapered overlap-add) across the image and
   apply the same fan filter locally within each window.

Only one of these turned out to help.

In [ ]:
# ── Fixing the diagonal-band artifact: amplitude gate vs. windowed filter ────────────
# Reuses ez_crop / dtheta / m0 / kz / dz_px / dx_px / fan_taper_mask / FK_TAPER_DEG /
# F_raw / fk_variants from the f-k dip filtering cell above -- run that cell first.
from scipy.signal.windows import hann

ez_fk20 = dict(fk_variants)['20 deg fan']

# ---- Approach 1: amplitude/energy gate built from the raw (unfiltered) data ----
ENERGY_SIGMA_PX = 8   # broader than the sigma=1.5 smoothing used for the dip fit --
                       # this builds a spatial envelope, not a denoised copy of the data
energy_env = gaussian_filter(np.abs(ez_crop), sigma=ENERGY_SIGMA_PX)
gate = np.clip(energy_env / np.percentile(energy_env, 99), 0, 1)
ez_fk20_gated = ez_fk20 * gate

# ---- Approach 2: windowed (localised) fan filter along the radial axis ----
def windowed_fan_filter(img, kz, dx_px, m0, hw_deg, taper_deg, win_px, hop_px):
    Nz, Nx = img.shape
    out = np.zeros_like(img)
    weight = np.zeros(Nx)
    win = hann(win_px, sym=False)
    starts = list(range(0, Nx - win_px + 1, hop_px))
    if not starts or starts[-1] != Nx - win_px:
        starts.append(Nx - win_px)
    theta_t_loc = np.arctan2(1.0, -m0)
    kx_loc = np.fft.fftfreq(win_px, d=dx_px)
    KZ_loc, KX_loc = np.meshgrid(kz, kx_loc, indexing='ij')
    theta_k_loc = np.arctan2(KZ_loc, KX_loc)
    dtheta_loc = np.angle(np.exp(1j * 2.0 * (theta_k_loc - theta_t_loc))) / 2.0
    mask_loc = fan_taper_mask(dtheta_loc, hw_deg, taper_deg)
    for start in starts:
        end = start + win_px
        patch_f = np.real(np.fft.ifft2(np.fft.fft2(img[:, start:end]) * mask_loc))
        out[:, start:end] += patch_f * win[np.newaxis, :]
        weight[start:end] += win
    weight[weight == 0] = 1.0
    return out / weight[np.newaxis, :]

WIN_M, HOP_M = 3.0, 1.0
win_px = int(round(WIN_M / dx_px))
hop_px = int(round(HOP_M / dx_px))
ez_fk20_windowed = windowed_fan_filter(ez_crop, kz, dx_px, m0, 20, FK_TAPER_DEG, win_px, hop_px)

fig, axes = plt.subplots(1, 4, figsize=(17, 6))
fig.suptitle('Diagonal-band artifact: amplitude gate vs. windowed filter (20 deg fan)', y=1.02)
gate_panels = [
    ('raw', ez_crop),
    ('global fan 20deg', ez_fk20),
    ('global fan 20deg\n+ amplitude gate', ez_fk20_gated),
    (f'windowed fan 20deg\n(win={WIN_M}m, hop={HOP_M}m)', ez_fk20_windowed),
]
for ax, (title, img) in zip(axes, gate_panels):
    clim = np.percentile(np.abs(img), 100)
    ax.imshow(img, aspect='auto', cmap='seismic',
              extent=[radial_crop[0], radial_crop[-1], depth_crop[0], depth_crop[-1]],
              origin='lower', vmin=-clim, vmax=clim)
    ax.invert_yaxis()
    ax.set_ylim(BH_DEPTH_RANGE[1], BH_DEPTH_RANGE[0])
    ax.set_xlim(0, 12)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('Radial [m]')
axes[0].set_ylabel('Depth [m]')
plt.tight_layout()
out_path = bh_out / 'borehole_prof_1_dx02_fk_dip_gate_vs_windowed.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show(); plt.close(fig)
print(f'Saved {out_path}')
print()
print('RESULT: the amplitude gate works well -- it removes essentially all of the far-')
print('field (radial > 6 m) diagonal banding while leaving the real reflector and the')
print('near-source region untouched, since it is built from the energy of the raw data,')
print('not from the (already noise-contaminated) fan-filtered output.')
print()
print('The windowed filter did NOT meaningfully help -- diagonal streaking still spans')
print('most of the radial extent. Localising the FFT limits how far a single streak can')
print('run, but it does not stop locally-oriented noise from being reconstructed within')
print('each window; the artifact is a property of *what survives the dip criterion*, not')
print('of the FFT window size. The gate targets the actual cause (no real energy there)')
print('rather than a proxy for it, which is why it works and windowing alone does not.')
print()
print('Recommendation: pair the 20 deg global fan filter with the amplitude gate; drop')
print('the windowed variant.')

## Per-Profile Fan Tuning + kz-kx Spectrum Diagnostic

Applying the profile-1-tuned 20 deg fan to all five profiles did not generalise: the
final processed images for profiles 8, 20 and 38 lose the main reflection almost
entirely. The automatic dip fit (per-column peak pick + linear fit) explains why --
profile 1 (m0=2.13) and profile 3 (m0=2.25) agree well, but profiles 8 (m0=0.08), 20
(m0=0.66) and 38 (m0=0.85) come out dramatically different. Near-source clutter appears
to dominate the per-column peak pick strongly enough in those three profiles to pull the
linear fit toward a much shallower slope than the true reflector's -- so the fan filter
ends up centred on the wrong orientation entirely, not just too narrow.

This cell visualises, per profile: the raw image with the fitted dip line drawn on top
(so a wrong dip fit is visible directly, not just inferred from the final result), and
the 2D FK magnitude spectrum with the current fan filter's pass-band contour overlaid in
red, so you can see whether the filter actually sits on the reflector's energy ridge in
k-space.

`FAN_HW_BY_RUN` and `DIP_OVERRIDE_BY_RUN` below are per-profile controls -- edit them and
re-run this cell, then the final-recipe cell after it (which now reads from the same two
dicts), to test different fan widths or override a bad automatic dip fit directly.

Two corrections from the first version of this diagnostic: (1) the crop used to include
the fully-zeroed padding inside the injection-halo mask (radial 0-1.0 m), producing a
hard internal step straight from zero to real signal that an edge-only Tukey taper can't
reach (it only smooths the array's outer boundary) -- the crop now starts at the mask
boundary itself, which removed a horizontal artefact band that was otherwise dominating
the spectrum. (2) both axes of this spectrum are wavenumbers ($k_z$, $k_x$), not
frequency and wavenumber -- the input is a single-time focused *image* over 2D space
(depth, radial), not a time-space B-scan, so a 2D FFT of it has no frequency axis at
all. Labelled "kz-kx spectrum" below rather than the seismic-jargon "F-K spectrum" that
conventionally implies a frequency axis.

In [ ]:
# ── Per-profile fan tuning: kz-kx spectrum + filter overlay diagnostic ───────────────
# Edit FAN_HW_BY_RUN / DIP_OVERRIDE_BY_RUN below and re-run this cell (and the final
# recipe cell after it, which reads from the same dicts) to test different values.
# Reuses gaussian_filter / fan_taper_mask / FK_TAPER_DEG / BH_DEPTH_RANGE /
# BH_RADIAL_RANGE / GAUSS_SIGMA_PX / _best_borehole_focus_variant / bh_out from the cells
# above -- run those first. Defines PROFILES_FINAL, reused by the final recipe cell below.
PROFILES_FINAL = [1, 3, 8, 20, 38]

FAN_HW_BY_RUN = {1: 20, 3: 20, 8: 20, 20: 20, 38: 20}   # [deg] per-profile fan half-width
DIP_OVERRIDE_BY_RUN = {1: 2.2, 3: 2.2, 8: 2.2, 20: 2.2, 38: 2.2}  # testing: all profiles at the shared dip

from scipy.signal.windows import tukey as _tukey_edge

def edge_taper_2d(shape, alpha=0.2):
    """2D Tukey taper -- suppresses the hard crop edges before any FFT. Without this,
    the bottom depth-crop edge (still ~20% of peak amplitude when cut) and the internal
    near-source mask's hard zero step both leak into the kz-kx spectrum as a bright cross
    along kz=0/kx=0, and contaminate the fan filter's output the same way."""
    win_z = _tukey_edge(shape[0], alpha=alpha)
    win_x = _tukey_edge(shape[1], alpha=alpha)
    return np.outer(win_z, win_x)

def _crop_and_dip(run, dip_override_map=DIP_OVERRIDE_BY_RUN):
    slug = f'borehole_prof_{run}_dx02'
    frame = _best_borehole_focus_variant(slug, run)
    if frame is None:
        return None
    ez_r, depth_r, radial_r, snap_idx, snap_t = frame
    dmask = (depth_r >= BH_DEPTH_RANGE[0]) & (depth_r <= BH_DEPTH_RANGE[1])
    rmask = (radial_r >= max(BH_RADIAL_RANGE[0], BH_NEAR_MASK_M)) & (radial_r <= BH_RADIAL_RANGE[1])
    ez_c = ez_r[np.ix_(dmask, rmask)]
    depth_c = depth_r[dmask]
    radial_c = radial_r[rmask]
    ez_s = gaussian_filter(ez_c, sigma=GAUSS_SIGMA_PX)
    sub_rmask = (radial_c >= 0.0) & (radial_c <= 6.0)
    sub = np.abs(ez_s[:, sub_rmask])
    valid_cols = sub.max(axis=0) > 1e-12
    depth_peak_idx = np.argmax(sub[:, valid_cols], axis=0)
    depth_peak = depth_c[depth_peak_idx]
    radial_sub = radial_c[sub_rmask][valid_cols]
    m0_fit, b0_fit = np.polyfit(radial_sub, depth_peak, 1)
    if run in dip_override_map:
        m0 = dip_override_map[run]
        b0 = float(np.median(depth_peak - m0 * radial_sub))
    else:
        m0, b0 = m0_fit, b0_fit
    return ez_c, depth_c, radial_c, m0, b0, m0_fit, depth_peak, radial_sub

fig, axes = plt.subplots(2, len(PROFILES_FINAL), figsize=(4.4 * len(PROFILES_FINAL), 9))
fig.suptitle('Per-profile dip fit + kz-kx spectrum / filter overlay', y=1.01)

for col, run in enumerate(PROFILES_FINAL):
    result = _crop_and_dip(run)
    ax_img, ax_fk = axes[0, col], axes[1, col]
    if result is None:
        for ax in (ax_img, ax_fk):
            ax.text(0.5, 0.5, f'prof_{run}\nnot found', ha='center', va='center', transform=ax.transAxes)
        continue
    ez_c, depth_c, radial_c, m0, b0, m0_fit, depth_peak, radial_sub = result
    hw_deg = FAN_HW_BY_RUN.get(run, 20)
    overridden = run in DIP_OVERRIDE_BY_RUN
    print(f'prof_{run}: auto dip m0={m0_fit:.3f}' +
          (f'  -> OVERRIDDEN to {m0:.3f}' if overridden else ' (used as-is)') +
          f'  fan_hw={hw_deg} deg')

    clim = np.percentile(np.abs(ez_c), 100)
    ax_img.imshow(ez_c, aspect='auto', cmap='seismic',
                  extent=[radial_c[0], radial_c[-1], depth_c[0], depth_c[-1]],
                  origin='lower', vmin=-clim, vmax=clim)
    ax_img.plot(radial_sub, depth_peak, '.', color='lime', ms=2, alpha=0.5, label='per-column peak pick')
    ax_img.plot(radial_c, m0 * radial_c + b0, '--', color='black', lw=1.2,
                label=f'dip fit (m0={m0:.2f}{" override" if overridden else ""})')
    ax_img.invert_yaxis()
    ax_img.set_ylim(BH_DEPTH_RANGE[1], BH_DEPTH_RANGE[0])
    ax_img.set_xlim(0, 12)
    ax_img.set_title(f'prof_{run}: raw + dip fit', fontsize=9)
    ax_img.legend(fontsize=6, loc='lower right')

    Nz_d, Nx_d = ez_c.shape
    dz_d = depth_c[1] - depth_c[0]
    dx_d = radial_c[1] - radial_c[0]
    kz_d = np.fft.fftfreq(Nz_d, d=dz_d)
    kx_d = np.fft.fftfreq(Nx_d, d=dx_d)
    KZ_d, KX_d = np.meshgrid(kz_d, kx_d, indexing='ij')
    theta_k_d = np.arctan2(KZ_d, KX_d)
    theta_t_d = np.arctan2(1.0, -m0)
    dtheta_d = np.angle(np.exp(1j * 2.0 * (theta_k_d - theta_t_d))) / 2.0
    mask_d = fan_taper_mask(dtheta_d, hw_deg, FK_TAPER_DEG)

    win2d_d = edge_taper_2d(ez_c.shape, alpha=0.2)
    mag_d = np.fft.fftshift(np.log1p(np.abs(np.fft.fft2(ez_c * win2d_d))))
    kz_s = np.fft.fftshift(kz_d)
    kx_s = np.fft.fftshift(kx_d)
    ax_fk.imshow(mag_d, aspect='auto', cmap='gray',
                 extent=[kx_s[0], kx_s[-1], kz_s[0], kz_s[-1]], origin='lower')
    ax_fk.contour(kx_s, kz_s, np.fft.fftshift(mask_d), levels=[0.5], colors='red', linewidths=1.2)
    ax_fk.set_xlim(-15, 15); ax_fk.set_ylim(-15, 15)
    ax_fk.set_title(f'prof_{run}: kz-kx spectrum, {hw_deg} deg fan', fontsize=9)
    ax_fk.set_xlabel('$k_x$ [1/m]')

axes[0, 0].set_ylabel('Depth [m]')
axes[1, 0].set_ylabel('$k_z$ [1/m]')
plt.tight_layout()
out_path = bh_out / 'borehole_dx02_fk_spectrum_diag.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show(); plt.close(fig)
print(f'\nSaved {out_path}')

## Final Post-Processing Recipe: Per-Profile Fan + Amplitude Gate + Near-Borehole Taper

Adopted recipe for cleaning up the borehole back-propagation focus image, combining the
pieces validated above, applied across all five profiles used in this chapter (1, 3, 8,
20, 38):

1. **f-k dip (fan) filter, 5 deg cosine taper, per-profile half-width** -- separates the
   reflector from clutter by orientation rather than sparsity (RPCA above was rejected:
   the target is spatially extended, not point-like, so it behaved as low-rank and
   RPCA could not isolate it, even after sweeping lambda down). A single 20 deg width
   tuned on profile 1 did not generalise -- profiles 8, 20 and 38 lost the main
   reflection entirely, traced to their automatic dip fits landing far from profile
   1/3's (see the diagnostic cell above). The half-width (and, where needed, the dip
   itself) is now a per-profile setting via `FAN_HW_BY_RUN` / `DIP_OVERRIDE_BY_RUN`,
   tuned using the kz-kx-spectrum-plus-filter-overlay diagnostic above. The auto-fit
   turned out to be the real problem, not the fan width: it was locking onto near-source
   clutter for profiles 8, 20 and 38 instead of the true reflector (auto m0 of -0.08,
   0.50 and 0.65 respectively, versus profile 1/3's consistent ~2.2-2.3). Overriding all
   three to `m0 = 2.2` via `DIP_OVERRIDE_BY_RUN` recovered a clean, coherent reflection
   in all three -- confirming the diagnosis and validating the fix.
2. **Amplitude gate** built from the raw data's own local energy envelope -- removes the
   diagonal-band artifact the fan filter alone introduces (incoherent noise sharing the
   target's orientation, reconstructed everywhere in the image since the fan filter is
   global/translation-invariant). The windowed/localised fan-filter alternative was
   tested and rejected: it didn't address the underlying cause (what survives the dip
   criterion), so the streaking persisted regardless of window size.
3. **Near-borehole radial taper**: a cosine ramp from 0 at radial <= 1.0 m to full pass
   by radial = 3.0 m (widened from an initial 2.5 m after visual inspection showed
   clutter persisting slightly past that point). The existing injection-halo mask
   already hard-zeros radial < 1.0 m; this extends the suppression zone with a soft edge
   (avoiding a second hard cutoff) rather than a hard mask.

The dip fit, fan filter, gate and taper are all recomputed independently per profile
(each profile's reflector sits at a different position/dip), via the
`compute_final_processed(run)` helper below, which now reads `FAN_HW_BY_RUN` and
`DIP_OVERRIDE_BY_RUN` from the diagnostic cell above. Profiles 3, 8, 20 and 38 need their
`dx=0.02` gprMax runs completed before this cell will find any data for them -- they
print `not found` and are skipped gracefully until then.

**Pending / not yet done:** `DIP_OVERRIDE_BY_RUN = {8: 2.2, 20: 2.2, 38: 2.2}` is a
manual, per-profile override, not a fix to the automatic dip-fit algorithm itself --
deliberately left as-is for now (robustifying the auto-fit, e.g. via a tighter or
outlier-resistant sub-window, is deferred until/unless more profiles beyond this
chapter's five are processed, since each new profile would otherwise need this same
diagnose-and-override cycle by hand). This recipe is otherwise an *output-side*
post-processing step applied to the completed back-propagation focus image -- it does
not change the gprMax injection file, which still only uses the peak-normalisation
choice from earlier. Deciding whether/how to fold the result into the time-lapse
analysis cells further down this notebook (which currently only consume raw,
unprocessed back-propagation snapshots) is still pending.

In [ ]:
# ── Final recipe: per-profile fan + amplitude gate + near-borehole radial taper (<3.0m) ─
# Applied across all 5 profiles used in this chapter. Reuses ez_crop-style cropping /
# gaussian_filter / fan_taper_mask / FK_TAPER_DEG / BH_DEPTH_RANGE / BH_RADIAL_RANGE /
# GAUSS_SIGMA_PX / _best_borehole_focus_variant / bh_out / PROFILES_FINAL /
# FAN_HW_BY_RUN / DIP_OVERRIDE_BY_RUN / edge_taper_2d from the cells above -- run those
# first. edge_taper_2d suppresses the hard crop-edge leakage identified in the spectrum
# diagnostic above (bottom depth edge + internal near-source mask step), which otherwise
# contaminates the fan filter's FFT the same way it contaminated the spectrum display.
RADIAL_TAPER_START_M, RADIAL_TAPER_END_M = 1.0, 3.0   # [m] cosine ramp, 0 -> full pass
FK_FINAL_HW_DEG = 20   # fallback default if a profile is missing from FAN_HW_BY_RUN
DIP_EST_RADIAL_RANGE_FINAL = (0.0, 6.0)
GATE_SIGMA_PX_FINAL = 8

def compute_final_processed(run, radial_taper_start_m=RADIAL_TAPER_START_M,
                             radial_taper_end_m=RADIAL_TAPER_END_M,
                             fan_halfwidth_deg=None, dip_override=None):
    slug = f'borehole_prof_{run}_dx02'
    frame = _best_borehole_focus_variant(slug, run)
    if frame is None:
        return None
    ez_r, depth_r, radial_r, snap_idx, snap_t = frame

    dmask = (depth_r >= BH_DEPTH_RANGE[0]) & (depth_r <= BH_DEPTH_RANGE[1])
    rmask = (radial_r >= max(BH_RADIAL_RANGE[0], BH_NEAR_MASK_M)) & (radial_r <= BH_RADIAL_RANGE[1])
    ez_c = ez_r[np.ix_(dmask, rmask)]
    depth_c = depth_r[dmask]
    radial_c = radial_r[rmask]

    ez_s = gaussian_filter(ez_c, sigma=GAUSS_SIGMA_PX)

    sub_rmask = (radial_c >= DIP_EST_RADIAL_RANGE_FINAL[0]) & (radial_c <= DIP_EST_RADIAL_RANGE_FINAL[1])
    sub = np.abs(ez_s[:, sub_rmask])
    valid_cols = sub.max(axis=0) > 1e-12
    if valid_cols.sum() < 5:
        print(f'  run {run}: too few valid columns for a dip fit ({int(valid_cols.sum())}), skipping')
        return None
    depth_peak_idx = np.argmax(sub[:, valid_cols], axis=0)
    depth_peak = depth_c[depth_peak_idx]
    radial_sub = radial_c[sub_rmask][valid_cols]
    m0_fit, b0_fit = np.polyfit(radial_sub, depth_peak, 1)

    dip_override_val = dip_override if dip_override is not None else DIP_OVERRIDE_BY_RUN.get(run)
    if dip_override_val is not None:
        m0_r = dip_override_val
        b0_r = float(np.median(depth_peak - m0_r * radial_sub))
    else:
        m0_r, b0_r = m0_fit, b0_fit

    hw_deg = fan_halfwidth_deg if fan_halfwidth_deg is not None else FAN_HW_BY_RUN.get(run, FK_FINAL_HW_DEG)

    Nz_r, Nx_r = ez_c.shape
    dz_r = depth_c[1] - depth_c[0]
    dx_r = radial_c[1] - radial_c[0]
    kz_r = np.fft.fftfreq(Nz_r, d=dz_r)
    kx_r = np.fft.fftfreq(Nx_r, d=dx_r)
    KZ_r, KX_r = np.meshgrid(kz_r, kx_r, indexing='ij')
    theta_k_r = np.arctan2(KZ_r, KX_r)
    theta_t_r = np.arctan2(1.0, -m0_r)
    dtheta_r = np.angle(np.exp(1j * 2.0 * (theta_k_r - theta_t_r))) / 2.0
    mask_r = fan_taper_mask(dtheta_r, hw_deg, FK_TAPER_DEG)
    win2d_r = edge_taper_2d(ez_c.shape, alpha=0.2)
    ez_fan = np.real(np.fft.ifft2(np.fft.fft2(ez_c * win2d_r) * mask_r))

    energy_env_r = gaussian_filter(np.abs(ez_c), sigma=GATE_SIGMA_PX_FINAL)
    gate_r = np.clip(energy_env_r / np.percentile(energy_env_r, 80), 0, 1)
    ez_gated = ez_fan * gate_r

    radial_taper_r = np.ones(Nx_r)
    radial_taper_r[radial_c < radial_taper_start_m] = 0.0
    ramp_r = (radial_c >= radial_taper_start_m) & (radial_c < radial_taper_end_m)
    radial_taper_r[ramp_r] = 0.5 * (1 - np.cos(np.pi * (radial_c[ramp_r] - radial_taper_start_m) /
                                                (radial_taper_end_m - radial_taper_start_m)))
    ez_fin = ez_gated * radial_taper_r[np.newaxis, :]

    out_path = bh_out / f'borehole_prof_{run}_dx02_final_processed.npz'
    np.savez(out_path, image=ez_fin, depth_axis=depth_c, radial_axis=radial_c,
             snap_idx=snap_idx, snap_t_ns=snap_t, dip_m0=m0_r, dip_b0=b0_r, dip_m0_auto=m0_fit,
             fan_halfwidth_deg=hw_deg, fan_taper_deg=FK_TAPER_DEG,
             gate_energy_sigma_px=GATE_SIGMA_PX_FINAL,
             radial_taper_start_m=radial_taper_start_m, radial_taper_end_m=radial_taper_end_m)

    return dict(raw=ez_c, final=ez_fin, depth_axis=depth_c, radial_axis=radial_c,
                m0=m0_r, b0=b0_r, m0_auto=m0_fit, fan_hw_deg=hw_deg, path=out_path)

final_results = {}
for run in PROFILES_FINAL:
    res = compute_final_processed(run)
    if res is None:
        print(f'run {run:>3}: no gprMax snapshot data found for borehole_prof_{run}_dx02 -- run it through gprMax first.')
        continue
    final_results[run] = res
    override_note = ' [dip overridden]' if run in DIP_OVERRIDE_BY_RUN else ''
    print(f'run {run:>3}: saved {res["path"].name}  (dip m0={res["m0"]:.2f}{override_note}, '
          f'fan_hw={res["fan_hw_deg"]} deg, radial taper {RADIAL_TAPER_START_M}-{RADIAL_TAPER_END_M} m)')

print(f'\n{len(final_results)}/{len(PROFILES_FINAL)} profiles processed.')

In [ ]:
# ── Visualise the final processed result for every available profile ────────────────
# Third row: consecutive time-lapse differences (prof_n - prof_{n-1} in PROFILES_FINAL
# order), showing what changed between each pair of profiles shown -- consistent with
# the "Strategy 1: Consecutive Profiles" differencing already used for Kirchhoff/Gazdag
# and back-propagation elsewhere in this notebook. The first column has no earlier
# profile in this set, so it's left as a placeholder rather than diffed against nothing.
n_cols = len(PROFILES_FINAL)
fig, axes = plt.subplots(3, n_cols, figsize=(4.2 * n_cols, 16))
fig.suptitle('Raw vs. final processed vs. consecutive time-lapse difference '
             '(20deg fan + gate + near-borehole taper)', y=1.005)

for col, run in enumerate(PROFILES_FINAL):
    ax_raw, ax_fin, ax_diff = axes[0, col], axes[1, col], axes[2, col]
    if run not in final_results:
        for ax in (ax_raw, ax_fin, ax_diff):
            ax.text(0.5, 0.5, f'prof_{run}\nnot found', ha='center', va='center', transform=ax.transAxes)
            ax.set_xticks([]); ax.set_yticks([])
        continue
    res = final_results[run]
    ext = [res['radial_axis'][0], res['radial_axis'][-1], res['depth_axis'][0], res['depth_axis'][-1]]

    clim_raw = np.percentile(np.abs(res['raw']), 100)
    ax_raw.imshow(res['raw'], aspect='auto', cmap='seismic', extent=ext, origin='lower',
                  vmin=-clim_raw, vmax=clim_raw)
    ax_raw.set_title(f'prof_{run}: raw', fontsize=9)

    clim_fin = np.percentile(np.abs(res['final']), 100)
    ax_fin.imshow(res['final'], aspect='auto', cmap='seismic', extent=ext, origin='lower',
                  vmin=-clim_fin, vmax=clim_fin)
    ax_fin.set_title(f'prof_{run}: final processed', fontsize=9)

    if col == 0:
        ax_diff.text(0.5, 0.5, 'no earlier profile\nin this set', ha='center', va='center',
                     transform=ax_diff.transAxes, fontsize=8)
        ax_diff.set_xticks([]); ax_diff.set_yticks([])
    else:
        prev_run = PROFILES_FINAL[col - 1]
        if prev_run not in final_results:
            ax_diff.text(0.5, 0.5, f'prof_{prev_run}\nnot found', ha='center', va='center',
                         transform=ax_diff.transAxes, fontsize=8)
            ax_diff.set_xticks([]); ax_diff.set_yticks([])
        else:
            prev_res = final_results[prev_run]
            if prev_res['final'].shape != res['final'].shape:
                ax_diff.text(0.5, 0.5, 'shape mismatch', ha='center', va='center',
                             transform=ax_diff.transAxes, fontsize=8)
                ax_diff.set_xticks([]); ax_diff.set_yticks([])
            else:
                diff = res['final'] - prev_res['final']
                clim_diff = np.percentile(np.abs(diff), 100)
                ax_diff.imshow(diff, aspect='auto', cmap='seismic', extent=ext, origin='lower',
                               vmin=-clim_diff, vmax=clim_diff)
                ax_diff.set_title(f'prof_{run} - prof_{prev_run}', fontsize=9)

    for ax in (ax_raw, ax_fin, ax_diff):
        ax.invert_yaxis()
        ax.set_ylim(BH_DEPTH_RANGE[1], BH_DEPTH_RANGE[0])
        ax.set_xlim(0, 12)
    ax_diff.set_xlabel('Radial [m]')

axes[0, 0].set_ylabel('Depth [m]')
axes[1, 0].set_ylabel('Depth [m]')
axes[2, 0].set_ylabel('Depth [m]')
plt.tight_layout()
out_path = bh_out / 'borehole_dx02_final_processed_all_profiles.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show(); plt.close(fig)
print(f'Saved {out_path}')

In [ ]:
# Frequency spectrum of excitation vs. gprMax dispersion limit
from helper_functions.migration import dispersion_limited_cutoff

exc_path = OUT_DIR / 'backprop' / 'prof_2' / 'excitation.txt'
exc_data = np.loadtxt(exc_path, skiprows=1)
time_s   = exc_data[:, 0]
traces   = exc_data[:, 1:]
dt_s_exc = float(time_s[1] - time_s[0])

# Spectrum of the middle trace (representative)
mid      = traces.shape[1] // 2
spectrum = np.abs(np.fft.rfft(traces[:, mid]))
freq_hz  = np.fft.rfftfreq(len(time_s), dt_s_exc)
freq_ghz = freq_hz * 1e-9

# gprMax dispersion limits for the half-velocity ice material
eps_r_val      = (0.299792458 / v) ** 2
eps_r_half_val = 4.0 * eps_r_val
f_limit_hz  = dispersion_limited_cutoff(eps_r_half_val, bp_dx, safety_factor=1.0)
f_safe_hz   = dispersion_limited_cutoff(eps_r_half_val, bp_dx, safety_factor=0.7)
f_limit_ghz = f_limit_hz * 1e-9
f_safe_ghz  = f_safe_hz  * 1e-9

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(freq_ghz, spectrum / (spectrum.max() + 1e-30), label='middle trace spectrum')
ax.axvline(f_limit_ghz, color='r',      linestyle='--', label=f'hard limit  {f_limit_ghz:.3f} GHz')
ax.axvline(f_safe_ghz,  color='orange', linestyle='--', label=f'safe cutoff {f_safe_ghz:.3f} GHz')
ax.set_xlabel('Frequency (GHz)')
ax.set_ylabel('Normalised amplitude')
ax.set_title(f'Excitation spectrum  (dx={bp_dx:.3f} m, eps_r_half={eps_r_half_val:.1f})')
ax.legend()
ax.set_xlim(0, 5 * f_limit_ghz)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(OUT_DIR / 'excitation_spectrum.png', dpi=150)
plt.show()

print(f'gprMax hard limit : {f_limit_ghz:.4f} GHz')
print(f'Safe cutoff       : {f_safe_ghz:.4f} GHz')
print(f'dx={bp_dx:.4f} m   eps_r_half={eps_r_half_val:.2f}   v_half={v/2:.4f} m/ns')

# Back-propagation Pre-processing and Quality Improvement

## Why noise appears at small radial distances

In time-reversal back-propagation every gprMax source is injected at `y ≈ bp_src_y ≈ 0 m`
(the borehole wall). At the focus time the **coherent** part of the field constructively
interferes at the reflector radius (≈ 5–7 m). Near `y = 0` the destructive interference
among all sources is **never complete**, leaving a residual "injection halo" regardless
of how much the input data is pre-processed — this is a structural property of single-sided
time-reversal from a borehole geometry.

The halo is suppressed in display by zeroing the first `BP_NEAR_MASK_M = 2.0 m` of the
radial axis in the snapshot, consecutive-difference, and reference-difference cells.
Increase `BP_NEAR_MASK_M` if the halo extends further.

**Note — trace-by-trace RMS normalisation was tested and reverted.**
Normalising each receiver trace by its RMS value amplifies low-SNR traces (e.g. receivers
far from the fluid front that contain mostly noise). These amplified noisy traces back-propagate
coherently toward `y = 0` (the borehole axis), making the injection halo *worse*, not better.

## Current pre-processing chain

| Step | Operation | Parameter |
|---:|---|---|
| 1 | Band-pass filter | 0.02–0.2 GHz |
| 2 | DC / direct-wave removal | `remove_mean` samples 299–517 |
| 3 | Trace alignment | `align_traces`, upsample ×5 |
| 4 | SVD rank-1 removal | ranks 0–1 |
| 5 | Difference from reference | `d_svd − ref_svd` |
| 6 | Linear gain | compensate geometric spreading |
| 7 | **Spatial Tukey taper** ✓ | `alpha=0.30` — 15 % of receivers tapered each side |
| 8 | **Temporal Tukey taper** ✓ | `alpha=0.10` — 5 % of time samples tapered each end |
| 9 | **f-kz dip filter** ✓ | zero bins where `|kz| > f / v_mig` |
| 10 | Low-pass filter on excitation | below gprMax dispersion limit |
| 11 | Edge source zeroing | `bp_edge_exclude = 5` |

**PML = 15 cells** (increased from 10); `bp_domain_y` and `bp_src_y` update automatically.

## Remaining avenues

| Technique | Expected benefit |
|---|---|
| Increase `BP_NEAR_MASK_M` beyond 2 m | Widen display exclusion if halo extends further |
| `sign_bit=True` in `write_backprop_files` | Uniform ±1 amplitude injection — test vs peak-normalised |
| Increase `bp_edge_exclude` to 10–15 | Zero more boundary sources at grazing PML angles |
| Spiking deconvolution per trace | Compress wavelet before injection (analogous to delta-wavelet Kirchhoff) |
| `bp_pml = 20` | Further reduce boundary reflections at cost of larger domain |

# Sign-Flip / Water-Level Validation: Isolated Single Reflector

Before trusting `apply_3d_to_2d_correction()` inside the full field-data pipeline, validate its two uncertain design choices in isolation, the same way the hertzian-dipole polarity flip in `write_backprop_files()` was validated (single synthetic reflector + correlation test, not a full-pipeline visual comparison):

1. **`phase_sign`** -- the Hankel-asymptotic derivation gives `e^{+i*pi/4}`; the code currently defaults to `e^{-i*pi/4}`, chosen by comparing full back-propagation output against Kirchhoff/Gazdag images (a confounded test -- SVD, tapers, the dip filter, and the *separate* dipole-current polarity flip all sit in between). Tested here directly against an analytic single-reflector Kirchhoff/Gazdag "truth".
2. **`water_level`** -- the filter only hard-zeros the exact DC bin; the next few low-frequency bins still get very large `1/sqrt(omega)` gain, which could be amplifying noise rather than signal. Tested here as an alternative regularisation.

**Method:** forward-model a single 3D point reflector (1/r spreading, exploding-reflector zero-offset convention) at a known `(x, z)`, migrate it with Kirchhoff and Gazdag to get a ground-truth focus location (these operate on travel-time/amplitude directly and don't depend on 2D-vs-3D Green's-function physics, so they're a fair reference), then apply `apply_3d_to_2d_correction()` with every combination of `phase_sign` and `water_level` and write gprMax back-propagation input files for each variant. After running gprMax on each `.in` file externally (same workflow as the field-data back-propagation runs above), the comparison cell loads the focus snapshots and scores each variant's position error against the Kirchhoff/Gazdag truth.

In [ ]:
# ── Isolated single-reflector validation: synthetic forward model + Kirchhoff/Gazdag truth ──
# Self-contained (does not depend on the field-data cells above, only on np/plt/OUT_DIR
# from the very first cell). Forward-models a single 3D point reflector in a zero-offset
# borehole geometry (1/r spreading, exploding-reflector convention), migrates it with
# Kirchhoff and Gazdag to get a ground-truth focus location -- these use travel-time /
# amplitude directly and don't depend on 2D-vs-3D Green's-function physics, so they're a
# fair reference -- then writes gprMax back-propagation files for every (phase_sign,
# water_level) combination of apply_3d_to_2d_correction() so each can be run through
# gprMax and compared in the cell below.
import importlib
from scipy.signal import fftconvolve
import helper_functions.KirchhoffPylopsZeroOffset as KirchhoffPylopsZeroOffset
importlib.reload(KirchhoffPylopsZeroOffset)
from pylops.utils.wavelets import ricker
from helper_functions.migration import (
    gazdag_migration, apply_3d_to_2d_correction,
    write_backprop_files, lowpass_filter_excitation, dispersion_limited_cutoff,
)

# --- Geometry: single point reflector, zero-offset borehole receivers ---
v_syn      = 0.10                          # [m/ns] background velocity
sf_syn     = 1.0                           # [GHz] sampling frequency
dt_syn     = 1.0 / sf_syn                  # [ns]
n_t_syn    = 512                           # samples per trace
n_rec_syn  = 120                           # zero-offset receivers along the borehole
dL_syn     = 0.05                          # [m] receiver spacing

z_bh_syn   = np.arange(n_rec_syn) * dL_syn         # receiver depths [m]
x_r_true   = 5.5                                    # [m] reflector radial distance (truth)
z_r_true   = z_bh_syn[n_rec_syn // 2]               # [m] reflector depth (truth)

t_syn      = np.arange(n_t_syn) * dt_syn            # [ns] -- sample 0 = true t=0 (no calibration offset)

f0_syn     = 0.1                                     # [GHz] Ricker centre frequency
_period    = 1.0 / f0_syn
_n_wav     = int(np.ceil(6 * _period / dt_syn))
if _n_wav % 2 == 0:
    _n_wav += 1
wav_syn, _, wcenter_syn = ricker(t_syn[:_n_wav], f0=f0_syn)

# Forward-model the 3D point-reflector response: 1/r spreading (exploding-reflector
# convention, matching the sqrt(r) correction's assumption), convolved with the wavelet.
b_scan_3d = np.zeros((n_rec_syn, n_t_syn))
for i, z_i in enumerate(z_bh_syn):
    r_i = np.hypot(x_r_true, z_r_true - z_i)         # 3D straight-line range [m]
    t_i = 2.0 * r_i / v_syn                           # two-way travel time [ns]
    idx = int(round(t_i / dt_syn))
    if 0 <= idx < n_t_syn:
        b_scan_3d[i, idx] = 1.0 / max(r_i, 1e-3)
b_scan_3d = np.array([fftconvolve(tr, wav_syn, mode='same') for tr in b_scan_3d])

# Add broadband (white) noise: without it, water_level has nothing to suppress -- it
# only regularises the 1/sqrt(omega) pole, which matters for noise energy near DC, not
# for a clean signal. White noise has flat power across all frequencies (including the
# low end where the unregularised filter blows up), so it's the right stress-test.
# Seeded for reproducibility; ~15% of the clean peak amplitude, a modest but real SNR hit.
NOISE_STD_SYN = 0.15 * np.max(np.abs(b_scan_3d))
_rng = np.random.default_rng(0)
b_scan_3d = b_scan_3d + _rng.normal(0.0, NOISE_STD_SYN, b_scan_3d.shape)

fig_s, ax_s = plt.subplots(figsize=(6, 5))
lim_s = np.max(np.abs(b_scan_3d))
ax_s.imshow(b_scan_3d.T, aspect='auto', cmap='seismic', vmin=-lim_s, vmax=lim_s,
            extent=[z_bh_syn[0], z_bh_syn[-1], t_syn[-1], t_syn[0]])
ax_s.set_xlabel('Receiver depth (m)'); ax_s.set_ylabel('Time (ns)')
ax_s.set_title('Synthetic 3D point-reflector B-scan (+ noise)')
plt.tight_layout()
(OUT_DIR / 'sign_waterlevel_validation').mkdir(exist_ok=True)
fig_s.savefig(OUT_DIR / 'sign_waterlevel_validation' / 'synthetic_bscan.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close(fig_s)

# --- Ground-truth image: Kirchhoff + Gazdag migration of the synthetic B-scan ---
x_img_syn = np.linspace(0, v_syn * t_syn[-1] / 2, 200)
z_img_syn = np.linspace(z_bh_syn.max(), z_bh_syn.min(), n_rec_syn)
recs_syn  = np.vstack((np.zeros(n_rec_syn), z_bh_syn))

K_syn = KirchhoffPylopsZeroOffset.Kirchhoff(
    z=z_img_syn, x=x_img_syn, t=t_syn,
    srcs=recs_syn, recs=recs_syn,
    vel=v_syn, wav=wav_syn, wavcenter=wcenter_syn,
    mode='analytic', dynamic=False,
)
m_kir_syn = (K_syn.H @ b_scan_3d.flatten()).reshape(len(x_img_syn), len(z_img_syn)).T
m_gaz_syn = gazdag_migration(b_scan_3d.T, z_bh_syn, t_syn, x_img_syn, v_syn)  # (n_radial, n_depth)

def _peak_xz(img, x_ax, z_ax):
    """Location and (signed) value of the peak absolute amplitude in a (z, x) image."""
    iz, ix = np.unravel_index(np.argmax(np.abs(img)), img.shape)
    return x_ax[ix], z_ax[iz], img[iz, ix]

x_kir, z_kir, a_kir = _peak_xz(m_kir_syn, x_img_syn, z_img_syn)
x_gaz, z_gaz, a_gaz = _peak_xz(m_gaz_syn.T, x_img_syn, z_bh_syn)
print(f'Truth reflector:  x={x_r_true:.3f} m, z={z_r_true:.3f} m')
print(f'Kirchhoff peak:   x={x_kir:.3f} m, z={z_kir:.3f} m  (err {100*np.hypot(x_kir-x_r_true, z_kir-z_r_true):.1f} cm, sign={np.sign(a_kir):+.0f})')
print(f'Gazdag peak:      x={x_gaz:.3f} m, z={z_gaz:.3f} m  (err {100*np.hypot(x_gaz-x_r_true, z_gaz-z_r_true):.1f} cm, sign={np.sign(a_gaz):+.0f})')

fig_m, (ax_k, ax_gm) = plt.subplots(1, 2, figsize=(11, 5))
for ax, img, x_ax, z_ax, title in (
    (ax_k,  m_kir_syn,   x_img_syn, z_img_syn, 'Kirchhoff'),
    (ax_gm, m_gaz_syn.T, x_img_syn, z_bh_syn,  'Gazdag'),
):
    lim = max(1e-30, np.max(np.abs(img)))
    ax.imshow(img, aspect='equal', cmap='seismic', vmin=-lim, vmax=lim,
              extent=[x_ax[0], x_ax[-1], z_ax[-1], z_ax[0]])
    ax.plot(x_r_true, z_r_true, 'g+', ms=14, mew=2, label='truth')
    ax.invert_yaxis(); ax.legend()
    ax.set_xlabel('Radial distance (m)'); ax.set_ylabel('Depth (m)'); ax.set_title(title)
plt.tight_layout()
plt.savefig(OUT_DIR / 'sign_waterlevel_validation_truth.png', dpi=150)
plt.show()

# --- Write gprMax back-propagation files for every (phase_sign, water_level) variant ---
SIGN_VARIANTS = {'signpos': +1.0, 'signneg': -1.0}   # +1 = formula as derived; -1 = current default
WL_VARIANTS   = {'wl0': 0.0, 'wl02': 0.02}            # 0 = old hard-DC-zero-only; 0.02 = regularised

SYN_ROOT = OUT_DIR / 'sign_waterlevel_validation'
SYN_ROOT.mkdir(exist_ok=True)

eps_r_syn       = (0.299792458 / v_syn) ** 2
eps_r_half_syn  = 4.0 * eps_r_syn
bp_dx_syn       = v_syn / (f0_syn * 20)
bp_pml_syn      = 15
bp_src_y_syn    = (bp_pml_syn + 1) * bp_dx_syn
bp_domain_y_syn = float(x_img_syn[-1]) + 2 * bp_pml_syn * bp_dx_syn
f_cut_syn       = dispersion_limited_cutoff(eps_r_half_syn, bp_dx_syn)

# write_backprop_files silently shifts sources (and the domain) by x_offset_syn
# whenever min(x_midpoints) sits closer to 0 than the PML thickness -- see its
# docstring -- because #pml_cells applies a PML band on *both* x edges, and
# z_bh_syn starts at exactly 0 m here (unlike real borehole depths, which are
# tens of metres and never trip this). Recompute the same value independently
# (rather than relying on the function to return it, which would break the
# other notebooks that unpack write_backprop_files() as a 4-tuple) so the
# comparison cell below can map gprMax x back to the true z_bh_syn frame.
x_offset_syn = max(0.0, bp_pml_syn * bp_dx_syn - float(np.min(z_bh_syn)))

# t0_ns here only sets write_backprop_files' snapshot-timing heuristic (t_focus = T -
# t0); size it from the known max two-way travel time so the focus snapshot window
# actually brackets the synthetic reflector's arrival (time_zero_idx=0 above, for the
# correction itself, stays the true physical zero used for the Kirchhoff/Gazdag truth).
SNAP_WIN_SYN    = 2.0
T_ns_syn        = n_t_syn * dt_syn
t_max_travel_ns = 2.0 * np.max(np.hypot(x_r_true, z_r_true - z_bh_syn)) / v_syn
t0_ns_syn       = max(0.0, T_ns_syn - t_max_travel_ns - SNAP_WIN_SYN)

variant_paths = {}
b_2d_variants = {}   # kept for the cheap correlation/energy diagnostic below (no gprMax needed)
for sign_name, sign_val in SIGN_VARIANTS.items():
    for wl_name, wl_val in WL_VARIANTS.items():
        slug = f'{sign_name}_{wl_name}'
        b_2d = apply_3d_to_2d_correction(
            b_scan_3d, dt=dt_syn * 1e-9, velocity=v_syn * 1e9,
            time_zero_idx=0, phase_sign=sign_val, water_level=wl_val,
        )
        b_2d_variants[slug] = b_2d
        in_path, n_src, n_snaps, t_focus_ns = write_backprop_files(
            SYN_ROOT, label=f'single-reflector {slug}', slug=slug,
            tapered_ntr_nt=b_2d, dt_ns=dt_syn, x_midpoints=z_bh_syn,
            t0_ns=t0_ns_syn, eps_r=eps_r_syn, v_ice=v_syn,
            dx=bp_dx_syn, domain_y=bp_domain_y_syn, src_y=bp_src_y_syn,
            pml_cells=bp_pml_syn, snap_win=SNAP_WIN_SYN,
        )
        exc_file = SYN_ROOT / 'backprop' / slug / 'excitation.txt'
        lowpass_filter_excitation(exc_file, f_cut_syn)
        variant_paths[slug] = in_path
        print(f'  wrote {slug}: {in_path.name}  (n_src={n_src}, n_snaps={n_snaps}, t_focus={t_focus_ns:.1f} ns)')

print(f'\nRun each .in file above through gprMax (outside this notebook, same as the '
      f'field-data back-propagation runs), then use the comparison cell below.\n'
      f'gprMax x=0 corresponds to true borehole depth {-x_offset_syn:.3f} m '
      f'(sources shifted by +{x_offset_syn:.3f} m to clear the x-min PML band).\n'
      f'Truth reflector: x={x_r_true:.2f} m, z={z_r_true:.2f} m  |  '
      f'Kirchhoff truth: x={x_kir:.2f} m, z={z_kir:.2f} m (sign={np.sign(a_kir):+.0f})  |  '
      f'Gazdag truth: x={x_gaz:.2f} m, z={z_gaz:.2f} m (sign={np.sign(a_gaz):+.0f})')

# --- Cheap pre-gprMax diagnostic: how different are the 4 corrected traces really? ---
# A phase_sign flip is a pure allpass rotation (same |H(omega)|, only the phase changes),
# so it preserves total energy and does NOT change where a time-reversal focus forms --
# only the pulse *shape* at focus. Expect near-zero correlation (a quadrature-like
# relationship) but near-identical RMS between signpos/signneg. water_level only
# regularises the near-DC gain, so with real signal+noise present it should show up as
# a modest but real difference between wl0/wl02 (it had ~0.999997 correlation on the
# clean signal alone, so most of any wl0-vs-wl02 gap now comes from noise handling).
def _corr(a, b):
    return float(np.corrcoef(a.ravel(), b.ravel())[0, 1])

def _rms(a):
    return float(np.sqrt(np.mean(a ** 2)))

print('\nPre-gprMax diagnostic on the corrected traces (no simulation needed):')
for wl_name in WL_VARIANTS:
    a, b = b_2d_variants[f'signpos_{wl_name}'], b_2d_variants[f'signneg_{wl_name}']
    print(f'  signpos vs signneg ({wl_name}):  corr={_corr(a, b):+.4f}  '
          f'rms_signpos={_rms(a):.4g}  rms_signneg={_rms(b):.4g}')
for sign_name in SIGN_VARIANTS:
    a, b = b_2d_variants[f'{sign_name}_wl0'], b_2d_variants[f'{sign_name}_wl02']
    print(f'  wl0 vs wl02 ({sign_name}):        corr={_corr(a, b):+.6f}  '
          f'rms_wl0={_rms(a):.4g}  rms_wl02={_rms(b):.4g}')

In [ ]:
# ── Compare gprMax back-propagation variants against the Kirchhoff/Gazdag truth ──
# Run this after the .in files from the cell above have each been run through gprMax
# (produces backprop_<slug>_snaps/bp_snap*.vti under SYN_ROOT/backprop/<slug>/).
#
# Scans every available snapshot per variant instead of trusting a single guessed
# focus index (see the "Save focus-frame snapshots" cell's empirically-tuned
# FOCUS_IDX_OFFSET -- a fixed offset isn't safe to reuse with a different domain/
# timing setup). The near-source region is masked out first (BP_NEAR_MASK_M
# convention used elsewhere in this notebook) so the source-injection halo can't be
# mistaken for the reflector focus.
#
# Two things a raw position/|amplitude| comparison cannot see (confirmed directly on
# the corrected traces in the cell above, no gprMax needed): a phase_sign flip is a
# pure allpass rotation -- same energy, same focus location, only a reshaped pulse --
# so it shows up as a *polarity* difference at the focus pixel, not a position or
# strength difference. water_level only regularises noise near DC, so it shows up (if
# at all) as reduced clutter *away* from the focus, not a sharper peak. This cell
# reports both explicitly instead of only position error.
import pyvista

BP_NEAR_MASK_M_SYN   = 2.0   # zero radial < this [m] before scoring/plotting (injection halo)
CLUTTER_EXCLUDE_M_SYN = 0.5  # radius around the detected focus excluded from the clutter-RMS metric

def _best_focus_frame(slug, near_mask_m=BP_NEAR_MASK_M_SYN):
    snap_dir = SYN_ROOT / 'backprop' / slug / f'backprop_{slug}_snaps'
    snap_files = sorted(snap_dir.glob('bp_snap*.vti'),
                         key=lambda p: int(p.stem.replace('bp_snap', '')))
    if not snap_files:
        return None

    t_focus_ns = T_ns_syn - t0_ns_syn
    t_start_ns = max(dt_syn, t_focus_ns - SNAP_WIN_SYN)
    snap_step  = max(1, int((T_ns_syn - t_start_ns) / (max(1, 30 - 1) * dt_syn)))
    snap_times = t_start_ns + np.arange(len(snap_files)) * snap_step * dt_syn

    best = None
    for k, f in enumerate(snap_files):
        mesh  = pyvista.read(str(f))
        nx_c  = mesh.dimensions[0] - 1
        ny_c  = mesh.dimensions[1] - 1
        dx_m  = float(mesh.spacing[0])
        ez    = np.array(mesh['E-field'])[:, 2].reshape(ny_c, nx_c).T   # (depth, radial)
        mask_px = max(1, round(near_mask_m / dx_m))
        ez_masked = ez.copy()
        ez_masked[:, :mask_px] = 0.0
        peak = np.max(np.abs(ez_masked))
        if best is None or peak > best[0]:
            best = (peak, k, ez_masked, nx_c, ny_c, dx_m)

    peak, k, ez_masked, nx_c, ny_c, dx_m = best
    depth_axis  = np.arange(nx_c) * dx_m - x_offset_syn   # undo the PML source shift
    radial_axis = np.arange(ny_c) * dx_m
    return ez_masked, depth_axis, radial_axis, k, snap_times[k]

def _clutter_rms(ez, depth_axis, radial_axis, focus_xz, exclude_radius_m=CLUTTER_EXCLUDE_M_SYN):
    """RMS amplitude away from the detected focus -- what water_level is meant to reduce."""
    x_f, z_f = focus_xz
    zz, xx = np.meshgrid(depth_axis, radial_axis, indexing='ij')
    far = np.hypot(xx - x_f, zz - z_f) > exclude_radius_m
    return float(np.sqrt(np.mean(ez[far] ** 2))) if far.any() else float('nan')

sign_kir = np.sign(a_kir)
results = {}
for slug in variant_paths:
    frame = _best_focus_frame(slug)
    if frame is None:
        print(f'{slug}: no snapshots found yet -- run gprMax on {variant_paths[slug]} first.')
        continue
    ez, depth_axis, radial_axis, snap_idx, snap_t = frame
    iz, ir = np.unravel_index(np.argmax(np.abs(ez)), ez.shape)
    x_bp, z_bp, a_bp = radial_axis[ir], depth_axis[iz], ez[iz, ir]
    err_cm  = 100 * np.hypot(x_bp - x_r_true, z_bp - z_r_true)
    clutter = _clutter_rms(ez, depth_axis, radial_axis, (x_bp, z_bp))
    results[slug] = dict(ez=ez, depth_axis=depth_axis, radial_axis=radial_axis,
                          x=x_bp, z=z_bp, amp=a_bp, err_cm=err_cm,
                          snap_idx=snap_idx, snap_t=snap_t, clutter=clutter)
    polarity_note = 'matches' if np.sign(a_bp) == sign_kir else 'OPPOSITE to'
    print(f'{slug:16s}  x={x_bp:.3f} m  z={z_bp:.3f} m  err={err_cm:5.1f} cm  '
          f'Ez={a_bp:+.3g} ({polarity_note} Kirchhoff sign={sign_kir:+.0f})  '
          f'clutter_rms={clutter:.3g}  best snap #{snap_idx} @ t={snap_t:.2f} ns')

if results:
    fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 5), squeeze=False)
    for ax, (slug, r) in zip(axes[0], results.items()):
        clim = np.percentile(np.abs(r['ez']), 99.5)
        ax.imshow(r['ez'], aspect='auto', cmap='seismic', origin='lower',
                   extent=[r['radial_axis'][0], r['radial_axis'][-1],
                           r['depth_axis'][0], r['depth_axis'][-1]],
                   vmin=-clim, vmax=clim)
        ax.plot(x_r_true, z_r_true, 'g+', ms=14, mew=2, label='truth')
        ax.plot(x_kir, z_kir, 'y+', ms=10, mew=1.5, label='Kirchhoff truth')
        ax.set_title(f'{slug}\nerr={r["err_cm"]:.1f} cm  clutter={r["clutter"]:.2g}')
        ax.set_xlabel('Radial (m)'); ax.set_ylabel('Depth (m)')
        ax.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(SYN_ROOT / 'sign_waterlevel_comparison.png', dpi=150)
    plt.show()

    print(f'\nTruth (Kirchhoff/Gazdag): x={x_kir:.2f} m, z={z_kir:.2f} m, sign={sign_kir:+.0f}')
    best_pos = min(results, key=lambda s: results[s]['err_cm'])
    print(f'Best variant by position error:  {best_pos} ({results[best_pos]["err_cm"]:.1f} cm)')
    polarity_matches = [s for s in results if np.sign(results[s]['amp']) == sign_kir]
    print(f'Variant(s) matching Kirchhoff polarity: {polarity_matches or "none"}')
    for wl_name in ('wl0', 'wl02'):
        pair = [s for s in results if s.endswith(wl_name)]
        if pair:
            avg_clutter = np.mean([results[s]['clutter'] for s in pair])
            print(f'Mean clutter RMS across {wl_name} variants: {avg_clutter:.3g}')
else:
    print('No completed gprMax runs found yet.')

In [ ]:
# Save focus-frame snapshots for completed peak-normalised back-propagation runs
# Shared infra for the [Superseded] Homogeneous-Domain Back-Propagation Results below
# AND for the Thesis Figure Compilation cells further down (both depend on `completed`/
# `t_focus_ns`/`t_start_ns`/`snap_step`/`dt_ns_bp` set here) -- do not remove or move.
import pyvista

# Timing parameters — must match write_backprop_files defaults
N_SNAP         = 30
SNAP_WIN       = 1.0    # ns
BP_NEAR_MASK_M = 1.0    # hide radial dist < this [m] (source injection artifact)

n_t_bp    = rad_cut
dt_ns_bp  = 1.0 / sf
T_ns_bp   = n_t_bp * dt_ns_bp
t0_ns_bp  = 299.0 / sf
dt_s_bp   = dt_ns_bp * 1e-9

t_focus_ns = T_ns_bp - t0_ns_bp
t_start_ns = max(dt_ns_bp, t_focus_ns - SNAP_WIN)
t_start_s  = t_start_ns * 1e-9
snap_step  = max(1, int((T_ns_bp * 1e-9 - t_start_s) / (max(1, N_SNAP - 1) * dt_s_bp)))

print(f't_focus = {t_focus_ns:.4f} ns  |  t_start = {t_start_ns:.4f} ns  |  snap_step = {snap_step}')

bp_root  = OUT_DIR / 'backprop'
snap_out = OUT_DIR / 'backprop_snapshots'
snap_out.mkdir(exist_ok=True)

# Collect completed peak-normalised homogeneous-domain runs (skip sign-bit and skip
# the borehole-geometry runs -- e.g. borehole_prof_1_dx02 -- which now live alongside
# these under the same backprop/ root but use a different domain geometry entirely
# (this cell's ylim(60,85)/mesh-reading logic assumes the old homogeneous layout and
# would silently produce wrong output if it tried to process those). Only directories
# matching the exact prof_<N> pattern are homogeneous-domain runs.
import re
_PROF_DIR_RE = re.compile(r"^prof_(\d+)$")

def _prof_dir_num(p):
    m = _PROF_DIR_RE.match(p.name)
    return int(m.group(1)) if (p.is_dir() and m) else None

completed = {}
for d in sorted((p for p in bp_root.iterdir() if _prof_dir_num(p) is not None), key=_prof_dir_num):
    snap_dir   = d / f'backprop_{d.name}_snaps'
    snap_files = sorted(snap_dir.glob('bp_snap*.vti'),
                        key=lambda p: int(p.stem.replace('bp_snap', ''))) if snap_dir.exists() else []
    if snap_files:
        completed[d.name] = snap_files
        print(f'  {d.name}: {len(snap_files)} snapshots found')

print(f'\nProcessing {len(completed)} completed run(s)...')

for slug, snap_files in completed.items():
    snap_times_ns = t_start_ns + np.arange(len(snap_files)) * snap_step * dt_ns_bp
    idx_focus     = int(np.argmin(np.abs(snap_times_ns - t_focus_ns))) + 28
    t_actual_ns   = snap_times_ns[idx_focus]

    mesh   = pyvista.read(str(snap_files[idx_focus]))
    nx_c   = mesh.dimensions[0] - 1   # depth cells  (gprMax x)
    ny_c   = mesh.dimensions[1] - 1   # radial cells (gprMax y)
    dx_m   = mesh.spacing[0]
    domain_x = nx_c * dx_m            # borehole depth extent [m]
    domain_y = ny_c * dx_m            # radial extent [m]

    # x-first cell ordering → reshape (ny_c, nx_c), then transpose to (nx_c, ny_c)
    # so rows = depth, cols = radial for imshow
    e_data = np.array(mesh['E-field'])          # (nx_c*ny_c, 3)
    ez     = e_data[:, 2].reshape(ny_c, nx_c).T   # (nx_c, ny_c): rows=depth, cols=radial
    mag    = np.linalg.norm(e_data, axis=1).reshape(ny_c, nx_c).T

    # extent: [radial_min, radial_max, depth_min, depth_max]
    extent = [0, domain_y, 0, domain_x]

    # Mask source injection zone (radial < BP_NEAR_MASK_M) for display only.
    # gprMax sources are at y ≈ 0; their near-field dominates there regardless
    # of preprocessing — the reflector focus always lies at larger radial dist.
    _mask_px = max(1, round(BP_NEAR_MASK_M / dx_m))
    ez_disp  = ez.copy();  ez_disp[:, :_mask_px]  = 0.0
    mag_disp = mag.copy(); mag_disp[:, :_mask_px] = 0.0
    clim_ez  = (np.percentile(np.abs(ez_disp[ez_disp != 0]), 100)
                if ez_disp.any() else 1.0)

    fig, axes = plt.subplots(1, 2, figsize=(10, 8))

    im0 = axes[0].imshow(mag_disp, aspect='auto', cmap='inferno',
                          extent=extent, origin='lower')
    plt.colorbar(im0, ax=axes[0], label='|E| [V/m]')
    axes[0].invert_yaxis()
    axes[0].set_title(f'{slug}  |  |E|  |  t={t_actual_ns:.3f} ns (snap {idx_focus})')
    axes[0].set_xlabel('Radial distance [m]')
    axes[0].set_ylabel('Depth [m]')
    axes[0].set_ylim(60, 85)
    axes[0].set_xlim(BP_NEAR_MASK_M, domain_y)
    axes[0].invert_yaxis()

    im1 = axes[1].imshow(ez_disp, aspect='auto', cmap='seismic',
                          extent=extent, origin='lower',
                          vmin=-clim_ez, vmax=clim_ez)
    plt.colorbar(im1, ax=axes[1], label='Ez [V/m]')
    axes[1].invert_yaxis()
    axes[1].set_title(f'{slug}  |  Ez  |  t={t_actual_ns:.3f} ns')
    axes[1].set_xlabel('Radial distance [m]')
    axes[1].set_ylabel('Depth [m]')
    axes[1].set_ylim(60, 85)
    axes[1].set_xlim(BP_NEAR_MASK_M, domain_y)
    axes[1].invert_yaxis()

    plt.tight_layout()
    out_path = snap_out / f'{slug}_focus.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'  Saved {out_path.name}')

print('Done.')

In [ ]:
# Compare migrated images (Kirchhoff, Gazdag) with back-propagation Ez focus frames
# Homogeneous-domain diagnostic (dx=0.05 m); no other cell depends on this one's
# outputs, safe to leave in place.
# for the key stage-boundary profiles. Use this to judge which BP snapshot index
# gives the best match to the migrated images, and to verify that the BP focus
# lands on the same reflectors as the migration.
#
# Part 1 — Side-by-side grid for all KEY_PROFILES at CMP_FOCUS_OFFSET.
# Part 2 — Offset scan: Kirchhoff + Gazdag + BP at each CMP_SCAN_OFFSETS for
#           one reference profile.
#
# CMP_KIRCHHOFF_METHOD / CMP_GAZDAG_METHOD : filename prefix of the saved .npy
#   (e.g. 'kirchhoff' → OUT_DIR/'migrated'/'kirchhoff_{run}.npy')
# CMP_FOCUS_OFFSET   : snapshot index offset used for the Part 1 grid
# CMP_DEPTH_RANGE    : depth window [m] shown in all panels
# CMP_RADIAL_RANGE   : radial window [m] shown in all panels
# CMP_SCAN_OFFSETS   : list of offsets to try in the Part 2 scan
# CMP_SCAN_PROFILE   : profile number used for the offset scan
# ─────────────────────────────────────────────────────────────────────────────────

KEY_PROFILES         = [1,3,4,8,9,20,21,38]
CMP_KIRCHHOFF_METHOD = 'kirchhoff_bp'   # change to 'kirchhoff_bp' if your files are named that way
CMP_GAZDAG_METHOD    = 'gazdag'
CMP_FOCUS_OFFSET     = 28    # snapshot offset for the grid comparison
CMP_DEPTH_RANGE      = (60, 85)    # [m]
CMP_RADIAL_RANGE     = (1.0, 14.0) # [m]
CMP_SCAN_OFFSETS     = [18, 20, 22, 24, 26, 28, 30]
CMP_SCAN_PROFILE     = 1

# ─────────────────────────────────────────────────────────────────────────────────

cmp_out = OUT_DIR / 'comparison'
cmp_out.mkdir(exist_ok=True)

def _cmp_load_mig(method, run):
    p = OUT_DIR / 'migrated' / f'{method}_{run}.npy'
    return np.load(p) if p.exists() else None

def _cmp_load_bp(slug, offset):
    snap_files = completed.get(slug)
    if not snap_files:
        return None, None
    snap_times_ns = t_start_ns + np.arange(len(snap_files)) * snap_step * dt_ns_bp
    idx = min(int(np.argmin(np.abs(snap_times_ns - t_focus_ns))) + offset,
              len(snap_files) - 1)
    mesh   = pyvista.read(str(snap_files[idx]))
    nx_c   = mesh.dimensions[0] - 1
    ny_c   = mesh.dimensions[1] - 1
    dx_m   = float(mesh.spacing[0])
    e_data = np.array(mesh['E-field'])
    ez     = e_data[:, 2].reshape(ny_c, nx_c).T   # (depth, radial)
    return ez, (nx_c * dx_m, ny_c * dx_m, snap_times_ns[idx])

# ── Part 1: grid comparison ───────────────────────────────────────────────────────
n_prof = len(KEY_PROFILES)
fig1, axes1 = plt.subplots(n_prof, 3, figsize=(13, 3.2 * n_prof),
                            sharex=True, sharey=True)
axes1[0, 0].set_title(CMP_KIRCHHOFF_METHOD.capitalize())
axes1[0, 1].set_title(CMP_GAZDAG_METHOD.capitalize())
axes1[0, 2].set_title(f'Back-propagation Ez  (offset={CMP_FOCUS_OFFSET})')
fig1.suptitle('Migrated vs back-propagation: key profiles', y=1.01)

for row, run in enumerate(KEY_PROFILES):
    slug = f'prof_{run}'

    # Kirchhoff
    img_k = _cmp_load_mig(CMP_KIRCHHOFF_METHOD, run)
    ax_k  = axes1[row, 0]
    if img_k is not None:
        n     = img_k.shape[0]
        ext_k = [x_img[0], x_img[-1], depth[n - 1], depth[0]]
        clim  = np.percentile(np.abs(img_k), 100)
        ax_k.imshow(img_k, aspect='auto', cmap='seismic',
                    extent=ext_k, origin='upper', vmin=-clim, vmax=clim)
        ax_k.invert_yaxis()
    else:
        ax_k.text(0.5, 0.5, 'not found', ha='center', va='center',
                  transform=ax_k.transAxes)
    ax_k.set_ylabel(f'prof_{run}')
    ax_k.set_ylim(*CMP_DEPTH_RANGE)
    ax_k.set_xlim(*CMP_RADIAL_RANGE)

    # Gazdag
    img_g = _cmp_load_mig(CMP_GAZDAG_METHOD, run)
    ax_g  = axes1[row, 1]
    if img_g is not None:
        n     = img_g.shape[0]
        ext_g = [x_img[0], x_img[-1], depth[n - 1], depth[0]]
        clim  = np.percentile(np.abs(img_g), 100)
        ax_g.imshow(img_g, aspect='auto', cmap='seismic',
                    extent=ext_g, origin='upper', vmin=-clim, vmax=clim)
        ax_g.invert_yaxis()
    else:
        ax_g.text(0.5, 0.5, 'not found', ha='center', va='center',
                  transform=ax_g.transAxes)
    ax_g.set_ylim(*CMP_DEPTH_RANGE)
    ax_g.set_xlim(*CMP_RADIAL_RANGE)

    # Back-propagation
    ez_bp, meta = _cmp_load_bp(slug, CMP_FOCUS_OFFSET)
    ax_bp = axes1[row, 2]
    if ez_bp is not None:
        dom_x, dom_y, t_ns = meta
        clim  = np.percentile(np.abs(ez_bp), 100)
        ax_bp.imshow(ez_bp, aspect='auto', cmap='seismic',
                     extent=[0, dom_y, 0, dom_x], origin='lower',
                     vmin=-clim, vmax=clim)
        ax_bp.invert_yaxis()
        ax_bp.text(0.98, 0.02, f't={t_ns:.2f} ns',
                   ha='right', va='bottom', transform=ax_bp.transAxes,
                   fontsize=7, color='white')
    else:
        ax_bp.text(0.5, 0.5, 'not found', ha='center', va='center',
                   transform=ax_bp.transAxes)
    ax_bp.set_ylim(*CMP_DEPTH_RANGE)
    ax_bp.set_xlim(*CMP_RADIAL_RANGE)

    if row == n_prof - 1:
        for ax in axes1[row]:
            ax.set_xlabel('Radial distance [m]')

plt.tight_layout()
fig1.savefig(cmp_out / f'comparison_grid_offset{CMP_FOCUS_OFFSET}.png', dpi=150, bbox_inches='tight')
plt.show();  plt.close(fig1)
print(f'Saved comparison_grid_offset{CMP_FOCUS_OFFSET}.png')

# ── Part 2: offset scan for one reference profile ─────────────────────────────────
slug_scan = f'prof_{CMP_SCAN_PROFILE}'
img_k_scan = _cmp_load_mig(CMP_KIRCHHOFF_METHOD, CMP_SCAN_PROFILE)
img_g_scan = _cmp_load_mig(CMP_GAZDAG_METHOD,    CMP_SCAN_PROFILE)

n_off  = len(CMP_SCAN_OFFSETS)
n_cols = 2 + n_off
fig2, axes2 = plt.subplots(1, n_cols, figsize=(3.5 * n_cols, 7), sharey=True)
fig2.suptitle(f'BP snapshot offset scan — {slug_scan}  '
              f'(depth {CMP_DEPTH_RANGE[0]}–{CMP_DEPTH_RANGE[1]} m)', y=1.01)

# Kirchhoff reference
ax0 = axes2[0]
if img_k_scan is not None:
    n     = img_k_scan.shape[0]
    ext_k = [x_img[0], x_img[-1], depth[n - 1], depth[0]]
    clim  = np.percentile(np.abs(img_k_scan), 100)
    ax0.imshow(img_k_scan, aspect='auto', cmap='seismic',
               extent=ext_k, origin='upper', vmin=-clim, vmax=clim)
    ax0.invert_yaxis()
ax0.set_ylim(*CMP_DEPTH_RANGE);  ax0.set_xlim(*CMP_RADIAL_RANGE)
ax0.set_title(CMP_KIRCHHOFF_METHOD.capitalize())
ax0.set_xlabel('Radial [m]');  ax0.set_ylabel('Depth [m]')

# Gazdag reference
ax1 = axes2[1]
if img_g_scan is not None:
    n     = img_g_scan.shape[0]
    ext_g = [x_img[0], x_img[-1], depth[n - 1], depth[0]]
    clim  = np.percentile(np.abs(img_g_scan), 100)
    ax1.imshow(img_g_scan, aspect='auto', cmap='seismic',
               extent=ext_g, origin='upper', vmin=-clim, vmax=clim)
    ax1.invert_yaxis()
ax1.set_ylim(*CMP_DEPTH_RANGE);  ax1.set_xlim(*CMP_RADIAL_RANGE)
ax1.set_title(CMP_GAZDAG_METHOD.capitalize())
ax1.set_xlabel('Radial [m]')

# BP at each offset
for j, offset in enumerate(CMP_SCAN_OFFSETS):
    ez_bp, meta = _cmp_load_bp(slug_scan, offset)
    ax_j = axes2[2 + j]
    if ez_bp is not None:
        dom_x, dom_y, t_ns = meta
        clim = np.percentile(np.abs(ez_bp), 100)
        ax_j.imshow(ez_bp, aspect='auto', cmap='seismic',
                    extent=[0, dom_y, 0, dom_x], origin='lower',
                    vmin=-clim, vmax=clim)
        ax_j.invert_yaxis()
        ax_j.set_title(f'BP  offset={offset}\nt={t_ns:.2f} ns')
    else:
        ax_j.text(0.5, 0.5, 'not found', ha='center', va='center',
                  transform=ax_j.transAxes)
        ax_j.set_title(f'BP  offset={offset}')
    ax_j.set_ylim(*CMP_DEPTH_RANGE);  ax_j.set_xlim(*CMP_RADIAL_RANGE)
    ax_j.set_xlabel('Radial [m]')

plt.tight_layout()
fig2.savefig(cmp_out / f'bp_offset_scan_{slug_scan}.png', dpi=150, bbox_inches='tight')
plt.show();  plt.close(fig2)
print(f'Saved bp_offset_scan_{slug_scan}.png')

print('Done.')


# Time-lapse Differencing t_n - t_n-1

In [ ]:
# Kirchhoff & Gazdag: image(t_n+1) - image(t_n) for consecutive runs
# Requires the main loop (previous section) to have been re-run so that
# kirchhoff_{run}.npy / gazdag_{run}.npy exist alongside the .png files.

import matplotlib.ticker as ticker

diff_dir = OUT_DIR / 'difference'
diff_dir.mkdir(exist_ok=True)

DIFF_SC        = 1.2    # color-limit scale factor
TICK_SPACING_X = 0.5    # radial-axis tick spacing [m]
TICK_SPACING_Z = 1.0    # depth-axis tick spacing [m]
NOISE_SUPPRESS = 0.15   # suppress values below this fraction of each image's peak

def _consecutive_pairs(method):
    """Yield (run_a, run_b, img_a, img_b) for runs with saved .npy arrays, in data_runs order."""
    avail = [r for r in data_runs if (OUT_DIR / 'migrated' / f'{method}_{r}.npy').exists()]
    for run_a, run_b in zip(avail[:-1], avail[1:]):
        img_a = np.load(OUT_DIR / 'migrated' / f'{method}_{run_a}.npy')
        img_b = np.load(OUT_DIR / 'migrated' / f'{method}_{run_b}.npy')
        n_common = min(img_a.shape[0], img_b.shape[0])
        yield run_a, run_b, img_a[:n_common], img_b[:n_common]

for method, label in [('kirchhoff', 'Kirchhoff'), ('gazdag', 'Gazdag')]:
    pairs = list(_consecutive_pairs(method))
    if not pairs:
        print(f'[{label}] No saved .npy arrays found — re-run the main processing loop first.')
        continue
    print(f'[{label}] {len(pairs)} consecutive pair(s) found')

    for run_a, run_b, img_a, img_b in pairs:
        diff      = img_b - img_a
        z_common  = depth[:diff.shape[0]]
        lim       = max(DIFF_SC * np.max(np.abs(diff)), 1.0)
        diff_disp = np.where(np.abs(diff) < NOISE_SUPPRESS * np.max(np.abs(diff)), 0.0, diff)

        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(diff_disp, aspect='auto', cmap='seismic',
                  extent=[x_img[0], x_img[-1], z_common[-1], z_common[0]],
                  vmin=-lim, vmax=lim)
        ax.invert_yaxis()
        ax.xaxis.set_major_locator(ticker.MultipleLocator(TICK_SPACING_X))
        ax.yaxis.set_major_locator(ticker.MultipleLocator(TICK_SPACING_Z))
        ax.grid(True, color='k', linewidth=0.3, alpha=0.4)
        ax.set_xlabel('Radial distance from borehole (m)')
        ax.set_ylabel('Depth (m)')
        ax.set_title(f'{label} difference: prof_{run_b} − prof_{run_a}')
        out_path = diff_dir / f'{method}_diff_{run_a}_to_{run_b}.png'
        fig.savefig(out_path, dpi=150)
        plt.close(fig)
        print(f'  Saved {out_path.name}')

print('Done.')

In [ ]:
# Back-propagation: Ez_focus(t_n+1) - Ez_focus(t_n) for consecutive completed runs
# Requires the snapshot cell above to have been run first (reuses `completed`,
# t_focus_ns, t_start_ns, snap_step, dt_ns_bp).

FOCUS_IDX_OFFSET  = 28    # must match the offset used in the snapshot cell above
BP_DIFF_PCT       = 97    # percentile used for difference colour limits
BP_NOISE_SUPPRESS = 0.15  # suppress values below this fraction of each image's peak
BP_NEAR_MASK_M    = 3.0   # zero radial < this [m] for display (source injection zone)

bp_diff_dir = OUT_DIR / 'backprop_snapshots' / 'difference'
bp_diff_dir.mkdir(parents=True, exist_ok=True)

def _load_focus_ez(snap_files):
    snap_times_ns = t_start_ns + np.arange(len(snap_files)) * snap_step * dt_ns_bp
    idx = min(int(np.argmin(np.abs(snap_times_ns - t_focus_ns))) + FOCUS_IDX_OFFSET,
              len(snap_files) - 1)
    mesh = pyvista.read(str(snap_files[idx]))
    nx_c = mesh.dimensions[0] - 1
    ny_c = mesh.dimensions[1] - 1
    e_data = np.array(mesh['E-field'])
    ez = e_data[:, 2].reshape(ny_c, nx_c).T          # rows=depth, cols=radial
    domain_x = nx_c * mesh.spacing[0]
    domain_y = ny_c * mesh.spacing[0]
    return ez, snap_times_ns[idx], domain_x, domain_y

avail_bp = sorted(completed.keys(), key=lambda s: int(s.split('_')[1]))
if len(avail_bp) < 2:
    print('Fewer than 2 completed back-propagation runs — nothing to difference yet.')
else:
    print(f'{len(avail_bp)} completed run(s): {avail_bp}')
    for slug_a, slug_b in zip(avail_bp[:-1], avail_bp[1:]):
        ez_a, t_a, dom_x, dom_y = _load_focus_ez(completed[slug_a])
        ez_b, t_b, _, _         = _load_focus_ez(completed[slug_b])

        n_x = min(ez_a.shape[0], ez_b.shape[0])
        n_y = min(ez_a.shape[1], ez_b.shape[1])
        diff = ez_b[:n_x, :n_y] - ez_a[:n_x, :n_y]

        extent    = [0, dom_y, 0, dom_x]
        clim      = np.percentile(np.abs(diff), BP_DIFF_PCT)
        diff_disp = np.where(np.abs(diff) < BP_NOISE_SUPPRESS * np.max(np.abs(diff)), 0.0, diff)
        _dx_bp = dom_y / ez_a.shape[1]   # [m/px]
        _mpx   = max(1, round(BP_NEAR_MASK_M / _dx_bp))
        diff_disp[:, :_mpx] = 0.0

        fig, ax = plt.subplots(figsize=(6, 8))
        im = ax.imshow(diff_disp, aspect='auto', cmap='seismic',
                       extent=extent, origin='lower', vmin=-clim, vmax=clim)
        plt.colorbar(im, ax=ax, label='ΔEz [V/m]')
        ax.set_ylim(60, 85)
        ax.set_xlim(BP_NEAR_MASK_M, dom_y)
        ax.invert_yaxis()
        ax.set_xlabel('Radial distance [m]')
        ax.set_ylabel('Depth [m]')
        ax.set_title(f'Back-propagation ΔEz: {slug_b} (t={t_b:.2f} ns) − {slug_a} (t={t_a:.2f} ns)')

        plt.tight_layout()
        out_path = bp_diff_dir / f'{slug_a}_to_{slug_b}_diff.png'
        plt.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f'  Saved {out_path.name}')

print('Done.')

# Time-lapse Differencing t_n - t_reference

In [ ]:
# Kirchhoff & Gazdag: image(t_n) - image(t_reference) for all runs vs a fixed baseline.
# Reuses DIFF_SC, TICK_SPACING_X/Z, ticker from the consecutive-pair cell above.

REF_RUN_DIFF   = 1     # prof1 (first injection measurement)
NOISE_SUPPRESS = 0.00  # suppress values below this fraction of each image's peak

ref_diff_dir = OUT_DIR / 'difference_from_ref'
ref_diff_dir.mkdir(exist_ok=True)

for method, label in [('kirchhoff', 'Kirchhoff'), ('gazdag', 'Gazdag')]:
    ref_path = OUT_DIR / 'migrated' / f'{method}_{REF_RUN_DIFF}.npy'
    if not ref_path.exists():
        print(f'[{label}] Reference {ref_path.name} not found — skipping.')
        continue
    img_ref = np.load(ref_path)

    avail = [r for r in data_runs
             if r != REF_RUN_DIFF and (OUT_DIR / 'migrated' / f'{method}_{r}.npy').exists()]
    if not avail:
        print(f'[{label}] No non-reference .npy arrays found.')
        continue
    print(f'[{label}] {len(avail)} run(s) vs ref=prof_{REF_RUN_DIFF}')

    for run in avail:
        img_n    = np.load(OUT_DIR / 'migrated' / f'{method}_{run}.npy')
        n_common = min(img_ref.shape[0], img_n.shape[0])
        diff     = img_n[:n_common] - img_ref[:n_common]
        z_common = depth[:n_common]

        lim       = max(DIFF_SC * np.max(np.abs(diff)), 1.0)
        diff_disp = np.where(np.abs(diff) < NOISE_SUPPRESS * np.max(np.abs(diff)), 0.0, diff)

        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(diff_disp, aspect='auto', cmap='seismic',
                  extent=[x_img[0], x_img[-1], z_common[-1], z_common[0]],
                  vmin=-lim, vmax=lim)
        ax.invert_yaxis()
        ax.xaxis.set_major_locator(ticker.MultipleLocator(TICK_SPACING_X))
        ax.yaxis.set_major_locator(ticker.MultipleLocator(TICK_SPACING_Z))
        ax.grid(True, color='k', linewidth=0.3, alpha=0.4)
        ax.set_xlabel('Radial distance from borehole (m)')
        ax.set_ylabel('Depth (m)')
        ax.set_title(f'{label} difference: prof_{run} − prof_{REF_RUN_DIFF}')
        out_path = ref_diff_dir / f'{method}_diff_ref{REF_RUN_DIFF}_to_{run}.png'
        fig.savefig(out_path, dpi=150)
        plt.close(fig)
        print(f'  Saved {out_path.name}')

print('Done.')

In [ ]:
# Back-propagation: Ez_focus(t_n) - Ez_focus(t_reference) for all completed runs.
# Requires the snapshot cell to have been run (reuses completed, t_focus_ns, etc.).

BP_REF_SLUG       = 'prof_1'   # slug for prof1 (first injection measurement)
BP_REF_DIFF_PCT   = 97         # percentile for colour limits
BP_NOISE_SUPPRESS = 0.15       # suppress values below this fraction of each image's peak
BP_NEAR_MASK_M    = 3.0        # zero radial < this [m] for display (source injection zone)

bp_ref_diff_dir = OUT_DIR / 'backprop_snapshots' / 'difference_from_ref'
bp_ref_diff_dir.mkdir(parents=True, exist_ok=True)

if BP_REF_SLUG not in completed:
    print(f'{BP_REF_SLUG} not in completed dict — run the snapshot cell first.')
else:
    ez_ref, t_ref, dom_x_ref, dom_y_ref = _load_focus_ez(completed[BP_REF_SLUG])
    _dx_bp_ref = dom_y_ref / ez_ref.shape[1]   # spatial pixel size [m/px]
    avail_bp_ref = sorted(
        [s for s in completed if s != BP_REF_SLUG],
        key=lambda s: int(s.split('_')[1])
    )
    print(f'{len(avail_bp_ref)} run(s) vs ref={BP_REF_SLUG}')

    for slug in avail_bp_ref:
        ez_n, t_n, dom_x, dom_y = _load_focus_ez(completed[slug])

        n_x = min(ez_ref.shape[0], ez_n.shape[0])
        n_y = min(ez_ref.shape[1], ez_n.shape[1])
        diff = ez_n[:n_x, :n_y] - ez_ref[:n_x, :n_y]

        extent    = [0, dom_y, 0, dom_x]
        clim      = np.percentile(np.abs(diff), BP_REF_DIFF_PCT)
        diff_disp = np.where(np.abs(diff) < BP_NOISE_SUPPRESS * np.max(np.abs(diff)), 0.0, diff)
        _mpx_r = max(1, round(BP_NEAR_MASK_M / _dx_bp_ref))
        diff_disp[:, :_mpx_r] = 0.0

        fig, ax = plt.subplots(figsize=(6, 8))
        im = ax.imshow(diff_disp, aspect='auto', cmap='seismic',
                       extent=extent, origin='lower', vmin=-clim, vmax=clim)
        plt.colorbar(im, ax=ax, label='ΔEz [V/m]')
        ax.set_ylim(60, 85)
        ax.set_xlim(BP_NEAR_MASK_M, dom_y)
        ax.invert_yaxis()
        ax.set_xlabel('Radial distance [m]')
        ax.set_ylabel('Depth [m]')
        ax.set_title(f'Back-propagation ΔEz: {slug} (t={t_n:.2f} ns) − {BP_REF_SLUG} (t={t_ref:.2f} ns)')

        plt.tight_layout()
        out_path = bp_ref_diff_dir / f'{BP_REF_SLUG}_to_{slug}_diff.png'
        plt.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f'  Saved {out_path.name}')

print('Done.')

# Time-lapse Differencing by Phase

Phase-total differences: **end-of-phase minus start-of-phase**.
Each image captures what changed over the full duration of that pumping stage.

| Phase | Start profile | End profile |
|---|---|---|
| Ref.     | prof     |          |
| Pushing  | prof\_1  | prof\_3  |
| Chasing  | prof\_4  | prof\_8  |
| Waiting  | prof\_9  | prof\_20 |
| Pulling  | prof\_21 | prof\_38 |


In [ ]:
# Kirchhoff & Gazdag: phase-total differences — end_of_phase − start_of_phase

PHASE_PAIRS = [
    ('Pushing',  1,  3),
    ('Chasing',  4,  8),
    ('Waiting',  9, 20),
    ('Pulling', 21, 38),
]

DIFF_SC        = 1.2    # colour-limit scale factor
TICK_SPACING_X = 0.5    # radial tick spacing [m]
TICK_SPACING_Z = 1.0    # depth tick spacing [m]
NOISE_SUPPRESS = 0.00   # suppress values below this fraction of the image peak

phase_diff_dir = OUT_DIR / 'difference_by_phase'
phase_diff_dir.mkdir(exist_ok=True)

for method, label in [('kirchhoff', 'Kirchhoff'), ('gazdag', 'Gazdag')]:
    for phase_name, run_start, run_end in PHASE_PAIRS:
        path_a = OUT_DIR / 'migrated' / f'{method}_{run_start}.npy'
        path_b = OUT_DIR / 'migrated' / f'{method}_{run_end}.npy'
        if not path_a.exists() or not path_b.exists():
            print(f'[{label}] {phase_name}: missing prof_{run_start} or prof_{run_end} — skipping.')
            continue
        img_a = np.load(path_a)
        img_b = np.load(path_b)
        n     = min(img_a.shape[0], img_b.shape[0])
        diff  = img_b[:n] - img_a[:n]
        z_common = depth[:n]

        lim       = max(DIFF_SC * np.max(np.abs(diff)), 1.0)
        diff_disp = np.where(np.abs(diff) < NOISE_SUPPRESS * np.max(np.abs(diff)), 0.0, diff)

        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(diff_disp, aspect='auto', cmap='seismic',
                  extent=[x_img[0], x_img[-1], z_common[-1], z_common[0]],
                  vmin=-lim, vmax=lim)
        ax.invert_yaxis()
        ax.xaxis.set_major_locator(ticker.MultipleLocator(TICK_SPACING_X))
        ax.yaxis.set_major_locator(ticker.MultipleLocator(TICK_SPACING_Z))
        ax.grid(True, color='k', linewidth=0.3, alpha=0.4)
        ax.set_xlabel('Radial distance from borehole (m)')
        ax.set_ylabel('Depth (m)')
        ax.set_title(f'{label} — {phase_name}: prof_{run_end} − prof_{run_start}')
        out_path = phase_diff_dir / f'{method}_phase_{phase_name.lower()}.png'
        fig.savefig(out_path, dpi=150)
        plt.close(fig)
        print(f'  Saved {out_path.name}')

print('Done.')


In [ ]:
# Back-propagation: Ez phase-total differences — end_of_phase − start_of_phase
# Reuses PHASE_PAIRS and _load_focus_ez() from the cells above.

BP_PHASE_DIFF_PCT   = 100    # percentile for colour limits
BP_PHASE_NOISE_SUPP = 0  # suppress values below this fraction of the image peak
BP_PHASE_NEAR_MASK  = 3.0   # zero radial < this [m] for display (source zone)

bp_phase_diff_dir = OUT_DIR / 'backprop_snapshots' / 'difference_by_phase'
bp_phase_diff_dir.mkdir(parents=True, exist_ok=True)

for phase_name, run_start, run_end in PHASE_PAIRS:
    slug_a = f'prof_{run_start}'
    slug_b = f'prof_{run_end}'
    if slug_a not in completed or slug_b not in completed:
        missing = slug_a if slug_a not in completed else slug_b
        print(f'[{phase_name}] {missing} not in completed — skipping.')
        continue

    ez_a, t_a, dom_x, dom_y = _load_focus_ez(completed[slug_a])
    ez_b, t_b, _,    _      = _load_focus_ez(completed[slug_b])

    n_x  = min(ez_a.shape[0], ez_b.shape[0])
    n_y  = min(ez_a.shape[1], ez_b.shape[1])
    diff = ez_b[:n_x, :n_y] - ez_a[:n_x, :n_y]

    extent    = [0, dom_y, 0, dom_x]
    clim      = np.percentile(np.abs(diff), BP_PHASE_DIFF_PCT)
    diff_disp = np.where(np.abs(diff) < BP_PHASE_NOISE_SUPP * np.max(np.abs(diff)),
                         0.0, diff)
    _dx_bp = dom_y / ez_a.shape[1]
    _mpx   = max(1, round(BP_PHASE_NEAR_MASK / _dx_bp))
    diff_disp[:, :_mpx] = 0.0

    fig, ax = plt.subplots(figsize=(6, 8))
    im = ax.imshow(diff_disp, aspect='auto', cmap='seismic',
                   extent=extent, origin='lower', vmin=-clim, vmax=clim)
    plt.colorbar(im, ax=ax, label='ΔEz [V/m]')
    ax.set_ylim(60, 85)
    ax.set_xlim(BP_PHASE_NEAR_MASK, dom_y)
    ax.invert_yaxis()
    ax.set_xlabel('Radial distance [m]')
    ax.set_ylabel('Depth [m]')
    ax.set_title(f'Back-prop — {phase_name}: {slug_b} (t={t_b:.2f} ns) − {slug_a} (t={t_a:.2f} ns)')

    plt.tight_layout()
    out_path = bp_phase_diff_dir / f'bp_phase_{phase_name.lower()}.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved {out_path.name}')

print('Done.')


# Fluid Front Movement Estimate

Pushing phase: profiles 1-3

Chasing phase: profiles 4-8

Waiting phase: profiles 9-20

Pulling phase: profiles 21-38

In [ ]:
# ── Helper functions: unified loader, WLS phase-plane fit, Riesz/monogenic/ROI ──────
from scipy.signal.windows import tukey
from matplotlib.patches import Rectangle
from helper_functions.migration import load_migrated_image, wls_phase_plane_fit

# ── Unified data-loading layer (protocol: see load_migrated_image docstring) ───────
# Every technique (gazdag / kirchhoff_bp / backprop) is normalised at load time to
# ONE convention: row index increases with physical depth, column index increases
# with radial distance from the borehole. This is what let the sign-convention bug
# found this session (back-propagation's depth axis ran opposite to Kirchhoff/
# Gazdag's) be fixed architecturally instead of per-cell -- no downstream code
# (rectangular window, sliding window, k-space/depth-radial picking) ever needs a
# technique-specific sign correction again.
def _load_img(method, run):
    """Legacy-contract shim: returns a bare (n_depth, n_radial) array in the
    ORIGINAL decreasing-depth-with-row-index convention (row 0 = 85 m, deepest),
    or None if missing -- the exact contract every pre-existing cell in this
    notebook (Strategy 1-3b, Thesis Figure Compilations) already depends on.
    New cells should call load_migrated_image directly instead, which returns a
    MigratedImage in the unified increasing-depth convention plus dz/dx/kz_cent/
    meta -- this shim exists only so cells written before this refactor keep
    working unmodified.
    """
    mi = load_migrated_image(method, run, migrated_dir=OUT_DIR / 'migrated', borehole_dir=bh_out,
                              depth_gk=depth, x_img_gk=x_img, dL_gk=dL, f0_mig=f0_mig, v=v)
    if mi is None:
        return None
    return mi.image[::-1, :] if method != 'backprop' else mi.image


def _phys_to_pix(depth_axis, radial_axis, z_min, z_max, x_min, x_max):
    """Convert a physical ROI (metres) to pixel indices. Works regardless of
    whether depth_axis increases or decreases with row index -- thresholding a
    monotonic array always yields a contiguous index range either way, so this one
    function serves both the legacy decreasing-depth arrays (_load_img's `depth`)
    and the unified increasing-depth arrays (load_migrated_image's depth_axis).
    Returns (z0, z1, x0, x1), z1/x1 exclusive.
    """
    rows = np.where((depth_axis >= z_min) & (depth_axis <= z_max))[0]
    cols = np.where((radial_axis >= x_min) & (radial_axis <= x_max))[0]
    if rows.size == 0 or cols.size == 0:
        raise ValueError(
            f'ROI ({z_min}-{z_max} m depth, {x_min}-{x_max} m radial) '
            f'does not intersect the image grid. '
            f'Depth: [{depth_axis.min():.1f}, {depth_axis.max():.1f}] m  '
            f'Radial: [{radial_axis.min():.2f}, {radial_axis.max():.2f}] m'
        )
    return int(rows[0]), int(rows[-1]) + 1, int(cols[0]), int(cols[-1]) + 1


# ── Shared grid constants for the ROI/WLS workflow (moved here so they no longer
# depend on 0a10689e having run first to leave them in scope -- the same execution-
# order fragility this refactor fixes for _load_img/_phys_to_pix). All derived from
# state already available by this point (dL/x_img/f0_mig/v, set in cbdbb920 above).
dz_g = dL
dx_g = float(x_img[1] - x_img[0])
kz_c = 2.0 * np.pi * f0_mig / v


# ── Per-technique WLS fit defaults (established codebase THING_BY_RUN convention,
# c.f. DIP_OVERRIDE_BY_RUN / FAN_HW_BY_RUN elsewhere in this notebook) ──────────────
# gazdag/kirchhoff_bp share identical settings: verified empirically this session
# that Kirchhoff-BP's previously-separate (tukey taper, no data padding, linear
# weighting) settings were accidental drift, not deliberate tuning -- adopting
# Gazdag's settings for both *improves* cross-technique agreement on the Chase pair
# (3->8) from ~7% to ~1% relative difference in Delta_z, consistent with this
# chapter's existing expectation that the two agree closely since they operate on
# the same underlying differenced B-scan. backprop keeps its own settings
# (lower amplitude threshold, tighter kx band), carried over from this session's
# validated borehole_final_wls_harvest work -- back-propagation's focus frames have
# different cross-spectrum characteristics (see bp_wls_difficulty_md below) and were
# never tested against Gazdag's settings.
WLS_FIT_DEFAULTS_BY_METHOD = {
    'gazdag':       dict(taper='edge', data_pad=8, wls_pow=3, kz_band_fac=0.5, kx_band_fac=2.0, amp_thr=0.20),
    'kirchhoff_bp': dict(taper='edge', data_pad=8, wls_pow=3, kz_band_fac=0.5, kx_band_fac=2.0, amp_thr=0.20),
    'backprop':     dict(taper='edge', data_pad=8, wls_pow=3, kz_band_fac=0.5, kx_band_fac=1.5, amp_thr=0.10),
}
# Per-pair band overrides, seeded from Gazdag's pre-existing tuning and applied to
# kirchhoff_bp too (same underlying B-scan/noise characteristics); revisit if
# kirchhoff_bp's diagnostics look off for a given pair. backprop has none yet -- no
# rectangular-window run has been tuned on it before this refactor.
MANUAL_KZ_BAND_FAC_BY_METHOD = {
    'gazdag':       {(1, 3): 0.5, (8, 20): 0.35, (20, 38): 0.45},
    'kirchhoff_bp': {(1, 3): 0.5, (8, 20): 0.35, (20, 38): 0.45},
    'backprop':     {},
}
MANUAL_KX_BAND_FAC_BY_METHOD = {
    'gazdag':       {(3, 8): 2.0},
    'kirchhoff_bp': {(3, 8): 2.0},
    'backprop':     {},
}


# ── Shared per-pair rectangular ROIs, keyed by METHOD (established codebase THING_BY_X
# convention). Used by the Rectangular Window, K-Space Picking, and Depth-Radial
# Picking sections below -- defined once here so none of them depend on another
# having run first. backprop's boxes are seeded from Gazdag/Kirchhoff-BP's (same real
# borehole depth/radial range) -- no rectangular-window run existed on this pipeline
# before this refactor, so expect to retune.
_ROI_1_3, _ROI_3_8, _ROI_8_20, _ROI_20_38 = (71, 78, 5.0, 6.5), (70, 78, 4.5, 7.5), (70, 77, 4.5, 6.0), (70, 78, 4.5, 6.5)
MANUAL_ROIS_BY_METHOD = {
    'gazdag':       {(1, 3): _ROI_1_3, (3, 8): _ROI_3_8, (8, 20): _ROI_8_20, (20, 38): _ROI_20_38},
    'kirchhoff_bp': {(1, 3): _ROI_1_3, (3, 8): _ROI_3_8, (8, 20): _ROI_8_20, (20, 38): _ROI_20_38},
    'backprop':     {(1, 3): _ROI_1_3, (3, 8): _ROI_3_8, (8, 20): _ROI_8_20, (20, 38): _ROI_20_38},
}


def _riesz_2d(img):
    """Full 2D Riesz transform.
    Returns (R_z, R_x): the two real-valued spatial components of the monogenic signal.
      R_z = IFFT2(-i * kz/|k| * FFT2(img))
      R_x = IFFT2(-i * kx/|k| * FFT2(img))
    Uses the same FFT convention as estimate_shift_2d (numpy, no fftshift).
    """
    nz, nx = img.shape
    KZ, KX = np.meshgrid(np.fft.fftfreq(nz), np.fft.fftfreq(nx), indexing='ij')
    K = np.sqrt(KZ**2 + KX**2)
    K[0, 0] = 1.0      # avoid DC divide-by-zero; DC component maps to 0 anyway
    F  = np.fft.fft2(img)
    Rz = np.real(np.fft.ifft2((-1j * KZ / K) * F))
    Rx = np.real(np.fft.ifft2((-1j * KX / K) * F))
    return Rz, Rx

def _monogenic_envelope(img):
    """Amplitude of the monogenic signal: sqrt(f^2 + Rz^2 + Rx^2).
    This is the 2D generalisation of the 1D Hilbert envelope; it is
    phase-invariant and highlights the spatial extent of coherent energy
    regardless of whether the wavelet is at a zero-crossing or a peak.
    """
    Rz, Rx = _riesz_2d(img)
    return np.sqrt(img**2 + Rz**2 + Rx**2)

def _roi_from_envelope(env, threshold_frac=0.5):
    """Tight bounding box of the high-energy region in the envelope image.
    Pixels with env >= threshold_frac * env.max() define the ROI mask;
    returns (z0, z1, x0, x1) as integer row/col indices (z1, x1 exclusive).
    Falls back to the full image if no pixels survive the threshold.
    """
    mask = env >= threshold_frac * env.max()
    rows, cols = np.where(mask)
    if rows.size == 0:
        return 0, env.shape[0], 0, env.shape[1]
    return int(rows.min()), int(rows.max()) + 1, int(cols.min()), int(cols.max()) + 1

def _estimate_shift_2d(base, mon, dz_g, dx_g, kz_cent, force_dz_zero=False, pad_fac=4):
    """WLS 2D phase-plane fit — mirrors estimate_shift_2d from TimeLapse_Processing.ipynb.

    Fits phi(kz, kx) = kz*dz + kx*dx + phi_0 to the cross-spectrum of base and mon,
    weighted by spectral amplitude and restricted to a band around kz_cent.

    Parameters
    ----------
    base, mon   : 2D arrays (n_depth, n_radial), pre-cropped to the ROI
    dz_g        : depth grid spacing [m]   (row spacing, axis 0)
    dx_g        : radial grid spacing [m]  (col spacing, axis 1)
    kz_cent     : dominant wavenumber [rad/m] = 2π * f0 / v
    force_dz_zero : if True, fit only (dx, phi_0) — use for purely lateral motion
    pad_fac     : zero-padding factor before FFT2 for denser kx/kz sampling (default 4)

    Returns (dz_est, dx_est, phi_0) all as floats [m, m, rad].
    """
    Nz, Nx = base.shape
    Nz_pad, Nx_pad = Nz * pad_fac, Nx * pad_fac
    kz_ax = np.fft.fftfreq(Nz_pad, d=dz_g) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx_pad, d=dx_g) * 2 * np.pi
    KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')
    taper = np.outer(tukey(Nz, alpha=0.15), tukey(Nx, alpha=0.15))
    XS  = (np.fft.fft2(base * taper, s=(Nz_pad, Nx_pad)) *
           np.conj(np.fft.fft2(mon  * taper, s=(Nz_pad, Nx_pad))))
    w   = np.abs(XS)
    phi = np.angle(XS)
    band = (np.abs(KZ) < 1.4 * kz_cent) & (np.abs(KX) < 1.4 * kz_cent)
    mask = (w > 0.10 * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0)
    W = w[mask]
    if force_dz_zero:
        A = np.column_stack([KX[mask], np.ones(mask.sum())])
        c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
        return 0.0, float(c[0]), float(c[1])
    A = np.column_stack([KZ[mask], KX[mask], np.ones(mask.sum())])
    c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
    return float(c[0]), float(c[1]), float(c[2])


## WLS Cross-Spectrum Phase Fitting — Parameter Reference

### Method overview
For each pair of migrated profiles the cross-spectrum
$\mathrm{XS}(k_z, k_x) = \mathcal{F}(\mathrm{base}) \cdot \mathcal{F}(\mathrm{mon})^*$
is computed on a zero-padded ROI crop (`pad_fac = 10`), windowed by one of two
tapers (`wls_phase_plane_fit`'s `taper` parameter):
- `'edge'`: a sin² ramp over exactly `data_pad` pixels of real data surrounding the
  ROI, so the taper rolls off through genuine data instead of attenuating the ROI
  signal itself. Requires `roi_px` (crops with the data margin included).
- `'tukey'`: a Tukey(`alpha`) window over the array as passed, no extra margin.

A **weighted least-squares plane** is fitted to the cross-spectrum phase:
$$\varphi(k_z, k_x) = k_z \,\Delta z + k_x \,\Delta x + \varphi_0$$
with $|\mathrm{XS}|^{\text{wls\_pow}}$ as weights. $\Delta z$ is displacement along
the borehole (depth), $\Delta x$ is radial displacement. This single function
(`wls_phase_plane_fit`, `helper_functions/migration.py`) now backs every ROI
strategy in this chapter -- rectangular window, sliding window, and both manual
picking variants -- replacing what were previously ~4 separately-maintained,
quietly-diverged copies of the same fit.

---

### Per-technique defaults (`WLS_FIT_DEFAULTS_BY_METHOD`)

| Method | `taper` | `data_pad` | `wls_pow` | `kz_band_fac` | `kx_band_fac` | `amp_thr` |
|---|---|---|---|---|---|---|
| `gazdag` | `'edge'` | 8 | 3 | 0.5 | 2.0 | 0.20 |
| `kirchhoff_bp` | `'edge'` | 8 | 3 | 0.5 | 2.0 | 0.20 |
| `backprop` | `'edge'` | 8 | 3 | 0.5 | **1.5** | **0.10** |

Gazdag and Kirchhoff-BP share identical settings: an empirical cross-check this
session (fit the Chase pair 3→8 both ways) confirmed Kirchhoff-BP's previously
separate settings (`taper='tukey'`, no data padding, `wls_pow=1`) were accidental
drift rather than deliberate tuning -- unifying to Gazdag's settings *improved*
cross-technique agreement from ~7% to ~1% relative difference in $\Delta z$,
consistent with both operating on the same underlying differenced B-scan.
Back-propagation keeps its own, lower amplitude threshold and tighter kx band --
its focus frames have a flatter cross-spectrum with a large near-DC component (see
the "Why $\Delta z$ Estimation Is Harder for Back-Propagation" note below), so a
Gazdag-level threshold would exclude most of the usable k-space.

**Why kz and kx bands are separated:** the signal energy sits in two horizontal
lobes at $k_x \approx \pm 8\,\text{rad/m},\; k_z \approx 0$ (sub-horizontal
borehole reflectors). The kx band must stay wide enough to include these lobes.
The kz band can be narrowed independently for stages where only the low-$k_z$
region is coherent, without cutting the signal in $k_x$.

**Phase-wrapping limit:** wrapping occurs when $k_{z,\text{max}} \times |\Delta z| > \pi$.
At `kz_band_fac = 0.5` the limit depends on $k_{z,c}$. The Push and Chase stages
($\Delta z \approx 1\text{--}2\,\text{m}$) may approach this limit, but the
cross-spectrum amplitude drops below `amp_thr` before the worst wrapping points,
so wrapped cells are largely excluded automatically.

---

### Per-pair kz band overrides (`MANUAL_KZ_BAND_FAC_BY_METHOD`)

Applied to both `gazdag` and `kirchhoff_bp` (same underlying B-scan/noise
characteristics); `backprop` has none yet.

| Stage | Pair | `kz_band_fac` | Reason |
|---|---|---|---|
| Push | 1 → 3 | 0.5 (default) | Strong coherent signal; kz-phase ramp is linear |
| Chase | 3 → 8 | 0.5 (default) | Good linearity across the kz band |
| Wait | 8 → 20 | **0.35** | S-curve artifact: restrict to $|k_z| < 0.35\,k_{z,c}$ |
| Pull | 20 → 38 | **0.45** | Oscillatory 1-D profile: same restriction |

---

### Per-pair kx band overrides (`MANUAL_KX_BAND_FAC_BY_METHOD`)

| Stage | Pair | `kx_band_fac` | Reason |
|---|---|---|---|
| Chase | 3 → 8 | **2.0** (== default; effectively a no-op override, kept for documentation) | Off-lobe kx cells can bias the sign of $\Delta x$ if narrowed further |

---

### Amplitude weight exponent (`wls_pow`)

The WLS minimises $\sum_i w_i (\hat\varphi_i - \varphi_i)^2$ where
$w_i = |\mathrm{XS}_i|^p$ and $p$ is `wls_pow`.

| `wls_pow` | Effect |
|---|---|
| 1 | Linear amplitude weights — standard amplitude-weighted least squares. |
| 2 | Squared: a cell at 50 % of peak gets $\tfrac{1}{4}$ relative influence. |
| 3 | Cubed (current default for all three methods): the same cell gets $\tfrac{1}{8}$ the influence. |

**Why cubing is needed here:** the signal energy is concentrated in two narrow lobes at
$k_x \approx \pm 8\,\text{rad/m},\; k_z \approx 0$.
Between and beyond these lobes there are many k-cells with moderate amplitude
(passing the `amp_thr` gate) that carry no coherent displacement signal.
With linear weights these peripheral cells collectively outweigh the small number of
dominant signal cells, biasing — and sometimes flipping the sign of — the fitted slope.
Cubing the weights multiplies the signal-to-peripheral influence ratio by
$(w_\text{signal}/w_\text{peripheral})^2$,
concentrating the fit on the high-amplitude lobe cells where the displacement
information actually lives.

---

### Residual phase in the 1-D diagnostic profiles

The raw cross-spectrum phase at $(k_z, k_x)$ is:
$$\varphi(k_z, k_x) = k_z \,\Delta z + k_x \,\Delta x + \varphi_0$$

Collapsing to a 1-D profile by averaging over one axis mixes both displacement
components, making it hard to judge fit quality in each direction separately.
A large $k_z \Delta z$ term causes phase wrapping at cells away from $k_z = 0$,
producing a systematic bias in the kx-marginal average.

**Partial residuals.** Before averaging, the contribution of the *other* axis is
subtracted using the current WLS estimates:

| Profile | Residual computed | Remaining slope | Fitted line overlay |
|---|---|---|---|
| 1-D $k_z$ (panel 4) | $\varphi - k_x\,\hat\Delta x$ | $k_z\,\Delta z + \varphi_0$ | $k_z\,\hat\Delta z + \hat\varphi_0$ |
| 1-D $k_x$ (panel 5) | $\varphi - k_z\,\hat\Delta z$ | $k_x\,\Delta x + \varphi_0$ | $k_x\,\hat\Delta x + \hat\varphi_0$ |

With residuals each profile shows only its own axis contribution.
The fitted line **should pass through the high-weight data points**.
If it does not — particularly if the slope is reversed — the WLS plane is being
pulled by low-amplitude peripheral cells; try increasing `wls_pow` or narrowing
the band with `MANUAL_KX_BAND_FAC_BY_METHOD` / `MANUAL_KZ_BAND_FAC_BY_METHOD`.

---

### Interpreting the diagnostic panels

**Panel 1 — Cross-spectrum phase:** amplitude-masked phase in $(k_z, k_x)$ space.
Two lobes at $k_x \approx \pm 8\,\text{rad/m}$ carry opposite sign when $\Delta x \neq 0$.
A kz-dependent phase ramp within each lobe indicates $\Delta z$.

**Panel 2 — Energy:** $|\mathrm{XS}|$ inside the band. Cyan contour = `amp_thr` level.
Points outside the contour are excluded from the WLS fit.

**Panel 3 — Fitted plane:** the WLS result $k_z\Delta z + k_x\Delta x + \varphi_0$.
Should resemble the phase panel if the linear model is a good fit.

**Panel 4 — 1-D kz profile (kx-corrected residual):** weighted mean of $\varphi - k_x\hat\Delta x$
collapsed over $k_x$. Dot colour = amplitude weight (viridis: purple = low, yellow = high).
A linear slope matching the red line confirms $\Delta z$.

| 1-D kz pattern | Cause | Interpretation |
|---|---|---|
| Linear ramp, slope matches fit | Good coherent signal | $\Delta z$ reliable |
| S-curve $(-k_z^3 + k_z)$ through origin | Two anti-phase kx lobes cancel unevenly | $\Delta z \approx 0$ |
| Oscillatory, multiple extrema | High-kz incoherence from long time gap | Reduce `MANUAL_KZ_BAND_FAC_BY_METHOD` |
| Flat scatter, no trend | Near-zero SNR | Treat displacement as $\approx 0$ |

**Panel 5 — 1-D kx profile (kz-corrected residual):** weighted mean of $\varphi - k_z\hat\Delta z$
collapsed over $k_z$. Slope $\approx \Delta x$.
If the high-weight dots follow a different slope than the red line,
peripheral cells are dominating — see `wls_pow` and `MANUAL_KX_BAND_FAC_BY_METHOD`.

---

### Sign convention

Every method's images are loaded through `load_migrated_image`, which normalises
row index to increase with physical depth regardless of the underlying technique's
native convention (Gazdag/Kirchhoff-BP's cached arrays run the opposite way
natively and are flipped at load time). A positive raw $\Delta z$ from this fit
therefore always means the same physical direction across all three methods --
downward, i.e. increasing depth. See @sec:hyp3-fd-bp-corrected in the thesis
chapter for how this was verified.

---

### Physical trajectory summary (a priori expectation, not the finding)

| Phase | Profiles | Expected $\Delta z$ | Expected $\Delta x$ |
|---|---|---|---|
| Pushing | 1→3 | $> 0$ (fluid injected downward along fracture) | Small |
| Chasing | 3→8 | $> 0$ (continued downward migration) | Small |
| Waiting | 8→20 | $\approx 0$ (no pumping) | $\approx 0$ |
| Pulling | 20→38 | $< 0$ (fluid partially returns upward) | Opposite to push |

The actual, sign-verified result reverses both the depth and radial direction from
this a priori table -- see @tab:fielddata-stages and @sec:hyp3-fd-interpretation in
the thesis chapter.


## Rectangular Window <sec:hyp3-fd-roi-rect>

Hand-picked axis-aligned ROI in depth/radial space, tuned per pair against the
monogenic envelope of the time-lapse difference image, with a fixed cross-spectrum
band and amplitude threshold selecting the $(k_z, k_x)$ cells inside it (per-method
defaults: `WLS_FIT_DEFAULTS_BY_METHOD`, see the "WLS Cross-Spectrum Phase Fitting"
reference above). `METHOD` below selects which migration technique to analyse --
`'gazdag'`, `'kirchhoff_bp'`, or `'backprop'` (the borehole-geometry, dx=0.02
pipeline -- the older homogeneous-domain back-propagation results are archived
further down, not selectable here). Replaces what were previously two
separately-maintained cells (one per technique, with quietly-diverged fit
settings) with one cell parameterised by technique.


In [ ]:
# ── Rectangular Window ROI + WLS phase-plane fit — all three migration techniques ────
#
# METHOD        — 'gazdag' | 'kirchhoff_bp' | 'backprop'. Selects which migration
#                 technique load_migrated_image() reads for every pair below.
# PAIRS         — list of (run_a, run_b) to analyse.
# COMPARE_TO    — None  → use pairs as written (run_a vs run_b)
#                 int   → override run_a for every pair to this run number,
#                         so you compare every profile against one fixed reference.
# MANUAL_ROIS   — per-pair ROI in physical metres, keyed by METHOD. Key is the
#                 ORIGINAL (run_a, run_b) regardless of COMPARE_TO.
#                 Format: (z_min_m, z_max_m, x_min_m, x_max_m)
#                 None → auto-detect from monogenic envelope.
#                 backprop's boxes are seeded from Gazdag/Kirchhoff-BP's (same real
#                 borehole depth/radial range) -- no rectangular-window run had been
#                 tuned on this pipeline before this refactor, so expect to retune.
# SHOW_DIAG     — True: plot cross-spectrum phase + fitted plane for every pair.
# FORCE_DZ_ZERO — True: fit only Δx + φ₀ (lateral fluid front).
# ROI_THRESH    — envelope threshold for auto-detected pairs.
# ────────────────────────────────────────────────────────────────────────────────────

METHOD = 'gazdag'   # 'gazdag' | 'kirchhoff_bp' | 'backprop'

PAIRS = [
    (1,  3),
    (3,  8),
    (8,  20),
    (20, 38),
]

COMPARE_TO = None

MANUAL_ROIS = MANUAL_ROIS_BY_METHOD[METHOD]   # shared dict, defined in bedd9933 above

FORCE_DZ_ZERO = False
ROI_THRESH    = 0.5
SHOW_DIAG     = True   # set False to skip cross-spectrum phase plots

# ────────────────────────────────────────────────────────────────────────────────────

def _load(run):
    return load_migrated_image(METHOD, run, migrated_dir=OUT_DIR / 'migrated', borehole_dir=bh_out,
                                depth_gk=depth, x_img_gk=x_img, dL_gk=dL, f0_mig=f0_mig, v=v)

roi_out = OUT_DIR / 'roi_phase'
roi_out.mkdir(exist_ok=True)

results = {}
for run_a, run_b in PAIRS:
    ref_run = COMPARE_TO if COMPARE_TO is not None else run_a
    img_a = _load(ref_run)
    img_b = _load(run_b)
    if img_a is None or img_b is None:
        print(f'[{ref_run}→{run_b}] Missing data for method={METHOD} — skipping.')
        continue

    n         = min(img_a.image.shape[0], img_b.image.shape[0])
    a_arr     = img_a.image[:n];  b_arr = img_b.image[:n]
    z_common  = img_a.depth_axis[:n]
    x_common  = img_a.radial_axis
    diff      = b_arr - a_arr
    env       = _monogenic_envelope(diff)

    # ROI — look up by original key (run_a, run_b) first, then (ref_run, run_b)
    roi_spec = MANUAL_ROIS.get((run_a, run_b)) or MANUAL_ROIS.get((ref_run, run_b))
    if roi_spec is not None:
        z0, z1, x0, x1 = _phys_to_pix(z_common, x_common, *roi_spec)
        roi_source = 'manual'
    else:
        z0, z1, x0, x1 = _roi_from_envelope(env, ROI_THRESH)
        roi_source = f'auto (thresh={ROI_THRESH})'

    z_roi_top = float(z_common[z0]);     z_roi_bot = float(z_common[z1 - 1])
    x_roi_lo  = float(x_common[x0]);     x_roi_hi  = float(x_common[x1 - 1])

    # ── Difference + envelope figure ────────────────────────────────────────────
    # z_common increases with row index (unified convention, see load_migrated_image);
    # this extent + origin='upper' combination renders shallow-at-top/deep-at-bottom
    # without an invert_yaxis() call -- verified empirically, do not add one back.
    extent = [x_common[0], x_common[-1], z_common[-1], z_common[0]]
    vmax_d = np.percentile(np.abs(diff), 98)
    vmax_e = np.percentile(env, 98)

    fig, (ax_d, ax_e) = plt.subplots(1, 2, figsize=(14, 5))
    im_d = ax_d.imshow(diff, aspect='auto', cmap='RdBu_r',
                        extent=extent, origin='upper',
                        vmin=-vmax_d, vmax=vmax_d)
    plt.colorbar(im_d, ax=ax_d, label='Δ amplitude [a.u.]')
    ax_d.add_patch(Rectangle((x_roi_lo, z_roi_bot),
                              width=x_roi_hi - x_roi_lo, height=z_roi_top - z_roi_bot,
                              lw=1.5, edgecolor='yellow', facecolor='none'))
    ax_d.xaxis.set_major_locator(ticker.MultipleLocator(0.5))
    ax_d.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_d.grid(True, color='white', lw=0.3, alpha=0.4)
    ax_d.set_title(f'{METHOD} diff: prof_{run_b} − prof_{ref_run}  [{roi_source}]')
    ax_d.set_xlabel('Radial distance (m)');  ax_d.set_ylabel('Depth (m)')

    im_e = ax_e.imshow(env, aspect='auto', cmap='inferno',
                        extent=extent, origin='upper', vmin=0, vmax=vmax_e)
    plt.colorbar(im_e, ax=ax_e, label='Monogenic envelope [a.u.]')
    ax_e.add_patch(Rectangle((x_roi_lo, z_roi_bot),
                              width=x_roi_hi - x_roi_lo, height=z_roi_top - z_roi_bot,
                              lw=1.5, edgecolor='cyan', facecolor='none'))
    ax_e.xaxis.set_major_locator(ticker.MultipleLocator(0.5))
    ax_e.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_e.grid(True, color='white', lw=0.3, alpha=0.4)
    ax_e.set_title(f'Monogenic envelope  [{roi_source}]')
    ax_e.set_xlabel('Radial distance (m)');  ax_e.set_ylabel('Depth (m)')

    plt.tight_layout()
    fig.savefig(roi_out / f'{METHOD}_roi_{ref_run}_to_{run_b}.png', dpi=150)
    plt.show();  plt.close(fig)

    # ── WLS phase-plane fit ──────────────────────────────────────────────────────
    _defaults = WLS_FIT_DEFAULTS_BY_METHOD[METHOD]
    _kz_fac = (MANUAL_KZ_BAND_FAC_BY_METHOD[METHOD].get((run_a, run_b))
               or MANUAL_KZ_BAND_FAC_BY_METHOD[METHOD].get((ref_run, run_b))
               or _defaults['kz_band_fac'])
    _kx_fac = (MANUAL_KX_BAND_FAC_BY_METHOD[METHOD].get((run_a, run_b))
               or MANUAL_KX_BAND_FAC_BY_METHOD[METHOD].get((ref_run, run_b))
               or _defaults['kx_band_fac'])

    dz_est, dx_est, phi_0, n_mask, diag = wls_phase_plane_fit(
        a_arr, b_arr, img_a.dz, img_a.dx, img_a.kz_cent,
        roi_px=(z0, z1, x0, x1), data_pad=8, taper=_defaults['taper'],
        kz_band_fac=_kz_fac, kx_band_fac=_kx_fac, amp_thr=_defaults['amp_thr'],
        wls_pow=_defaults['wls_pow'], pad_fac=10, force_dz_zero=FORCE_DZ_ZERO,
        return_diagnostics=True)
    Nz, Nx = z1 - z0, x1 - x0

    # ── Diagnostic: cross-spectrum phase + fitted plane ──────────────────────────
    if SHOW_DIAG and n_mask >= 3:
        KZ, KX, w, phi = diag['KZ'], diag['KX'], diag['w'], diag['phi']
        band, fitted, kz_ax, kx_ax = diag['mask'], diag['fitted'], diag['kz_ax'], diag['kx_ax']
        Nz_pad, Nx_pad = KZ.shape
        kz_c = img_a.kz_cent

        # Shift to centre-zero frequency for display
        phi_shift = np.fft.fftshift(phi)
        w_shift   = np.fft.fftshift(w)
        kz_disp   = np.fft.fftshift(kz_ax)
        kx_disp   = np.fft.fftshift(kx_ax)
        fitted_shift = np.fft.fftshift(fitted)

        # 1-D kz profile: subtract kx contribution first so the slope shows Dz cleanly
        phi_kz_resid = phi - KX * dx_est
        phi_1d  = np.zeros(Nz_pad)
        w_1d    = np.zeros(Nz_pad)
        band_kx = np.abs(kx_ax) < _kx_fac * kz_c
        for i_row in range(Nz_pad):
            sel = band_kx & (w[i_row, :] > _defaults['amp_thr'] * w.max())
            if sel.sum() > 0:
                phi_1d[i_row] = np.average(phi_kz_resid[i_row, :][sel], weights=w[i_row, :][sel])
                w_1d[i_row]   = w[i_row, :][sel].sum()
        kz_1d_s  = np.fft.fftshift(kz_ax)
        phi_1d_s = np.fft.fftshift(phi_1d)
        w_1d_s   = np.fft.fftshift(w_1d)

        # 1-D kx profile: subtract kz contribution first so the slope shows Dx cleanly
        phi_kx_resid = phi - KZ * dz_est
        phi_1d_kx   = np.zeros(Nx_pad)
        w_1d_kx     = np.zeros(Nx_pad)
        band_kz_fit = np.abs(kz_ax) < _kz_fac * kz_c
        for j_col in range(Nx_pad):
            sel_kz = band_kz_fit & (w[:, j_col] > _defaults['amp_thr'] * w.max())
            if sel_kz.sum() > 0:
                phi_1d_kx[j_col] = np.average(phi_kx_resid[:, j_col][sel_kz],
                                               weights=w[:, j_col][sel_kz])
                w_1d_kx[j_col]   = w[:, j_col][sel_kz].sum()
        kx_1d_s     = np.fft.fftshift(kx_ax)
        phi_1d_kx_s = np.fft.fftshift(phi_1d_kx)
        w_1d_kx_s   = np.fft.fftshift(w_1d_kx)

        fig_d, axes_d = plt.subplots(1, 5, figsize=(28, 4))

        # Panel 1: cross-spectrum phase (fftshifted, band only)
        band_shift = np.fft.fftshift(band)
        phi_masked = np.where(band_shift & (np.fft.fftshift(w) > _defaults['amp_thr'] * w.max()),
                              phi_shift, np.nan)
        im_ph = axes_d[0].imshow(phi_masked, aspect='auto', cmap='RdBu_r',
                                  extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]],
                                  vmin=-np.pi, vmax=np.pi, origin='upper')
        plt.colorbar(im_ph, ax=axes_d[0], label='phase [rad]')
        axes_d[0].set_title(f'Cross-spectrum phase  (mask: {n_mask} px)')
        axes_d[0].set_xlabel('kx [rad/m]');  axes_d[0].set_ylabel('kz [rad/m]')
        axes_d[0].set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3)
        axes_d[0].set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

        # Panel 2: cross-spectrum energy with WLS amplitude threshold contour
        w_plot = np.where(band_shift, w_shift, np.nan)
        im_en = axes_d[1].imshow(w_plot, aspect='auto', cmap='inferno',
                                  extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]],
                                  origin='upper')
        plt.colorbar(im_en, ax=axes_d[1], label='|XS| [a.u.]')
        axes_d[1].contour(kx_disp, kz_disp, w_shift,
                          levels=[_defaults['amp_thr'] * w.max()], colors='cyan', linewidths=0.8)
        axes_d[1].set_title(f'Energy  (thr={_defaults["amp_thr"]:.2f}×max — cyan contour)')
        axes_d[1].set_xlabel('kx [rad/m]');  axes_d[1].set_ylabel('kz [rad/m]')
        axes_d[1].set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3)
        axes_d[1].set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

        # Panel 3: fitted plane
        fitted_masked = np.where(band_shift, fitted_shift, np.nan)
        im_fit = axes_d[2].imshow(fitted_masked, aspect='auto', cmap='RdBu_r',
                                   extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]],
                                   vmin=-np.pi, vmax=np.pi, origin='upper')
        plt.colorbar(im_fit, ax=axes_d[2], label='phase [rad]')
        axes_d[2].set_title(f'Fitted plane  Δz={dz_est:+.4f} m  Δx={dx_est:+.4f} m  φ₀={phi_0:+.3f} rad')
        axes_d[2].set_xlabel('kx [rad/m]');  axes_d[2].set_ylabel('kz [rad/m]')
        axes_d[2].set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3)
        axes_d[2].set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

        # Panel 4: 1-D kz slice — measured vs fitted
        in_band = np.abs(kz_1d_s) < _kz_fac * kz_c
        sc_kz = axes_d[3].scatter(kz_1d_s[in_band & (w_1d_s > 0)],
                          phi_1d_s[in_band & (w_1d_s > 0)],
                          c=w_1d_s[in_band & (w_1d_s > 0)],
                          cmap='viridis', s=20, label='measured (weighted)')
        plt.colorbar(sc_kz, ax=axes_d[3], label='weight [a.u.]')
        kz_fit = kz_1d_s[in_band]
        axes_d[3].plot(kz_fit, kz_fit * dz_est + phi_0, 'r-', lw=1.5, label='fitted slope')
        axes_d[3].axhline(0, color='k', lw=0.5, ls='--')
        axes_d[3].set_xlabel('kz [rad/m]');  axes_d[3].set_ylabel('phase [rad]')
        axes_d[3].set_title('1-D kz profile  (kx-corrected residual, slope = Δz)')
        axes_d[3].legend(fontsize=8)

        # Panel 5: 1-D kx slice — measured vs fitted
        in_band_kx = np.abs(kx_1d_s) < _kx_fac * kz_c
        sc_kx = axes_d[4].scatter(kx_1d_s[in_band_kx & (w_1d_kx_s > 0)],
                          phi_1d_kx_s[in_band_kx & (w_1d_kx_s > 0)],
                          c=w_1d_kx_s[in_band_kx & (w_1d_kx_s > 0)],
                          cmap='viridis', s=20, label='measured (weighted)')
        plt.colorbar(sc_kx, ax=axes_d[4], label='weight [a.u.]')
        kx_fit = kx_1d_s[in_band_kx]
        axes_d[4].plot(kx_fit, kx_fit * dx_est + phi_0, 'r-', lw=1.5, label='fitted slope')
        axes_d[4].axhline(0, color='k', lw=0.5, ls='--')
        axes_d[4].set_xlabel('kx [rad/m]');  axes_d[4].set_ylabel('phase [rad]')
        axes_d[4].set_title('1-D kx profile  (kz-corrected residual, slope = Δx)')
        axes_d[4].legend(fontsize=8)
        axes_d[4].set_xlim(-_kx_fac * kz_c, _kx_fac * kz_c)

        plt.suptitle(f'Diagnostic ({METHOD}): prof_{ref_run} → prof_{run_b}', y=1.02)
        plt.tight_layout()
        fig_d.savefig(roi_out / f'{METHOD}_diag_{ref_run}_to_{run_b}.png',
                      dpi=150, bbox_inches='tight')
        plt.show();  plt.close(fig_d)
    elif SHOW_DIAG:
        print(f'[{ref_run}→{run_b}] WARNING: only {n_mask} pixels pass the WLS mask '
              f'(crop {Nz}×{Nx} px).  ROI may be too small or SNR too low.')

    results[(ref_run, run_b)] = {
        'dx': dx_est, 'dz': dz_est, 'phi_0': phi_0,
        'roi_m':  (z_roi_bot, z_roi_top, x_roi_lo, x_roi_hi),
        'roi_px': (z0, z1, x0, x1),
        'n_mask': n_mask, 'source': roi_source,
    }
    print(f'prof_{ref_run}→{run_b}  [{roi_source}]:  '
          f'Δx={dx_est:+.4f} m  Δz={dz_est:+.4f} m  φ₀={phi_0:+.4f} rad  '
          f'mask={n_mask} px  crop={Nz}×{Nx} px  '
          f'ROI depth=[{z_roi_bot:.1f},{z_roi_top:.1f}] m  '
          f'radial=[{x_roi_lo:.2f},{x_roi_hi:.2f}] m')

# ── Summary plot ─────────────────────────────────────────────────────────────────────
if results:
    pair_labels = [f'{a}→{b}' for a, b in results]
    dx_vals  = [results[k]['dx']    for k in results]
    phi_vals = [results[k]['phi_0'] for k in results]

    fig2, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    dz_vals = [results[k]['dz'] for k in results]
    from matplotlib.patches import Patch as _Patch
    _PHASE_DEFS = [('Pushing', 1, 3, '#aed6f1'), ('Chasing', 4, 8, '#a9dfbf'),
                   ('Waiting', 9, 20, '#f9e79f'), ('Pulling', 21, 38, '#f1948a')]
    _rn_col = {n: c for _nm, lo, hi, c in _PHASE_DEFS for n in range(lo, hi + 1)}
    _pkeys  = list(results.keys())
    for _bax in (ax1, ax2):
        for _i, _k in enumerate(_pkeys):
            _c = _rn_col.get(_k[1])
            if _c:
                _bax.axvspan(_i - 0.5, _i + 0.5, color=_c, alpha=0.3, zorder=0, lw=0)
    _ph_hdl = [_Patch(facecolor=c, alpha=0.6, label=nm, edgecolor='grey', lw=0.5)
               for nm, lo, hi, c in _PHASE_DEFS
               if any(_rn_col.get(_k[1]) == c for _k in _pkeys)]
    ax1.legend(handles=_ph_hdl, loc='best', fontsize=8, framealpha=0.7)
    ax1.plot(dx_vals, 'o-', color='steelblue')
    ax1.axhline(0, color='k', lw=0.6, ls='--')
    ax1.set_ylabel('Δx (radial) [m]')
    mode_str = f'vs prof_{COMPARE_TO}' if COMPARE_TO is not None else 'consecutive pairs'
    ax1.set_title(f'WLS fluid-front shift — {METHOD} ({mode_str})')
    ax2.plot(dz_vals, '^-', color='seagreen')
    ax2.axhline(0, color='k', lw=0.6, ls='--')
    ax2.set_ylabel('Δz (depth) [m]')
    plt.tight_layout()
    fig2.savefig(roi_out / f'{METHOD}_dx_summary.png', dpi=150)
    plt.show();  plt.close(fig2)

print(f'\nDone — {len(results)} pair(s).  Output: {roi_out}')


## Sliding-Window ROI Scan <sec:hyp3-fd-roi-sliding>

Instead of one manually-drawn ROI per pair, slide a fixed-size window across the
whole difference B-scan and run the same WLS phase-plane fit in every window
position. This trades the hand-picked rectangle above for a 2-D map of the fit,
so spatial patterns in the estimated displacement become visible instead of a
single averaged number per pair. `METHOD` below selects the migration technique,
same as the Rectangular Window section above.

- **Window size** — set as a multiple of the dominant wavelength
  (`WINDOW_LAMBDA_FRAC`), so it scales automatically if `f0_mig` or `v` change.
  The physical window footprint (metres) is kept the same across all three
  techniques -- only the WLS fit's internal k-space band (relative to each
  technique's own reference wavenumber) varies by method.
- **Stride** — `STRIDE_Z_PX` / `STRIDE_X_PX` pixels the window centre advances
  each step. `None` defaults to a non-overlapping stride (window width) to keep
  the scan fast; shrink it for a finer map once the QC figure below confirms the
  window size looks reasonable.
- **Output** — two maps, **Δz** (depth / vertical) and **Δx** (radial /
  lateral), plus a QC figure with the scanned window grid drawn on the
  difference B-scan so you can judge if the window is too big or too small.


In [ ]:
# ── Sliding-window ROI scan: Δz / Δx displacement maps ──────────────────────────────
#
# METHOD — 'gazdag' | 'kirchhoff_bp' | 'backprop'. Same selector as the
#          Rectangular Window section above; this cell is self-contained (defines
#          its own METHOD/PAIRS rather than relying on that section having just run).
# PAIRS / COMPARE_TO — same semantics as the Rectangular Window section.
#
# WINDOW_Z_LAMBDA_FRAC / WINDOW_X_LAMBDA_FRAC — window depth/radial extent, each as a
#                       multiple of the dominant wavelength (2*pi/kz_c, full-velocity
#                       convention, kept fixed across all three METHOD choices so the
#                       window's physical footprint in metres doesn't change just
#                       because a different technique is selected). These are
#                       independent because the signal is anisotropic: reflectors are
#                       sub-horizontal (kz ~ 0, long coherence length in depth) while
#                       the radial lobes sit at kx ~ +/-8 rad/m (short wavelength,
#                       ~0.8 m) -- the manually-tuned ROIs in the Rectangular Window
#                       section reflect this, running ~7-8 m deep but only ~1.5-3 m
#                       radial. A window narrower than ~1 radial wavelength cannot
#                       resolve a stable phase gradient in x, and just fits noise
#                       (this is what produced salt-and-pepper dz/dx maps with
#                       WINDOW_LAMBDA_FRAC=0.5 in an earlier version). Defaults below
#                       reproduce the manually-tuned ROI footprint.
# STRIDE_Z_PX / STRIDE_X_PX — pixels the window centre advances each step.
#                       None -> full window width (non-overlapping), which keeps the
#                       scan fast. Shrink once the QC figure confirms the window size,
#                       and only once WINDOW_*_LAMBDA_FRAC is large enough to give a
#                       stable per-window fit -- a fine stride cannot fix a window
#                       that's fundamentally too small.
# SCAN_Z_RANGE_M / SCAN_X_RANGE_M — physical sub-region to scan (None -> full domain).
# WIN_MIN_MASK_PX   — minimum WLS-mask pixel count; windows below this are left as
#                      NaN in the maps (too few k-cells passed the amplitude gate).
# WIN_ENERGY_THR    — the WLS amplitude gate is relative to *each window's own* peak,
#                      so a window sitting in pure noise still clears WIN_MIN_MASK_PX
#                      with a "confident" but meaningless fit. WIN_ENERGY_THR adds an
#                      absolute gate: a window's mean |Δ amplitude| must reach this
#                      fraction of the pair's global 98th-percentile |Δ| to be trusted.
#                      Windows failing either gate are left as NaN in the maps.
# Other WIN_* params default to WLS_FIT_DEFAULTS_BY_METHOD[METHOD] (bedd9933 above) so
# the fit itself matches the Rectangular Window section -- only the ROI placement is
# now automatic.
# ────────────────────────────────────────────────────────────────────────────────────

METHOD = 'gazdag'   # 'gazdag' | 'kirchhoff_bp' | 'backprop'

PAIRS = [
    (1,  3),
    # (3,  8),
    # (8,  20),
    # (20, 38),
]
COMPARE_TO = None

WINDOW_Z_LAMBDA_FRAC = 3.0   # -> ~7.5 m depth, matching the manual ROIs' 7-8 m
WINDOW_X_LAMBDA_FRAC = 0.9   # -> ~2.0 m radial, matching the manual ROIs' 1.5-3 m
STRIDE_Z_PX = 4
STRIDE_X_PX = 4
SCAN_Z_RANGE_M = None
SCAN_X_RANGE_M = None
WIN_DATA_PAD_PX = 8
_defaults = WLS_FIT_DEFAULTS_BY_METHOD[METHOD]
WIN_KZ_BAND_FAC = _defaults['kz_band_fac']
WIN_KX_BAND_FAC = _defaults['kx_band_fac']
WIN_AMP_THR = _defaults['amp_thr']
WIN_WLS_POW = _defaults['wls_pow']
WIN_PAD_FAC = 6
WIN_MIN_MASK_PX = 3
WIN_ENERGY_THR = 0.50
WIN_FORCE_DZ_ZERO = False
QC_PAIR_INDEX = 0   # which PAIRS entry to draw the QC window-grid figure for

# Physical window footprint (metres) -- fixed across all three METHOD choices, using
# the full-velocity kz_c (Gazdag/Kirchhoff-BP convention) as the reference wavelength
# regardless of which technique is selected below.
wavelength_m = 2.0 * np.pi / kz_c
window_z_m   = WINDOW_Z_LAMBDA_FRAC * wavelength_m
window_x_m   = WINDOW_X_LAMBDA_FRAC * wavelength_m
print(f'Dominant wavelength = {wavelength_m:.3f} m  ->  window = {window_z_m:.2f} m (depth, '
      f'{WINDOW_Z_LAMBDA_FRAC:.1f} x lambda) x {window_x_m:.2f} m (radial, '
      f'{WINDOW_X_LAMBDA_FRAC:.1f} x lambda)')


def sliding_window_scan(img_a, img_b, diff, z_common, x_img_arr, dz_g, dx_g, kz_c,
                         window_z_m, window_x_m, stride_z_px=None, stride_x_px=None,
                         z_range_m=None, x_range_m=None, data_pad=8,
                         kz_band_fac=0.5, kx_band_fac=2.0, amp_thr=0.20,
                         wls_pow=3, pad_fac=6, force_dz_zero=False, min_mask_px=3):
    """Slide a fixed-size window across (img_a, img_b), running the WLS phase-plane
    fit (wls_phase_plane_fit, helper_functions/migration.py) in each window.
    window_z_m / window_x_m are the window's physical depth and radial extent
    (independent -- see WINDOW_Z_LAMBDA_FRAC / WINDOW_X_LAMBDA_FRAC above for why
    they shouldn't be equal). Returns dz_map, dx_map, n_map, e_map (2-D; e_map is
    the window's mean |diff| amplitude, for the absolute energy gate), z_centers_m,
    x_centers_m (1-D map axes, physical units), and window_px (list of
    (z0,z1,x0,x1) pixel boxes, for QC plotting)."""
    Nz_img, Nx_img = img_a.shape
    nz_half = max(1, int(round((window_z_m / 2) / dz_g)))
    nx_half = max(1, int(round((window_x_m / 2) / dx_g)))
    if stride_z_px is None:
        stride_z_px = 2 * nz_half
    if stride_x_px is None:
        stride_x_px = 2 * nx_half

    if z_range_m is None:
        z0_lim, z1_lim = 0, Nz_img
    else:
        rows = np.where((z_common >= z_range_m[0]) & (z_common <= z_range_m[1]))[0]
        z0_lim, z1_lim = int(rows.min()), int(rows.max()) + 1
    if x_range_m is None:
        x0_lim, x1_lim = 0, Nx_img
    else:
        cols = np.where((x_img_arr >= x_range_m[0]) & (x_img_arr <= x_range_m[1]))[0]
        x0_lim, x1_lim = int(cols.min()), int(cols.max()) + 1

    z_centers_px = list(range(z0_lim + nz_half, z1_lim - nz_half + 1, stride_z_px))
    x_centers_px = list(range(x0_lim + nx_half, x1_lim - nx_half + 1, stride_x_px))
    if not z_centers_px or not x_centers_px:
        raise ValueError('Scan region is smaller than one window -- widen the range '
                          'or shrink WINDOW_Z_LAMBDA_FRAC / WINDOW_X_LAMBDA_FRAC.')

    dz_map = np.full((len(z_centers_px), len(x_centers_px)), np.nan)
    dx_map = np.full_like(dz_map, np.nan)
    n_map  = np.zeros_like(dz_map, dtype=int)
    e_map  = np.zeros_like(dz_map)
    window_px = []
    for iz, zc in enumerate(z_centers_px):
        for ix, xc in enumerate(x_centers_px):
            z0, z1 = zc - nz_half, zc + nz_half
            x0, x1 = xc - nx_half, xc + nx_half
            dz_est, dx_est, _phi0, n_mask = wls_phase_plane_fit(
                img_a, img_b, dz_g, dx_g, kz_c, roi_px=(z0, z1, x0, x1),
                data_pad=data_pad, taper='edge', kz_band_fac=kz_band_fac,
                kx_band_fac=kx_band_fac, amp_thr=amp_thr, wls_pow=wls_pow,
                pad_fac=pad_fac, force_dz_zero=force_dz_zero)
            dz_map[iz, ix] = dz_est if n_mask >= min_mask_px else np.nan
            dx_map[iz, ix] = dx_est if n_mask >= min_mask_px else np.nan
            n_map[iz, ix]  = n_mask
            e_map[iz, ix]  = np.abs(diff[z0:z1, x0:x1]).mean()
            window_px.append((z0, z1, x0, x1))

    z_centers_m = z_common[z_centers_px]
    x_centers_m = x_img_arr[x_centers_px]
    return dz_map, dx_map, n_map, e_map, z_centers_m, x_centers_m, window_px, (nz_half, nx_half)


def _plot_window_qc(diff, z_common, x_img_arr, window_px, nz_half, nx_half, dz_g, dx_g,
                     title, save_path, max_boxes=250, thesis_name=None):
    """QC figure: difference B-scan with every scanned window drawn as a thin rectangle,
    plus one window highlighted in green so its size is easy to judge against the data
    (too big -> averages out the front; too small -> noisy / too few k-cells).
    z_common increases with row index (unified convention) -- this extent +
    origin='upper' combination renders shallow-at-top/deep-at-bottom without an
    invert_yaxis() call, verified empirically (see Rectangular Window section)."""
    extent = [x_img_arr[0], x_img_arr[-1], z_common[-1], z_common[0]]
    vmax_d = np.percentile(np.abs(diff), 98)
    fig, ax = plt.subplots(figsize=(9, 6))
    im = ax.imshow(diff, aspect='auto', cmap='RdBu_r', extent=extent, origin='upper',
                    vmin=-vmax_d, vmax=vmax_d)
    plt.colorbar(im, ax=ax, label='Delta amplitude [a.u.]')

    step = max(1, len(window_px) // max_boxes)
    for (z0, z1, x0, x1) in window_px[::step]:
        z_top, z_bot = z_common[z0], z_common[z1 - 1]
        x_lo, x_hi   = x_img_arr[x0], x_img_arr[x1 - 1]
        ax.add_patch(Rectangle((x_lo, z_bot), width=x_hi - x_lo, height=z_top - z_bot,
                                lw=0.6, edgecolor='yellow', facecolor='none', alpha=0.6))

    z0, z1, x0, x1 = window_px[len(window_px) // 2]
    z_top, z_bot = z_common[z0], z_common[z1 - 1]
    x_lo, x_hi   = x_img_arr[x0], x_img_arr[x1 - 1]
    ax.add_patch(Rectangle((x_lo, z_bot), width=x_hi - x_lo, height=z_top - z_bot,
                            lw=2.0, edgecolor='lime', facecolor='none'))
    ax.set_title(f'{title}\none window = {2*nz_half*dz_g:.2f} m (depth) x '
                 f'{2*nx_half*dx_g:.2f} m (radial)  --  {len(window_px)} windows total')
    ax.set_xlabel('Radial distance (m)'); ax.set_ylabel('Depth (m)')
    plt.tight_layout()
    fig.savefig(save_path, dpi=150)
    if thesis_name is not None:
        save_fig(fig, thesis_name, study='FieldData_Study', prefix='FD_', category='Compilations')
    plt.show(); plt.close(fig)


def _plot_disp_maps(dz_map, dx_map, valid, z_centers_m, x_centers_m,
                     title_prefix, save_path, thesis_name=None):
    """Delta-z and Delta-x maps side-by-side; windows failing the valid mask
    (WIN_MIN_MASK_PX and/or WIN_ENERGY_THR) are blanked. z_centers_m increases with
    row index -- same no-invert-needed convention as _plot_window_qc above."""
    dz_plot = np.where(valid, dz_map, np.nan)
    dx_plot = np.where(valid, dx_map, np.nan)
    extent = [x_centers_m[0], x_centers_m[-1], z_centers_m[-1], z_centers_m[0]]

    fig, (ax_z, ax_x) = plt.subplots(1, 2, figsize=(13, 5))
    for ax, m, lbl, cmap in ((ax_z, dz_plot, 'Delta z (depth) [m]', 'RdBu_r'),
                              (ax_x, dx_plot, 'Delta x (radial) [m]', 'PuOr_r')):
        finite = m[np.isfinite(m)]
        vmax = np.percentile(np.abs(finite), 98) if finite.size else 1.0
        vmax = vmax if vmax > 0 else 1.0
        im = ax.imshow(m, aspect='auto', cmap=cmap, extent=extent, origin='upper',
                        vmin=-vmax, vmax=vmax)
        plt.colorbar(im, ax=ax, label=lbl)
        ax.set_xlabel('Radial distance (m)'); ax.set_ylabel('Depth (m)')
        ax.set_title(f'{title_prefix}: {lbl}')
    plt.tight_layout()
    fig.savefig(save_path, dpi=150)
    if thesis_name is not None:
        save_fig(fig, thesis_name, study='FieldData_Study', prefix='FD_', category='Compilations')
    plt.show(); plt.close(fig)


# ── Run the scan for the same pairs as the Rectangular Window section above ────────
# THESIS_QC_NAME / THESIS_MAPS_NAME_FMT — pass a name to also export the QC figure and
# each pair's dz/dx maps to TimeLapse_Figures/FieldData_Study/Compilations/ (protocol:
# .wiki/FIGURES_PROTOCOL.md), in addition to the ad-hoc roi_out save above. None skips
# the thesis export. Kept exactly as cited by the thesis chapter (@fig:fd-sliding-qc /
# @fig:fd-sliding-maps, captioned "Gazdag migration") only for METHOD=='gazdag';
# suffixed with the method name otherwise so a non-Gazdag run never overwrites the
# cited Gazdag figures.
if METHOD == 'gazdag':
    THESIS_QC_NAME       = 'sliding_window_qc'
    THESIS_MAPS_NAME_FMT = 'sliding_window_maps_{ref_run}_to_{run_b}'
else:
    THESIS_QC_NAME       = f'sliding_window_qc_{METHOD}'
    THESIS_MAPS_NAME_FMT = 'sliding_window_maps_{ref_run}_to_{run_b}_' + METHOD

roi_out = OUT_DIR / 'roi_phase'
roi_out.mkdir(exist_ok=True)

def _load(run):
    return load_migrated_image(METHOD, run, migrated_dir=OUT_DIR / 'migrated', borehole_dir=bh_out,
                                depth_gk=depth, x_img_gk=x_img, dL_gk=dL, f0_mig=f0_mig, v=v)

scan_results = {}
for i_pair, (run_a, run_b) in enumerate(PAIRS):
    ref_run = COMPARE_TO if COMPARE_TO is not None else run_a
    img_a = _load(ref_run)
    img_b = _load(run_b)
    if img_a is None or img_b is None:
        print(f'[{ref_run}->{run_b}] Missing data for method={METHOD} -- skipping sliding-window scan.')
        continue
    n = min(img_a.image.shape[0], img_b.image.shape[0])
    a_arr, b_arr = img_a.image[:n], img_b.image[:n]
    z_common = img_a.depth_axis[:n]
    x_common = img_a.radial_axis
    diff = b_arr - a_arr

    (dz_map, dx_map, n_map, e_map, z_centers_m, x_centers_m, window_px,
     (nz_half, nx_half)) = sliding_window_scan(
        a_arr, b_arr, diff, z_common, x_common, img_a.dz, img_a.dx, img_a.kz_cent,
        window_z_m, window_x_m,
        stride_z_px=STRIDE_Z_PX, stride_x_px=STRIDE_X_PX,
        z_range_m=SCAN_Z_RANGE_M, x_range_m=SCAN_X_RANGE_M,
        data_pad=WIN_DATA_PAD_PX, kz_band_fac=WIN_KZ_BAND_FAC, kx_band_fac=WIN_KX_BAND_FAC,
        amp_thr=WIN_AMP_THR, wls_pow=WIN_WLS_POW, pad_fac=WIN_PAD_FAC,
        force_dz_zero=WIN_FORCE_DZ_ZERO, min_mask_px=WIN_MIN_MASK_PX)

    vmax_d_global = np.percentile(np.abs(diff), 98)
    valid = (n_map >= WIN_MIN_MASK_PX) & (e_map >= WIN_ENERGY_THR * vmax_d_global)

    scan_results[(ref_run, run_b)] = dict(dz_map=dz_map, dx_map=dx_map, n_map=n_map,
                                           e_map=e_map, valid=valid,
                                           z_centers_m=z_centers_m, x_centers_m=x_centers_m)

    print(f'prof_{ref_run}->{run_b}: {len(window_px)} windows scanned '
          f'({len(z_centers_m)} x {len(x_centers_m)} grid), {int(valid.sum())} pass '
          f'both WIN_MIN_MASK_PX and WIN_ENERGY_THR gates')

    if i_pair == QC_PAIR_INDEX:
        _plot_window_qc(diff, z_common, x_common, window_px, nz_half, nx_half, img_a.dz, img_a.dx,
                         title=f'Sliding-window ROI grid -- {METHOD} prof_{ref_run}->{run_b}',
                         save_path=roi_out / f'{METHOD}_scan_qc_{ref_run}_to_{run_b}.png',
                         thesis_name=THESIS_QC_NAME)

    _plot_disp_maps(dz_map, dx_map, valid, z_centers_m, x_centers_m,
                     title_prefix=f'prof_{ref_run}->{run_b}',
                     save_path=roi_out / f'{METHOD}_scan_maps_{ref_run}_to_{run_b}.png',
                     thesis_name=(THESIS_MAPS_NAME_FMT.format(ref_run=ref_run, run_b=run_b)
                                  if THESIS_MAPS_NAME_FMT else None))

print(f'\nSliding-window scan done -- {len(scan_results)} pair(s). Output: {roi_out}')


## Manual k-Space Pixel Selection (Napari)

The automatic WLS fit above selects k-cells with a band (`KZ_BAND_FAC`/`KX_BAND_FAC`)
and a relative amplitude gate (`WLS_AMP_THR`). Here we investigate the alternative:
picking the cross-spectrum pixels by hand, in an interactive [napari](https://napari.org)
viewer, and refitting the WLS plane on exactly the pixels you paint.

This is two cells because napari is interactive and can't run to completion inside a
single non-interactive cell execution:

1. **Launch** — computes the cross-spectrum for a chosen pair/ROI (same crop + taper
   recipe as the manual-ROI cell above) and opens a napari viewer with the amplitude
   as an image layer and an empty, paintable `manual_mask` Labels layer on top. Paint
   over the k-cells you want to include (label 1); leave everything else at 0.
2. **Harvest** — run *after* you've painted. Reads `manual_mask` back, refits the WLS
   plane on your selected pixels, and plots it side by side with the automatic
   band+amplitude selection for the same window so you can compare the two directly.


In [ ]:
# ── Manual k-space pixel selection (napari) — 1: build cross-spectrum & launch viewer ──
#
# METHOD       — 'gazdag' | 'kirchhoff_bp' | 'backprop'. Same selector as the
#                Rectangular Window / Sliding Window sections above; self-contained.
# NAPARI_PAIR  — which (run_a, run_b) to investigate.
# NAPARI_ROI_M — ROI in physical metres (z_min, z_max, x_min, x_max); None -> reuse the
#                shared MANUAL_ROIS_BY_METHOD[METHOD] (bedd9933 above) for this pair,
#                falling back to the monogenic-envelope auto-ROI if there isn't one.
# NAPARI_KDISP_FAC — half-width of the displayed k-space crop, as a multiple of the
#                technique's own reference wavenumber. Only affects what's
#                shown/paintable, not the fit itself.
#
# Paint over the 'manual_mask' layer (paintbrush, label 1) to mark the k-cells you want
# in the WLS fit -- everything left at 0 is excluded. Zoom/pan freely; painting only
# writes where you actually paint. Run cell 2 once you're done.
# ─────────────────────────────────────────────────────────────────────────────────────

import napari

METHOD           = 'gazdag'   # 'gazdag' | 'kirchhoff_bp' | 'backprop'

roi_out = OUT_DIR / 'roi_phase'
roi_out.mkdir(exist_ok=True)
NAPARI_PAIR      = (1, 3)
NAPARI_ROI_M     = None
NAPARI_KDISP_FAC = 3.0
NAPARI_FORCE_DZ_ZERO = False

run_a, run_b = NAPARI_PAIR
ref_run = run_a
_defaults = WLS_FIT_DEFAULTS_BY_METHOD[METHOD]

img_a_mi = load_migrated_image(METHOD, ref_run, migrated_dir=OUT_DIR / 'migrated', borehole_dir=bh_out,
                                depth_gk=depth, x_img_gk=x_img, dL_gk=dL, f0_mig=f0_mig, v=v)
img_b_mi = load_migrated_image(METHOD, run_b, migrated_dir=OUT_DIR / 'migrated', borehole_dir=bh_out,
                                depth_gk=depth, x_img_gk=x_img, dL_gk=dL, f0_mig=f0_mig, v=v)
if img_a_mi is None or img_b_mi is None:
    raise RuntimeError(f'Missing data for method={METHOD}, pair {NAPARI_PAIR}')

n_kspace = min(img_a_mi.image.shape[0], img_b_mi.image.shape[0])
a_arr_kspace = img_a_mi.image[:n_kspace]
b_arr_kspace = img_b_mi.image[:n_kspace]
z_common_kspace = img_a_mi.depth_axis[:n_kspace]
x_common_kspace = img_a_mi.radial_axis

roi_spec = (NAPARI_ROI_M or MANUAL_ROIS_BY_METHOD[METHOD].get((run_a, run_b)))
if roi_spec is not None:
    z0, z1, x0, x1 = _phys_to_pix(z_common_kspace, x_common_kspace, *roi_spec)
else:
    z0, z1, x0, x1 = _roi_from_envelope(_monogenic_envelope(b_arr_kspace - a_arr_kspace), ROI_THRESH)

# Automatic-fit cross-spectrum (reused for both the napari display below and the
# "automatic" comparison fit in the harvest cell -- no separate recomputation needed).
_kz_fac = MANUAL_KZ_BAND_FAC_BY_METHOD[METHOD].get((run_a, run_b)) or _defaults['kz_band_fac']
_kx_fac = MANUAL_KX_BAND_FAC_BY_METHOD[METHOD].get((run_a, run_b)) or _defaults['kx_band_fac']
dz_auto, dx_auto, phi0_auto, n_auto, _diag = wls_phase_plane_fit(
    a_arr_kspace, b_arr_kspace, img_a_mi.dz, img_a_mi.dx, img_a_mi.kz_cent,
    roi_px=(z0, z1, x0, x1), data_pad=8, taper=_defaults['taper'],
    kz_band_fac=_kz_fac, kx_band_fac=_kx_fac, amp_thr=_defaults['amp_thr'],
    wls_pow=_defaults['wls_pow'], pad_fac=10, force_dz_zero=NAPARI_FORCE_DZ_ZERO,
    return_diagnostics=True)
w, phi, KZ, KX = _diag['w'], _diag['phi'], _diag['KZ'], _diag['KX']
kz_ax, kx_ax = _diag['kz_ax'], _diag['kx_ax']
kz_c_kspace = img_a_mi.kz_cent

# Centre-zero display, cropped to +/- NAPARI_KDISP_FAC * kz_c so the viewer isn't 90%
# empty high-frequency padding. NAPARI_IZ0/IX0 remember where this crop sits inside the
# full (unshifted) KZ/KX grid, so cell 2 can map the painted mask back correctly.
kz_disp = np.fft.fftshift(kz_ax);  kx_disp = np.fft.fftshift(kx_ax)
_iz = np.where(np.abs(kz_disp) < NAPARI_KDISP_FAC * kz_c_kspace)[0]
_ix = np.where(np.abs(kx_disp) < NAPARI_KDISP_FAC * kz_c_kspace)[0]
NAPARI_IZ0, NAPARI_IZ1 = int(_iz.min()), int(_iz.max()) + 1
NAPARI_IX0, NAPARI_IX1 = int(_ix.min()), int(_ix.max()) + 1

w_shift   = np.fft.fftshift(w)[NAPARI_IZ0:NAPARI_IZ1, NAPARI_IX0:NAPARI_IX1]
phi_shift = np.fft.fftshift(phi)[NAPARI_IZ0:NAPARI_IZ1, NAPARI_IX0:NAPARI_IX1]
kz_crop   = kz_disp[NAPARI_IZ0:NAPARI_IZ1]
kx_crop   = kx_disp[NAPARI_IX0:NAPARI_IX1]

napari_viewer = napari.Viewer(title=f'k-space manual selection -- {METHOD} prof_{ref_run}->{run_b}')
napari_viewer.add_image(w_shift, name='amplitude |XS|', colormap='inferno')
napari_viewer.add_image(phi_shift, name='phase (rad)', colormap='twilight', visible=False)
manual_labels = napari_viewer.add_labels(
    np.zeros(w_shift.shape, dtype=np.uint8), name='manual_mask')
napari_viewer.layers.selection.active = manual_labels
manual_labels.mode = 'paint'
manual_labels.brush_size = max(1, min(w_shift.shape) // 20)

print(f'Viewer open for {METHOD} prof_{ref_run}->{run_b}.  k-space crop: '
      f'{w_shift.shape[0]} x {w_shift.shape[1]} px  (|kz|,|kx| < {NAPARI_KDISP_FAC * kz_c_kspace:.1f} rad/m).')
print(f'Automatic fit for comparison: dz={dz_auto:+.4f} m  dx={dx_auto:+.4f} m  n={n_auto} px')
print("Paint the 'manual_mask' layer (label 1) over the k-cells you want in the fit, "
      "then run the next cell.")


In [ ]:
# ── Manual k-space pixel selection (napari) — 2: harvest selection & refit ─────────────
#
# Run this AFTER painting the 'manual_mask' layer in the viewer from cell 1. Refits the
# WLS plane using only your painted k-cells (wls_phase_plane_fit with an explicit
# mask=), and compares it against the automatic band+amplitude fit already computed
# in cell 1 (dz_auto/dx_auto/phi0_auto/n_auto).
# ─────────────────────────────────────────────────────────────────────────────────────

manual_mask_crop = manual_labels.data > 0
n_manual = int(manual_mask_crop.sum())
print(f'{n_manual} k-cells painted.')
if n_manual < 3:
    raise RuntimeError("Paint at least 3 k-cells in the 'manual_mask' layer "
                        "(napari_viewer, cell 1) before running this cell.")

# Map the painted (cropped, fftshifted) mask back onto the full, unshifted KZ/KX grid.
manual_mask_shift = np.zeros(np.fft.fftshift(w).shape, dtype=bool)
manual_mask_shift[NAPARI_IZ0:NAPARI_IZ1, NAPARI_IX0:NAPARI_IX1] = manual_mask_crop
manual_mask_full = np.fft.ifftshift(manual_mask_shift)

dz_man, dx_man, phi0_man, n_man = wls_phase_plane_fit(
    a_arr_kspace, b_arr_kspace, img_a_mi.dz, img_a_mi.dx, img_a_mi.kz_cent,
    roi_px=(z0, z1, x0, x1), data_pad=8, taper=_defaults['taper'],
    mask=manual_mask_full, wls_pow=_defaults['wls_pow'], pad_fac=10,
    force_dz_zero=NAPARI_FORCE_DZ_ZERO)

print(f'Manual    ({n_man:5d} px):  dz={dz_man:+.4f} m  dx={dx_man:+.4f} m  phi0={phi0_man:+.3f} rad')
print(f'Automatic ({n_auto:5d} px):  dz={dz_auto:+.4f} m  dx={dx_auto:+.4f} m  phi0={phi0_auto:+.3f} rad')

# ── Comparison figure: phase panel with each selection outlined ────────────────────────
_kz_fac_h = MANUAL_KZ_BAND_FAC_BY_METHOD[METHOD].get((run_a, run_b)) or _defaults['kz_band_fac']
_kx_fac_h = MANUAL_KX_BAND_FAC_BY_METHOD[METHOD].get((run_a, run_b)) or _defaults['kx_band_fac']
auto_band = (np.abs(KZ) < _kz_fac_h * kz_c_kspace) & (np.abs(KX) < _kx_fac_h * kz_c_kspace)
auto_mask = (w > _defaults['amp_thr'] * w.max()) & auto_band & ((np.abs(KZ) + np.abs(KX)) > 0)

extent_k = [kx_crop[0], kx_crop[-1], kz_crop[-1], kz_crop[0]]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, mask_full, dz_e, dx_e, n_e, title in (
        (axes[0], manual_mask_full, dz_man, dx_man, n_man, 'Manual selection'),
        (axes[1], auto_mask, dz_auto, dx_auto, n_auto, 'Automatic band + amplitude gate')):
    mask_shift_crop = np.fft.fftshift(mask_full)[NAPARI_IZ0:NAPARI_IZ1, NAPARI_IX0:NAPARI_IX1]
    im = ax.imshow(phi_shift, aspect='auto', cmap='RdBu_r', extent=extent_k,
                    vmin=-np.pi, vmax=np.pi, origin='upper')
    ax.contour(kx_crop, kz_crop, mask_shift_crop.astype(float), levels=[0.5],
               colors='lime', linewidths=1.2)
    ax.set_title(f'{title}\ndz={dz_e:+.4f} m  dx={dx_e:+.4f} m  n={n_e} px')
    ax.set_xlabel('kx [rad/m]');  ax.set_ylabel('kz [rad/m]')
plt.colorbar(im, ax=axes, label='phase [rad]', shrink=0.8)
plt.suptitle(f'Manual vs automatic k-space selection -- {METHOD} prof_{ref_run}->{run_b}', y=1.03)
fig.savefig(roi_out / f'{METHOD}_napari_manual_vs_auto_{ref_run}_to_{run_b}.png',
            dpi=150, bbox_inches='tight')
# THESIS_NAME -- also export to TimeLapse_Figures/FieldData_Study/Compilations/ (protocol:
# .wiki/FIGURES_PROTOCOL.md). Set to None to skip; picks a fixed thesis-facing name so
# re-running for a different pair overwrites rather than accumulating stale copies.
# Kept exactly as cited by the thesis chapter (@fig:fd-napari-kspace, Gazdag/pair 3->8)
# only for METHOD=='gazdag'; suffixed otherwise so a non-Gazdag run never overwrites it.
THESIS_NAME = 'napari_kspace_manual_vs_auto' if METHOD == 'gazdag' else f'napari_kspace_manual_vs_auto_{METHOD}'
if THESIS_NAME is not None:
    save_fig(fig, THESIS_NAME, study='FieldData_Study', prefix='FD_', category='Compilations')
plt.show(); plt.close(fig)


## Depth-Radial (B-Scan) Picking <sec:hyp3-fd-roi-picking-bscan>

The k-space picking above selects $(k_z, k_x)$ cells directly; this variant instead
paints directly on the *difference image* (depth/radial space), letting the ROI
trace the actual shape of the reflection rather than an axis-aligned box or a
k-space lobe. `METHOD` below selects the migration technique -- the same three
options as the sections above.

Because a hand-painted mask has hard edges that would otherwise leak spectral
energy, the painted mask is Gaussian-softened into a spatial window before the
FFT; the resulting cross-spectrum still goes through the same automatic
band+amplitude gate as the other sections (`WLS_FIT_DEFAULTS_BY_METHOD`), rather
than restricting the fit to painted *k-space* cells the way the k-space picking
variant does.

This section replaces two previously-separate implementations: a generic
depth-radial picking cell that regressed to a hardcoded, never-populated
migration-technique identifier (confirmed via git history -- it silently stopped
working and produced nothing), and a back-propagation-specific version built
during this session's borehole-pipeline correction work. Both are unified here,
parameterised by `METHOD`.


In [ ]:
# ── Depth-Radial (B-Scan) Picking (napari) — 1: launch ─────────────────────────────────
# Paint over the 'manual_roi_mask' layer (paintbrush, label 1) directly on the
# difference image to mark where the real reflection change sits. Run the harvest cell
# below once you're done.
#
# METHOD — 'gazdag' | 'kirchhoff_bp' | 'backprop'. Same selector as the other
#          ROI-workflow sections above; self-contained.
# PICK_PAIR — (run_a, run_b) to investigate.
# ─────────────────────────────────────────────────────────────────────────────────────
import napari

METHOD     = 'gazdag'   # 'gazdag' | 'kirchhoff_bp' | 'backprop'
PICK_PAIR  = (3, 8)

_run_a, _run_b = PICK_PAIR
_pick_img_a = load_migrated_image(METHOD, _run_a, migrated_dir=OUT_DIR / 'migrated', borehole_dir=bh_out,
                                   depth_gk=depth, x_img_gk=x_img, dL_gk=dL, f0_mig=f0_mig, v=v)
_pick_img_b = load_migrated_image(METHOD, _run_b, migrated_dir=OUT_DIR / 'migrated', borehole_dir=bh_out,
                                   depth_gk=depth, x_img_gk=x_img, dL_gk=dL, f0_mig=f0_mig, v=v)
if _pick_img_a is None or _pick_img_b is None:
    raise RuntimeError(f'Missing data for method={METHOD}, pair {PICK_PAIR}')

_nd = min(_pick_img_a.image.shape[0], _pick_img_b.image.shape[0])
_nr = min(_pick_img_a.image.shape[1], _pick_img_b.image.shape[1])
pick_img_a = _pick_img_a.image[:_nd, :_nr]
pick_img_b = _pick_img_b.image[:_nd, :_nr]
pick_depth_axis  = _pick_img_a.depth_axis[:_nd]
pick_radial_axis = _pick_img_a.radial_axis[:_nr]
pick_diff = pick_img_b - pick_img_a

vmax_pick = np.percentile(np.abs(pick_diff), 98)
diff_cmap_pick = napari.utils.Colormap(['blue', 'white', 'red'], name='diff_rdbu_pick')
pick_viewer = napari.Viewer(title=f'Depth-radial manual ROI -- {METHOD} prof_{_run_a}->{_run_b}')
pick_viewer.add_image(pick_diff, name='difference', colormap=diff_cmap_pick,
                       contrast_limits=[-vmax_pick, vmax_pick])
pick_manual_roi_labels = pick_viewer.add_labels(
    np.zeros(pick_diff.shape, dtype=np.uint8), name='manual_roi_mask')
pick_viewer.layers.selection.active = pick_manual_roi_labels
pick_manual_roi_labels.mode = 'paint'
pick_manual_roi_labels.brush_size = max(1, min(pick_diff.shape) // 40)

print(f'Viewer open for {METHOD} prof_{_run_a}->{_run_b}.  Image shape (depth px x radial px): {pick_diff.shape}.')
print(f'Depth axis:  {pick_depth_axis[0]:.2f} m (row 0)  ->  {pick_depth_axis[-1]:.2f} m (row {pick_diff.shape[0]-1}).')
print(f'Radial axis: {pick_radial_axis[0]:.2f} m (col 0) ->  {pick_radial_axis[-1]:.2f} m (col {pick_diff.shape[1]-1}).')
print("Paint the 'manual_roi_mask' layer (label 1) over the real reflection change, then run the next cell.")


In [ ]:
# ── Depth-Radial (B-Scan) Picking (napari) — 2: harvest ────────────────────────────────
# Run this AFTER painting the 'manual_roi_mask' layer in the viewer from the launch cell.
# Uses wls_phase_plane_fit (helper_functions/migration.py) with the painted region
# converted into a Gaussian-softened spatial window (taper='none', since the window is
# already baked into the cropped image before the fit's own FFT) -- the fit itself still
# uses the standard automatic band+amplitude gate (WLS_FIT_DEFAULTS_BY_METHOD[METHOD]),
# same as the other ROI-workflow sections; only the *spatial* extent of the crop comes
# from painting, not a k-space selection (contrast with the K-Space Picking section
# above, which paints (kz,kx) cells directly via an explicit mask=).
# ─────────────────────────────────────────────────────────────────────────────────────
from scipy.ndimage import gaussian_filter as _gaussian_filter_pick

PICK_GAUSS_SIGMA_PX = 3.0    # softens the painted mask's hard 0/1 edges before FFT windowing

mask_full_pick = pick_manual_roi_labels.data > 0
n_painted_pick = int(mask_full_pick.sum())
print(f'{n_painted_pick} image px painted.')
if n_painted_pick < 20:
    raise RuntimeError("Paint a larger region in the 'manual_roi_mask' layer "
                        "(pick_viewer, launch cell) before running this cell.")

rows_pick, cols_pick = np.where(mask_full_pick)
z0_pick, z1_pick = int(rows_pick.min()), int(rows_pick.max()) + 1
x0_pick, x1_pick = int(cols_pick.min()), int(cols_pick.max()) + 1

_margin_pick = int(np.ceil(3 * PICK_GAUSS_SIGMA_PX))
_z0p_pick = max(0, z0_pick - _margin_pick);  _z1p_pick = min(pick_img_a.shape[0], z1_pick + _margin_pick)
_x0p_pick = max(0, x0_pick - _margin_pick);  _x1p_pick = min(pick_img_a.shape[1], x1_pick + _margin_pick)
base_crop_pick = pick_img_a[_z0p_pick:_z1p_pick, _x0p_pick:_x1p_pick]
mon_crop_pick  = pick_img_b[_z0p_pick:_z1p_pick, _x0p_pick:_x1p_pick]
mask_crop_pick = mask_full_pick[_z0p_pick:_z1p_pick, _x0p_pick:_x1p_pick].astype(float)

soft_win_pick = _gaussian_filter_pick(mask_crop_pick, sigma=PICK_GAUSS_SIGMA_PX)
if soft_win_pick.max() > 0:
    soft_win_pick = soft_win_pick / soft_win_pick.max()

_defaults = WLS_FIT_DEFAULTS_BY_METHOD[METHOD]
dz_pick, dx_pick, phi0_pick, n_mask_pick = wls_phase_plane_fit(
    base_crop_pick * soft_win_pick, mon_crop_pick * soft_win_pick,
    _pick_img_a.dz, _pick_img_a.dx, _pick_img_a.kz_cent,
    roi_px=None, taper='none', kz_band_fac=_defaults['kz_band_fac'],
    kx_band_fac=_defaults['kx_band_fac'], amp_thr=_defaults['amp_thr'],
    wls_pow=_defaults['wls_pow'], pad_fac=10, force_dz_zero=False)

if n_mask_pick < 3:
    print(f'WARNING: only {n_mask_pick} k-space px pass the WLS mask. Painted region may '
          f'be too small, or WLS_FIT_DEFAULTS_BY_METHOD[{METHOD!r}] needs loosening.')

print(f'\n{METHOD} prof_{_run_a} -> prof_{_run_b}  (painted mask, {n_painted_pick} img px, {n_mask_pick} k px):')
print(f'  Delta_z (depth)  = {dz_pick:+.4f} m')
print(f'  Delta_x (radial) = {dx_pick:+.4f} m')
print(f'  phi_0            = {phi0_pick:+.4f} rad')

# ── Comparison figure: painted-selection diff + contour ──────────────────────────────
# pick_depth_axis increases with row index (unified convention) -- this extent +
# origin='upper' combination renders shallow-at-top/deep-at-bottom without an
# invert_yaxis() call, verified empirically (see Rectangular Window section).
extent_pick = [pick_radial_axis[0], pick_radial_axis[-1], pick_depth_axis[-1], pick_depth_axis[0]]
vmax_d_pick = np.percentile(np.abs(pick_diff), 98)

fig_pick, ax_pick = plt.subplots(figsize=(7, 7))
im_pick = ax_pick.imshow(pick_diff, aspect='auto', cmap='RdBu_r', extent=extent_pick, origin='upper',
                          vmin=-vmax_d_pick, vmax=vmax_d_pick)
plt.colorbar(im_pick, ax=ax_pick, label='Delta amplitude [a.u.]')
ax_pick.contour(pick_radial_axis, pick_depth_axis, mask_full_pick.astype(float), levels=[0.5],
                colors='lime', linewidths=1.2)
ax_pick.set_xlabel('Radial distance (m)'); ax_pick.set_ylabel('Depth (m)')
ax_pick.set_title(f'{METHOD} prof_{_run_a} -> prof_{_run_b}\n'
                   f'Delta_z={dz_pick:+.4f} m  Delta_x={dx_pick:+.4f} m  (painted ROI)')
plt.tight_layout()
out_path_pick = roi_out / f'{METHOD}_napari_bscan_{_run_a}_to_{_run_b}.png'
fig_pick.savefig(out_path_pick, dpi=150, bbox_inches='tight')

# THESIS_NAME -- also export to TimeLapse_Figures/FieldData_Study/Compilations/ (protocol:
# .wiki/FIGURES_PROTOCOL.md). Kept exactly as cited by the thesis chapter for the two
# methods it already documents (@fig:fd-napari-bscan for gazdag/pair 3->8;
# @fig:fd-borehole-napari-pairs for backprop, per-pair names) -- kirchhoff_bp has no
# established citation yet, so it gets a per-pair name too.
if METHOD == 'gazdag':
    THESIS_NAME = 'napari_bscan_manual_vs_rect'
elif METHOD == 'backprop':
    THESIS_NAME = f'borehole_napari_wls_{_run_a}_to_{_run_b}'
else:
    THESIS_NAME = f'napari_bscan_manual_vs_rect_{METHOD}_{_run_a}_to_{_run_b}'
save_fig(fig_pick, THESIS_NAME, study='FieldData_Study', prefix='FD_', category='Compilations')
plt.show(); plt.close(fig_pick)
print(f'\nSaved {out_path_pick}')


# [Superseded] Homogeneous-Domain Back-Propagation Results

The cells below use the earlier homogeneous-domain back-propagation model
(dx=0.05 m, no explicit borehole material) -- **not** the borehole-geometry
pipeline (dx=0.02 m, explicit water-filled borehole) that the Region of Influence
Workflow section above now uses as the canonical `'backprop'` technique for new
displacement-estimation work.

Retained, not deleted: the Thesis Figure Compilation cells further below
(`@fig:fd-profile-grid`, `@fig:fd-disp-backprop`) call functions defined in this
block (`_load_bp_frame_wls`, `BP_FOCUS_IDX_OFFSET`) and depend on it staying
executable in its current position. The thesis chapter documents these figures'
back-propagation panels as deliberate pre-fix historical results -- do not delete
or move this block, and do not extend it with new profiles or analysis; use the
Region of Influence Workflow section above instead.


## Why Δz Estimation Is Harder for Back-Propagation

Three structural differences make the WLS phase fit less reliable for BP Ez focus
frames than for Kirchhoff/Gazdag B-scans.

### 1. Flat cross-spectrum amplitude — no natural band

Kirchhoff/Gazdag migrate energy from diffraction hyperbolas into localized
reflectors, but the image still contains a wavelet envelope spread over many rows.
This gives useful kz content at non-zero wavenumbers and natural amplitude lobes
that guide band selection.

The BP Ez frame is a *focused* field: energy collapses to a small spatial blob.
A localized blob has a near-flat cross-spectrum amplitude (the Fourier transform of
an approximate delta function is approximately flat), so the amplitude weighting
loses its discriminatory power and there are no clear lobes to restrict the band to.

### 2. Near-DC contamination

The focused peak has a large DC component ($k_z \approx 0$, $k_x \approx 0$) that
dominates the cross-spectrum amplitude. The `(|KZ| + |KX|) > 0` mask excludes the
exact origin, but the near-DC region still carries high amplitude and little phase
information — diluting the $\Delta z$ signal for the WLS.

### 3. Focus-quality sensitivity

If the snapshot offset is slightly off, the difference between two BP frames contains
*defocusing artifacts* on top of the spatial shift. Unlike a B-scan where defocus
simply blurs the image, a defocused BP frame has a non-trivial phase field: the
cross-spectrum phase is no longer a clean linear ramp and the WLS fit is pulled
toward a biased solution.

---

### Approach: Tight ROI + Reduced Amplitude Threshold (Option C)

We stay with the WLS method but improve conditioning by two complementary changes.

**Tighter ROI** — restrict the analysis window to the immediate neighbourhood of the
focus peak (≈ 2–3 m depth × 1 m radial). A smaller crop excludes background noise
that dilutes the cross-spectrum, raises the effective SNR of the phase information,
and reduces contamination from off-focus reflectors.

**Lower `BP_WLS_AMP_THR`** — reducing the threshold from 0.20 to 0.10 allows more
k-space cells to contribute to the WLS fit. Because the cross-spectrum amplitude is
nearly flat (see point 1), a high threshold would exclude most of the k-space,
leaving very few WLS constraints.

These two changes act in opposite directions in terms of included data: the tighter
ROI makes the cross-spectrum *cleaner* (less noise energy), while the lower threshold
*expands* the k-space mask on that cleaner spectrum. Together they keep the WLS mask
well-populated with cells that carry genuine phase information.

> **Practical note:** start with the comparison grid (`mig_bp_comparison` cell) to
> locate the focus peak for each pair, then centre `BP_MANUAL_ROIS` on that peak.
> A window of ±1.5 m in depth and ±0.5 m in radial is a reasonable starting point.


In [ ]:
# ── Manual pair selection + ROI definition — Back-propagation Ez (full WLS) ─────────────
#
# BP_MANUAL_PAIRS       — list of (run_a, run_b) int tuples (same numbering as Kirchhoff).
#
# BP_COMPARE_TO         — None  → use pairs as written
#                         int   → override run_a for every pair to this run number.
#
# BP_MANUAL_ROIS        — per-pair ROI (z_min_m, z_max_m, x_min_m, x_max_m).
#                         Depth/radial in gprMax physical coordinates (metres).
#
# BP_SHOW_DIAG          — True: plot 5-panel cross-spectrum diagnostic per pair.
# BP_FORCE_DZ_ZERO      — True: fit only Δx + φ₀.
# BP_ROI_THRESH         — envelope threshold for auto-ROI.
# BP_FOCUS_IDX_OFFSET   — snapshot index offset (must match snapshot cell).
# BP_KZ_BAND_FAC        — default kz band factor (overridden per-pair by MANUAL_KZ_BAND_FAC_BP).
# BP_KX_BAND_FAC        — kx band factor.
# BP_WLS_AMP_THR        — amplitude gate for WLS mask.
# MANUAL_KZ_BAND_FAC_BP — per-pair override of BP_KZ_BAND_FAC.
# ────────────────────────────────────────────────────────────────────────────────────

BP_MANUAL_PAIRS = [
    # (1,  2), (2,  3), ...   # uncomment for consecutive pairs
    (1,  3),
    (4,  8),
    (9,  20),
    (21, 38),
]

BP_COMPARE_TO    = 1      # all pairs vs prof_1 (first injection measurement)
BP_FORCE_DZ_ZERO = False
BP_ROI_THRESH    = 0.5
BP_SHOW_DIAG     = True
BP_FOCUS_IDX_OFFSET = 28    # must match value in the snapshot cell

# Tight ROIs centred on the focus peak.
# BP focus is localised (~1-2 m depth x ~1 m radial); wider ROIs dilute the
# cross-spectrum with background noise. Adjust after inspecting mig_bp_comparison.
BP_MANUAL_ROIS = {
    (1,  3):  (71, 72, 4.5, 5.5),
    (4,  8):  (70, 79, 4.5, 6.5),
    (9,  20): (70, 78, 5.0, 7.0),
    (21, 38): (70, 79, 4.5, 6.5),
}

BP_KZ_BAND_FAC = 0.5     # default; overridden per-pair by MANUAL_KZ_BAND_FAC_BP
BP_KX_BAND_FAC = 1.5     # kx band: |kx| < BP_KX_BAND_FAC * kz_c_bp
# Lowered from 0.20: BP cross-spectrum amplitude is nearly flat (focused blob
# ~ delta function in space), so a high threshold excludes most of k-space.
BP_WLS_AMP_THR = 0.10
MANUAL_KZ_BAND_FAC_BP = {
    (9,  20): 0.25,
    (21, 38): 0.15,
}

# ────────────────────────────────────────────────────────────────────────────────────

bp_wls_out = OUT_DIR / 'roi_phase' / 'backprop_wls'
bp_wls_out.mkdir(parents=True, exist_ok=True)

kz_c_bp = 2.0 * np.pi * f0_mig / (v / 2)    # half-velocity for gprMax snapshots

def _load_bp_frame_wls(run_num, focus_offset):
    """Load the focus-frame Ez array for integer run number."""
    slug = f'prof_{run_num}'
    snap_files = completed.get(slug)
    if not snap_files:
        return None
    import pyvista
    snap_times_ns = t_start_ns + np.arange(len(snap_files)) * snap_step * dt_ns_bp
    idx = min(int(np.argmin(np.abs(snap_times_ns - t_focus_ns))) + focus_offset,
              len(snap_files) - 1)
    mesh   = pyvista.read(str(snap_files[idx]))
    nx_c   = mesh.dimensions[0] - 1
    ny_c   = mesh.dimensions[1] - 1
    dx_m   = float(mesh.spacing[0])
    e_data = np.array(mesh['E-field'])
    ez     = e_data[:, 2].reshape(ny_c, nx_c).T      # (nx_c, ny_c): rows=depth, cols=radial
    depth_axis  = np.linspace(0, nx_c * dx_m, nx_c)
    radial_axis = np.linspace(0, ny_c * dx_m, ny_c)
    return ez, depth_axis, radial_axis, dx_m

def _phys_to_pix_bp_wls(depth_axis, radial_axis, d_min, d_max, r_min, r_max):
    """Physical ROI → pixel indices. depth_axis is INCREASING (row 0 = 0 m)."""
    rows = np.where((depth_axis  >= d_min) & (depth_axis  <= d_max))[0]
    cols = np.where((radial_axis >= r_min) & (radial_axis <= r_max))[0]
    if rows.size == 0 or cols.size == 0:
        raise ValueError(
            f'ROI ({d_min}–{d_max} m depth, {r_min}–{r_max} m radial) '
            f'outside grid depth=[0,{depth_axis[-1]:.1f}] radial=[0,{radial_axis[-1]:.1f}]'
        )
    return int(rows[0]), int(rows[-1]) + 1, int(cols[0]), int(cols[-1]) + 1

bp_wls_results = {}
for run_a, run_b in BP_MANUAL_PAIRS:
    ref_run = BP_COMPARE_TO if BP_COMPARE_TO is not None else run_a
    res_a = _load_bp_frame_wls(ref_run, BP_FOCUS_IDX_OFFSET)
    res_b = _load_bp_frame_wls(run_b,   BP_FOCUS_IDX_OFFSET)
    if res_a is None or res_b is None:
        missing = ref_run if res_a is None else run_b
        print(f'[{ref_run}→{run_b}] prof_{missing} not in completed — skipping.')
        continue

    ez_a, depth_ax, radial_ax, dx_m = res_a
    ez_b, _,        _,          _   = res_b

    nd = min(ez_a.shape[0], ez_b.shape[0])
    nr = min(ez_a.shape[1], ez_b.shape[1])
    ez_a, ez_b = ez_a[:nd, :nr], ez_b[:nd, :nr]
    depth_ax   = depth_ax[:nd]
    radial_ax  = radial_ax[:nr]
    diff = ez_b - ez_a
    env  = _monogenic_envelope(diff)

    roi_spec = BP_MANUAL_ROIS.get((run_a, run_b)) or BP_MANUAL_ROIS.get((ref_run, run_b))
    if roi_spec is not None:
        d0, d1, r0, r1 = _phys_to_pix_bp_wls(depth_ax, radial_ax, *roi_spec)
        roi_source = 'manual'
    else:
        d0, d1, r0, r1 = _roi_from_envelope(env, BP_ROI_THRESH)
        roi_source = f'auto (thresh={BP_ROI_THRESH})'

    d_roi_lo = float(depth_ax[d0]);   d_roi_hi = float(depth_ax[d1 - 1])
    r_roi_lo = float(radial_ax[r0]);  r_roi_hi = float(radial_ax[r1 - 1])

    # ── Difference + envelope figure ──────────────────────────────────────────────────────────────────
    extent_bp = [0, float(radial_ax[-1]), 0, float(depth_ax[-1])]
    vmax_d = np.percentile(np.abs(diff), 99.9)
    vmax_e = np.percentile(env, 99.9)

    fig, (ax_d, ax_e) = plt.subplots(1, 2, figsize=(14, 5))
    im_d = ax_d.imshow(diff, aspect='auto', cmap='RdBu_r',
                        extent=extent_bp, origin='lower', vmin=-vmax_d, vmax=vmax_d)
    plt.colorbar(im_d, ax=ax_d, label='ΔEz [V/m]')
    ax_d.invert_yaxis()
    ax_d.add_patch(Rectangle((r_roi_lo, d_roi_lo),
                              width=r_roi_hi - r_roi_lo, height=d_roi_hi - d_roi_lo,
                              lw=1.5, edgecolor='yellow', facecolor='none'))
    ax_d.xaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_d.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_d.grid(True, color='white', lw=0.3, alpha=0.4)
    ax_d.set_title(f'Back-prop ΔEz: prof_{run_b} − prof_{ref_run}  [{roi_source}]')
    ax_d.set_xlabel('Radial distance (m)'); ax_d.set_ylabel('Depth (m)')
    ax_d.set_ylim(62, 85)

    im_e = ax_e.imshow(env, aspect='auto', cmap='inferno',
                        extent=extent_bp, origin='lower', vmin=0, vmax=vmax_e)
    plt.colorbar(im_e, ax=ax_e, label='Monogenic envelope [a.u.]')
    ax_e.invert_yaxis()
    ax_e.add_patch(Rectangle((r_roi_lo, d_roi_lo),
                              width=r_roi_hi - r_roi_lo, height=d_roi_hi - d_roi_lo,
                              lw=1.5, edgecolor='cyan', facecolor='none'))
    ax_e.xaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_e.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_e.grid(True, color='white', lw=0.3, alpha=0.4)
    ax_e.set_title(f'Monogenic envelope  [{roi_source}]')
    ax_e.set_xlabel('Radial distance (m)'); ax_e.set_ylabel('Depth (m)')
    ax_e.set_ylim(62, 85)

    plt.tight_layout()
    fig.savefig(bp_wls_out / f'backprop_roi_{ref_run}_to_{run_b}.png', dpi=150)
    plt.show(); plt.close(fig)

    # ── WLS phase-plane fit (zero-padded for denser kx/kz sampling) ───────────────────────────
    base_crop = ez_a[d0:d1, r0:r1]
    mon_crop  = ez_b[d0:d1, r0:r1]
    Nz, Nx    = base_crop.shape
    pad_fac   = 10
    Nz_pad, Nx_pad = Nz * pad_fac, Nx * pad_fac

    dz_bp = dx_bp = dx_m    # uniform gprMax grid: same spacing in both axes
    kz_ax = np.fft.fftfreq(Nz_pad, d=dz_bp) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx_pad, d=dx_bp) * 2 * np.pi
    KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')
    taper  = np.outer(tukey(Nz, alpha=0.15), tukey(Nx, alpha=0.15))
    XS     = (np.fft.fft2(base_crop * taper, s=(Nz_pad, Nx_pad)) *
               np.conj(np.fft.fft2(mon_crop * taper, s=(Nz_pad, Nx_pad))))
    w      = np.abs(XS);  phi = np.angle(XS)
    _kz_fac = (MANUAL_KZ_BAND_FAC_BP.get((run_a, run_b))
               or MANUAL_KZ_BAND_FAC_BP.get((ref_run, run_b))
               or BP_KZ_BAND_FAC)
    band   = (np.abs(KZ) < _kz_fac * kz_c_bp) & (np.abs(KX) < BP_KX_BAND_FAC * kz_c_bp)
    mask   = (w > BP_WLS_AMP_THR * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0)

    n_mask = int(mask.sum())
    if n_mask < 3:
        print(f'[{ref_run}→{run_b}] WARNING: only {n_mask} pixels pass WLS mask '
              f'(crop {Nz}×{Nx} px, padded {Nz_pad}×{Nx_pad}).  ROI may be too small.')
        dz_est = dx_est = phi_0 = 0.0
    else:
        W = w[mask]
        if BP_FORCE_DZ_ZERO:
            A = np.column_stack([KX[mask], np.ones(n_mask)])
        else:
            A = np.column_stack([KZ[mask], KX[mask], np.ones(n_mask)])
        c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
        if BP_FORCE_DZ_ZERO:
            dz_est, dx_est, phi_0 = 0.0, float(c[0]), float(c[1])
        else:
            dz_est, dx_est, phi_0 = float(c[0]), float(c[1]), float(c[2])

    # ── Diagnostic: cross-spectrum phase + fitted plane ──────────────────────────────────
    if BP_SHOW_DIAG and n_mask >= 3:
        phi_shift    = np.fft.fftshift(phi)
        w_shift      = np.fft.fftshift(w)
        kz_disp      = np.fft.fftshift(kz_ax)
        kx_disp      = np.fft.fftshift(kx_ax)
        fitted       = KX * dx_est + KZ * dz_est + phi_0
        fitted_shift = np.fft.fftshift(fitted)

        # 1-D kz profile
        phi_1d  = np.zeros(Nz_pad)
        w_1d    = np.zeros(Nz_pad)
        band_kx = (kx_ax > 0) & (kx_ax < BP_KX_BAND_FAC * kz_c_bp)  # positive kx only
        for i_row in range(Nz_pad):
            sel = band_kx & (w[i_row, :] > BP_WLS_AMP_THR * w.max())
            if sel.sum() > 0:
                phi_1d[i_row] = np.average(phi[i_row, :][sel], weights=w[i_row, :][sel])
                w_1d[i_row]   = w[i_row, :][sel].sum()
        kz_1d_s  = np.fft.fftshift(kz_ax)
        phi_1d_s = np.fft.fftshift(phi_1d)
        w_1d_s   = np.fft.fftshift(w_1d)

        # 1-D kx profile
        phi_1d_kx   = np.zeros(Nx_pad)
        w_1d_kx     = np.zeros(Nx_pad)
        band_kz_fit = np.abs(kz_ax) < _kz_fac * kz_c_bp
        for j_col in range(Nx_pad):
            sel_kz = band_kz_fit & (w[:, j_col] > BP_WLS_AMP_THR * w.max())
            if sel_kz.sum() > 0:
                phi_1d_kx[j_col] = np.average(phi[:, j_col][sel_kz],
                                               weights=w[:, j_col][sel_kz])
                w_1d_kx[j_col]   = w[:, j_col][sel_kz].sum()
        kx_1d_s     = np.fft.fftshift(kx_ax)
        phi_1d_kx_s = np.fft.fftshift(phi_1d_kx)
        w_1d_kx_s   = np.fft.fftshift(w_1d_kx)

        fig_d, axes_d = plt.subplots(1, 5, figsize=(28, 4))
        band_shift = np.fft.fftshift(band)
        phi_masked = np.where(band_shift & (np.fft.fftshift(w) > BP_WLS_AMP_THR * w.max()),
                              phi_shift, np.nan)

        # Panel 1: cross-spectrum phase (full kx/kz shown; WLS uses kx>=0 only)
        im_ph = axes_d[0].imshow(phi_masked, aspect='auto', cmap='RdBu_r',
                                  extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]],
                                  vmin=-np.pi, vmax=np.pi, origin='upper')
        plt.colorbar(im_ph, ax=axes_d[0], label='phase [rad]')
        axes_d[0].set_title(f'Cross-spectrum phase  (mask: {n_mask} px)')
        axes_d[0].set_xlabel('kx [rad/m]'); axes_d[0].set_ylabel('kz [rad/m]')
        axes_d[0].set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3)
        axes_d[0].set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

        # Panel 2: cross-spectrum energy with WLS amplitude threshold contour
        w_plot = np.where(band_shift, w_shift, np.nan)
        im_en = axes_d[1].imshow(w_plot, aspect='auto', cmap='inferno',
                                  extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]],
                                  origin='upper')
        plt.colorbar(im_en, ax=axes_d[1], label='|XS| [a.u.]')
        axes_d[1].contour(kx_disp, kz_disp, w_shift,
                          levels=[BP_WLS_AMP_THR * w.max()], colors='cyan', linewidths=0.8)
        axes_d[1].set_title(f'Energy  (thr={BP_WLS_AMP_THR:.2f}×max — cyan contour)')
        axes_d[1].set_xlabel('kx [rad/m]'); axes_d[1].set_ylabel('kz [rad/m]')
        axes_d[1].set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3)
        axes_d[1].set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

        # Panel 3: fitted plane
        fitted_masked = np.where(band_shift, fitted_shift, np.nan)
        im_fit = axes_d[2].imshow(fitted_masked, aspect='auto', cmap='RdBu_r',
                                   extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]],
                                   vmin=-np.pi, vmax=np.pi, origin='upper')
        plt.colorbar(im_fit, ax=axes_d[2], label='phase [rad]')
        axes_d[2].set_title(f'Fitted plane  Δz={dz_est:+.4f} m  Δx={dx_est:+.4f} m  φ₀={phi_0:+.3f} rad')
        axes_d[2].set_xlabel('kx [rad/m]'); axes_d[2].set_ylabel('kz [rad/m]')
        axes_d[2].set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3)
        axes_d[2].set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

        # Panel 4: 1-D kz slice — measured vs fitted
        in_band = np.abs(kz_1d_s) < _kz_fac * kz_c_bp
        sc_kz = axes_d[3].scatter(kz_1d_s[in_band & (w_1d_s > 0)],
                          phi_1d_s[in_band & (w_1d_s > 0)],
                          c=w_1d_s[in_band & (w_1d_s > 0)],
                          cmap='viridis', s=20, label='measured (weighted)')
        plt.colorbar(sc_kz, ax=axes_d[3], label='weight [a.u.]')
        kz_fit_ax = kz_1d_s[in_band]
        axes_d[3].plot(kz_fit_ax, kz_fit_ax * dz_est + phi_0, 'r-', lw=1.5, label='fitted slope')
        axes_d[3].axhline(0, color='k', lw=0.5, ls='--')
        axes_d[3].set_xlabel('kz [rad/m]'); axes_d[3].set_ylabel('phase [rad]')
        axes_d[3].set_title('1-D kz phase profile (kx>0 only, both kz±)')
        axes_d[3].legend(fontsize=8)

        # Panel 5: 1-D kx slice — measured vs fitted
        in_band_kx = (kx_1d_s > 0) & (kx_1d_s < BP_KX_BAND_FAC * kz_c_bp)  # positive kx only
        sc_kx = axes_d[4].scatter(kx_1d_s[in_band_kx & (w_1d_kx_s > 0)],
                          phi_1d_kx_s[in_band_kx & (w_1d_kx_s > 0)],
                          c=w_1d_kx_s[in_band_kx & (w_1d_kx_s > 0)],
                          cmap='viridis', s=20, label='measured (weighted)')
        plt.colorbar(sc_kx, ax=axes_d[4], label='weight [a.u.]')
        kx_fit_ax = kx_1d_s[in_band_kx]
        axes_d[4].plot(kx_fit_ax, kx_fit_ax * dx_est + phi_0, 'r-', lw=1.5, label='fitted slope')
        axes_d[4].axhline(0, color='k', lw=0.5, ls='--')
        axes_d[4].set_xlabel('kx [rad/m]'); axes_d[4].set_ylabel('phase [rad]')
        axes_d[4].set_title('1-D kx phase profile (positive kx only, kz± collapsed)')
        axes_d[4].legend(fontsize=8)

        plt.suptitle(f'Diagnostic (back-prop): prof_{ref_run} → prof_{run_b}', y=1.02)
        plt.tight_layout()
        fig_d.savefig(bp_wls_out / f'backprop_diag_{ref_run}_to_{run_b}.png',
                      dpi=150, bbox_inches='tight')
        plt.show(); plt.close(fig_d)

    bp_wls_results[(ref_run, run_b)] = {
        'dx': dx_est, 'dz': dz_est, 'phi_0': phi_0,
        'roi_m':  (d_roi_lo, d_roi_hi, r_roi_lo, r_roi_hi),
        'roi_px': (d0, d1, r0, r1),
        'n_mask': n_mask, 'source': roi_source,
    }
    print(f'prof_{ref_run}→{run_b}  [{roi_source}]:  '
          f'Δx={dx_est:+.4f} m  Δz={dz_est:+.4f} m  φ₀={phi_0:+.4f} rad  '
          f'mask={n_mask} px  crop={Nz}×{Nx} px  '
          f'ROI depth=[{d_roi_lo:.1f},{d_roi_hi:.1f}] m  '
          f'radial=[{r_roi_lo:.2f},{r_roi_hi:.2f}] m')

# ── Summary plot ─────────────────────────────────────────────────────────────────────────────────────
if bp_wls_results:
    pair_labels = [f'{a}→{b}' for a, b in bp_wls_results]
    dx_vals = [bp_wls_results[k]['dx'] for k in bp_wls_results]
    dz_vals = [bp_wls_results[k]['dz'] for k in bp_wls_results]

    fig2, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
    from matplotlib.patches import Patch as _Patch
    _PHASE_DEFS = [('Pushing', 1, 3, '#aed6f1'), ('Chasing', 4, 8, '#a9dfbf'),
                   ('Waiting', 9, 20, '#f9e79f'), ('Pulling', 21, 38, '#f1948a')]
    _rn_col = {n: c for _nm, lo, hi, c in _PHASE_DEFS for n in range(lo, hi + 1)}
    _pkeys  = list(bp_wls_results.keys())
    for _bax in (ax1, ax2):
        for _i, _k in enumerate(_pkeys):
            _c = _rn_col.get(_k[1])
            if _c:
                _bax.axvspan(_i - 0.5, _i + 0.5, color=_c, alpha=0.3, zorder=0, lw=0)
    _ph_hdl = [_Patch(facecolor=c, alpha=0.6, label=nm, edgecolor='grey', lw=0.5)
               for nm, lo, hi, c in _PHASE_DEFS
               if any(_rn_col.get(_k[1]) == c for _k in _pkeys)]
    ax1.legend(handles=_ph_hdl, loc='best', fontsize=8, framealpha=0.7)
    ax1.plot(dx_vals, 'o-', color='steelblue')
    ax1.axhline(0, color='k', lw=0.6, ls='--')
    ax1.set_ylabel('Δx (radial) [m]')
    mode_str = f'vs prof_{BP_COMPARE_TO}' if BP_COMPARE_TO is not None else 'stage pairs'
    ax1.set_title(f'WLS fluid-front shift — Back-propagation ({mode_str})')
    ax2.plot(dz_vals, '^-', color='seagreen')
    ax2.axhline(0, color='k', lw=0.6, ls='--')
    ax2.set_ylabel('Δz (depth) [m]')
    ax2.set_xticks(range(len(pair_labels)))
    ax2.set_xticklabels(pair_labels, rotation=30, ha='right')
    plt.tight_layout()
    fig2.savefig(bp_wls_out / 'backprop_wls_summary.png', dpi=150)
    plt.show(); plt.close(fig2)

print(f'\nDone — {len(bp_wls_results)} pair(s).  Output: {bp_wls_out}')


In [ ]:
# ── Manual pair selection + ROI definition — Back-propagation Ez focus frames ────────
#
# MANUAL_PAIRS_BP  — list of (slug_a, slug_b) string tuples from the `completed` dict.
#                    e.g. ('prof_1', 'prof_4').
#
# MANUAL_ROIS_BP   — per-pair ROI in gprMax physical coordinates (metres):
#                      (slug_a, slug_b): (depth_min_m, depth_max_m, radial_min_m, radial_max_m)
#                    depth_min/max refer to the gprMax x-axis (borehole source positions,
#                    0 → 86 m; sources sit between ~62 m and 85 m in this dataset).
#                    radial_min/max refer to the gprMax y-axis (0 → ~14 m from borehole).
#                    Set a pair's value to None to auto-detect from the monogenic envelope.
#
# FOCUS_IDX_OFFSET_BP — snapshot index offset, must match the value in the snapshot cell.
# BP_FORCE_DZ_ZERO    — True: fit only Δx + φ₀ (fluid front is lateral only).
# BP_ROI_THRESH       — envelope threshold for auto-detected pairs (0–1).
# ─────────────────────────────────────────────────────────────────────────────────────

MANUAL_PAIRS_BP = [
    ('prof_1',  'prof_2'),
    ('prof_2',  'prof_3'),
    ('prof_3',  'prof_4'),
    ('prof_4', 'prof_5'),
    ('prof_5', 'prof_7'),
    ('prof_7', 'prof_8'),
    ('prof_8', 'prof_9'),
    ('prof_9', 'prof_10')
]

MANUAL_ROIS_BP = {
    ('prof_1',  'prof_2'): (70, 78, 5.0, 6.5),
    ('prof_2',  'prof_3'): (70, 78, 5.0, 6.5),
    ('prof_3',  'prof_4'): (70, 78, 5.0, 6.5),
    ('prof_4', 'prof_5'): (70, 78, 5.0, 6.5),
    ('prof_5', 'prof_7'): (70, 78, 5.0, 6.5),
    ('prof_7', 'prof_8'): (70, 78, 5.0, 6.5),
    ('prof_8', 'prof_9'): (70, 78, 5.0, 6.5),
    ('prof_9', 'prof_10'): (70, 78, 5.0, 6.5)
}

FOCUS_IDX_OFFSET_BP = 26    # must match offset used in the snapshot cell above
BP_FORCE_DZ_ZERO    = True
BP_ROI_THRESH       = 0.5

# ─────────────────────────────────────────────────────────────────────────────────────

bp_roi_out = OUT_DIR / 'roi_phase' / 'backprop'
bp_roi_out.mkdir(parents=True, exist_ok=True)

def _load_bp_ez(slug):
    """Load the focus-frame Ez array for a given backprop slug.
    Returns (ez, t_actual_ns, depth_axis_m, radial_axis_m, dx_m).
    """
    snap_files = completed.get(slug)
    if not snap_files:
        return None
    import pyvista
    snap_times_ns = t_start_ns + np.arange(len(snap_files)) * snap_step * dt_ns_bp
    idx = min(int(np.argmin(np.abs(snap_times_ns - t_focus_ns))) + FOCUS_IDX_OFFSET_BP,
              len(snap_files) - 1)
    mesh   = pyvista.read(str(snap_files[idx]))
    nx_c   = mesh.dimensions[0] - 1
    ny_c   = mesh.dimensions[1] - 1
    dx_m   = float(mesh.spacing[0])
    e_data = np.array(mesh['E-field'])
    ez     = e_data[:, 2].reshape(ny_c, nx_c).T      # (nx_c, ny_c): rows=depth, cols=radial
    depth_axis  = np.linspace(0, nx_c * dx_m, nx_c)  # gprMax x: 0 → domain_x [m]
    radial_axis = np.linspace(0, ny_c * dx_m, ny_c)  # gprMax y: 0 → domain_y [m]
    return ez, snap_times_ns[idx], depth_axis, radial_axis, dx_m

def _phys_to_pix_bp(depth_axis, radial_axis, d_min, d_max, r_min, r_max):
    """Convert physical-coordinate ROI to pixel indices for the backprop Ez array.
    depth_axis is INCREASING (row 0 = 0 m, last row = domain_x ≈ 86 m).
    Returns (d0, d1, r0, r1) as exclusive-end integer indices.
    """
    row_mask = (depth_axis  >= d_min) & (depth_axis  <= d_max)
    col_mask = (radial_axis >= r_min) & (radial_axis <= r_max)
    rows = np.where(row_mask)[0]
    cols = np.where(col_mask)[0]
    if rows.size == 0 or cols.size == 0:
        raise ValueError(
            f'ROI ({d_min}–{d_max} m depth, {r_min}–{r_max} m radial) '
            f'is outside grid  depth=[0, {depth_axis[-1]:.1f}] m  '
            f'radial=[0, {radial_axis[-1]:.1f}] m'
        )
    return int(rows[0]), int(rows[-1]) + 1, int(cols[0]), int(cols[-1]) + 1

bp_results = {}
for slug_a, slug_b in MANUAL_PAIRS_BP:
    res_a = _load_bp_ez(slug_a)
    res_b = _load_bp_ez(slug_b)
    if res_a is None or res_b is None:
        missing = slug_a if res_a is None else slug_b
        print(f'[{slug_a}→{slug_b}] {missing} not in completed dict — skipping.')
        continue

    ez_a, t_a, depth_ax, radial_ax, dx_m = res_a
    ez_b, t_b, _,        _,         _    = res_b

    # Align shapes (should be identical, but guard against edge cases)
    nd = min(ez_a.shape[0], ez_b.shape[0])
    nr = min(ez_a.shape[1], ez_b.shape[1])
    ez_a, ez_b   = ez_a[:nd, :nr], ez_b[:nd, :nr]
    depth_ax     = depth_ax[:nd]
    radial_ax    = radial_ax[:nr]
    diff = ez_b - ez_a
    env  = _monogenic_envelope(diff)

    # Resolve ROI
    roi_spec = MANUAL_ROIS_BP.get((slug_a, slug_b), None)
    if roi_spec is not None:
        d_min, d_max, r_min, r_max = roi_spec
        d0, d1, r0, r1 = _phys_to_pix_bp(depth_ax, radial_ax, d_min, d_max, r_min, r_max)
        roi_source = 'manual'
    else:
        d0, d1, r0, r1 = _roi_from_envelope(env, BP_ROI_THRESH)
        roi_source = f'auto (thresh={BP_ROI_THRESH})'

    d_roi_lo = float(depth_ax[d0])
    d_roi_hi = float(depth_ax[d1 - 1])
    r_roi_lo = float(radial_ax[r0])
    r_roi_hi = float(radial_ax[r1 - 1])

    # extent convention: [x_min, x_max, y_min, y_max] with origin='lower' + invert_yaxis
    # → x = radial, y = depth with 0 at top after invert
    extent = [0, float(radial_ax[-1]), 0, float(depth_ax[-1])]
    vmax_d = np.percentile(np.abs(diff), 98)
    vmax_e = np.percentile(env, 98)

    fig, (ax_d, ax_e) = plt.subplots(1, 2, figsize=(14, 5))

    im_d = ax_d.imshow(diff, aspect='auto', cmap='RdBu_r',
                        extent=extent, origin='lower',
                        vmin=-vmax_d, vmax=vmax_d)
    plt.colorbar(im_d, ax=ax_d, label='ΔEz [V/m]')
    ax_d.invert_yaxis()
    ax_d.add_patch(Rectangle((r_roi_lo, d_roi_lo),
                              width=r_roi_hi - r_roi_lo,
                              height=d_roi_hi - d_roi_lo,
                              lw=1.5, edgecolor='yellow', facecolor='none'))
    ax_d.xaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_d.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_d.grid(True, color='white', linewidth=0.3, alpha=0.4)
    ax_d.set_title(f'Back-prop ΔEz: {slug_b} − {slug_a}  [{roi_source}]')
    ax_d.set_xlabel('Radial distance (m)')
    ax_d.set_ylabel('Depth (m)')

    im_e = ax_e.imshow(env, aspect='auto', cmap='inferno',
                        extent=extent, origin='lower',
                        vmin=0, vmax=vmax_e)
    plt.colorbar(im_e, ax=ax_e, label='Monogenic envelope [a.u.]')
    ax_e.invert_yaxis()
    ax_e.add_patch(Rectangle((r_roi_lo, d_roi_lo),
                              width=r_roi_hi - r_roi_lo,
                              height=d_roi_hi - d_roi_lo,
                              lw=1.5, edgecolor='cyan', facecolor='none'))
    ax_e.xaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_e.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
    ax_e.grid(True, color='white', linewidth=0.3, alpha=0.4)
    ax_e.set_title(f'Monogenic envelope  [{roi_source}]')
    ax_e.set_xlabel('Radial distance (m)')
    ax_e.set_ylabel('Depth (m)')

    plt.tight_layout()
    fname = f'bp_roi_{slug_a}_to_{slug_b}.png'
    fig.savefig(bp_roi_out / fname, dpi=150)
    plt.show()
    plt.close(fig)

    # WLS phase-plane fit — uniform grid so dz_g = dx_g = dx_m
    # kz_c uses half-velocity (eps_r * 4 → v_half = v/2) since gprMax runs at v/2
    kz_c_bp = 2.0 * np.pi * f0_mig / (v / 2)
    base_crop = ez_a[d0:d1, r0:r1]
    mon_crop  = ez_b[d0:d1, r0:r1]
    dz_est, dx_est, phi_0 = _estimate_shift_2d(
        base_crop, mon_crop, dx_m, dx_m, kz_c_bp, force_dz_zero=False)

    bp_results[(slug_a, slug_b)] = {
        'dx': dx_est, 'dz': dz_est, 'phi_0': phi_0,
        'roi_m':  (d_roi_lo, d_roi_hi, r_roi_lo, r_roi_hi),
        'roi_px': (d0, d1, r0, r1),
        'source': roi_source,
    }
    print(f'{slug_a}→{slug_b}  [{roi_source}]:  '
          f'Δx={dx_est:+.4f} m  Δz={dz_est:+.4f} m  φ₀={phi_0:+.4f} rad  '
          f'ROI depth=[{d_roi_lo:.1f}, {d_roi_hi:.1f}] m  '
          f'radial=[{r_roi_lo:.2f}, {r_roi_hi:.2f}] m  '
          f'size={d1-d0}×{r1-r0} px')

# ── Summary plot ─────────────────────────────────────────────────────────────────────
if bp_results:
    pair_labels = [f'{a.split("_")[1]}→{b.split("_")[1]}' for a, b in bp_results]
    dx_vals  = [bp_results[k]['dx']    for k in bp_results]
    phi_vals = [bp_results[k]['phi_0'] for k in bp_results]

    fig2, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    dz_vals = [bp_results[k]['dz'] for k in bp_results]
    from matplotlib.patches import Patch as _Patch
    _PHASE_DEFS = [('Pushing', 1, 3, '#aed6f1'), ('Chasing', 4, 8, '#a9dfbf'),
                   ('Waiting', 9, 20, '#f9e79f'), ('Pulling', 21, 38, '#f1948a')]
    _rn_col = {n: c for _nm, lo, hi, c in _PHASE_DEFS for n in range(lo, hi + 1)}
    _pkeys  = list(bp_results.keys())
    for _bax in (ax1, ax2, ax3):
        for _i, _k in enumerate(_pkeys):
            _c = _rn_col.get(int(_k[1].split('_')[1]))
            if _c:
                _bax.axvspan(_i - 0.5, _i + 0.5, color=_c, alpha=0.3, zorder=0, lw=0)
    _ph_hdl = [_Patch(facecolor=c, alpha=0.6, label=nm, edgecolor='grey', lw=0.5)
               for nm, lo, hi, c in _PHASE_DEFS
               if any(_rn_col.get(int(_k[1].split('_')[1])) == c for _k in _pkeys)]
    ax1.legend(handles=_ph_hdl, loc='best', fontsize=8, framealpha=0.7)
    ax1.plot(dx_vals, 'o-', color='steelblue')
    ax1.axhline(0, color='k', lw=0.6, ls='--')
    ax1.set_ylabel('Δx (radial) [m]')
    ax1.set_title('Localised WLS fluid-front shift — Back-propagation Ez')
    ax2.plot(dz_vals, '^-', color='seagreen')
    ax2.axhline(0, color='k', lw=0.6, ls='--')
    ax2.set_ylabel('Δz (depth) [m]')
    ax3.plot(np.degrees(phi_vals), 's-', color='darkorange')
    ax3.axhline(0, color='k', lw=0.6, ls='--')
    ax3.set_ylabel('φ₀ [°]')
    ax3.set_xlabel('Pair')
    ax3.set_xticks(range(len(pair_labels)))
    ax3.set_xticklabels(pair_labels, rotation=30, ha='right')
    plt.tight_layout()
    fig2.savefig(bp_roi_out / 'bp_dx_summary.png', dpi=150)
    plt.show()
    plt.close(fig2)

print(f'\nDone — {len(bp_results)} pair(s) analysed.  Output: {bp_roi_out}')

# Fluid Front Movement Estimate:

## 1\. Consecutive Profiles ($1 \rightarrow 2, 2 \rightarrow 3, \dots, 37 \rightarrow 38$)

Pushing phase: profiles 1-3

Chasing phase: profiles 4-8

Waiting phase: profiles 9-20

Pulling phase: profiles 21-38

In [ ]:
# ── Strategy 1: Consecutive Profiles (n → n+1) ───────────────────────────────────
# Pro: δx ≪ λ/4 ⇒ phase always in [−π, +π], high waveform correlation.
# Con: integrating 37 independent noise terms ⇒ random-walk drift ∝ √N.
# Summary shows CUMULATIVE displacement (sum of incremental δx values).
# ─────────────────────────────────────────────────────────────────────────────────────

S1_ROI          = (70, 79, 4.5, 7.5)   # [z_min, z_max, r_min, r_max] m
S1_METHOD       = 'gazdag'
S1_FORCE_DZ_ZERO = False

pairs_s1   = [(n, n + 1) for n in range(1, 38)]
results_s1 = {}

for _ra, _rb in pairs_s1:
    _a = _load_img(S1_METHOD, _ra);  _b = _load_img(S1_METHOD, _rb)
    if _a is None or _b is None:
        print(f'[{_ra}->{_rb}] missing .npy — skipped')
        continue
    _n = min(_a.shape[0], _b.shape[0])
    _a, _b = _a[:_n], _b[:_n]
    _z0, _z1, _x0, _x1 = _phys_to_pix(depth[:_n], x_img, *S1_ROI)
    _dz, _dx, _phi = _estimate_shift_2d(
        _a[_z0:_z1, _x0:_x1], _b[_z0:_z1, _x0:_x1],
        dz_g, dx_g, kz_c, force_dz_zero=S1_FORCE_DZ_ZERO)
    results_s1[(_ra, _rb)] = {'dx': _dx, 'dz': _dz, 'phi_0': _phi}
    print(f'[{_ra}->{_rb}]  dx={_dx:+.4f} m  dz={_dz:+.4f} m  phi0={_phi:+.4f} rad')

# ── Summary: cumulative (integrated) displacement ─────────────────────────────────
if results_s1:
    from matplotlib.patches import Patch as _Patch
    _PHASE_DEFS = [('Pushing', 1, 3, '#aed6f1'), ('Chasing', 4, 8, '#a9dfbf'),
                   ('Waiting', 9, 20, '#f9e79f'), ('Pulling', 21, 38, '#f1948a')]
    _rn_col = {n: c for _nm, lo, hi, c in _PHASE_DEFS for n in range(lo, hi + 1)}
    _pkeys = list(results_s1.keys())
    dx_cum = np.cumsum([results_s1[k]['dx'] for k in _pkeys])
    dz_cum = np.cumsum([results_s1[k]['dz'] for k in _pkeys])
    phi_v  = [results_s1[k]['phi_0'] for k in _pkeys]
    labels = [f'{a}->{b}' for a, b in _pkeys]

    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
    for _bax in (ax1, ax2, ax3):
        for _i, _k in enumerate(_pkeys):
            _c = _rn_col.get(_k[1])
            if _c:
                _bax.axvspan(_i - 0.5, _i + 0.5, color=_c, alpha=0.3, zorder=0, lw=0)
    _ph_hdl = [_Patch(facecolor=c, alpha=0.6, label=nm, edgecolor='grey', lw=0.5)
               for nm, lo, hi, c in _PHASE_DEFS if any(_rn_col.get(_k[1]) == c for _k in _pkeys)]
    ax1.plot(dx_cum, 'o-', color='steelblue')
    ax1.axhline(0, color='k', lw=0.6, ls='--')
    ax1.set_ylabel('Cumulative Δx (radial) [m]')
    ax1.set_title(f'Strategy 1: Consecutive  [{S1_METHOD}]  (σ_drift ∝ √N)')
    ax1.legend(handles=_ph_hdl, loc='best', fontsize=8)
    ax2.plot(dz_cum, '^-', color='seagreen')
    ax2.axhline(0, color='k', lw=0.6, ls='--')
    ax2.set_ylabel('Cumulative Δz (depth) [m]')
    ax3.plot(np.degrees(phi_v), 's-', color='darkorange')
    ax3.axhline(0, color='k', lw=0.6, ls='--')
    ax3.set_ylabel('φ₀ [°]')
    ax3.set_xlabel('Pair')
    ax3.set_xticks(range(len(labels)))
    ax3.set_xticklabels(labels, rotation=45, ha='right', fontsize=6)
    plt.tight_layout()
    fig.savefig(OUT_DIR / 'roi_phase' / 'strat1_consecutive_summary.png', dpi=150)
    save_fig(fig, 'strat1_consecutive_summary', study='FieldData_Study', prefix='FD_')
    plt.show();  plt.close(fig)


## 2\. Fixed Global Baseline ($1 \rightarrow 2, 1 \rightarrow 3, \dots, 1 \rightarrow 38$)

In [ ]:
# ── Strategy 2: Fixed Global Baseline (1 → n for all n) ─────────────────────────
# Pro: independent estimates ⇒ zero error accumulation.
# Con: phase wrapping when net shift > λ/4; waveform decorrelation in late profiles.
# Summary shows DIRECT displacement relative to the pre-injection baseline (Profile 1).
# ─────────────────────────────────────────────────────────────────────────────────────

S2_ROI          = (70, 79, 4.5, 7.5)
S2_METHOD       = 'gazdag'
S2_FORCE_DZ_ZERO = False

pairs_s2   = [(1, n) for n in range(2, 39)]
results_s2 = {}

for _ra, _rb in pairs_s2:
    _a = _load_img(S2_METHOD, _ra);  _b = _load_img(S2_METHOD, _rb)
    if _a is None or _b is None:
        print(f'[{_ra}->{_rb}] missing .npy — skipped')
        continue
    _n = min(_a.shape[0], _b.shape[0])
    _a, _b = _a[:_n], _b[:_n]
    _z0, _z1, _x0, _x1 = _phys_to_pix(depth[:_n], x_img, *S2_ROI)
    _dz, _dx, _phi = _estimate_shift_2d(
        _a[_z0:_z1, _x0:_x1], _b[_z0:_z1, _x0:_x1],
        dz_g, dx_g, kz_c, force_dz_zero=S2_FORCE_DZ_ZERO)
    results_s2[(_ra, _rb)] = {'dx': _dx, 'dz': _dz, 'phi_0': _phi}
    print(f'[{_ra}->{_rb}]  dx={_dx:+.4f} m  dz={_dz:+.4f} m  phi0={_phi:+.4f} rad')

# ── Summary: direct displacement from profile 1 ───────────────────────────────────
if results_s2:
    from matplotlib.patches import Patch as _Patch
    _PHASE_DEFS = [('Pushing', 1, 3, '#aed6f1'), ('Chasing', 4, 8, '#a9dfbf'),
                   ('Waiting', 9, 20, '#f9e79f'), ('Pulling', 21, 38, '#f1948a')]
    _rn_col = {n: c for _nm, lo, hi, c in _PHASE_DEFS for n in range(lo, hi + 1)}
    _pkeys = list(results_s2.keys())
    dx_v   = [results_s2[k]['dx'] for k in _pkeys]
    dz_v   = [results_s2[k]['dz'] for k in _pkeys]
    phi_v  = [results_s2[k]['phi_0'] for k in _pkeys]
    labels = [f'1->{b}' for _, b in _pkeys]

    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
    for _bax in (ax1, ax2, ax3):
        for _i, _k in enumerate(_pkeys):
            _c = _rn_col.get(_k[1])
            if _c:
                _bax.axvspan(_i - 0.5, _i + 0.5, color=_c, alpha=0.3, zorder=0, lw=0)
    _ph_hdl = [_Patch(facecolor=c, alpha=0.6, label=nm, edgecolor='grey', lw=0.5)
               for nm, lo, hi, c in _PHASE_DEFS if any(_rn_col.get(_k[1]) == c for _k in _pkeys)]
    ax1.plot(dx_v, 'o-', color='steelblue')
    ax1.axhline(0, color='k', lw=0.6, ls='--')
    ax1.set_ylabel('Δx from baseline (radial) [m]')
    ax1.set_title(f'Strategy 2: Fixed Baseline (1→n)  [{S2_METHOD}]  (wrapping risk at large shifts)')
    ax1.legend(handles=_ph_hdl, loc='best', fontsize=8)
    ax2.plot(dz_v, '^-', color='seagreen')
    ax2.axhline(0, color='k', lw=0.6, ls='--')
    ax2.set_ylabel('Δz from baseline (depth) [m]')
    ax3.plot(np.degrees(phi_v), 's-', color='darkorange')
    ax3.axhline(0, color='k', lw=0.6, ls='--')
    ax3.set_ylabel('φ₀ [°]')
    ax3.set_xlabel('Profile')
    ax3.set_xticks(range(len(labels)))
    ax3.set_xticklabels(labels, rotation=45, ha='right', fontsize=6)
    plt.tight_layout()
    fig.savefig(OUT_DIR / 'roi_phase' / 'strat2_baseline_summary.png', dpi=150)
    save_fig(fig, 'strat2_baseline_summary', study='FieldData_Study', prefix='FD_')
    plt.show();  plt.close(fig)


## 3\. Stage-Anchored Tracking ($1 \rightarrow 4$ \[Push\], $5 \rightarrow 9$ \[Chase\], $10 \rightarrow 20$ \[Wait\], etc.)

In [ ]:
# ── Strategy 3: Stage-Anchored Hybrid — RECOMMENDED ─────────────────────────────────
# Reference reset at the start of each hydraulic stage:
#   Pushing  (profs 2–4 vs 1)    — sub-wavelength increments, zero wrapping
#   Chasing  (profs 6–9 vs 5)    — controlled drift, stage-isolated physics
#   Waiting  (profs 11–20 vs 10) — diffusion-dominated; ref reset prevents
#                                   decorrelation bleed-through from push phase
#   Pulling  (profs 22–38 vs 21) — symmetric to push, separate drift budget
# Kinematic chaining: boundary pairs (1→5, 5→10, 10→21) bridge stage offsets
# so all intra-stage tracks are stitched into one global trajectory X_total(t).
# ─────────────────────────────────────────────────────────────────────────────────────

S3_ROI          = (70, 79, 4.5, 7.5)
S3_METHOD       = 'gazdag'
S3_FORCE_DZ_ZERO = False

# Intra-stage pairs (local reference → profile)
_push_pairs  = [(1,  n) for n in range(2, 5)]    # ref=1: profs 2,3,4
_chase_pairs = [(5,  n) for n in range(6, 10)]   # ref=5: profs 6,7,8,9
_wait_pairs  = [(10, n) for n in range(11, 21)]  # ref=10: profs 11–20
_pull_pairs  = [(21, n) for n in range(22, 39)]  # ref=21: profs 22–38
# Inter-stage boundary pairs (bridge stage transitions)
_boundary_pairs = [(1, 5), (5, 10), (10, 21)]

results_s3 = {}
for _ra, _rb in _push_pairs + _chase_pairs + _wait_pairs + _pull_pairs + _boundary_pairs:
    _a = _load_img(S3_METHOD, _ra);  _b = _load_img(S3_METHOD, _rb)
    if _a is None or _b is None:
        print(f'[{_ra}->{_rb}] missing .npy — skipped')
        continue
    _n = min(_a.shape[0], _b.shape[0])
    _a, _b = _a[:_n], _b[:_n]
    _z0, _z1, _x0, _x1 = _phys_to_pix(depth[:_n], x_img, *S3_ROI)
    _dz, _dx, _phi = _estimate_shift_2d(
        _a[_z0:_z1, _x0:_x1], _b[_z0:_z1, _x0:_x1],
        dz_g, dx_g, kz_c, force_dz_zero=S3_FORCE_DZ_ZERO)
    results_s3[(_ra, _rb)] = {'dx': _dx, 'dz': _dz, 'phi_0': _phi}
    tag = ' [boundary]' if (_ra, _rb) in _boundary_pairs else ''
    print(f'[{_ra}->{_rb}]{tag}  dx={_dx:+.4f} m  dz={_dz:+.4f} m  phi0={_phi:+.4f} rad')

# ── Kinematic chaining: add boundary offsets to intra-stage tracks ──────────────
def _bnd(key, comp):
    return results_s3.get(key, {}).get(comp, 0.0)

# Cumulative boundary offsets: push → chase → wait → pull
_off_dx = [0.0,
            _bnd((1, 5), 'dx'),
            _bnd((1, 5), 'dx') + _bnd((5, 10), 'dx'),
            _bnd((1, 5), 'dx') + _bnd((5, 10), 'dx') + _bnd((10, 21), 'dx')]
_off_dz = [0.0,
            _bnd((1, 5), 'dz'),
            _bnd((1, 5), 'dz') + _bnd((5, 10), 'dz'),
            _bnd((1, 5), 'dz') + _bnd((5, 10), 'dz') + _bnd((10, 21), 'dz')]

global_prof, global_dx, global_dz, global_phi, global_col = [], [], [], [], []
from matplotlib.patches import Patch as _Patch
_PHASE_DEFS = [('Pushing', 1, 3, '#aed6f1'), ('Chasing', 4, 8, '#a9dfbf'),
               ('Waiting', 9, 20, '#f9e79f'), ('Pulling', 21, 38, '#f1948a')]
_rn_col = {n: c for _nm, lo, hi, c in _PHASE_DEFS for n in range(lo, hi + 1)}

for _stage_pairs, _odx, _odz in zip(
        [_push_pairs, _chase_pairs, _wait_pairs, _pull_pairs],
        _off_dx, _off_dz):
    for _k in _stage_pairs:
        if _k not in results_s3:
            continue
        global_prof.append(_k[1])
        global_dx.append(_odx + results_s3[_k]['dx'])
        global_dz.append(_odz + results_s3[_k]['dz'])
        global_phi.append(results_s3[_k]['phi_0'])
        global_col.append(_rn_col.get(_k[1], '#cccccc'))

# ── Summary: chained global trajectory ────────────────────────────────────────────
if global_dx:
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
    for _bax in (ax1, ax2, ax3):
        for _i, _c in enumerate(global_col):
            _bax.axvspan(_i - 0.5, _i + 0.5, color=_c, alpha=0.3, zorder=0, lw=0)
        # Dotted vertical lines at stage-boundary transitions
        for _b_prof in (5, 10, 21):
            _bi = [_j for _j, p in enumerate(global_prof) if p == _b_prof]
            if _bi:
                _bax.axvline(_bi[0] - 0.5, color='k', lw=1.0, ls=':', zorder=1)
    _ph_hdl = [_Patch(facecolor=c, alpha=0.6, label=nm, edgecolor='grey', lw=0.5)
               for nm, lo, hi, c in _PHASE_DEFS]
    ax1.plot(global_dx, 'o-', color='steelblue')
    ax1.axhline(0, color='k', lw=0.6, ls='--')
    ax1.set_ylabel('Global Δx (radial) [m]')
    ax1.set_title(f'Strategy 3: Stage-Anchored Hybrid  [{S3_METHOD}]  (chained global trajectory)')
    ax1.legend(handles=_ph_hdl, loc='best', fontsize=8)
    ax2.plot(global_dz, '^-', color='seagreen')
    ax2.axhline(0, color='k', lw=0.6, ls='--')
    ax2.set_ylabel('Global Δz (depth) [m]')
    ax3.plot(np.degrees(global_phi), 's-', color='darkorange')
    ax3.axhline(0, color='k', lw=0.6, ls='--')
    ax3.set_ylabel('φ₀ [°]')
    ax3.set_xlabel('Profile number')
    ax3.set_xticks(range(len(global_prof)))
    ax3.set_xticklabels(global_prof, fontsize=7)
    plt.tight_layout()
    fig.savefig(OUT_DIR / 'roi_phase' / 'strat3_stage_anchored_summary.png', dpi=150)
    save_fig(fig, 'strat3_stage_anchored_summary', study='FieldData_Study', prefix='FD_')
    plt.show();  plt.close(fig)


## 3b\. Stage-Anchored (Coarse) — Total Stage Displacements

A coarser variant of strategy 3: instead of comparing every profile within a stage
to its stage reference, we take a **single large WLS step per stage**, using the
same four representative consecutive pairs used throughout the Region of Influence
Workflow above (1→3, 3→8, 8→20, 20→38) rather than the original stage-boundary
profiles (1→4, 5→9, 10→20, 21→38). This maximises cross-spectrum SNR — larger Δx
means a stronger, less ambiguous phase ramp — while keeping the reference close
enough to prevent severe decorrelation, and keeps this table's pairs consistent
with the rest of the chapter's analysis.

| Step | Pair | Physics |
|---|---|---|
| Push | 1 → 3 | Mechanical injection displacement |
| Chase | 3 → 8 | Continued injection at a lower rate |
| Wait | 8 → 20 | Net diffusion / dispersion spread during paused injection |
| Pull | 20 → 38 | Withdrawal displacement |

Because these four pairs are consecutive (each pair's end profile is the next
pair's start profile), the global trajectory **X_total(t)** is simply their
cumulative sum — no separate boundary/bridging pairs are needed, unlike the
original version of this analysis.


In [ ]:
# ── Strategy 3b: Stage-Anchored (Coarse) — Total Stage Displacements ─────────────
# One WLS estimate per stage, using the same four representative consecutive pairs
# used throughout the ROI workflow above (1->3 Push, 3->8 Chase, 8->20 Wait,
# 20->38 Pull) -- updated from the original stage-BOUNDARY pairs (1->4, 5->9,
# 10->20, 21->38 plus 3 separate bridging pairs), which no longer matched any
# other part of this chapter's analysis. Because these four pairs are consecutive
# (each pair's end is the next pair's start), no separate boundary/bridging pairs
# are needed -- the global trajectory is just their cumulative sum. ROIs reused
# from the rectangular-window MANUAL_ROIS (0a10689e) for the same pairs.
# ─────────────────────────────────────────────────────────────────────────────────────

S3B_PAIR_ROIS = {
    (1,  3):  (71, 78, 5.0, 6.5),   # Push
    (3,  8):  (70, 78, 4.5, 7.5),   # Chase
    (8,  20): (70, 77, 4.5, 6.0),   # Wait
    (20, 38): (70, 78, 4.5, 6.5),   # Pull
}
S3B_METHOD        = 'gazdag'
S3B_FORCE_DZ_ZERO = False

_stage_end_s3b = [(1, 3), (3, 8), (8, 20), (20, 38)]

results_s3b = {}
for _ra, _rb in _stage_end_s3b:
    _a = _load_img(S3B_METHOD, _ra);  _b = _load_img(S3B_METHOD, _rb)
    if _a is None or _b is None:
        print(f'[{_ra}->{_rb}] missing .npy — skipped')
        continue
    _n = min(_a.shape[0], _b.shape[0])
    _a, _b = _a[:_n], _b[:_n]
    _z0, _z1, _x0, _x1 = _phys_to_pix(depth[:_n], x_img, *S3B_PAIR_ROIS[(_ra, _rb)])
    _dz, _dx, _phi = _estimate_shift_2d(
        _a[_z0:_z1, _x0:_x1], _b[_z0:_z1, _x0:_x1],
        dz_g, dx_g, kz_c, force_dz_zero=S3B_FORCE_DZ_ZERO)
    results_s3b[(_ra, _rb)] = {'dx': _dx, 'dz': _dz, 'phi_0': _phi}
    print(f'[{_ra}->{_rb}]  dx={_dx:+.4f} m  dz={_dz:+.4f} m  phi0={_phi:+.4f} rad')

# ── Kinematic chaining: cumulative sum (pairs are consecutive, no bridging needed) ──
_traj = []
_cdx = _cdz = 0.0
for (_ra, _rb) in _stage_end_s3b:
    if (_ra, _rb) not in results_s3b:
        continue
    _cdx += results_s3b[(_ra, _rb)]['dx']
    _cdz += results_s3b[(_ra, _rb)]['dz']
    _traj.append((_rb, _cdx, _cdz))

print('\nGlobal trajectory (profile | Δx | Δz):')
for _p, _dx, _dz in _traj:
    print(f'  prof {_p:>2}  Δx={_dx:+.4f} m  Δz={_dz:+.4f} m')

# ── Summary plot ──────────────────────────────────────────────────────────────────
if _traj:
    from matplotlib.patches import Patch as _Patch
    _PHASE_DEFS = [('Pushing', 1, 3, '#aed6f1'), ('Chasing', 4, 8, '#a9dfbf'),
                   ('Waiting', 9, 20, '#f9e79f'), ('Pulling', 21, 38, '#f1948a')]
    _rn_col = {n: c for _nm, lo, hi, c in _PHASE_DEFS for n in range(lo, hi + 1)}

    _profs = [t[0] for t in _traj]
    _gdx   = [t[1] for t in _traj]
    _gdz   = [t[2] for t in _traj]
    _cols  = [_rn_col.get(p, '#cccccc') for p in _profs]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    for _bax in (ax1, ax2):
        for _i, _c in enumerate(_cols):
            _bax.axvspan(_i - 0.5, _i + 0.5, color=_c, alpha=0.3, zorder=0, lw=0)

    _ph_hdl = [_Patch(facecolor=c, alpha=0.6, label=nm, edgecolor='grey', lw=0.5)
               for nm, lo, hi, c in _PHASE_DEFS]

    ax1.plot(_gdx, 'o-', color='steelblue', lw=1.2, zorder=2)
    ax1.axhline(0, color='k', lw=0.6, ls='--')
    ax1.set_ylabel('Global Δx (radial) [m]')
    ax1.set_title(f'Strategy 3b: Stage-Anchored Coarse  [{S3B_METHOD}]  (4-point global trajectory)')
    ax1.legend(handles=_ph_hdl, loc='best', fontsize=8)

    ax2.plot(_gdz, '^-', color='seagreen', lw=1.2, zorder=2)
    ax2.axhline(0, color='k', lw=0.6, ls='--')
    ax2.set_ylabel('Global Δz (depth) [m]')
    ax2.set_xlabel('Profile number')
    ax2.set_xticks(range(len(_profs)))
    ax2.set_xticklabels(_profs)
    plt.tight_layout()
    fig.savefig(OUT_DIR / 'roi_phase' / 'strat3b_coarse_summary.png', dpi=150)
    save_fig(fig, 'strat3b_coarse_summary', study='FieldData_Study', prefix='FD_')
    plt.show();  plt.close(fig)


# Thesis Figure Compilations (Chapter 7 -- Hypothesis 3, field data)

The three figure sets below compile the field-data results for
`07_hypothesis3_complex.typ`. They reuse the migrated `.npy` products
(`OUT_DIR/migrated/`), the back-propagation snapshots (`completed`, built in
the "Save focus-frame snapshots" cell above), and the monogenic-envelope /
WLS phase-plane helpers defined in the "Manual pair selection + ROI
definition" cells above. Run this section after the full notebook has been
executed at least once (so the cached `.npy` / `.vti` products exist) --
none of it re-runs Kirchhoff, Gazdag, or gprMax.

Stage boundaries (Push 1->4, Chase 5->9, Wait 10->20, Pull 21->38) and the
fixed ROI (depth 70-79 m, radial 4.5-7.5 m) match the ones already used in
`sec:hyp3-fd-setup` / `sec:hyp3-fd-phaseplane` of the thesis text.

In [ ]:
# ── Thesis figure compilations: shared setup ────────────────────────────────
ROI_PHYS = (70, 79, 4.5, 7.5)   # fixed ROI, matches sec:hyp3-fd-phaseplane
PHASE_STAGES = [
    ('Pushing', 1, 4),
    ('Chasing', 5, 9),
    ('Waiting', 10, 20),
    ('Pulling', 21, 38),
]

thesis_fig_dir = OUT_DIR / 'thesis_figures'
thesis_fig_dir.mkdir(exist_ok=True)


def _load_processed_Dt(run):
    """Recomputes the processed B-scan (Dt) for one profile (cheap: I/O + filtering)."""
    data, _ = load_mala(str(DATA / _prof_name(run)), return_object=False)
    n = min(data.shape[1], n_traces)
    d_bp = filter_data(data[:, :n], fq=(0.02, 0.2), sfreq=sf, btype='bandpass')
    d_dc, _ = remove_mean(d_bp, 299, 517)
    d_aligned, _, _ = align_traces(d_dc, ref_aligned[:, :n], upsample=5, normalize=True, align_reference=False)
    d_svd, _ = remove_svd(d_aligned, low_s=0, high_s=1)
    d_gain, _ = linear_gain(d_svd, t)
    return d_gain[:rad_cut, :].T   # (n, rad_cut)


def _wls_diag(base_crop, mon_crop, dz_g_, dx_g_, kz_cent, kz_fac, kx_fac,
              wls_thr, pad_fac=10, wls_pow=1):
    """Zero-padded WLS cross-spectrum phase-plane fit; returns fit + everything
    needed to draw the 5-panel diagnostic (mirrors the SHOW_DIAG blocks above)."""
    Nz, Nx = base_crop.shape
    Nz_pad, Nx_pad = Nz * pad_fac, Nx * pad_fac
    kz_ax = np.fft.fftfreq(Nz_pad, d=dz_g_) * 2 * np.pi
    kx_ax = np.fft.fftfreq(Nx_pad, d=dx_g_) * 2 * np.pi
    KZ, KX = np.meshgrid(kz_ax, kx_ax, indexing='ij')
    taper = np.outer(tukey(Nz, alpha=0.15), tukey(Nx, alpha=0.15))
    XS = (np.fft.fft2(base_crop * taper, s=(Nz_pad, Nx_pad)) *
          np.conj(np.fft.fft2(mon_crop * taper, s=(Nz_pad, Nx_pad))))
    w = np.abs(XS)
    phi = np.angle(XS)
    band = (np.abs(KZ) < kz_fac * kz_cent) & (np.abs(KX) < kx_fac * kz_cent)
    mask = (w > wls_thr * w.max()) & band & ((np.abs(KZ) + np.abs(KX)) > 0)
    n_mask = int(mask.sum())
    if n_mask < 3:
        dz_est = dx_est = phi_0 = 0.0
    else:
        W = w[mask] ** wls_pow
        A = np.column_stack([KZ[mask], KX[mask], np.ones(n_mask)])
        c = np.linalg.lstsq(A * W[:, None], phi[mask] * W, rcond=None)[0]
        dz_est, dx_est, phi_0 = float(c[0]), float(c[1]), float(c[2])

    fitted = KX * dx_est + KZ * dz_est + phi_0

    phi_shift, w_shift = np.fft.fftshift(phi), np.fft.fftshift(w)
    kz_disp, kx_disp = np.fft.fftshift(kz_ax), np.fft.fftshift(kx_ax)
    band_shift = np.fft.fftshift(band)
    fitted_shift = np.fft.fftshift(fitted)
    phi_masked = np.where(band_shift & (np.fft.fftshift(w) > wls_thr * w.max()), phi_shift, np.nan)
    w_plot = np.where(band_shift, w_shift, np.nan)
    fitted_masked = np.where(band_shift, fitted_shift, np.nan)

    # 1-D kz slice: subtract kx contribution, average over the kx band
    phi_kz_resid = phi - KX * dx_est
    phi_1d = np.zeros(Nz_pad); w_1d = np.zeros(Nz_pad)
    band_kx = np.abs(kx_ax) < kx_fac * kz_cent
    for i in range(Nz_pad):
        sel = band_kx & (w[i, :] > wls_thr * w.max())
        if sel.sum() > 0:
            phi_1d[i] = np.average(phi_kz_resid[i, :][sel], weights=w[i, :][sel])
            w_1d[i] = w[i, :][sel].sum()
    kz_1d_s, phi_1d_s, w_1d_s = np.fft.fftshift(kz_ax), np.fft.fftshift(phi_1d), np.fft.fftshift(w_1d)

    # 1-D kx slice: subtract kz contribution, average over the kz band
    phi_kx_resid = phi - KZ * dz_est
    phi_1d_kx = np.zeros(Nx_pad); w_1d_kx = np.zeros(Nx_pad)
    band_kz_fit = np.abs(kz_ax) < kz_fac * kz_cent
    for j in range(Nx_pad):
        sel = band_kz_fit & (w[:, j] > wls_thr * w.max())
        if sel.sum() > 0:
            phi_1d_kx[j] = np.average(phi_kx_resid[:, j][sel], weights=w[:, j][sel])
            w_1d_kx[j] = w[:, j][sel].sum()
    kx_1d_s, phi_1d_kx_s, w_1d_kx_s = np.fft.fftshift(kx_ax), np.fft.fftshift(phi_1d_kx), np.fft.fftshift(w_1d_kx)

    return dict(
        dz_est=dz_est, dx_est=dx_est, phi_0=phi_0, n_mask=n_mask,
        kz_disp=kz_disp, kx_disp=kx_disp, phi_masked=phi_masked, w_plot=w_plot,
        w_max=w.max(), wls_thr=wls_thr, fitted_masked=fitted_masked,
        kz_1d_s=kz_1d_s, phi_1d_s=phi_1d_s, w_1d_s=w_1d_s,
        kx_1d_s=kx_1d_s, phi_1d_kx_s=phi_1d_kx_s, w_1d_kx_s=w_1d_kx_s,
        kz_fac=kz_fac, kx_fac=kx_fac, kz_cent=kz_cent,
    )


print('Thesis-figure helpers ready.')


## Figure 1: Processed profiles + migrated / back-propagated counterparts

5 rows (profiles 1, 3, 8, 20, 38) x 4 columns (processed B-scan,
Kirchhoff-BP, Gazdag, back-propagation $E_z$ focus frame).

In [ ]:
# ── Figure 1: profile / migration / back-propagation compilation grid ──────
FIG1_PROFILES = [1, 3, 8, 20, 38]
FIG1_DEPTH_RANGE = (62, 86)
FIG1_RADIAL_RANGE = (0, float(x_img[-1]))
FIG1_BP_NEAR_MASK_M = 1.0
FIG1_SC = 1.2

fig1_col_titles = ['Processed B-scan', 'Kirchhoff-BP migration', 'Gazdag migration',
                    'Back-propagation ($E_z$ focus frame)']

fig1, fig1_axes = plt.subplots(len(FIG1_PROFILES), 4, figsize=(17, 3.6 * len(FIG1_PROFILES)),
                                sharex=True, sharey=True)

for row, run in enumerate(FIG1_PROFILES):
    Dt_row = _load_processed_Dt(run)
    n = Dt_row.shape[0]
    z_common = depth[:n]
    lim = max(FIG1_SC * np.max(np.abs(Dt_row)), 1.0)
    ax = fig1_axes[row, 0]
    ax.imshow(Dt_row, aspect='auto', cmap='seismic',
              extent=[x_img[0], x_img[-1], z_common[-1], z_common[0]],
              vmin=-lim, vmax=lim, origin='upper')
    ax.invert_yaxis()
    ax.set_ylabel(f'Profile {run}\nDepth (m)')

    for col, method in [(1, 'kirchhoff_bp'), (2, 'gazdag')]:
        img = _load_img(method, run)
        ax = fig1_axes[row, col]
        if img is None:
            ax.text(0.5, 0.5, 'missing', ha='center', va='center', transform=ax.transAxes)
            continue
        n2 = img.shape[0]
        z2 = depth[:n2]
        lim2 = max(FIG1_SC * np.max(np.abs(img)), 1.0)
        ax.imshow(img, aspect='auto', cmap='seismic',
                  extent=[x_img[0], x_img[-1], z2[-1], z2[0]],
                  vmin=-lim2, vmax=lim2, origin='upper')
        ax.invert_yaxis()

    ax = fig1_axes[row, 3]
    res = _load_bp_frame_wls(run, BP_FOCUS_IDX_OFFSET)
    if res is None:
        ax.text(0.5, 0.5, 'missing', ha='center', va='center', transform=ax.transAxes)
    else:
        ez, depth_axis, radial_axis, dx_m = res
        mask_px = max(1, round(FIG1_BP_NEAR_MASK_M / dx_m))
        ez_disp = ez.copy(); ez_disp[:, :mask_px] = 0.0
        clim = np.percentile(np.abs(ez_disp), 100) if ez_disp.any() else 1.0
        ax.imshow(ez_disp, aspect='auto', cmap='seismic',
                  extent=[0, float(radial_axis[-1]), 0, float(depth_axis[-1])],
                  vmin=-clim, vmax=clim, origin='lower')
        ax.invert_yaxis()

    for col in range(4):
        fig1_axes[row, col].set_ylim(FIG1_DEPTH_RANGE[1], FIG1_DEPTH_RANGE[0])
        fig1_axes[row, col].set_xlim(*FIG1_RADIAL_RANGE)
        fig1_axes[row, col].xaxis.set_major_locator(ticker.MultipleLocator(2))
        fig1_axes[row, col].yaxis.set_major_locator(ticker.MultipleLocator(5))
        if row == len(FIG1_PROFILES) - 1:
            fig1_axes[row, col].set_xlabel('Radial distance (m)')

for col, title in enumerate(fig1_col_titles):
    fig1_axes[0, col].set_title(title)

fig1.suptitle('Processed profiles and their migrated / back-propagated counterparts', y=1.005)
plt.tight_layout()
save_fig(fig1, 'profile_migration_grid', study='FieldData_Study', prefix='FD_', category='Compilations')
plt.show(); plt.close(fig1)


## Figure 2: Per-stage difference + monogenic envelope

One figure per stage (Pushing, Chasing, Waiting, Pulling); each has 3 rows
(Kirchhoff-BP, Gazdag, Back-propagation) x 2 columns (difference + ROI,
monogenic envelope + ROI).

In [ ]:
# ── Figure 2: per-stage difference + monogenic-envelope compilations ───────
FIG2_METHOD_ROWS = [('kirchhoff_bp', 'Kirchhoff-BP'), ('gazdag', 'Gazdag'),
                     ('backprop', 'Back-propagation')]


def _fig2_pair_arrays(method, run_a, run_b):
    if method == 'backprop':
        res_a = _load_bp_frame_wls(run_a, BP_FOCUS_IDX_OFFSET)
        res_b = _load_bp_frame_wls(run_b, BP_FOCUS_IDX_OFFSET)
        if res_a is None or res_b is None:
            return None
        ez_a, depth_axis, radial_axis, dx_m = res_a
        ez_b, _, _, _ = res_b
        nd = min(ez_a.shape[0], ez_b.shape[0]); nr = min(ez_a.shape[1], ez_b.shape[1])
        ez_a, ez_b = ez_a[:nd, :nr], ez_b[:nd, :nr]
        depth_axis, radial_axis = depth_axis[:nd], radial_axis[:nr]
        diff = ez_b - ez_a
        env = _monogenic_envelope(diff)
        d0, d1, r0, r1 = _phys_to_pix(depth_axis, radial_axis, *ROI_PHYS)
        return diff, env, depth_axis, radial_axis, (d0, d1, r0, r1), 'lower'
    else:
        img_a = _load_img(method, run_a); img_b = _load_img(method, run_b)
        if img_a is None or img_b is None:
            return None
        n = min(img_a.shape[0], img_b.shape[0])
        img_a, img_b = img_a[:n], img_b[:n]
        z_common = depth[:n]
        diff = img_b - img_a
        env = _monogenic_envelope(diff)
        z0, z1, x0, x1 = _phys_to_pix(z_common, x_img, *ROI_PHYS)
        return diff, env, z_common, x_img, (z0, z1, x0, x1), 'upper'


def _fig2_plot_stage(stage_name, run_a, run_b):
    fig, axes = plt.subplots(3, 2, figsize=(12, 12), sharex=True, sharey=True)
    z_min, z_max, x_min, x_max = ROI_PHYS

    for row, (method, label) in enumerate(FIG2_METHOD_ROWS):
        result = _fig2_pair_arrays(method, run_a, run_b)
        ax_d, ax_e = axes[row, 0], axes[row, 1]
        if result is None:
            for ax in (ax_d, ax_e):
                ax.text(0.5, 0.5, 'missing data', ha='center', va='center', transform=ax.transAxes)
            continue
        diff, env, z_axis, x_axis, roi_px, origin = result
        z0, z1, x0, x1 = roi_px
        z_roi_lo, z_roi_hi = float(z_axis[z0]), float(z_axis[z1 - 1])
        roi_bottom, roi_h = min(z_roi_lo, z_roi_hi), abs(z_roi_hi - z_roi_lo)
        extent = ([x_axis[0], x_axis[-1], z_axis[-1], z_axis[0]] if origin == 'upper'
                  else [x_axis[0], x_axis[-1], z_axis[0], z_axis[-1]])

        vmax_d = np.percentile(np.abs(diff), 98)
        vmax_e = np.percentile(env, 98)

        im_d = ax_d.imshow(diff, aspect='auto', cmap='RdBu_r', extent=extent,
                            origin=origin, vmin=-vmax_d, vmax=vmax_d)
        plt.colorbar(im_d, ax=ax_d, label='Δ amplitude [a.u.]', fraction=0.046, pad=0.04)
        ax_d.add_patch(Rectangle((x_min, roi_bottom), x_max - x_min, roi_h,
                                  lw=1.5, edgecolor='yellow', facecolor='none'))

        im_e = ax_e.imshow(env, aspect='auto', cmap='inferno', extent=extent,
                            origin=origin, vmin=0, vmax=vmax_e)
        plt.colorbar(im_e, ax=ax_e, label='Monogenic envelope [a.u.]', fraction=0.046, pad=0.04)
        ax_e.add_patch(Rectangle((x_min, roi_bottom), x_max - x_min, roi_h,
                                  lw=1.5, edgecolor='cyan', facecolor='none'))

        for ax in (ax_d, ax_e):
            ax.invert_yaxis()
            ax.set_ylim(86, 62)
            ax.set_xlim(0, float(x_img[-1]))
            ax.xaxis.set_major_locator(ticker.MultipleLocator(2))
            ax.yaxis.set_major_locator(ticker.MultipleLocator(5))
            ax.grid(True, color='grey', lw=0.3, alpha=0.4)
        ax_d.set_ylabel(f'{label}\nDepth (m)')

    axes[0, 0].set_title(f'Difference: prof_{run_b} − prof_{run_a}')
    axes[0, 1].set_title('Monogenic envelope')
    for ax in axes[-1]:
        ax.set_xlabel('Radial distance (m)')

    fig.suptitle(f'{stage_name} stage (profiles {run_a}→{run_b}): difference and monogenic '
                 f'envelope, ROI depth {ROI_PHYS[0]}–{ROI_PHYS[1]} m, radial {ROI_PHYS[2]}–{ROI_PHYS[3]} m', y=1.01)
    plt.tight_layout()
    return fig


for stage_name, run_a, run_b in PHASE_STAGES:
    fig2 = _fig2_plot_stage(stage_name, run_a, run_b)
    save_fig(fig2, f'stage_diff_envelope_{stage_name.lower()}',
             study='FieldData_Study', prefix='FD_', category='Compilations')
    plt.show(); plt.close(fig2)


## Figure 3: Displacement-estimate (WLS phase-plane) diagnostics per method

One figure per migration method (Kirchhoff-BP, Gazdag, Back-propagation);
each has 4 rows (Pushing, Chasing, Waiting, Pulling) x 5 columns
(cross-spectrum phase, cross-spectrum energy + WLS contour, fitted plane,
1-D $k_z$ slice, 1-D $k_x$ slice).

In [ ]:
# ── Figure 3: displacement-estimate (WLS phase-plane) diagnostics per method ──
FIG3_KZ_BAND_FAC = 0.5
FIG3_KX_BAND_FAC = 2.0
FIG3_WLS_AMP_THR = 0.20
FIG3_BP_KX_BAND_FAC = 1.5
FIG3_BP_WLS_AMP_THR = 0.10
FIG3_PAD_FAC = 10


def _fig3_get_crop(method, run_a, run_b):
    if method == 'backprop':
        res_a = _load_bp_frame_wls(run_a, BP_FOCUS_IDX_OFFSET)
        res_b = _load_bp_frame_wls(run_b, BP_FOCUS_IDX_OFFSET)
        if res_a is None or res_b is None:
            return None
        ez_a, depth_axis, radial_axis, dx_m = res_a
        ez_b, _, _, _ = res_b
        nd = min(ez_a.shape[0], ez_b.shape[0]); nr = min(ez_a.shape[1], ez_b.shape[1])
        ez_a, ez_b = ez_a[:nd, :nr], ez_b[:nd, :nr]
        depth_axis, radial_axis = depth_axis[:nd], radial_axis[:nr]
        d0, d1, r0, r1 = _phys_to_pix(depth_axis, radial_axis, *ROI_PHYS)
        return (ez_a[d0:d1, r0:r1], ez_b[d0:d1, r0:r1], dx_m, dx_m, kz_c_bp,
                FIG3_BP_KX_BAND_FAC, FIG3_BP_WLS_AMP_THR)
    else:
        img_a = _load_img(method, run_a); img_b = _load_img(method, run_b)
        if img_a is None or img_b is None:
            return None
        n = min(img_a.shape[0], img_b.shape[0])
        img_a, img_b = img_a[:n], img_b[:n]
        z_common = depth[:n]
        z0, z1, x0, x1 = _phys_to_pix(z_common, x_img, *ROI_PHYS)
        return (img_a[z0:z1, x0:x1], img_b[z0:z1, x0:x1], dz_g, dx_g, kz_c,
                FIG3_KX_BAND_FAC, FIG3_WLS_AMP_THR)


def _fig3_plot_panel_row(axes_row, diag, stage_name, run_a, run_b):
    kz_disp, kx_disp = diag['kz_disp'], diag['kx_disp']
    kz_fac, kx_fac, kz_cent = diag['kz_fac'], diag['kx_fac'], diag['kz_cent']

    ax = axes_row[0]
    im = ax.imshow(diag['phi_masked'], aspect='auto', cmap='RdBu_r',
                    extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]],
                    vmin=-np.pi, vmax=np.pi, origin='upper')
    plt.colorbar(im, ax=ax, label='phase [rad]', fraction=0.046, pad=0.04)
    ax.set_title(f'Cross-spectrum phase ({diag["n_mask"]} px)')
    ax.set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3); ax.set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

    ax = axes_row[1]
    im = ax.imshow(diag['w_plot'], aspect='auto', cmap='inferno',
                    extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]], origin='upper')
    plt.colorbar(im, ax=ax, label='|XS| [a.u.]', fraction=0.046, pad=0.04)
    ax.contour(kx_disp, kz_disp, np.nan_to_num(diag['w_plot']),
               levels=[diag['wls_thr'] * diag['w_max']], colors='cyan', linewidths=0.8)
    ax.set_title(f'Energy (thr={diag["wls_thr"]:.2f}×max)')
    ax.set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3); ax.set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

    ax = axes_row[2]
    im = ax.imshow(diag['fitted_masked'], aspect='auto', cmap='RdBu_r',
                    extent=[kx_disp[0], kx_disp[-1], kz_disp[-1], kz_disp[0]],
                    vmin=-np.pi, vmax=np.pi, origin='upper')
    plt.colorbar(im, ax=ax, label='phase [rad]', fraction=0.046, pad=0.04)
    ax.set_title(f'Fitted plane  Δz={diag["dz_est"]:+.3f} m  Δx={diag["dx_est"]:+.3f} m')
    ax.set_xlim(kx_disp[0] / 3, kx_disp[-1] / 3); ax.set_ylim(kz_disp[-1] / 3, kz_disp[0] / 3)

    ax = axes_row[3]
    kz_1d_s, phi_1d_s, w_1d_s = diag['kz_1d_s'], diag['phi_1d_s'], diag['w_1d_s']
    in_band = np.abs(kz_1d_s) < kz_fac * kz_cent
    sel = in_band & (w_1d_s > 0)
    sc = ax.scatter(kz_1d_s[sel], phi_1d_s[sel], c=w_1d_s[sel], cmap='viridis', s=14)
    plt.colorbar(sc, ax=ax, label='weight', fraction=0.046, pad=0.04)
    kz_fit = kz_1d_s[in_band]
    ax.plot(kz_fit, kz_fit * diag['dz_est'] + diag['phi_0'], 'r-', lw=1.3)
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.set_title('1-D $k_z$ slice: measured vs fitted')
    ax.set_xlabel('$k_z$ [rad/m]'); ax.set_ylabel('phase [rad]')

    ax = axes_row[4]
    kx_1d_s, phi_1d_kx_s, w_1d_kx_s = diag['kx_1d_s'], diag['phi_1d_kx_s'], diag['w_1d_kx_s']
    in_band_kx = np.abs(kx_1d_s) < kx_fac * kz_cent
    sel = in_band_kx & (w_1d_kx_s > 0)
    sc = ax.scatter(kx_1d_s[sel], phi_1d_kx_s[sel], c=w_1d_kx_s[sel], cmap='viridis', s=14)
    plt.colorbar(sc, ax=ax, label='weight', fraction=0.046, pad=0.04)
    kx_fit = kx_1d_s[in_band_kx]
    ax.plot(kx_fit, kx_fit * diag['dx_est'] + diag['phi_0'], 'r-', lw=1.3)
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.set_title('1-D $k_x$ slice: measured vs fitted')
    ax.set_xlabel('$k_x$ [rad/m]'); ax.set_ylabel('phase [rad]')

    axes_row[0].set_ylabel(f'{stage_name}\n(prof {run_a}$\\to${run_b})\n$k_z$ [rad/m]')


def _fig3_plot_method(method, label):
    fig, axes = plt.subplots(4, 5, figsize=(24, 16))
    for row, (stage_name, run_a, run_b) in enumerate(PHASE_STAGES):
        crop = _fig3_get_crop(method, run_a, run_b)
        if crop is None:
            for ax in axes[row]:
                ax.text(0.5, 0.5, 'missing data', ha='center', va='center', transform=ax.transAxes)
            continue
        base_crop, mon_crop, dz_gv, dx_gv, kz_cent, kx_fac, wls_thr = crop
        diag = _wls_diag(base_crop, mon_crop, dz_gv, dx_gv, kz_cent,
                          FIG3_KZ_BAND_FAC, kx_fac, wls_thr, pad_fac=FIG3_PAD_FAC)
        _fig3_plot_panel_row(axes[row], diag, stage_name, run_a, run_b)
        print(f'  [{label}] {stage_name} ({run_a}->{run_b}): '
              f'dz={diag["dz_est"]:+.4f} m  dx={diag["dx_est"]:+.4f} m  n_mask={diag["n_mask"]}')

    fig.suptitle(f'{label}: WLS cross-spectrum phase-plane diagnostics per stage '
                 f'(ROI depth {ROI_PHYS[0]}-{ROI_PHYS[1]} m, radial {ROI_PHYS[2]}-{ROI_PHYS[3]} m)', y=1.01)
    plt.tight_layout()
    return fig


for method, label in [('kirchhoff_bp', 'Kirchhoff-BP'), ('gazdag', 'Gazdag'),
                       ('backprop', 'Back-propagation')]:
    print(f'--- {label} ---')
    fig3 = _fig3_plot_method(method, label)
    save_fig(fig3, f'displacement_diagnostics_{method}',
             study='FieldData_Study', prefix='FD_', category='Compilations')
    plt.show(); plt.close(fig3)


# WLS vs RANSAC -- robustness of the phase-plane fit  *(key result)*

Given a picked ROI, does a robust (RANSAC) plane fit change the displacement
estimate versus plain WLS? This section runs the comparison across all four
stages x three migration techniques, in **two picking domains**:

- **k-space picking** (below): the painted `(k_z, k_x)` cells *are* the fit
  population, so RANSAC is tested against a hand-curated cell set.
- **B-scan (amplitude-domain) picking** (further below): painting only fixes the
  spatial ROI; the `(k_z, k_x)` cells are chosen by the automatic band+amplitude
  gate, so RANSAC is tested against an automatically-populated set.

A cross-domain comparison at the end contrasts the two. These produce the
`FD_ransac_vs_wls_displacement_summary.png` and
`FD_ransac_vs_wls_bscan_displacement_summary.png` figures used in Chapter 7's
"WLS vs RANSAC" section.

## Manual k-Space Picking Across All Stages x Methods, with a RANSAC Phase-Plane Fit

The automatic and manual-picking WLS fits above (`wls_phase_plane_fit`,
`WLS_FIT_DEFAULTS_BY_METHOD`) minimise squared phase residual over every masked
k-cell, with no protection against a cell whose true phase has wrapped (or that
belongs to a different reflector/side-lobe) pulling the fitted plane -- especially
on the wider-band Chase/Wait/Pull pairs, where the phase ramp is more likely to
exceed +-pi across the fitting window than on Push's "sub-wavelength, zero
wrapping" pairs.

This section:

1. Extends the napari manual k-space picking workflow above to all
   **4 hydraulic stages x 3 migration methods = 12 picks**, using the same
   representative consecutive pairs as the rest of this chapter's ROI workflow
   (Push 1->3, Chase 3->8, Wait 8->20, Pull 20->38; c.f. `MANUAL_ROIS_BY_METHOD`,
   Strategy 3b).
2. Refits **both** `wls_phase_plane_fit` and the new `ransac_phase_plane_fit`
   (`helper_functions/migration.py`) on the *exact same* painted pixels for each
   pick, isolating the fitting method as the only difference between the two
   estimates.
3. Summarises Delta_z / Delta_x for every stage x method, WLS vs RANSAC, in one
   table and one comparison figure.

Run the **Launch** / **Harvest** cell pair once per queue entry (12 times total) --
napari is interactive and can't run to completion inside a single non-interactive
cell, same as the single-pair picking workflow above. `RANSAC_PICK_IDX` tracks
progress and auto-advances, so re-running the same two cells steps through the
whole queue; `ransac_picks` persists across reruns so you can pick a few, come back
later, and pick the rest.

In [ ]:
# ── RANSAC Trial: setup — 12-pick queue (4 stages x 3 methods) + persistent stores ──
from helper_functions.migration import ransac_phase_plane_fit, plot_phase_slice

RANSAC_STAGE_PAIRS = [   # (stage_name, run_a, run_b) -- same pairs as MANUAL_ROIS_BY_METHOD / Strategy 3b
    ('Pushing', 1, 3),
    ('Chasing', 3, 8),
    ('Waiting', 8, 20),
    ('Pulling', 20, 38),
]
RANSAC_METHODS = ['gazdag', 'kirchhoff_bp', 'backprop']

RANSAC_PICK_QUEUE = [(method, stage, ra, rb)
                      for stage, ra, rb in RANSAC_STAGE_PAIRS
                      for method in RANSAC_METHODS]

# Persistent across reruns of Launch/Harvest below -- only (re)initialise on first run
# in this kernel session, so re-running this setup cell doesn't wipe picks already made.
if 'ransac_picks' not in globals():
    ransac_picks = {}       # {(method, stage_name): {...}}, filled in by Harvest
if 'RANSAC_PICK_IDX' not in globals():
    RANSAC_PICK_IDX = 0     # index into RANSAC_PICK_QUEUE; advanced by Harvest

roi_out = OUT_DIR / 'roi_phase'
roi_out.mkdir(exist_ok=True)

print(f'{len(RANSAC_PICK_QUEUE)} (method, stage) picks queued.')
_next = 'done' if RANSAC_PICK_IDX >= len(RANSAC_PICK_QUEUE) else RANSAC_PICK_QUEUE[RANSAC_PICK_IDX]
print(f'{len(ransac_picks)} already picked; resuming at index {RANSAC_PICK_IDX} ({_next}).')

### Launch / Harvest — run this pair once per queue entry (x12)

In [ ]:
# ── RANSAC Trial — 1: launch napari for the next (method, stage) pick in the queue ──
#
# Paint over the 'manual_mask' layer (paintbrush, label 1) to mark the k-cells you
# want in the fit -- everything left at 0 is excluded. Run the Harvest cell once
# you're done; it refits WLS + RANSAC on exactly what you painted, stores the
# result, and advances the queue automatically.
# ─────────────────────────────────────────────────────────────────────────────────────

import napari

if RANSAC_PICK_IDX >= len(RANSAC_PICK_QUEUE):
    raise RuntimeError(f'All {len(RANSAC_PICK_QUEUE)} picks done -- see ransac_picks / the '
                        'summary cell below. Set RANSAC_PICK_IDX = 0 to redo picks.')

METHOD, STAGE_NAME, run_a, run_b = RANSAC_PICK_QUEUE[RANSAC_PICK_IDX]
NAPARI_KDISP_FAC = 3.0
NAPARI_FORCE_DZ_ZERO = False

ref_run = run_a
_defaults = WLS_FIT_DEFAULTS_BY_METHOD[METHOD]

img_a_mi = load_migrated_image(METHOD, ref_run, migrated_dir=OUT_DIR / 'migrated', borehole_dir=bh_out,
                                depth_gk=depth, x_img_gk=x_img, dL_gk=dL, f0_mig=f0_mig, v=v)
img_b_mi = load_migrated_image(METHOD, run_b, migrated_dir=OUT_DIR / 'migrated', borehole_dir=bh_out,
                                depth_gk=depth, x_img_gk=x_img, dL_gk=dL, f0_mig=f0_mig, v=v)
if img_a_mi is None or img_b_mi is None:
    raise RuntimeError(f'Missing data for method={METHOD}, pair ({run_a}, {run_b})')

n_kspace = min(img_a_mi.image.shape[0], img_b_mi.image.shape[0])
a_arr_kspace = img_a_mi.image[:n_kspace]
b_arr_kspace = img_b_mi.image[:n_kspace]
z_common_kspace = img_a_mi.depth_axis[:n_kspace]
x_common_kspace = img_a_mi.radial_axis

roi_spec = MANUAL_ROIS_BY_METHOD[METHOD].get((run_a, run_b))
if roi_spec is not None:
    z0, z1, x0, x1 = _phys_to_pix(z_common_kspace, x_common_kspace, *roi_spec)
else:
    z0, z1, x0, x1 = _roi_from_envelope(_monogenic_envelope(b_arr_kspace - a_arr_kspace), ROI_THRESH)

_kz_fac = MANUAL_KZ_BAND_FAC_BY_METHOD[METHOD].get((run_a, run_b)) or _defaults['kz_band_fac']
_kx_fac = MANUAL_KX_BAND_FAC_BY_METHOD[METHOD].get((run_a, run_b)) or _defaults['kx_band_fac']
dz_auto, dx_auto, phi0_auto, n_auto, _diag = wls_phase_plane_fit(
    a_arr_kspace, b_arr_kspace, img_a_mi.dz, img_a_mi.dx, img_a_mi.kz_cent,
    roi_px=(z0, z1, x0, x1), data_pad=8, taper=_defaults['taper'],
    kz_band_fac=_kz_fac, kx_band_fac=_kx_fac, amp_thr=_defaults['amp_thr'],
    wls_pow=_defaults['wls_pow'], pad_fac=10, force_dz_zero=NAPARI_FORCE_DZ_ZERO,
    return_diagnostics=True)
w, phi, KZ, KX = _diag['w'], _diag['phi'], _diag['KZ'], _diag['KX']
kz_ax, kx_ax = _diag['kz_ax'], _diag['kx_ax']
kz_c_kspace = img_a_mi.kz_cent

kz_disp = np.fft.fftshift(kz_ax);  kx_disp = np.fft.fftshift(kx_ax)
_iz = np.where(np.abs(kz_disp) < NAPARI_KDISP_FAC * kz_c_kspace)[0]
_ix = np.where(np.abs(kx_disp) < NAPARI_KDISP_FAC * kz_c_kspace)[0]
NAPARI_IZ0, NAPARI_IZ1 = int(_iz.min()), int(_iz.max()) + 1
NAPARI_IX0, NAPARI_IX1 = int(_ix.min()), int(_ix.max()) + 1

w_shift   = np.fft.fftshift(w)[NAPARI_IZ0:NAPARI_IZ1, NAPARI_IX0:NAPARI_IX1]
phi_shift = np.fft.fftshift(phi)[NAPARI_IZ0:NAPARI_IZ1, NAPARI_IX0:NAPARI_IX1]
kz_crop   = kz_disp[NAPARI_IZ0:NAPARI_IZ1]
kx_crop   = kx_disp[NAPARI_IX0:NAPARI_IX1]

napari_viewer = napari.Viewer(
    title=f'[{RANSAC_PICK_IDX+1}/{len(RANSAC_PICK_QUEUE)}] {STAGE_NAME} -- {METHOD} prof_{ref_run}->{run_b}')
napari_viewer.add_image(w_shift, name='amplitude |XS|', colormap='inferno')
napari_viewer.add_image(phi_shift, name='phase (rad)', colormap='twilight', visible=False)
manual_labels = napari_viewer.add_labels(
    np.zeros(w_shift.shape, dtype=np.uint8), name='manual_mask')
napari_viewer.layers.selection.active = manual_labels
manual_labels.mode = 'paint'
manual_labels.brush_size = max(1, min(w_shift.shape) // 20)

print(f'[{RANSAC_PICK_IDX+1}/{len(RANSAC_PICK_QUEUE)}] Viewer open for {STAGE_NAME} -- '
      f'{METHOD} prof_{ref_run}->{run_b}.  k-space crop: {w_shift.shape[0]} x {w_shift.shape[1]} px '
      f'(|kz|,|kx| < {NAPARI_KDISP_FAC * kz_c_kspace:.1f} rad/m).')
print(f'Automatic WLS fit for comparison: dz={dz_auto:+.4f} m  dx={dx_auto:+.4f} m  n={n_auto} px')
print("Paint the 'manual_mask' layer (label 1) over the k-cells you want in the fit, "
      "then run the Harvest cell.")

In [ ]:
# ── RANSAC Trial — 2: harvest the pick, refit WLS + RANSAC on the SAME pixels, store, advance ──
manual_mask_crop = manual_labels.data > 0
n_manual = int(manual_mask_crop.sum())
print(f'{n_manual} k-cells painted for {STAGE_NAME} / {METHOD} ({run_a}->{run_b}).')
if n_manual < 3:
    raise RuntimeError("Paint at least 3 k-cells in the 'manual_mask' layer "
                        "(napari_viewer, Launch cell) before running this cell.")

manual_mask_shift = np.zeros(np.fft.fftshift(w).shape, dtype=bool)
manual_mask_shift[NAPARI_IZ0:NAPARI_IZ1, NAPARI_IX0:NAPARI_IX1] = manual_mask_crop
manual_mask_full = np.fft.ifftshift(manual_mask_shift)

dz_wls, dx_wls, phi0_wls, n_wls = wls_phase_plane_fit(
    a_arr_kspace, b_arr_kspace, img_a_mi.dz, img_a_mi.dx, img_a_mi.kz_cent,
    roi_px=(z0, z1, x0, x1), data_pad=8, taper=_defaults['taper'],
    mask=manual_mask_full, wls_pow=_defaults['wls_pow'], pad_fac=10,
    force_dz_zero=NAPARI_FORCE_DZ_ZERO)

dz_ransac, dx_ransac, phi0_ransac, n_ransac, ransac_diag = ransac_phase_plane_fit(
    a_arr_kspace, b_arr_kspace, img_a_mi.dz, img_a_mi.dx, img_a_mi.kz_cent,
    roi_px=(z0, z1, x0, x1), data_pad=8, taper=_defaults['taper'],
    mask=manual_mask_full, final_wls_pow=_defaults['wls_pow'], pad_fac=10,
    n_iter=2000, residual_thr=0.35, force_dz_zero=NAPARI_FORCE_DZ_ZERO,
    random_state=0, return_diagnostics=True)

print(f'WLS    ({n_wls:5d} px):            dz={dz_wls:+.4f} m  dx={dx_wls:+.4f} m  phi0={phi0_wls:+.3f} rad')
print(f'RANSAC ({n_ransac:5d}/{n_manual} inliers):  dz={dz_ransac:+.4f} m  dx={dx_ransac:+.4f} m  '
      f'phi0={phi0_ransac:+.3f} rad')

ransac_picks[(METHOD, STAGE_NAME)] = dict(
    run_a=run_a, run_b=run_b, n_painted=n_manual,
    dz_wls=dz_wls, dx_wls=dx_wls, phi0_wls=phi0_wls, n_wls=n_wls,
    dz_ransac=dz_ransac, dx_ransac=dx_ransac, phi0_ransac=phi0_ransac, n_ransac=n_ransac,
    mask=manual_mask_full.copy(),
)

# ── Comparison figure: same painted phase panel, WLS selection vs RANSAC inliers/outliers ──
extent_k = [kx_crop[0], kx_crop[-1], kz_crop[-1], kz_crop[0]]
inlier_shift = np.fft.fftshift(ransac_diag['inlier_mask'])[NAPARI_IZ0:NAPARI_IZ1, NAPARI_IX0:NAPARI_IX1]
outlier_shift = np.fft.fftshift(ransac_diag['outlier_mask'])[NAPARI_IZ0:NAPARI_IZ1, NAPARI_IX0:NAPARI_IX1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
im = ax.imshow(phi_shift, aspect='auto', cmap='RdBu_r', extent=extent_k,
                vmin=-np.pi, vmax=np.pi, origin='upper')
ax.contour(kx_crop, kz_crop,
           manual_mask_shift[NAPARI_IZ0:NAPARI_IZ1, NAPARI_IX0:NAPARI_IX1].astype(float),
           levels=[0.5], colors='lime', linewidths=1.2)
ax.set_title(f'WLS on all painted px\ndz={dz_wls:+.4f} m  dx={dx_wls:+.4f} m  n={n_wls} px')
ax.set_xlabel('kx [rad/m]'); ax.set_ylabel('kz [rad/m]')

ax = axes[1]
im = ax.imshow(phi_shift, aspect='auto', cmap='RdBu_r', extent=extent_k,
                vmin=-np.pi, vmax=np.pi, origin='upper')
ax.contour(kx_crop, kz_crop, inlier_shift.astype(float), levels=[0.5],
           colors='lime', linewidths=1.2)
ax.contour(kx_crop, kz_crop, outlier_shift.astype(float), levels=[0.5],
           colors='red', linewidths=1.0, linestyles='--')
ax.set_title(f'RANSAC inliers (green) / rejected (red)\n'
             f'dz={dz_ransac:+.4f} m  dx={dx_ransac:+.4f} m  n={n_ransac}/{n_manual}')
ax.set_xlabel('kx [rad/m]'); ax.set_ylabel('kz [rad/m]')
plt.colorbar(im, ax=axes, label='phase [rad]', shrink=0.8)
plt.suptitle(f'WLS vs RANSAC on manually-picked k-cells -- {STAGE_NAME}, '
             f'{METHOD} prof_{ref_run}->{run_b}', y=1.03)
fig.savefig(roi_out / f'ransac_vs_wls_{METHOD}_{STAGE_NAME.lower()}_{ref_run}_to_{run_b}.png',
            dpi=150, bbox_inches='tight')
plt.show(); plt.close(fig)

# ── Diagnostic figure: 1-D phase slices -- does each fit's line track its points? ──
# Top row: WLS's line against all painted points. Bottom row: RANSAC's line against
# its inliers only, with rejected points marked (red x) so you can see whether they
# really did sit off the trend -- the direct answer to "is RANSAC a better fit
# through the phase points than WLS".
pop_mask = manual_mask_full
KZp, KXp, PHIp, Wp = KZ[pop_mask], KX[pop_mask], phi[pop_mask], w[pop_mask]
outlier_flags = ransac_diag['outlier_mask'][pop_mask]

fig2, axes2 = plt.subplots(2, 3, figsize=(15, 8))

ax = axes2[0, 0]
im2 = ax.imshow(phi_shift, aspect='auto', cmap='RdBu_r', extent=extent_k,
                 vmin=-np.pi, vmax=np.pi, origin='upper')
ax.contour(kx_crop, kz_crop,
           manual_mask_shift[NAPARI_IZ0:NAPARI_IZ1, NAPARI_IX0:NAPARI_IX1].astype(float),
           levels=[0.5], colors='lime', linewidths=1.2)
ax.set_title(f'WLS input ({n_wls} px)')
ax.set_xlabel('kx [rad/m]'); ax.set_ylabel('kz [rad/m]')

plot_phase_slice(axes2[0, 1], KZp, KXp, PHIp, Wp, dz_wls, dx_wls, phi0_wls,
                  xlabel='kz [rad/m]', color='tab:blue', line_label='WLS')
axes2[0, 1].set_title('kz slice')
plot_phase_slice(axes2[0, 2], KXp, KZp, PHIp, Wp, dx_wls, dz_wls, phi0_wls,
                  xlabel='kx [rad/m]', color='tab:blue', line_label='WLS')
axes2[0, 2].set_title('kx slice')

ax = axes2[1, 0]
ax.imshow(phi_shift, aspect='auto', cmap='RdBu_r', extent=extent_k,
          vmin=-np.pi, vmax=np.pi, origin='upper')
ax.contour(kx_crop, kz_crop, inlier_shift.astype(float), levels=[0.5],
           colors='lime', linewidths=1.2)
ax.contour(kx_crop, kz_crop, outlier_shift.astype(float), levels=[0.5],
           colors='red', linewidths=1.0, linestyles='--')
ax.set_title(f'RANSAC inliers ({n_ransac}/{n_manual})')
ax.set_xlabel('kx [rad/m]'); ax.set_ylabel('kz [rad/m]')

plot_phase_slice(axes2[1, 1], KZp, KXp, PHIp, Wp, dz_ransac, dx_ransac, phi0_ransac,
                  xlabel='kz [rad/m]', color='tab:orange', outlier=outlier_flags,
                  line_label='RANSAC')
axes2[1, 1].set_title('kz slice')
plot_phase_slice(axes2[1, 2], KXp, KZp, PHIp, Wp, dx_ransac, dz_ransac, phi0_ransac,
                  xlabel='kx [rad/m]', color='tab:orange', outlier=outlier_flags,
                  line_label='RANSAC')
axes2[1, 2].set_title('kx slice')

for _ax in axes2[:, 1:].ravel():
    _ax.legend(fontsize=8, loc='best')
plt.tight_layout()
plt.colorbar(im2, ax=axes2[:, 0], label='phase [rad]', shrink=0.8)
fig2.suptitle(f'Phase-plane fit diagnostics -- {STAGE_NAME}, {METHOD} prof_{ref_run}->{run_b}\n'
              f'top: WLS on all {n_wls} painted px   bottom: RANSAC, {n_ransac} inliers / '
              f'{n_manual - n_ransac} rejected (red x)', y=1.05)
fig2.savefig(roi_out / f'ransac_vs_wls_phase_slices_{METHOD}_{STAGE_NAME.lower()}_{ref_run}_to_{run_b}.png',
             dpi=150, bbox_inches='tight')
plt.show(); plt.close(fig2)

RANSAC_PICK_IDX += 1
if RANSAC_PICK_IDX < len(RANSAC_PICK_QUEUE):
    _next_m, _next_s, _next_a, _next_b = RANSAC_PICK_QUEUE[RANSAC_PICK_IDX]
    print(f'\n-- {RANSAC_PICK_IDX}/{len(RANSAC_PICK_QUEUE)} done. Next up: {_next_s} / {_next_m} '
          f'({_next_a}->{_next_b}). Re-run the Launch cell above.')
else:
    print(f'\n-- All {len(RANSAC_PICK_QUEUE)} picks done. Run the summary cell below.')

## Summary: Delta_z / Delta_x Across All Stages and Methods

In [ ]:
# ── RANSAC Trial — summary: dz / dx across all picked stages and methods ──
import pandas as pd
from matplotlib.patches import Patch as _Patch

_stage_order = {s: i for i, (s, _, _) in enumerate(RANSAC_STAGE_PAIRS)}
_rows = [dict(method=method, stage=stage, run_a=r['run_a'], run_b=r['run_b'],
              n_painted=r['n_painted'],
              dz_wls=r['dz_wls'], dx_wls=r['dx_wls'], n_wls=r['n_wls'],
              dz_ransac=r['dz_ransac'], dx_ransac=r['dx_ransac'], n_ransac=r['n_ransac'])
         for (method, stage), r in ransac_picks.items()]
summary_df = pd.DataFrame(_rows)
if not summary_df.empty:
    summary_df = summary_df.assign(_so=summary_df['stage'].map(_stage_order)) \
                            .sort_values(['_so', 'method']).drop(columns='_so') \
                            .reset_index(drop=True)
display(summary_df)

_missing = [q for q in RANSAC_PICK_QUEUE if (q[0], q[1]) not in ransac_picks]
if _missing:
    print(f'{len(_missing)} pick(s) not yet done: '
          + ', '.join(f'{s}/{m}' for m, s, _, _ in _missing))

summary_csv = roi_out / 'ransac_vs_wls_summary.csv'
summary_df.to_csv(summary_csv, index=False)
print(f'[saved] {summary_csv}')

# ── Summary figure: Δz / Δx per stage, grouped by method -- hatched=WLS, solid=RANSAC ──
if not summary_df.empty:
    _stages = [s for s, _, _ in RANSAC_STAGE_PAIRS if s in summary_df['stage'].values]
    _methods = [m for m in RANSAC_METHODS if m in summary_df['method'].values]
    _method_labels = {'gazdag': 'Gazdag', 'kirchhoff_bp': 'Kirchhoff-BP', 'backprop': 'Back-propagation'}
    _method_colors = {'gazdag': '#0072B2', 'kirchhoff_bp': '#E69F00', 'backprop': '#009E73'}

    _n_method = len(_methods)
    _group_w = 0.8
    _bar_w = _group_w / (_n_method * 2)
    _x = np.arange(len(_stages))

    fig, (ax_dz, ax_dx) = plt.subplots(1, 2, figsize=(13, 5))
    for _mi, _method in enumerate(_methods):
        _sub = summary_df[summary_df['method'] == _method].set_index('stage')
        _dz_w = np.array([_sub['dz_wls'].get(s, np.nan) for s in _stages])
        _dz_r = np.array([_sub['dz_ransac'].get(s, np.nan) for s in _stages])
        _dx_w = np.array([_sub['dx_wls'].get(s, np.nan) for s in _stages])
        _dx_r = np.array([_sub['dx_ransac'].get(s, np.nan) for s in _stages])
        _pos_w = _x - _group_w / 2 + (2 * _mi) * _bar_w + _bar_w / 2
        _pos_r = _pos_w + _bar_w
        _c = _method_colors[_method]
        ax_dz.bar(_pos_w, _dz_w, width=_bar_w, color=_c, hatch='///', edgecolor='white', linewidth=0.6)
        ax_dz.bar(_pos_r, _dz_r, width=_bar_w, color=_c, edgecolor='white', linewidth=0.6)
        ax_dx.bar(_pos_w, _dx_w, width=_bar_w, color=_c, hatch='///', edgecolor='white', linewidth=0.6)
        ax_dx.bar(_pos_r, _dx_r, width=_bar_w, color=_c, edgecolor='white', linewidth=0.6)

    for _ax, _ylabel, _title in ((ax_dz, 'Δz (depth) [m]', 'Δz per stage'),
                                  (ax_dx, 'Δx (radial) [m]', 'Δx per stage')):
        _ax.axhline(0, color='#888888', lw=0.7, ls='--', zorder=0)
        _ax.set_xticks(_x); _ax.set_xticklabels(_stages)
        _ax.set_ylabel(_ylabel); _ax.set_title(_title)
        _ax.spines['top'].set_visible(False)
        _ax.spines['right'].set_visible(False)

    _method_handles = [_Patch(facecolor=_method_colors[m], edgecolor='white', label=_method_labels[m])
                        for m in _methods]
    _fit_handles = [_Patch(facecolor='#999999', hatch='///', edgecolor='white', label='WLS'),
                    _Patch(facecolor='#999999', edgecolor='white', label='RANSAC')]
    _leg1 = ax_dx.legend(handles=_method_handles, loc='upper left', bbox_to_anchor=(1.02, 1.0),
                          fontsize=9, title='Method', frameon=False)
    ax_dx.add_artist(_leg1)
    ax_dx.legend(handles=_fit_handles, loc='upper left', bbox_to_anchor=(1.02, 0.55),
                 fontsize=9, title='Fit', frameon=False)
    fig.suptitle('WLS vs RANSAC displacement estimate per stage and migration method '
                 '(hatched = WLS, solid = RANSAC, same picked pixels)', y=1.03)
    plt.tight_layout()
    fig.savefig(roi_out / 'ransac_vs_wls_summary_all_stages_methods.png', dpi=150, bbox_inches='tight')
    save_fig(fig, 'ransac_vs_wls_displacement_summary', study='FieldData_Study', prefix='FD_',
             category='Compilations')
    plt.show(); plt.close(fig)

## B-Scan (Amplitude-Domain) Picking Across All Stages x Methods, WLS vs RANSAC

The k-space picking section above paints $(k_z, k_x)$ cells directly, so the
painted selection *is* the fit population -- RANSAC's robustness there is being
tested against a hand-curated set of cells. This section is the amplitude-domain
counterpart to that (mirroring the existing Depth-Radial / B-Scan Picking cells
above): painting happens on the spatial *difference image* instead, tracing the
actual shape of the reflection change. The paint only sets the spatial crop and a
Gaussian-softened window fed into the FFT; which $(k_z, k_x)$ cells enter the fit
is still decided by the same automatic band + amplitude gate every other
non-k-space section in this chapter uses (`WLS_FIT_DEFAULTS_BY_METHOD`).

That makes this a genuinely different test of RANSAC than the k-space section: any
WLS-vs-RANSAC divergence here comes from the *automatic* gate's cell population,
not a hand-picked one, so it isolates how much the fitting method itself matters
once the spatial ROI is fixed by hand but the k-space selection is left automatic.

Same structure as the k-space section: a 12-entry queue (4 stages x 3 methods),
a Launch/Harvest cell pair to run once per entry, persistent `bscan_ransac_picks`
/ `BSCAN_PICK_IDX` so progress survives reruns, and a summary table + figure at the
end. A final cell compares the two RANSAC estimates (k-space-picked vs
b-scan-picked) side by side.

In [ ]:
# ── B-Scan RANSAC Trial: setup — 12-pick queue (4 stages x 3 methods) + persistent stores ──
# Reuses RANSAC_STAGE_PAIRS / RANSAC_METHODS from the k-space section above if that
# section has already run in this kernel session; otherwise (re)defines them here so
# this section is self-contained.
from helper_functions.migration import ransac_phase_plane_fit, plot_phase_slice

if 'RANSAC_STAGE_PAIRS' not in globals():
    RANSAC_STAGE_PAIRS = [
        ('Pushing', 1, 3),
        ('Chasing', 3, 8),
        ('Waiting', 8, 20),
        ('Pulling', 20, 38),
    ]
if 'RANSAC_METHODS' not in globals():
    RANSAC_METHODS = ['gazdag', 'kirchhoff_bp', 'backprop']

BSCAN_PICK_QUEUE = [(method, stage, ra, rb)
                     for stage, ra, rb in RANSAC_STAGE_PAIRS
                     for method in RANSAC_METHODS]

# Persistent across reruns of Launch/Harvest below -- distinct names from the k-space
# section's ransac_picks/RANSAC_PICK_IDX so the two workflows don't clobber each other.
if 'bscan_ransac_picks' not in globals():
    bscan_ransac_picks = {}    # {(method, stage_name): {...}}, filled in by Harvest
if 'BSCAN_PICK_IDX' not in globals():
    BSCAN_PICK_IDX = 0         # index into BSCAN_PICK_QUEUE; advanced by Harvest

roi_out = OUT_DIR / 'roi_phase'
roi_out.mkdir(exist_ok=True)

print(f'{len(BSCAN_PICK_QUEUE)} (method, stage) picks queued.')
_next = 'done' if BSCAN_PICK_IDX >= len(BSCAN_PICK_QUEUE) else BSCAN_PICK_QUEUE[BSCAN_PICK_IDX]
print(f'{len(bscan_ransac_picks)} already picked; resuming at index {BSCAN_PICK_IDX} ({_next}).')

### Launch / Harvest — run this pair once per queue entry (x12)

In [ ]:
# ── B-Scan RANSAC Trial — 1: launch napari for the next (method, stage) pick ──
# Paint over the 'manual_roi_mask' layer (paintbrush, label 1) directly on the
# difference image to mark where the real reflection change sits. Run the Harvest
# cell once you're done.
# ─────────────────────────────────────────────────────────────────────────────────────
import napari

if BSCAN_PICK_IDX >= len(BSCAN_PICK_QUEUE):
    raise RuntimeError(f'All {len(BSCAN_PICK_QUEUE)} picks done -- see bscan_ransac_picks / '
                        'the summary cell below. Set BSCAN_PICK_IDX = 0 to redo picks.')

METHOD, STAGE_NAME, run_a, run_b = BSCAN_PICK_QUEUE[BSCAN_PICK_IDX]

bscan_img_a_mi = load_migrated_image(METHOD, run_a, migrated_dir=OUT_DIR / 'migrated', borehole_dir=bh_out,
                                      depth_gk=depth, x_img_gk=x_img, dL_gk=dL, f0_mig=f0_mig, v=v)
bscan_img_b_mi = load_migrated_image(METHOD, run_b, migrated_dir=OUT_DIR / 'migrated', borehole_dir=bh_out,
                                      depth_gk=depth, x_img_gk=x_img, dL_gk=dL, f0_mig=f0_mig, v=v)
if bscan_img_a_mi is None or bscan_img_b_mi is None:
    raise RuntimeError(f'Missing data for method={METHOD}, pair ({run_a}, {run_b})')

_nd_bscan = min(bscan_img_a_mi.image.shape[0], bscan_img_b_mi.image.shape[0])
_nr_bscan = min(bscan_img_a_mi.image.shape[1], bscan_img_b_mi.image.shape[1])
bscan_img_a = bscan_img_a_mi.image[:_nd_bscan, :_nr_bscan]
bscan_img_b = bscan_img_b_mi.image[:_nd_bscan, :_nr_bscan]
bscan_depth_axis = bscan_img_a_mi.depth_axis[:_nd_bscan]
bscan_radial_axis = bscan_img_a_mi.radial_axis[:_nr_bscan]
bscan_diff = bscan_img_b - bscan_img_a

_vmax_bscan = np.percentile(np.abs(bscan_diff), 98)
_diff_cmap_bscan = napari.utils.Colormap(['blue', 'white', 'red'], name='diff_rdbu_bscan')
bscan_viewer = napari.Viewer(
    title=f'[{BSCAN_PICK_IDX+1}/{len(BSCAN_PICK_QUEUE)}] {STAGE_NAME} -- {METHOD} prof_{run_a}->{run_b}')
bscan_viewer.add_image(bscan_diff, name='difference', colormap=_diff_cmap_bscan,
                        contrast_limits=[-_vmax_bscan, _vmax_bscan])
bscan_manual_roi_labels = bscan_viewer.add_labels(
    np.zeros(bscan_diff.shape, dtype=np.uint8), name='manual_roi_mask')
bscan_viewer.layers.selection.active = bscan_manual_roi_labels
bscan_manual_roi_labels.mode = 'paint'
bscan_manual_roi_labels.brush_size = max(1, min(bscan_diff.shape) // 40)

print(f'[{BSCAN_PICK_IDX+1}/{len(BSCAN_PICK_QUEUE)}] Viewer open for {STAGE_NAME} -- '
      f'{METHOD} prof_{run_a}->{run_b}.  Image shape (depth px x radial px): {bscan_diff.shape}.')
print(f'Depth axis:  {bscan_depth_axis[0]:.2f} m (row 0)  ->  {bscan_depth_axis[-1]:.2f} m '
      f'(row {bscan_diff.shape[0]-1}).')
print(f'Radial axis: {bscan_radial_axis[0]:.2f} m (col 0) ->  {bscan_radial_axis[-1]:.2f} m '
      f'(col {bscan_diff.shape[1]-1}).')
print("Paint the 'manual_roi_mask' layer (label 1) over the real reflection change, "
      "then run the Harvest cell.")

In [ ]:
# ── B-Scan RANSAC Trial — 2: harvest the pick, refit WLS + RANSAC, store, advance ──
# Same recipe as the existing Depth-Radial (B-Scan) Picking harvest cell above: the
# painted mask is Gaussian-softened into a spatial window (taper='none', since the
# window is already baked into the cropped image), and the fit's own k-cell selection
# still comes from the automatic band+amplitude gate (WLS_FIT_DEFAULTS_BY_METHOD) --
# only the spatial extent of the crop comes from painting. Here that same windowed
# crop is fit twice: once with wls_phase_plane_fit, once with ransac_phase_plane_fit.
# ─────────────────────────────────────────────────────────────────────────────────────
from scipy.ndimage import gaussian_filter as _gaussian_filter_bscan

BSCAN_GAUSS_SIGMA_PX = 3.0   # softens the painted mask's hard 0/1 edges before FFT windowing

bscan_mask_full = bscan_manual_roi_labels.data > 0
n_painted_bscan = int(bscan_mask_full.sum())
print(f'{n_painted_bscan} image px painted for {STAGE_NAME} / {METHOD} ({run_a}->{run_b}).')
if n_painted_bscan < 20:
    raise RuntimeError("Paint a larger region in the 'manual_roi_mask' layer "
                        "(bscan_viewer, Launch cell) before running this cell.")

_rows_bscan, _cols_bscan = np.where(bscan_mask_full)
_z0_bscan, _z1_bscan = int(_rows_bscan.min()), int(_rows_bscan.max()) + 1
_x0_bscan, _x1_bscan = int(_cols_bscan.min()), int(_cols_bscan.max()) + 1

_margin_bscan = int(np.ceil(3 * BSCAN_GAUSS_SIGMA_PX))
_z0p_bscan = max(0, _z0_bscan - _margin_bscan)
_z1p_bscan = min(bscan_img_a.shape[0], _z1_bscan + _margin_bscan)
_x0p_bscan = max(0, _x0_bscan - _margin_bscan)
_x1p_bscan = min(bscan_img_a.shape[1], _x1_bscan + _margin_bscan)
base_crop_bscan = bscan_img_a[_z0p_bscan:_z1p_bscan, _x0p_bscan:_x1p_bscan]
mon_crop_bscan = bscan_img_b[_z0p_bscan:_z1p_bscan, _x0p_bscan:_x1p_bscan]
mask_crop_bscan = bscan_mask_full[_z0p_bscan:_z1p_bscan, _x0p_bscan:_x1p_bscan].astype(float)

soft_win_bscan = _gaussian_filter_bscan(mask_crop_bscan, sigma=BSCAN_GAUSS_SIGMA_PX)
if soft_win_bscan.max() > 0:
    soft_win_bscan = soft_win_bscan / soft_win_bscan.max()

_defaults = WLS_FIT_DEFAULTS_BY_METHOD[METHOD]
_win_a_bscan = base_crop_bscan * soft_win_bscan
_win_b_bscan = mon_crop_bscan * soft_win_bscan

dz_wls_b, dx_wls_b, phi0_wls_b, n_wls_b, wls_diag_b = wls_phase_plane_fit(
    _win_a_bscan, _win_b_bscan, bscan_img_a_mi.dz, bscan_img_a_mi.dx, bscan_img_a_mi.kz_cent,
    roi_px=None, taper='none', kz_band_fac=_defaults['kz_band_fac'],
    kx_band_fac=_defaults['kx_band_fac'], amp_thr=_defaults['amp_thr'],
    wls_pow=_defaults['wls_pow'], pad_fac=10, force_dz_zero=False,
    return_diagnostics=True)

dz_ransac_b, dx_ransac_b, phi0_ransac_b, n_ransac_b, ransac_diag_b = ransac_phase_plane_fit(
    _win_a_bscan, _win_b_bscan, bscan_img_a_mi.dz, bscan_img_a_mi.dx, bscan_img_a_mi.kz_cent,
    roi_px=None, taper='none', kz_band_fac=_defaults['kz_band_fac'],
    kx_band_fac=_defaults['kx_band_fac'], amp_thr=_defaults['amp_thr'],
    final_wls_pow=_defaults['wls_pow'], pad_fac=10, n_iter=2000, residual_thr=0.35,
    force_dz_zero=False, random_state=0, return_diagnostics=True)

if n_wls_b < 3 or n_ransac_b < 3:
    print(f'WARNING: WLS n={n_wls_b}, RANSAC n={n_ransac_b} k-space px pass the automatic gate. '
          f'Painted region may be too small, or WLS_FIT_DEFAULTS_BY_METHOD[{METHOD!r}] needs loosening.')

print(f'WLS    ({n_wls_b:5d} px):            dz={dz_wls_b:+.4f} m  dx={dx_wls_b:+.4f} m  '
      f'phi0={phi0_wls_b:+.3f} rad')
print(f'RANSAC ({n_ransac_b:5d}/{n_wls_b:5d} gated):    dz={dz_ransac_b:+.4f} m  dx={dx_ransac_b:+.4f} m  '
      f'phi0={phi0_ransac_b:+.3f} rad')

bscan_ransac_picks[(METHOD, STAGE_NAME)] = dict(
    run_a=run_a, run_b=run_b, n_painted=n_painted_bscan,
    dz_wls=dz_wls_b, dx_wls=dx_wls_b, phi0_wls=phi0_wls_b, n_wls=n_wls_b,
    dz_ransac=dz_ransac_b, dx_ransac=dx_ransac_b, phi0_ransac=phi0_ransac_b, n_ransac=n_ransac_b,
)

# ── Comparison figure: painted difference image, WLS vs RANSAC fit annotated ───────
# bscan_depth_axis increases with row index (unified convention) -- this extent +
# origin='upper' combination renders shallow-at-top/deep-at-bottom, c.f. the existing
# Depth-Radial (B-Scan) Picking harvest cell above.
extent_bscan = [bscan_radial_axis[0], bscan_radial_axis[-1], bscan_depth_axis[-1], bscan_depth_axis[0]]
vmax_d_bscan = np.percentile(np.abs(bscan_diff), 98)

fig_b, ax_b = plt.subplots(figsize=(7, 7))
im_b = ax_b.imshow(bscan_diff, aspect='auto', cmap='RdBu_r', extent=extent_bscan, origin='upper',
                    vmin=-vmax_d_bscan, vmax=vmax_d_bscan)
plt.colorbar(im_b, ax=ax_b, label='Delta amplitude [a.u.]')
ax_b.contour(bscan_radial_axis, bscan_depth_axis, bscan_mask_full.astype(float), levels=[0.5],
             colors='lime', linewidths=1.2)
ax_b.set_xlabel('Radial distance (m)'); ax_b.set_ylabel('Depth (m)')
ax_b.set_title(f'{STAGE_NAME} -- {METHOD} prof_{run_a}->{run_b}  (painted ROI)\n'
                f'WLS: dz={dz_wls_b:+.4f} m dx={dx_wls_b:+.4f} m   '
                f'RANSAC: dz={dz_ransac_b:+.4f} m dx={dx_ransac_b:+.4f} m')
plt.tight_layout()
fig_b.savefig(roi_out / f'ransac_vs_wls_bscan_{METHOD}_{STAGE_NAME.lower()}_{run_a}_to_{run_b}.png',
              dpi=150, bbox_inches='tight')
plt.show(); plt.close(fig_b)

# ── Diagnostic figure: 2-D phase map + 1-D phase slices -- does each fit's line
# track its points? Mirrors the k-space section's diagnostic figure; here the
# k-cell population comes from the automatic band+amplitude gate (both fits see the
# identical population, since they're called with the same kz_band_fac/kx_band_fac/
# amp_thr on the same windowed images), not a hand-painted k-space mask -- so this
# shows whether RANSAC still finds (and benefits from rejecting) outliers even when
# the k-space selection itself was never painted by hand.
_pop_mask_b = wls_diag_b['mask']
KZb, KXb = wls_diag_b['KZ'], wls_diag_b['KX']
PHIb, Wb_full = wls_diag_b['phi'], wls_diag_b['w']
KZp_b, KXp_b = KZb[_pop_mask_b], KXb[_pop_mask_b]
PHIp_b, Wp_b = PHIb[_pop_mask_b], Wb_full[_pop_mask_b]
outlier_flags_b = ransac_diag_b['outlier_mask'][_pop_mask_b]

_kdisp_fac_b = 3.0
_kz_disp_b = np.fft.fftshift(wls_diag_b['kz_ax'])
_kx_disp_b = np.fft.fftshift(wls_diag_b['kx_ax'])
_iz_b = np.where(np.abs(_kz_disp_b) < _kdisp_fac_b * bscan_img_a_mi.kz_cent)[0]
_ix_b = np.where(np.abs(_kx_disp_b) < _kdisp_fac_b * bscan_img_a_mi.kz_cent)[0]
_iz0_b, _iz1_b = int(_iz_b.min()), int(_iz_b.max()) + 1
_ix0_b, _ix1_b = int(_ix_b.min()), int(_ix_b.max()) + 1
_phi_shift_b = np.fft.fftshift(PHIb)[_iz0_b:_iz1_b, _ix0_b:_ix1_b]
_pop_shift_b = np.fft.fftshift(_pop_mask_b)[_iz0_b:_iz1_b, _ix0_b:_ix1_b]
_inlier_shift_b = np.fft.fftshift(ransac_diag_b['inlier_mask'])[_iz0_b:_iz1_b, _ix0_b:_ix1_b]
_outlier_shift_b = np.fft.fftshift(ransac_diag_b['outlier_mask'])[_iz0_b:_iz1_b, _ix0_b:_ix1_b]
_kz_crop_b, _kx_crop_b = _kz_disp_b[_iz0_b:_iz1_b], _kx_disp_b[_ix0_b:_ix1_b]
_extent_kb = [_kx_crop_b[0], _kx_crop_b[-1], _kz_crop_b[-1], _kz_crop_b[0]]

fig2b, axes2b = plt.subplots(2, 3, figsize=(15, 8))

ax = axes2b[0, 0]
im2b = ax.imshow(_phi_shift_b, aspect='auto', cmap='RdBu_r', extent=_extent_kb,
                  vmin=-np.pi, vmax=np.pi, origin='upper')
ax.contour(_kx_crop_b, _kz_crop_b, _pop_shift_b.astype(float), levels=[0.5],
           colors='lime', linewidths=1.2)
ax.set_title(f'WLS input ({n_wls_b} px, automatic gate)')
ax.set_xlabel('kx [rad/m]'); ax.set_ylabel('kz [rad/m]')

plot_phase_slice(axes2b[0, 1], KZp_b, KXp_b, PHIp_b, Wp_b, dz_wls_b, dx_wls_b, phi0_wls_b,
                  xlabel='kz [rad/m]', color='tab:blue', line_label='WLS')
axes2b[0, 1].set_title('kz slice')
plot_phase_slice(axes2b[0, 2], KXp_b, KZp_b, PHIp_b, Wp_b, dx_wls_b, dz_wls_b, phi0_wls_b,
                  xlabel='kx [rad/m]', color='tab:blue', line_label='WLS')
axes2b[0, 2].set_title('kx slice')

ax = axes2b[1, 0]
ax.imshow(_phi_shift_b, aspect='auto', cmap='RdBu_r', extent=_extent_kb,
          vmin=-np.pi, vmax=np.pi, origin='upper')
ax.contour(_kx_crop_b, _kz_crop_b, _inlier_shift_b.astype(float), levels=[0.5],
           colors='lime', linewidths=1.2)
ax.contour(_kx_crop_b, _kz_crop_b, _outlier_shift_b.astype(float), levels=[0.5],
           colors='red', linewidths=1.0, linestyles='--')
ax.set_title(f'RANSAC inliers ({n_ransac_b}/{n_wls_b})')
ax.set_xlabel('kx [rad/m]'); ax.set_ylabel('kz [rad/m]')

plot_phase_slice(axes2b[1, 1], KZp_b, KXp_b, PHIp_b, Wp_b, dz_ransac_b, dx_ransac_b, phi0_ransac_b,
                  xlabel='kz [rad/m]', color='tab:orange', outlier=outlier_flags_b,
                  line_label='RANSAC')
axes2b[1, 1].set_title('kz slice')
plot_phase_slice(axes2b[1, 2], KXp_b, KZp_b, PHIp_b, Wp_b, dx_ransac_b, dz_ransac_b, phi0_ransac_b,
                  xlabel='kx [rad/m]', color='tab:orange', outlier=outlier_flags_b,
                  line_label='RANSAC')
axes2b[1, 2].set_title('kx slice')

for _ax in axes2b[:, 1:].ravel():
    _ax.legend(fontsize=8, loc='best')
plt.tight_layout()
plt.colorbar(im2b, ax=axes2b[:, 0], label='phase [rad]', shrink=0.8)
fig2b.suptitle(f'Phase-plane fit diagnostics (B-scan picking) -- {STAGE_NAME}, {METHOD} '
               f'prof_{run_a}->{run_b}\ntop: WLS on all {n_wls_b} automatically-gated px   '
               f'bottom: RANSAC, {n_ransac_b} inliers / {n_wls_b - n_ransac_b} rejected (red x)', y=1.05)
fig2b.savefig(roi_out / f'ransac_vs_wls_bscan_phase_slices_{METHOD}_{STAGE_NAME.lower()}_{run_a}_to_{run_b}.png',
              dpi=150, bbox_inches='tight')
plt.show(); plt.close(fig2b)

BSCAN_PICK_IDX += 1
if BSCAN_PICK_IDX < len(BSCAN_PICK_QUEUE):
    _next_m, _next_s, _next_a, _next_b = BSCAN_PICK_QUEUE[BSCAN_PICK_IDX]
    print(f'\n-- {BSCAN_PICK_IDX}/{len(BSCAN_PICK_QUEUE)} done. Next up: {_next_s} / {_next_m} '
          f'({_next_a}->{_next_b}). Re-run the Launch cell above.')
else:
    print(f'\n-- All {len(BSCAN_PICK_QUEUE)} picks done. Run the summary cell below.')

## Summary: Delta_z / Delta_x Across All Stages and Methods — B-Scan (Amplitude-Domain) Picking

In [ ]:
# ── B-Scan RANSAC Trial — summary: dz / dx across all picked stages and methods ──
import pandas as pd
from matplotlib.patches import Patch as _Patch

_stage_order = {s: i for i, (s, _, _) in enumerate(RANSAC_STAGE_PAIRS)}
_rows_b = [dict(method=method, stage=stage, run_a=r['run_a'], run_b=r['run_b'],
                n_painted=r['n_painted'],
                dz_wls=r['dz_wls'], dx_wls=r['dx_wls'], n_wls=r['n_wls'],
                dz_ransac=r['dz_ransac'], dx_ransac=r['dx_ransac'], n_ransac=r['n_ransac'])
           for (method, stage), r in bscan_ransac_picks.items()]
bscan_summary_df = pd.DataFrame(_rows_b)
if not bscan_summary_df.empty:
    bscan_summary_df = bscan_summary_df.assign(_so=bscan_summary_df['stage'].map(_stage_order)) \
                                        .sort_values(['_so', 'method']).drop(columns='_so') \
                                        .reset_index(drop=True)
display(bscan_summary_df)

_missing_b = [q for q in BSCAN_PICK_QUEUE if (q[0], q[1]) not in bscan_ransac_picks]
if _missing_b:
    print(f'{len(_missing_b)} pick(s) not yet done: '
          + ', '.join(f'{s}/{m}' for m, s, _, _ in _missing_b))

bscan_summary_csv = roi_out / 'ransac_vs_wls_bscan_summary.csv'
bscan_summary_df.to_csv(bscan_summary_csv, index=False)
print(f'[saved] {bscan_summary_csv}')

# ── Summary figure: Δz / Δx per stage, grouped by method -- hatched=WLS, solid=RANSAC ──
if not bscan_summary_df.empty:
    _stages_b = [s for s, _, _ in RANSAC_STAGE_PAIRS if s in bscan_summary_df['stage'].values]
    _methods_b = [m for m in RANSAC_METHODS if m in bscan_summary_df['method'].values]
    _method_labels = {'gazdag': 'Gazdag', 'kirchhoff_bp': 'Kirchhoff-BP', 'backprop': 'Back-propagation'}
    _method_colors = {'gazdag': '#0072B2', 'kirchhoff_bp': '#E69F00', 'backprop': '#009E73'}

    _n_method_b = len(_methods_b)
    _group_w = 0.8
    _bar_w_b = _group_w / (_n_method_b * 2)
    _x_b = np.arange(len(_stages_b))

    fig, (ax_dz, ax_dx) = plt.subplots(1, 2, figsize=(13, 5))
    for _mi, _method in enumerate(_methods_b):
        _sub = bscan_summary_df[bscan_summary_df['method'] == _method].set_index('stage')
        _dz_w = np.array([_sub['dz_wls'].get(s, np.nan) for s in _stages_b])
        _dz_r = np.array([_sub['dz_ransac'].get(s, np.nan) for s in _stages_b])
        _dx_w = np.array([_sub['dx_wls'].get(s, np.nan) for s in _stages_b])
        _dx_r = np.array([_sub['dx_ransac'].get(s, np.nan) for s in _stages_b])
        _pos_w = _x_b - _group_w / 2 + (2 * _mi) * _bar_w_b + _bar_w_b / 2
        _pos_r = _pos_w + _bar_w_b
        _c = _method_colors[_method]
        ax_dz.bar(_pos_w, _dz_w, width=_bar_w_b, color=_c, hatch='///', edgecolor='white', linewidth=0.6)
        ax_dz.bar(_pos_r, _dz_r, width=_bar_w_b, color=_c, edgecolor='white', linewidth=0.6)
        ax_dx.bar(_pos_w, _dx_w, width=_bar_w_b, color=_c, hatch='///', edgecolor='white', linewidth=0.6)
        ax_dx.bar(_pos_r, _dx_r, width=_bar_w_b, color=_c, edgecolor='white', linewidth=0.6)

    for _ax, _ylabel, _title in ((ax_dz, 'Δz (depth) [m]', 'Δz per stage'),
                                  (ax_dx, 'Δx (radial) [m]', 'Δx per stage')):
        _ax.axhline(0, color='#888888', lw=0.7, ls='--', zorder=0)
        _ax.set_xticks(_x_b); _ax.set_xticklabels(_stages_b)
        _ax.set_ylabel(_ylabel); _ax.set_title(_title)
        _ax.spines['top'].set_visible(False)
        _ax.spines['right'].set_visible(False)

    _method_handles = [_Patch(facecolor=_method_colors[m], edgecolor='white', label=_method_labels[m])
                        for m in _methods_b]
    _fit_handles = [_Patch(facecolor='#999999', hatch='///', edgecolor='white', label='WLS'),
                    _Patch(facecolor='#999999', edgecolor='white', label='RANSAC')]
    _leg1 = ax_dx.legend(handles=_method_handles, loc='upper left', bbox_to_anchor=(1.02, 1.0),
                          fontsize=9, title='Method', frameon=False)
    ax_dx.add_artist(_leg1)
    ax_dx.legend(handles=_fit_handles, loc='upper left', bbox_to_anchor=(1.02, 0.55),
                 fontsize=9, title='Fit', frameon=False)
    fig.suptitle('WLS vs RANSAC displacement estimate per stage and migration method -- '
                 'B-scan (amplitude-domain) picking (hatched = WLS, solid = RANSAC)', y=1.03)
    plt.tight_layout()
    fig.savefig(roi_out / 'ransac_vs_wls_bscan_summary_all_stages_methods.png', dpi=150, bbox_inches='tight')
    save_fig(fig, 'ransac_vs_wls_bscan_displacement_summary', study='FieldData_Study', prefix='FD_',
             category='Compilations')
    plt.show(); plt.close(fig)

### Cross-Domain Comparison: K-Space Picking vs B-Scan Picking (RANSAC)

Both sections above produce an independent RANSAC displacement estimate for the
same 12 (method, stage) pairs -- one from a hand-picked k-space cell population,
one from a hand-picked spatial ROI with an automatically-gated k-space population.
Agreement between the two is a useful cross-check: it means the RANSAC estimate
isn't an artefact of *how* the k-cells were selected.

In [ ]:
# ── Cross-domain comparison: k-space-picked vs b-scan-picked RANSAC estimates ──
_common_keys = sorted(set(ransac_picks) & set(bscan_ransac_picks),
                       key=lambda k: (_stage_order[k[1]], k[0]))
if not _common_keys:
    print('No (method, stage) pair has been picked in both sections yet -- '
          'nothing to compare. Run both Launch/Harvest loops above first.')
else:
    _cross_rows = []
    for method, stage in _common_keys:
        _ks = ransac_picks[(method, stage)]
        _bs = bscan_ransac_picks[(method, stage)]
        _cross_rows.append(dict(
            method=method, stage=stage,
            dz_kspace=_ks['dz_ransac'], dx_kspace=_ks['dx_ransac'],
            dz_bscan=_bs['dz_ransac'], dx_bscan=_bs['dx_ransac'],
            dz_diff=_ks['dz_ransac'] - _bs['dz_ransac'],
            dx_diff=_ks['dx_ransac'] - _bs['dx_ransac'],
        ))
    cross_domain_df = pd.DataFrame(_cross_rows)
    display(cross_domain_df)

    cross_domain_csv = roi_out / 'ransac_kspace_vs_bscan_summary.csv'
    cross_domain_df.to_csv(cross_domain_csv, index=False)
    print(f'[saved] {cross_domain_csv}')
    print(f'\nMean |Delta_z| disagreement: {cross_domain_df["dz_diff"].abs().mean():.4f} m   '
          f'Mean |Delta_x| disagreement: {cross_domain_df["dx_diff"].abs().mean():.4f} m')